# V-polyhedral disjunctive cuts plotting worksheet
1. Table 1: Summary statistics for percent gap closed by VPCs --- avg (%) and number of strict wins (best by at least `EPS`), including set of all instances and set of ≥ 10% gap closed instances
2. Table 2: Average percent gap closed by num disj terms
3. Table 3: Summary statistics for time to solve instances with branch-and-bound

We select instances that meet the following criteria:
1. Belong to MIPLIB, NEOS, or COR@L
2. IP optimal value is known
3. ≤ 5000 variables and 5000 constraints (in presolved instance)
4. The partial branch-and-bound tree with 64 leaves does not find an IP optimal solution
5. The disjunctive lower bound is strictly less than the maximum objective value on any leaf node

There are some instances for which we do not have data for all 6 partial tree sizes. We include these instances in most tables, except if we are showing how some statistic changes as the disjunction increases in size.

# Section 0: Set variables, import whatever is needed, and read in data

### Global variables

In [2]:
## Global variables
EPS = 1e-7
GAP_DIFF_EPS = 1e-3
INFINITY = 1e+100
MAX_TIME = 3600.

## Set up variables containing relevant directories
import os
repos_key = 'REPOS_DIR'
try:
    REPOS_DIR = os.environ[repos_key]
    print("REPOS_DIR set to \"%s\"." % REPOS_DIR)
    HOME_DIR = os.environ['HOME']
    print("HOME_DIR set to \"%s\"." % HOME_DIR)
except KeyError:
    print("*** ERROR: %s not found!" % repos_key)

VPC_DIR = REPOS_DIR + "/vpc/"
RESULTS_DIR = VPC_DIR + "results/saved/"
# RESULTS_DIR = HOME_DIR + '/' + "results/saved/"
DATA_DIR = VPC_DIR + "data/"

ONLY_PURE_BINARY = False
ONLY_MIXED_BINARY = False

# What to multiply by when an instance times out for nodes and time
TIMEOUT_TIME_FACTOR = 2. # TIMEOUT_TIME_FACTOR * MAX_TIME
TIMEOUT_NODE_FACTOR = 2. # TIMEOUT_NODE_FACTOR * # nodes

REPOS_DIR set to "/Users/akazachkov/repos".
HOME_DIR set to "/Users/akazachkov".


### Import data processing, plotting, and export packages and functions

In [3]:
## Import data processing, plotting, and export packages and functions
from IPython.display import display

from plots_helper import * # this includes matplotlib (+ params), pandas, and custom LaTeX helper functions

import re # regular expressions

### `initialize_df`: common way to process each data frame that we need

In [4]:
## Common way to process each data frame that we need
def initialize_df(filename):
    """
    Create a multilevel index df out of data from file `filename`.
    """
    df = pd.read_csv(filename, sep=',', index_col=False, skiprows=1)
    df.sort_values(by = ['INSTANCE','disj_terms'], inplace=True)
    df.set_index(['INSTANCE','disj_terms'], inplace=True)
    df.replace({"\'-inf\'": -np.inf, "\'inf\'": np.inf}, inplace=True)
    return df

## Column name definitions

(It is possible / likely that we are not using all of these...)

In [5]:
# Types of stats reported
pct_gap_closed_stub = "% GAP CLOSED"
bound_stub = "BOUND"
obj_stub = "OBJ"
time_stub = "TIME"
time_w_cuts_stub = time_stub + " " + "W/CUTGEN"
nodes_stub = "NODES"

# Types of solvers
solver_stubs = [ "REF", "REF+V" ]
solver_stat_list = [ "FIRST", "AVG", "BEST", "MIN", "MAX" ]

# Chosen statistics for computing with reference solver
REF_TYPE = 'AVG'
REFV_TYPE = 'AVG'
selected_stat_list = [ REF_TYPE, REFV_TYPE ]

# Gap closed cut columns
reg_cut_type_long_list = [ "GMIC", "ROOT", "BEST DISJ", "VPC", "MAX(GMIC,VPC)", "VPC+GMIC" ]
reg_cut_type_short_list = [ "G", "R", "DB", "V", "max(G,V)", "V+G" ]

solver_cut_type_stubs = [ "FIRST_CUT_PASS", "LAST_CUT_PASS" ]
solver_type = solver_stubs[0]
ref_cut_type_long_list    = [ solver_type + ' ' + cut_type for cut_type in solver_cut_type_stubs ]
ref_cut_type_short_list   = [ "GurF", "GurL" ]
solver_type = solver_stubs[1]
refv_cut_type_long_list   = [ solver_type + ' ' + cut_type for cut_type in solver_cut_type_stubs ]
refv_cut_type_short_list  = [ "V+GurF", "V+GurL" ]

solver_cut_type_long_list  = [ [solver_type + ' ' + cut_type for cut_type in solver_cut_type_stubs] for solver_type in solver_stubs ]
solver_cut_type_short_list = [ ref_cut_type_short_list, refv_cut_type_short_list ]

col_num_passes_stub = "NUM PASSES"

# Objective value columns
col_lp_obj = 'LP OBJ'
col_ip_obj = 'IP OBJ'
col_best_disj_stub = 'BEST DISJ'
col_best_disj_obj = col_best_disj_stub + ' ' + obj_stub
col_best_disj_gap = col_best_disj_stub + ' ' + pct_gap_closed_stub
col_worst_disj_stub = 'WORST DISJ'
col_worst_disj_obj = col_worst_disj_stub + ' ' + obj_stub
col_worst_disj_gap = col_worst_disj_stub + ' ' + pct_gap_closed_stub

obj_val_col_list = \
  [ col_lp_obj, col_ip_obj, col_worst_disj_obj ] \
  + \
  [ cut_type + ' ' + obj_stub for cut_type in reg_cut_type_long_list if cut_type != "MAX(GMIC,VPC)" ] \
  + \
  [ stat_type + ' ' + solver_type + ' ' + cut_type
      for cut_type in solver_cut_type_stubs
      for stat_type in solver_stat_list[:3]
      for solver_type in solver_stubs
  ]

# Time closed columns
time_type_list = [ "TIME", "NODES" ]

# Generation time
col_vpc_gen_time = 'VPC_GEN_TIME'

# Number rows/columns/cuts
col_num_rows        = 'ROWS'
col_num_cols        = 'COLS'
col_num_vpc         = 'NUM VPC'
col_num_gmic        = 'NUM GMIC'
col_num_disj_terms  = 'NUM DISJ TERMS'
col_num_obj         = 'NUM OBJ'
col_num_fails       = 'NUM FAILS'

# Instance information
col_binary          = 'BINARY'
col_integer         = 'INTEGER'
col_continuous      = 'CONTINUOUS'
col_gen_int         = 'GEN INT'
col_pure_binary     = 'IS PURE BINARY'
col_mixed_binary    = 'IS MIXED BINARY'

# Run information
col_exit_reason     = 'ExitReason'

### Gap column names

In [6]:
### Gap column names

# These will be calculated from the data
col_gmic            = 'GMIC' + ' ' + pct_gap_closed_stub
col_root            = 'ROOT' + ' ' + pct_gap_closed_stub
col_best_disj       = 'BEST DISJ' + ' ' + pct_gap_closed_stub
col_vpc             = 'VPC' + ' ' + pct_gap_closed_stub
col_max_gmic_vpc    = 'MAX(GMIC,VPC)' + ' ' + pct_gap_closed_stub
col_vpc_gmic        = 'VPC+GMIC' + ' ' + pct_gap_closed_stub

col_best_disj_vpc                 = 'BEST VPC DISJ'
col_best_disj_gmic_vpc            = 'BEST GMIC+VPC DISJ'
col_best_disj_refv_first_cut_pass = 'Best ' + refv_cut_type_short_list[0] + ' DISJ'
col_best_disj_refv_last_cut_pass  = 'Best ' + refv_cut_type_short_list[1] + ' DISJ'

## First cut pass
col_stub = solver_cut_type_stubs[0]
col_first_cut_pass_stubs = [ 
    stat_type + ' ' + solver_type + ' ' + col_stub 
    for stat_type in solver_stat_list[:3]
    for solver_type in solver_stubs 
]
#col_refv_first_cut_pass_stubs = [ stat_type + ' ' + 'REF+V FIRST_CUT_PASS' for stat_type in ref_solver_stat_list ]

# ref
solver_type = solver_stubs[0]
col_first_cut_pass_bound_ref_first = 'FIRST' + ' ' + solver_type + ' ' + col_stub
col_first_cut_pass_bound_ref_avg   = 'AVG' + ' ' + solver_type + ' ' + col_stub
col_first_cut_pass_bound_ref_best  = 'BEST' + ' ' + solver_type + ' ' + col_stub
col_first_cut_pass_gap_ref_first   = 'FIRST' + ' ' + solver_type + ' ' + col_stub + ' ' + pct_gap_closed_stub
col_first_cut_pass_gap_ref_avg     = 'AVG' + ' ' + solver_type + ' ' + col_stub + ' ' + pct_gap_closed_stub
col_first_cut_pass_gap_ref_best    = 'BEST' + ' ' + solver_type + ' ' + col_stub + ' ' + pct_gap_closed_stub

col_first_cut_pass_bound_ref       = REF_TYPE + ' ' + solver_type + ' ' + col_stub
col_first_cut_pass_gap_ref         = REF_TYPE + ' ' + solver_type + ' ' + col_stub + ' ' + pct_gap_closed_stub

# refv
solver_type = solver_stubs[1]
col_first_cut_pass_bound_ref_v     = REFV_TYPE + ' ' + solver_type + ' ' + col_stub
col_first_cut_pass_gap_ref_v       = REFV_TYPE + ' ' + solver_type + ' ' + col_stub + ' ' + pct_gap_closed_stub

## Last cut pass
col_stub = solver_cut_type_stubs[1]
col_last_cut_pass_stubs = [ 
    stat_type + ' ' + solver_type + ' ' + col_stub
    for stat_type in solver_stat_list[:3]
    for solver_type in solver_stubs 
]

# col_ref_last_cut_pass_stubs = [ stat_type + ' ' + 'REF LAST_CUT_PASS' for stat_type in ref_solver_stat_list ]
# col_refv_last_cut_pass_stubs = [ stat_type + ' ' + 'REF+V LAST_CUT_PASS' for stat_type in ref_solver_stat_list ]

# ref
solver_type = solver_stubs[0]
col_last_cut_pass_bound_ref_first  = 'FIRST' + ' ' + solver_type + ' ' + col_stub
col_last_cut_pass_bound_ref_avg    = 'AVG' + ' ' + solver_type + ' ' + col_stub
col_last_cut_pass_bound_ref_best   = 'BEST' + ' ' + solver_type + ' ' + col_stub
col_last_cut_pass_gap_ref_first    = 'FIRST' + ' ' + solver_type + ' ' + col_stub + ' ' + pct_gap_closed_stub
col_last_cut_pass_gap_ref_avg      = 'AVG' + ' ' + solver_type + ' ' + col_stub + ' ' + pct_gap_closed_stub
col_last_cut_pass_gap_ref_best     = 'BEST' + ' ' + solver_type + ' ' + col_stub + ' ' + pct_gap_closed_stub

col_last_cut_pass_bound_ref        = REF_TYPE + ' ' + solver_type + ' ' + col_stub
col_last_cut_pass_gap_ref          = REF_TYPE + ' ' + solver_type + ' ' + col_stub + ' ' + pct_gap_closed_stub

# refv
solver_type = solver_stubs[1]
col_last_cut_pass_bound_ref_v      = REFV_TYPE + ' ' + solver_type + ' ' + col_stub
col_last_cut_pass_gap_ref_v        = REFV_TYPE + ' ' + solver_type + ' ' + col_stub + ' ' + pct_gap_closed_stub

## Set of all gap cols
gap_cols = \
    [ col_stub + ' ' + pct_gap_closed_stub for col_stub in reg_cut_type_long_list] \
    + \
    [ col_stub + ' ' + pct_gap_closed_stub for col_stub in col_first_cut_pass_stubs] \
    + \
    [ col_stub + ' ' + pct_gap_closed_stub for col_stub in col_last_cut_pass_stubs]

solver_type = solver_stubs[0]
ref_solver_gap_cols = [
    stat_type + ' ' + solver_type + ' ' + col_stub + ' ' + pct_gap_closed_stub
    for stat_type in solver_stat_list[:3]
    for col_stub in solver_cut_type_stubs
]
# refv_solver_gap_cols = [
#     col_first_cut_pass_gap_ref_v,
#     col_last_cut_pass_gap_ref_v,
# ]

# Create maps between short and long names for gap columns
map_short_to_cols_gap = {
    short_stub : long_stub + ' ' + pct_gap_closed_stub for short_stub, long_stub in zip(reg_cut_type_short_list, reg_cut_type_long_list)
}
# Add ref and refv for first/last cut pass
for solver_ind in range(len(solver_stubs)):
    for cut_type_ind in range(len(solver_cut_type_stubs)):
        short_stub = solver_cut_type_short_list[cut_type_ind][solver_ind]
        long_stub = selected_stat_list[solver_ind] + ' ' + solver_cut_type_long_list[cut_type_ind][solver_ind] + ' ' + pct_gap_closed_stub
        map_short_to_cols_gap[short_stub] = long_stub
        
map_cols_to_short_gap = {v: k for k, v in map_short_to_cols_gap.items()}

gap_cols_short = list(map_short_to_cols_gap.keys())


### Time column names

In [7]:
### Time column names

solver_ind = 0
solver_type = solver_stubs[solver_ind]
ref_time_stub   = solver_type + ' ' + time_stub
ref_nodes_stub  = solver_type + ' ' + nodes_stub
ref_time_col    = selected_stat_list[solver_ind] + ' ' + ref_time_stub
ref_nodes_col   = selected_stat_list[solver_ind] + ' ' + ref_nodes_stub

solver_ind = 1
solver_type = solver_stubs[solver_ind]
refv_time_stub        = solver_type + ' ' + time_stub
refv_w_cut_time_stub  = solver_type + ' ' + 'W/CUTGEN' + ' ' + time_stub
refv_nodes_stub       = solver_type + ' ' + nodes_stub
refv_time_col         = selected_stat_list[solver_ind] + ' ' + refv_time_stub
refv_w_cut_time_col   = selected_stat_list[solver_ind] + ' ' + refv_w_cut_time_stub
refv_nodes_col        = selected_stat_list[solver_ind] + ' ' + refv_nodes_stub

map_cols_to_short_time = {
    ref_time_col          : 'Gur',
    refv_time_col         : 'V',
    col_vpc_gen_time      : 'Gen',
}

map_cols_to_short_nodes = {
    ref_nodes_col        : 'Gur',
    refv_nodes_col       : 'V',
}

map_short_to_cols_time = {v: k for k, v in map_cols_to_short_time.items()}
map_short_to_cols_nodes = {v: k for k, v in map_cols_to_short_nodes.items()}

time_cols_short = list(map_short_to_cols_time.keys())
node_cols_short = list(map_short_to_cols_nodes.keys())
# display(time_cols, node_cols)

# Select a subset of columns for the "long" list used when updating the 0-row
time_cols_long = [map_short_to_cols_time[col] for col in time_cols_short]
node_cols_long = [map_short_to_cols_nodes[col] for col in node_cols_short]

# How to handle timeouts (do we take average across the seeds, do we take the best, do we account for the timeout multiplicative factor?)
#STAT_FOR_TIMEOUT = 'BEST'
STAT_FOR_TIMEOUT = 'AVG'
ref_timeout_col = STAT_FOR_TIMEOUT + ' ' + ref_time_stub
refv_timeout_col = STAT_FOR_TIMEOUT + ' ' + refv_time_stub


## `df_ipopt`: Retrieve best known IP objective values

In [8]:
## Best known IP objective values
df_ipopt = pd.read_csv(DATA_DIR + "ip_obj.csv")
df_ipopt = df_ipopt.set_index(df_ipopt[df_ipopt.columns[0]])
# df_ipopt.rename(columns = {'IP_OBJ' : col_ip_obj}, inplace=True) # for consistency with other dfs
# df_ipopt.rename(columns = {'IP Objective' : col_ip_obj}, inplace=True) # for consistency with other dfs
df_ipopt = df_ipopt[~df_ipopt.index.duplicated()]
display(df_ipopt.head())
display(df_ipopt[col_ip_obj]['bm23_presolved'])

,INSTANCE,IP OBJ,SET
INSTANCE,,,
22433,22433,21477,miplib2017
23588,23588,8090,miplib2017
10teams,10teams,924,miplib2017
2club200v15p5scn,2club200v15p5scn,-70,miplib2017
30_70_45_05_100,30_70_45_05_100,9,miplib2017


'34'

## `df_preprocess`: Results from preprocessing instances

In [9]:
## Results from preprocessing instances
df_preprocess = pd.read_csv(RESULTS_DIR + "vpc-preprocess.csv", sep=',', index_col=False, skiprows=1)
df_preprocess = df_preprocess.set_index(df_preprocess[df_preprocess.columns[0]])
display(df_preprocess.head())

col_cleaned_lp_obj = 'CLEANED LP OBJ'
display(df_preprocess.loc['bm23',col_cleaned_lp_obj])

,INSTANCE,STRATEGY,ORIG LP OBJ,CLEANED LP OBJ,ORIG FIRST GUR NODES,CLEANED FIRST GUR NODES,ORIG BEST GUR NODES,CLEANED BEST GUR NODES,ORIG FIRST GUR TIME,CLEANED FIRST GUR TIME,...,vpc_version,cbc_version,clp_version,gurobi_version,cplex_version,ExitReason,end_time_string,time elapsed,instname,Unnamed: 137
INSTANCE,,,,,,,,,,,,,,,,,,,,,
22433,22433,536,21240.526171,21240.52617079889751039445,9,12,9,12,0.236,0.137,...,#e5b66ee,#d4272be,#8294096,10.02,22.1.1,SUCCESS,Thu Nov 9 16:36:45 2023,1,22433,DONE
23588,23588,536,7649.866134,7649.86613382250197901158,1612,654,1612,654,1.463,0.807,...,#e5b66ee,#d4272be,#8294096,10.02,22.1.1,SUCCESS,Wed Nov 8 23:51:27 2023,3,23588,DONE
10teams,10teams,536,917.000000,917.00000000000034106051,1,1,1,1,0.723,0.526,...,#e5b66ee,#d4272be,#8294096,10.02,22.1.1,SUCCESS,Wed Nov 8 22:25:16 2023,1,10teams,DONE
2club200v15p5scn,2club200v15p5scn,536,-121.222222,-120.07692307692302335909,231910,137774,231910,137774,7200.002,7200.001,...,#e5b66ee,#d4272be,#8294096,10.02,22.1.1,SUCCESS,Thu Nov 9 12:06:26 2023,14401,2club200v15p5scn,DONE
30_70_45_05_100,30_70_45_05_100,536,8.100000,8.09999999998854036676,1,1,1,1,4.036,4.394,...,#e5b66ee,#d4272be,#8294096,10.02,22.1.1,SUCCESS,Wed Nov 8 23:26:24 2023,18,30_70_45_05_100,DONE


'20.57092176323557097817'

## `df_bb`: Results from generating VPCs for various number of disjunctive terms

In [10]:
## Results from generating VPCs for various number of disjunctive terms
df_bb = initialize_df(RESULTS_DIR + "vpc-bb0bb.csv")
display(df_bb.head())

cutlimit  gomory  mode  partial_bb_strategy  \
INSTANCE          disj_terms                                                
10teams_presolved 2                 -1      -1     0                    4   
                  4                 -1      -1     0                    4   
                  8                 -1      -1     0                    4   
                  16                -1      -1     0                    4   
                  32                -1      -1     0                    4   

                              partial_bb_keep_pruned_nodes  \
INSTANCE          disj_terms                                 
10teams_presolved 2                                      0   
                  4                                      0   
                  8                                      0   
                  16                                     0   
                  32                                     0   

                              partial_bb_num_strong  preprocess  \
INSTANCE          disj_terms                                      
10teams_presolved 2                               5           0   
                  4                               5           0   
                  8                               5           0   
                  16                              5           0   
                  32                              5           0   

                              prlp_flip_beta  rounds  bb_mode  ...  \
INSTANCE          disj_terms                                   ...   
10teams_presolved 2                        0       1       11  ...   
                  4                        0       1       11  ...   
                  8                        0       1       11  ...   
                  16                       0       1       11  ...   
                  32                       0       1       11  ...   

                              gurobi_version  cplex_version  hostname  \
INSTANCE          disj_terms                                            
10teams_presolved 2                    10.03            NaN   rupert2   
                  4                    10.03            NaN   rupert1   
                  8                    10.03            NaN   rupert0   
                  16                   10.03            NaN   rupert1   
                  32                   10.03            NaN   rupert4   

                                                         cpu_model  cpu_id  \
INSTANCE          disj_terms                                                 
10teams_presolved 2           13th Gen Intel(R) Core(TM) i9-13900K       8   
                  4           13th Gen Intel(R) Core(TM) i9-13900K      12   
                  8           13th Gen Intel(R) Core(TM) i9-13900K       2   
                  16          13th Gen Intel(R) Core(TM) i9-13900K       2   
                  32          13th Gen Intel(R) Core(TM) i9-13900K       4   

                              ExitReason           end_time_string  \
INSTANCE          disj_terms                                         
10teams_presolved 2              SUCCESS  Sat Jan 20 15:29:36 2024   
                  4           FAIL_LIMIT  Sat Jan 20 15:10:22 2024   
                  8           FAIL_LIMIT  Sun Jan 21 04:52:49 2024   
                  16             SUCCESS  Sun Jan 21 03:58:29 2024   
                  32          TIME_LIMIT  Sun Jan 21 13:26:30 2024   

                              time elapsed           instname   end  
INSTANCE          disj_terms                                         
10teams_presolved 2                     31  10teams_presolved  DONE  
                  4                     30  10teams_presolved  DONE  
                  8                     84  10teams_presolved  DONE  
                  16                  1558  10teams_presolved  DONE  
                  32                  3636  10teams_presolved  DONE  

[5 rows x 294 columns]

## `df_disjset`: Read in results from using cuts across all 6 trees

In [11]:
df_disjset = initialize_df(RESULTS_DIR + "vpc-disjset.csv")

# Set disj_terms index to be -1 for all instances
df_disjset.index = df_disjset.index.set_levels([-1], level='disj_terms')

display(df_disjset.head())

,,cutlimit,gomory,mode,partial_bb_strategy,partial_bb_keep_pruned_nodes,partial_bb_num_strong,preprocess,prlp_flip_beta,rounds,bb_mode,...,gurobi_version,cplex_version,hostname,cpu_model,cpu_id,ExitReason,end_time_string,time elapsed,instname,Unnamed: 294
INSTANCE,disj_terms,,,,,,,,,,,,,,,,,,,,,
10teams_presolved,-1,-1,-1,4,4,0,5,0,0,1,11,...,10.03,22.1.1,rupert1,13th Gen Intel(R) Core(TM) i9-13900K,6,PRLP_TIME_LIMIT,Thu Nov 30 12:49:40 2023,494,10teams_presolved,DONE
22433_presolved,-1,-1,-1,4,4,0,5,0,0,1,11,...,10.03,22.1.1,rupert1,13th Gen Intel(R) Core(TM) i9-13900K,2,OPTIMAL_SOLUTION_FOUND,Thu Nov 30 12:37:57 2023,2,22433_presolved,DONE
23588_presolved,-1,-1,-1,4,4,0,5,0,0,1,11,...,10.03,22.1.1,rupert0,13th Gen Intel(R) Core(TM) i9-13900K,8,CUT_LIMIT,Thu Nov 30 12:34:58 2023,50,23588_presolved,DONE
30n20b8_presolved,-1,-1,-1,4,4,0,5,0,0,1,11,...,10.03,22.1.1,rupert3,13th Gen Intel(R) Core(TM) i9-13900K,2,PRLP_TIME_LIMIT,Thu Nov 30 12:51:40 2023,965,30n20b8_presolved,DONE
50v-10_presolved,-1,-1,-1,4,4,0,5,0,0,1,11,...,10.03,22.1.1,rupert4,13th Gen Intel(R) Core(TM) i9-13900K,10,CUT_LIMIT,Thu Nov 30 22:45:39 2023,36601,50v-10_presolved,DONE


## `df`: Append to `df_bb` results from running baseline solver 7 times

In [12]:
## Append results from running baseline solver 7 times
#df = df_bb.append(initialize_df(RESULTS_DIR + "vpc-bb0.csv")) # deprecated
df = pd.concat([df_disjset, df_bb, initialize_df(RESULTS_DIR + "vpc-bb0.csv")])
df.sort_values(by = ['INSTANCE','disj_terms'], inplace=True)

col_list = [col_best_disj_obj, col_worst_disj_obj]
for col in col_list:
    df[col] = pd.to_numeric(df[col])

# Create new column for number of disjunctive terms since original one is now index
df[col_num_disj_terms] = df.index.get_level_values(1)

## Identify pure binary instances, which are those where 'CLEANED BINARY' column equals 'CLEANED COLS'
df[col_pure_binary] = (df[col_binary] == df[col_num_cols])

## Identify mixed binary instances, which are those where 'CLEANED GEN INT' column = 0
df[col_mixed_binary] = (df[col_gen_int] == 0)

# col_list = [col_num_disj_terms]
# for col in col_list:
#     df[col] = pd.to_numeric(df[col])

# start = 220
# end = start + 15
# print(df.columns[start:end])
# print(df.dtypes[start:end])

display(df.head())

cutlimit  gomory  mode  partial_bb_strategy  \
INSTANCE          disj_terms                                                
10teams_presolved -1                -1      -1     4                    4   
                   0                -1       0     0                    4   
                   2                -1      -1     0                    4   
                   4                -1      -1     0                    4   
                   8                -1      -1     0                    4   

                              partial_bb_keep_pruned_nodes  \
INSTANCE          disj_terms                                 
10teams_presolved -1                                     0   
                   0                                     0   
                   2                                     0   
                   4                                     0   
                   8                                     0   

                              partial_bb_num_strong  preprocess  \
INSTANCE          disj_terms                                      
10teams_presolved -1                              5           0   
                   0                              5           0   
                   2                              5           0   
                   4                              5           0   
                   8                              5           0   

                              prlp_flip_beta  rounds  bb_mode  ...  cpu_id  \
INSTANCE          disj_terms                                   ...           
10teams_presolved -1                       0       1       11  ...       6   
                   0                       0       1        1  ...       2   
                   2                       0       1       11  ...       8   
                   4                       0       1       11  ...      12   
                   8                       0       1       11  ...       2   

                                   ExitReason           end_time_string  \
INSTANCE          disj_terms                                              
10teams_presolved -1          PRLP_TIME_LIMIT  Thu Nov 30 12:49:40 2023   
                   0                  SUCCESS  Sat Dec  2 13:44:10 2023   
                   2                  SUCCESS  Sat Jan 20 15:29:36 2024   
                   4               FAIL_LIMIT  Sat Jan 20 15:10:22 2024   
                   8               FAIL_LIMIT  Sun Jan 21 04:52:49 2024   

                              time elapsed           instname  Unnamed: 294  \
INSTANCE          disj_terms                                                  
10teams_presolved -1                   494  10teams_presolved          DONE   
                   0                    15  10teams_presolved           NaN   
                   2                    31  10teams_presolved           NaN   
                   4                    30  10teams_presolved           NaN   
                   8                    84  10teams_presolved           NaN   

                              max_support_rel   end  IS PURE BINARY  \
INSTANCE          disj_terms                                          
10teams_presolved -1                      NaN   NaN            True   
                   0                      NaN  DONE            True   
                   2                      1.0  DONE            True   
                   4                      1.0  DONE            True   
                   8                      1.0  DONE            True   

                              IS MIXED BINARY  
INSTANCE          disj_terms                   
10teams_presolved -1                     True  
                   0                     True  
                   2                     True  
                   4                     True  
                   8                     True  

[5 rows x 297 columns]

In [13]:
# # get the count of each instance-disj_terms pair
# counts = df.groupby(level=[0]).size()

# # get the instances that have only one occurrence
# instances_with_one_occurrence = counts[counts == 1].index.get_level_values(0).unique()

# # filter df_bb to only include instances with one occurrence
# df_bb_one_occurrence = df.loc[instances_with_one_occurrence]

# # display the filtered dataframe
# display(df_bb_one_occurrence)

### Remove `stein*` instances (keep modified `stein*_nocard` instances)

In [14]:
# Remove unmodified stein instances from consideration
df.drop(index = ['stein09_presolved', 'stein15_presolved', 'stein27_presolved', 'stein45_presolved'], inplace=True)
df.index = df.index.remove_unused_levels()

### Modify `mas` instances

In [15]:
df_preprocess.rename(index={'mas74': 'mas074', 'mas76': 'mas076'}, inplace=True)

### Recompute average running time to account for timeouts

In [16]:
for solver in solver_stubs:
  # Split values in 'ALL REF TIME' into new columns for 'REF TIME (SEED)' where 'SEED' takes values 628 * [1,2,3,4,5,6,7]
  df_timing = df['ALL '+solver+' TIME'].str.split(';', expand=True)
  df_timing.columns = [solver+' TIME (%d)' % (i+1) for i in range(df_timing.shape[1])]
  df_timing = df_timing.astype(float)
  df_timing_cols = df_timing.columns[df_timing.columns.str.contains(re.escape(solver)+' TIME \(.+\)')]

  # Do the same for nodes
  df_nodes = df['ALL '+solver+' NODES'].str.split(';', expand=True)
  df_nodes.columns = [solver+' NODES (%d)' % (i+1) for i in range(df_nodes.shape[1])]
  df_nodes = df_nodes.astype(float)
  df_nodes_cols = df_nodes.columns[df_nodes.columns.str.contains(re.escape(solver)+' NODES \(.+\)')]

  # Select entries in which the max time is greater than 3600
  selected_entries = df_timing > (MAX_TIME - EPS)

  # Add min and max of 'REF TIME (SEED)' columns
  df[solver+' TIME MIN'] = df_timing[df_timing_cols].min(axis=1)
  df[solver+' TIME MAX'] = df_timing[df_timing_cols].max(axis=1)

  # Find average of 'REF TIME (SEED)' columns after adjusting for timeout
  df_timing[selected_entries] = TIMEOUT_TIME_FACTOR * MAX_TIME
  # df_timing[solver+' TIME AVG'] = df_timing.mean(axis=1)
  df['AVG '+solver+' TIME'] = df_timing[df_timing_cols].mean(axis=1)

  ## Repeat for nodes
  # Add min and max of 'REF NODES (SEED)' columns
  df[solver+' NODES MIN'] = df_nodes[df_nodes_cols].min(axis=1)
  df[solver+' NODES MAX'] = df_nodes[df_nodes_cols].max(axis=1)

  # Find average of 'REF NODES (SEED)' columns after adjusting for timeout
  selected_entries.columns = df_nodes_cols
  df_nodes[selected_entries] *= TIMEOUT_NODE_FACTOR
  # df_nodes[solver+' NODES AVG'] = df_nodes[df_nodes_cols].mean(axis=1)
  df['AVG '+solver+' NODES'] = df_nodes[df_nodes_cols].mean(axis=1)

  # Append df_timing to df
  df = pd.concat([df, df_timing], axis=1)

  # Append df_nodes to df
  df = pd.concat([df, df_nodes], axis=1)

df.head()

cutlimit  gomory  mode  partial_bb_strategy  \
INSTANCE          disj_terms                                                
10teams_presolved -1                -1      -1     4                    4   
                   0                -1       0     0                    4   
                   2                -1      -1     0                    4   
                   4                -1      -1     0                    4   
                   8                -1      -1     0                    4   

                              partial_bb_keep_pruned_nodes  \
INSTANCE          disj_terms                                 
10teams_presolved -1                                     0   
                   0                                     0   
                   2                                     0   
                   4                                     0   
                   8                                     0   

                              partial_bb_num_strong  preprocess  \
INSTANCE          disj_terms                                      
10teams_presolved -1                              5           0   
                   0                              5           0   
                   2                              5           0   
                   4                              5           0   
                   8                              5           0   

                              prlp_flip_beta  rounds  bb_mode  ...  \
INSTANCE          disj_terms                                   ...   
10teams_presolved -1                       0       1       11  ...   
                   0                       0       1        1  ...   
                   2                       0       1       11  ...   
                   4                       0       1       11  ...   
                   8                       0       1       11  ...   

                              REF+V TIME (5)  REF+V TIME (6)  REF+V TIME (7)  \
INSTANCE          disj_terms                                                   
10teams_presolved -1                4.006704        1.230319        1.703434   
                   0                     NaN             NaN             NaN   
                   2                0.443166        0.675966        5.114223   
                   4                0.727127        0.671572        5.106286   
                   8                0.731199        0.675403        5.134729   

                              REF+V NODES (1)  REF+V NODES (2)  \
INSTANCE          disj_terms                                     
10teams_presolved -1                     29.0              1.0   
                   0                      NaN              NaN   
                   2                    106.0            138.0   
                   4                    106.0            138.0   
                   8                    106.0            138.0   

                              REF+V NODES (3)  REF+V NODES (4)  \
INSTANCE          disj_terms                                     
10teams_presolved -1                    384.0            399.0   
                   0                      NaN              NaN   
                   2                    180.0            524.0   
                   4                    180.0            524.0   
                   8                    180.0            524.0   

                              REF+V NODES (5)  REF+V NODES (6)  \
INSTANCE          disj_terms                                     
10teams_presolved -1                   1039.0            132.0   
                   0                      NaN              NaN   
                   2                      1.0             52.0   
                   4                     19.0             52.0   
                   8                     19.0             52.0   

                              REF+V NODES (7)  
INSTANCE          disj_terms                   
10teams_presolved -1                    299.0  


### Add col for avg b&b time counting cut generation

In [17]:
### Add total time for running solver + generating cuts
df[refv_w_cut_time_col] = df[refv_time_col] + df[col_vpc_gen_time]

inst_set = ['a1c1s1_presolved']

# display(df.loc['30n20b8_presolved'][[ref_time_col,refv_time_col,col_vpc_gen_time,refv_w_cut_time_col]])
# display(df.loc['binkar10_1_presolved'][[ref_time_col,refv_time_col,col_vpc_gen_time,refv_w_cut_time_col]])
display(df.loc[inst_set][[col_num_vpc, ref_time_col,refv_time_col,col_vpc_gen_time,refv_w_cut_time_col]])

NUM VPC  AVG REF TIME  AVG REF+V TIME  \
INSTANCE         disj_terms                                          
a1c1s1_presolved -1               22    531.318307      605.310975   
                  0                0    617.083421             NaN   
                  2               15    519.457900      510.424348   
                  4                4    521.557748      499.265273   
                  8                3    523.138207      499.183995   
                  16               1    633.798969      602.985181   
                  32               0           NaN             NaN   
                  64               0           NaN             NaN   

                             VPC_GEN_TIME  AVG REF+V W/CUTGEN TIME  
INSTANCE         disj_terms                                         
a1c1s1_presolved -1                 11.06               616.370975  
                  0                  0.00                      NaN  
                  2                  0.16               510.584348  
                  4                  0.28               499.545273  
                  8                  0.54               499.723995  
                  16                 1.27               604.255181  
                  32                 2.12                      NaN  
                  64                 4.77                      NaN

### (Disabled; code fixed in commit 4ed946c) Fix mistake in code for one root pass containing wrong bound

In [18]:
# for prefix in solver_stat_list:
#     inst_set = df[prefix + " REF+V ROOT_PASSES"] == 1
#     tmp_df = df[inst_set]
#     tmp_df = tmp_df[[col_lp_obj,
#                      prefix + " REF+V BOUND", 
#                      prefix + " REF+V FIRST_CUT_PASS",
#                      prefix + " REF+V LAST_CUT_PASS"]]
#     tmp_df.tail(30)

#     tmp_tmp_df = tmp_df["LP OBJ"] - tmp_df[prefix + " REF+V FIRST_CUT_PASS"]
#     assert(tmp_tmp_df.max() < EPS)

#     refcol = prefix + " REF+V BOUND"
#     col = prefix + " REF+V FIRST_CUT_PASS"
#     df.loc[inst_set,col] = tmp_df[refcol].values
#     col = prefix + " REF+V LAST_CUT_PASS"
#     df.loc[inst_set,col] = tmp_df[refcol].values


# df.loc  [["misc02_presolved",'cap6000_presolved'],
#          ["FIRST REF+V BOUND",
#           "FIRST REF+V FIRST_CUT_PASS",
#           "FIRST REF+V LAST_CUT_PASS"
#          ]
#          +
#          ["BEST REF+V BOUND",
#           "BEST REF+V FIRST_CUT_PASS",
#           "BEST REF+V LAST_CUT_PASS"
#          ]
#         ]

## `instances`: get unique instance list

In [19]:
# Get unique instance list
if (ONLY_PURE_BINARY):
    # Select only instances in df in which column ['IS PURE BINARY'] is True
    tmp_df = df[df['IS PURE BINARY'] == True]
    tmp_df.index = tmp_df.index.remove_unused_levels()
    instances = tmp_df.index.levels[0]
elif (ONLY_MIXED_BINARY):
    tmp_df = df[df['IS MIXED BINARY'] == True]
    tmp_df.index = tmp_df.index.remove_unused_levels()
    instances = tmp_df.index.levels[0]
else:
    instances = df.index.levels[0]

instances.set_names(names = 'Instance', inplace=True)

pure_binary_instances = df[df[col_pure_binary] == True].index.get_level_values(0).unique().to_list()

print("Number of selected instances: ", len(instances))
print("Number of binary instances: {:d}/{:d}".format(len(pure_binary_instances), len(instances)))

Number of selected instances:  438
Number of binary instances: 88/438


## `df_rejection_reason`: Track why instances were not selected for our statistics

In [20]:
rejection_reasons = [
    'SELECTED_GAP', # *not* rejected for gap experiments; _must_ be first column
    'SELECTED_TIME', # *not* rejected for time experiments; _must_ be second column
    'SELECTED_6TREES', # *not* rejected for 6trees set for time experiments; _must_ be third column
    'NUM_WITH_OBJS', # number of attempts that successfully tried solving the PRLP
    'NUM_WITH_CUTS', # number of attempts that successfully yielded cuts
    'IP_OPT_UNKNOWN', # ip opt val must be known
    'TOO_MANY_ROWS_OR_COLS', # require max(nrows, ncols) ≤ 5K
    'OPTIMAL_SOLUTION_FOUND', # optimal solution should not be found by any of the partial trees
    'LP_OPT_IS_NOT_CUT', # check if lp opt < ip opt
    'DLB=DUB', # check if disj lb < disj ub
    'LP=DLB=DUB', # require either lp opt < disj lb or disj lb < disj ub
    'PRLP_INFEASIBLE', # require PRLP is feasible and solves within timelimit for at least one of the attempts
    'PRLP_TIME_LIMIT', # require PRLP solves within timelimit for at least one of the attempts
    'NO_CUTS', # there must be cuts from at least one of the partial b&b trees
    'NO_GAP', # require that ip opt != lp opt
    'GUR_TIMEOUT', # require Gur7 < 3600 (Gurobi is able to solve the instance to optimality within an hour either with or without using VPCs)'
    '<7_ATTEMPTS', # indicates not all partial trees were successfully run
]
df_rejection_reason = pd.DataFrame(index = instances, columns = rejection_reasons, dtype=bool)
df_rejection_reason.iloc[:,3:] = False # no rejection criteria at true

for col in ['OPTIMAL_SOLUTION_FOUND']:
    df_rejection_reason[col] = df_rejection_reason[col].astype(np.int64)
for col in ['NUM_WITH_OBJS', 'NUM_WITH_CUTS', 'LP_OPT_IS_NOT_CUT', 'DLB=DUB', 'LP=DLB=DUB', 'PRLP_INFEASIBLE', 'PRLP_TIME_LIMIT', 'GUR_TIMEOUT']:
    df_rejection_reason[col] = df_rejection_reason[col].astype(np.int8)
display(df_rejection_reason.head())

,SELECTED_GAP,SELECTED_TIME,SELECTED_6TREES,NUM_WITH_OBJS,NUM_WITH_CUTS,IP_OPT_UNKNOWN,TOO_MANY_ROWS_OR_COLS,OPTIMAL_SOLUTION_FOUND,LP_OPT_IS_NOT_CUT,DLB=DUB,LP=DLB=DUB,PRLP_INFEASIBLE,PRLP_TIME_LIMIT,NO_CUTS,NO_GAP,GUR_TIMEOUT,<7_ATTEMPTS
Instance,,,,,,,,,,,,,,,,,
10teams_presolved,True,True,True,0,0,False,False,0,0,0,0,0,0,False,False,0,False
22433_presolved,True,True,True,0,0,False,False,0,0,0,0,0,0,False,False,0,False
23588_presolved,True,True,True,0,0,False,False,0,0,0,0,0,0,False,False,0,False
30n20b8_presolved,True,True,True,0,0,False,False,0,0,0,0,0,0,False,False,0,False
50v-10_presolved,True,True,True,0,0,False,False,0,0,0,0,0,0,False,False,0,False


## `map_rejection_reason_to_number`: Reference paper's rejection criteria

In [21]:
# map_rejection_reason_to_number = {
#     'OPTIMAL_SOLUTION_FOUND':   '(3)',
#     'LP=DLB=DUB':               '(4a)',
#     'PRLP_INFEASIBLE':          '(4b)',
#     'PRLP_TIME_LIMIT':          '(4c)',
#     '<7_ATTEMPTS':              '(?)',
# }
map_rejection_reason_to_number = {
    'IP_OPT_UNKNOWN':           '\\ref{selection-criterion:ip-opt-known}',
    'NO_GAP':                   '\\ref{selection-criterion:ip-opt-known}',
    'TOO_MANY_ROWS_OR_COLS':    '\\ref{selection-criterion:max-instance-size}',
    'OPTIMAL_SOLUTION_FOUND':   '\\ref{selection-criterion:partial-tree-does-not-find-opt}',
    'LP=DLB=DUB':               '\\ref{selection-criterion:cuts-are-generated:not_lp=dlb=dub}',
    'PRLP_INFEASIBLE':          '\\ref{selection-criterion:cuts-are-generated:PRLP-primal-feasible}',
    'PRLP_TIME_LIMIT':          '\\ref{selection-criterion:cuts-are-generated:PRLP-time-limit}',
    'NO_CUTS':                  '\\ref{selection-criterion:cuts-are-generated:cuts-are-generated}',
    'GUR_TIMEOUT':              'G',
    '<7_ATTEMPTS':              '?',
}

## `df_status_by_depth`: Track success or failure reason by depth

In [22]:
sizes = [2, 4, 8, 16, 32, 64]
df_status_by_depth = pd.DataFrame(index = instances, columns = sizes, dtype=str)

DEFAULT_STATUS = map_rejection_reason_to_number['<7_ATTEMPTS']

df_status_by_depth[:] = DEFAULT_STATUS

display(df_status_by_depth.head())

,2,4,8,16,32,64
Instance,,,,,,
10teams_presolved,?,?,?,?,?,?
22433_presolved,?,?,?,?,?,?
23588_presolved,?,?,?,?,?,?
30n20b8_presolved,?,?,?,?,?,?
50v-10_presolved,?,?,?,?,?,?


### DEBUG

In [23]:
# col = "REF+V FIRST_CUT_PASS"
# tmp = df[col]
# display(tmp)

# for col in df.columns:
#     if str(col).endswith("FIRST_CUT_PASS"):
#         print("{}".format(col))

# inst = 'neos22_presolved'
# col = 'NUM DISJ TERMS'
# df.loc[inst][col]

# display(df.loc[('bppc4-08_presolved',2)]['LP OBJ'])
# display(df.loc[('bppc4-08_presolved',2)]['BEST DISJ OBJ'])
# display(df.loc[('bppc4-08_presolved',2)]['WORST DISJ OBJ'])
# display(df['BEST DISJ OBJ'])

# Section 1: Select instances

### `selected_gap_instances_dict` (original index, instance): Select instances for gap closed calculations

Criteria to filter gap closed instances:
* ip opt val is known
* lp opt < ip opt
* max(nrows, ncols) ≤ 5K
* optimal solution should not be found by any of the partial trees
* either lp opt < disj lb or disj lb < disj ub
* PRLP is feasible and solves within timelimit for at least one of the attempts

In [24]:
## Select instances for gap closed calculations
#
# Criteria to filter gap closed instances:
# * ip opt val is known
# * lp opt < ip opt
# * max(nrows, ncols) ≤ 5K
# * optimal solution should not be found by any of the partial trees
# * either lp opt < disj lb or disj lb < disj ub
# * PRLP is feasible and solves within timelimit for at least one of the attempts

# Constants
MAX_ROWS = 5000
MAX_COLS = MAX_ROWS
PRINT_SKIP_REASON = False

# Information to save
selected_gap_instances_dict = {} # dictionary of (original index, instance)
#selected_indices = []
num_gap_errors = 0

inst_set = instances
num_attempts = np.zeros(len(inst_set), dtype=int)

for i, inst in enumerate(inst_set):
    if inst == 'cod105_presolved':
        PRINT_SKIP_REASON = True
    else:
        PRINT_SKIP_REASON = False
    print("{}/{}".format(i+1,len(inst_set)), end='\r', flush=True)
    skip_instance = False
    curr_df = df.loc[inst]
    curr_df = curr_df[curr_df.index.get_level_values(0) >= 0]
    
    # Count number of times instance appears
    num_attempts[i] = len(curr_df)

    if num_attempts[i] < 7:
        print("*** ERROR: Instance {:d} -- {}: {:d} < 7 attempts.".format(i, inst, num_attempts[i]))
        skip_instance = True
        num_gap_errors += 1
        df_rejection_reason.loc[inst, '<7_ATTEMPTS'] = True

    # Check that LP opt < IP opt
    lp_obj = np.float64(df_preprocess.loc[remove_presolved_from_name(inst),col_cleaned_lp_obj])
    ip_obj = np.float64(df_ipopt.loc[inst,col_ip_obj])
    YES_GAP = (ip_obj - lp_obj) >= EPS
    if not YES_GAP:
        print("*** ERROR: Instance {:d} -- {}: not YES GAP (lp = {:.10f}; ip = {:.10f}, diff = {:.2f})".format(i, inst, lp_obj, ip_obj, ip_obj-lp_obj))
        skip_instance = True
        num_gap_errors += 1
        df_rejection_reason.loc[inst, 'NO_GAP'] = True
        
    # Check that ExitReason != OPTIMAL_SOLUTION_FOUND
    OPT_SOL_FOUND = False
    for curr_index, row in curr_df.iterrows():
        #print(i,j, curr_df['ExitReason'])
        curr_depth = int(curr_index)
        if curr_depth <= 0:
            continue
        exitreason = row[col_exit_reason]
        if exitreason == 'OPTIMAL_SOLUTION_FOUND' and not OPT_SOL_FOUND:
            if PRINT_SKIP_REASON:
                print("Skipping instance {:d} -- {}: optimal IP solution found at depth {:d}.".format(
                    i, inst, curr_depth
                ))
            skip_instance = True
            OPT_SOL_FOUND = True
            df_rejection_reason.loc[inst, 'OPTIMAL_SOLUTION_FOUND'] = curr_depth
        if OPT_SOL_FOUND:
            df_status_by_depth.loc[inst, curr_depth] = map_rejection_reason_to_number['OPTIMAL_SOLUTION_FOUND']
        else:
            df_status_by_depth.loc[inst, curr_depth] = ''

    # Check that best and worst bound on leaf nodes is not same (likely cause of primal infeasible PRLP)
    num_successful_attempts = 0
    has_zero = False # could check with "0 in curr_df['NUM DISJ TERMS'], but that would implicitly add an extra (short) loop per instance"
    terms = curr_df.index
    for curr_index in terms:
        if curr_df[col_num_disj_terms][curr_index] == 0:
            has_zero = True
            continue
            
        curr_lp_obj = curr_df[col_lp_obj][curr_index]
        curr_ip_obj = curr_df[col_ip_obj][curr_index]
        best_disj_obj = curr_df[col_best_disj_obj][curr_index]
        worst_disj_obj = curr_df[col_worst_disj_obj][curr_index]
        num_frac = curr_df['NUM FRAC'][curr_index]
        num_obj_tried = curr_df[col_num_obj][curr_index]
        num_cuts = curr_df[col_num_vpc][curr_index] # can be > 0 even if num_obj_tried = 0, b/c of OPTIMAL_SOLUTION_FOUND exit reason
        exitreason = curr_df[col_exit_reason][curr_index]

        # Quick double check that LP and IP objectives are correct
        if abs(curr_lp_obj - lp_obj) >= GAP_DIFF_EPS:
            raise ValueError(
                "*** ERROR: Instance {:d} -- {}: at depth {:d}, curr lp obj {:.10f} != lp obj {:.10f}".format(
                    i, inst, curr_index, curr_lp_obj, lp_obj
                )
            )
        if abs(curr_ip_obj - ip_obj) >= GAP_DIFF_EPS:
            raise ValueError(
                "*** ERROR: Instance {:d} -- {}: at depth {:d}, curr ip obj {:.10f} != ip obj {:.10f}".format(
                    i, inst, curr_index, curr_ip_obj, ip_obj
                )
            )

        curr_YES_GAP = (ip_obj - lp_obj) >= 1e-7
        LP_OPT_IS_CUT = (num_frac > 0) and curr_YES_GAP and abs(lp_obj - worst_disj_obj) >= 1e-7
        DLB_NE_DUB = (num_frac > 0) and abs(best_disj_obj - worst_disj_obj) >= 1e-7
        df_rejection_reason.loc[inst, 'NO_GAP'] += (not curr_YES_GAP)
        df_rejection_reason.loc[inst, 'LP_OPT_IS_NOT_CUT'] += (not LP_OPT_IS_CUT)
        df_rejection_reason.loc[inst, 'DLB=DUB'] += (not DLB_NE_DUB)
        df_rejection_reason.loc[inst, 'PRLP_INFEASIBLE'] += (exitreason == 'PRLP_INFEASIBLE')
        df_rejection_reason.loc[inst, 'PRLP_TIME_LIMIT'] += (exitreason == 'PRLP_TIME_LIMIT')
        # if not DLB_NE_DUB and num_obj_tried > 0:
        #     raise ValueError(
        #         "*** ERROR: Instance {:d} -- {}: at depth {:d}, num obj tried = {:d} (num cuts = {:d}) but lp opj {:.10f}, best_disj_obj {:.10f} = worst_disj_obj {:.10f} with exit reason {}".format(
        #             i, inst, curr_index, num_obj_tried, num_cuts, lp_obj, best_disj_obj, worst_disj_obj, curr_df['ExitReason'][curr_index]
        #         )
        #     )
        if LP_OPT_IS_CUT or DLB_NE_DUB:
            if (num_obj_tried == 0) and (exitreason not in ['PRLP_TIME_LIMIT','PRLP_INFEASIBLE','OPTIMAL_SOLUTION_FOUND','TIME_LIMIT']):
                # We should be trying objectives at this point, unless the initial PRLP timed out or was infeasible or an optimal solution was found
                raise ValueError(
                    "*** ERROR: Instance {:d} -- {}: at depth {:d}, num obj tried = 0 but lp opj {:.10f} < best_disj_obj {:.10f} < worst_disj_obj {:.10f} with exit reason {}".format(
                        i, inst, curr_index, lp_obj, best_disj_obj, worst_disj_obj, curr_df['ExitReason'][curr_index]
                    )
                )
            if num_obj_tried > 0:
                df_rejection_reason.loc[inst, 'NUM_WITH_OBJS'] += 1
                if num_cuts > 0:
                    num_successful_attempts += 1
                    df_rejection_reason.loc[inst, 'NUM_WITH_CUTS'] += 1
                else:
                    df_status_by_depth.loc[inst, curr_index] = map_rejection_reason_to_number['NO_CUTS']
            elif exitreason == 'PRLP_INFEASIBLE':
                df_status_by_depth.loc[inst, curr_index] = map_rejection_reason_to_number[exitreason]
            elif exitreason == 'PRLP_TIME_LIMIT':
                df_status_by_depth.loc[inst, curr_index] = map_rejection_reason_to_number[exitreason]
        else:
            # check that num obj tried is 0
            if (num_obj_tried > 0):
                raise ValueError(
                    "*** ERROR: Instance {:d} -- {}: at depth {:d}, num obj tried = {:d} > 0 but best_disj_obj {:f} = worst_disj_obj {:f}".format(
                        i, inst, curr_index, num_obj_tried, best_disj_obj, worst_disj_obj
                    )
                )
            df_rejection_reason.loc[inst, 'LP=DLB=DUB'] += 1
            df_status_by_depth.loc[inst, int(curr_index)] = map_rejection_reason_to_number['LP=DLB=DUB']

    if not has_zero:
        raise ValueError(
            "*** ERROR: Instance {:d} -- {}: has no bb0 entry.".format(
                i, inst, curr_index
            )
        )        
    
    if num_successful_attempts == 0 and not skip_instance:
        if PRINT_SKIP_REASON:
            print("Skipping instance {:d} -- {}: best and worst bound on leaf nodes coincide for all trees, no objectives ever tried, or no objectives successfully produced cuts.".format(
                i, inst, num_attempts[i]))
        skip_instance = True
        exitreason = 'NO_CUTS'
        df_rejection_reason.loc[inst, exitreason] = True
    else:        
        # Ensure IP objective value is known
        ip_obj = curr_df[col_ip_obj][curr_df.index[0]]
        if not isinstance(ip_obj,float):
            if PRINT_SKIP_REASON:
                print(
                    "Skipping instance {:d} -- {}: IP objective value ({}) is not detected to be a float value.".format(
                    i, inst, ip_obj))
            skip_instance = True
            df_rejection_reason.loc[inst, 'IP_OPT_UNKNOWN'] = True
            
        # Ensure nrows and ncols is not too many
        nrows = curr_df[col_num_rows][curr_df.index[0]]
        ncols = curr_df[col_num_cols][curr_df.index[0]]
        if (nrows > MAX_ROWS) or (ncols > MAX_COLS):
            if PRINT_SKIP_REASON:
                print("Skipping instance {:d} -- {}: nrows = {:d} > {:d} or ncols = {:d} > {:d}.".format(
                        i, inst, nrows, ncols, MAX_ROWS, MAX_COLS))
            skip_instance = True
            df_rejection_reason.loc[inst, 'TOO_MANY_ROWS_OR_COLS'] = True
    
    if not skip_instance:
        #selected_gap_instances_dict[len(selected_gap_instances_dict)] = inst
        selected_gap_instances_dict[inst] = i
    else:
        df_rejection_reason.loc[inst, 'SELECTED_GAP'] = False
        df_rejection_reason.loc[inst, 'SELECTED_TIME'] = False
        df_rejection_reason.loc[inst, 'SELECTED_6TREES'] = False

selected_gap_instances = selected_gap_instances_dict.keys()
num_selected_gap_instances = len(selected_gap_instances_dict)
print("Total number of errors: {}".format(num_gap_errors))
print("Total number of selected instances for gap closed reporting: {}/{:d}".format(num_selected_gap_instances,len(instances)))

# intersect the pure binary instances with the selected gap instances
binary_x_gap_instances = [inst for inst in pure_binary_instances if inst in selected_gap_instances]
print("Total number of binary instances for gap closed reporting: {}/{:d}".format(len(binary_x_gap_instances),num_selected_gap_instances))

*** ERROR: Instance 213 -- neos-3214367-sovi_presolved: 4 < 7 attempts.
*** ERROR: Instance 228 -- neos-3661949-lesse_presolved: 1 < 7 attempts.
*** ERROR: Instance 228 -- neos-3661949-lesse_presolved: not YES GAP (lp = 689000000.0000000000; ip = 688995225.0000000000, diff = -4775.00)
*** ERROR: Instance 230 -- neos-3734794-moppy_presolved: 1 < 7 attempts.
*** ERROR: Instance 253 -- neos-530627_presolved: 5 < 7 attempts.
Total number of errors: 5
Total number of selected instances for gap closed reporting: 332/438
Total number of binary instances for gap closed reporting: 65/332


In [25]:
# Retrieve all instances from df with df_rejection_reason '<7_ATTEMPTS' == True
instances_with_less_than_7_attempts = df_rejection_reason[df_rejection_reason['<7_ATTEMPTS'] == True].index.tolist()
display(instances_with_less_than_7_attempts)

# Get df_bb entries for instances_with_less_than_7_attempts
df_bb_with_less_than_7_attempts = df.loc[instances_with_less_than_7_attempts]

df_bb_with_less_than_7_attempts.loc[instances_with_less_than_7_attempts[0]]

['neos-3214367-sovi_presolved',
 'neos-3661949-lesse_presolved',
 'neos-3734794-moppy_presolved',
 'neos-530627_presolved']

,cutlimit,gomory,mode,partial_bb_strategy,partial_bb_keep_pruned_nodes,partial_bb_num_strong,preprocess,prlp_flip_beta,rounds,bb_mode,...,REF+V TIME (6),REF+V TIME (7),REF+V NODES (1),REF+V NODES (2),REF+V NODES (3),REF+V NODES (4),REF+V NODES (5),REF+V NODES (6),REF+V NODES (7),AVG REF+V W/CUTGEN TIME
disj_terms,,,,,,,,,,,,,,,,,,,,,
-1,-1,-1,4,4,0,5,0,0,1,11,...,7200.0,7200.0,14908.0,11266.0,12548.0,11728.0,12966.0,13942.0,18024.0,10834.43
0,-1,0,0,4,0,5,0,0,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-1,-1,0,4,0,5,0,0,1,11,...,7200.0,7200.0,14892.0,11282.0,12542.0,11730.0,12832.0,13824.0,18024.0,7287.19
4,-1,-1,0,4,0,5,0,0,1,11,...,7200.0,7200.0,12376.0,9352.0,10390.0,9512.0,10328.0,11446.0,14524.0,7297.84
8,-1,-1,0,4,0,5,0,0,1,11,...,7200.0,7200.0,14796.0,11210.0,12502.0,11682.0,12860.0,13934.0,18040.0,7383.20


In [26]:
max_num_vpc_df = df.loc[df.index.get_level_values(1) > 0]
max_num_vpc_df = max_num_vpc_df[col_num_vpc].groupby(level=0).max()
print("Across selected_gap_instances, minimum number of VPCs generated across all depths: ",max_num_vpc_df.loc[selected_gap_instances].min())

Across selected_gap_instances, minimum number of VPCs generated across all depths:  1


### `selected_time_instances_dict` and `all6_instances_dict` (original index, instance): Select instances for time tables

Criteria to filter instances for reporting time:
* ip opt val is known
* lp opt < ip opt
* max(nrows, ncols) ≤ 5K
* optimal solution should not be found by any of the partial trees
* either lp opt < disj lb or disj lb < disj ub
* PRLP is feasible and solves within timelimit for at least one of the attempts
* Gur7 < 3600 (Gurobi is able to solve the instance to optimality within an hour either with or without using VPCs)

6 trees set
* all six partial tree sizes produced VPCs and solved within timelimit (to make # of instances consistent across depths)

In [126]:
## Select instances for time tables
#
# Criteria to filter instances for reporting time:
# * ip opt val is known
# * lp opt < ip opt 
# * max(nrows, ncols) ≤ 5K
# * optimal solution should not be found by any of the partial trees
# * either lp opt < disj lb or disj lb < disj ub
# * PRLP is feasible and solves within timelimit for at least one of the attempts
# * min(Gur,Gur+V) < 3600 (Gurobi is able to solve the instance to optimality within an hour either with or without using VPCs)
#
# 6 trees set
# * all six partial tree sizes produced VPCs 
# * ** and solved within timelimit (to make number of instances consistent across depths)

# Constants
PRINT_SKIP_REASON = False

# Information to save
selected_time_instances_dict = {}   # dictionary of (original index, instance)
all6_instances_dict = {}            # dictionary of (original index, instance)
skipped_instances_dict = {}         # dictionary of (original index, instance)
error_instances_dict = {}           # dictionary of (original index, instance)

num_timeouts = 0
num_time_errors = 0

inst_set = selected_gap_instances
# inst_set = ['lotsize_presolved']
# inst_set = ['neos-3754480-nidda_presolved']
for i, inst in enumerate(inst_set):
    print("{}/{}".format(i+1,len(inst_set)), end='\r', flush=True)
    skip_instance = False
    curr_df = df.loc[inst]
    curr_df = curr_df[curr_df.index.get_level_values(0) >= 0]

    # Check Gur < 3600 (Gurobi is able to solve the instance to optimality within an hour without using VPCs)
    mintime_gur = float(curr_df.loc[2:64,ref_timeout_col].min())
    
    # Check V < 3600 (Gurobi is able to solve the instance to optimality within an hour either with using VPCs)
    mintime_gur7 = float(curr_df.loc[2:64,refv_timeout_col].min())

    mintime = min(mintime_gur, mintime_gur7)
    if mintime >= MAX_TIME:
        if PRINT_SKIP_REASON:
            print("{:d}: Skipping instance {:d} -- {}: Gurobi's best time (with or without VPCs) is {:.7f} >= {:.7f}.".format(
                    len(skipped_instances_dict), i, inst, mintime, MAX_TIME-EPS
                ))
        skip_instance = True
        skipped_instances_dict[inst] = i
        num_timeouts += 1
        df_rejection_reason.loc[inst, 'GUR_TIMEOUT'] += 1
        # df_status_by_depth.loc[inst, int(curr_index)] = map_rejection_reason_to_number['GUR_TIMEOUT']

    # Check how many times VPCs were successfully generated
    num_successful_attempts = 0
    num_timeouts = 0
    has_zero = False
    for curr_index, row in curr_df.iterrows():
        if row[col_num_disj_terms] == 0:
            has_zero = True
            continue

        num_vpc = float(row[col_num_vpc])
        num_successful_attempts += (num_vpc > 0)

        curr_time = float(curr_df.loc[curr_index,refv_timeout_col])
        if curr_time > MAX_TIME - EPS:
            num_timeouts += 1
            if df_status_by_depth.loc[inst, int(curr_index)] == DEFAULT_STATUS:
                df_status_by_depth.loc[inst, int(curr_index)] = map_rejection_reason_to_number['GUR_TIMEOUT']

    if not has_zero:
        raise ValueError(
            "*** ERROR: Instance {:d} -- {}: has no bb0 entry.".format(
                i, inst, curr_index
            )
        )        
    
    # if num_successful_attempts == 0 and not skip_instance:
    #     if PRINT_SKIP_REASON:
    #         print("Skipping instance {:d} -- {}: no VPCs generated successfully for any number of terms.".format(i, inst, num_attempts[i]))
    #     skip_instance = True
    #     skipped_instances_dict[inst] = i

    if not skip_instance:
        if num_successful_attempts == 6 and num_timeouts == 0:
            all6_instances_dict[inst] = i
        else:
            df_rejection_reason.loc[inst, 'SELECTED_6TREES'] = False
        selected_time_instances_dict[inst] = i
    else:
        df_rejection_reason.loc[inst, 'SELECTED_TIME'] = False
        df_rejection_reason.loc[inst, 'SELECTED_6TREES'] = False

selected_time_instances = selected_time_instances_dict.keys()
all6_instances = all6_instances_dict.keys()

num_selected_time_instances = len(selected_time_instances_dict)
num_all6_instances = len(all6_instances_dict)
print("Total number of errors: {}".format(num_time_errors))
print("Total number of timeouts: {}".format(num_timeouts))
print("Total number of instances for time reporting: {}".format(num_selected_time_instances))
print("Total number of \"6 trees\" instances: {}".format(num_all6_instances))

#list(set(selected_gap_instances_dict.keys()).intersection(pure_binary_instances))
#selected_pure_binary_instances.sort()

# print("Found {:d} pure binary instances".format(len(selected_pure_binary_instances)))

binary_x_time_instances = [inst for inst in pure_binary_instances if inst in selected_time_instances_dict]
print("Number of pure binary instances for time reporting: {}".format(len(binary_x_time_instances)))

all6_binary_x_time_instances = [inst for inst in all6_instances_dict.keys() if inst in binary_x_time_instances]
print("Number of pure binary instances in all-6 set: {}".format(len(all6_binary_x_time_instances)))

Total number of errors: 0
Total number of timeouts: 0
Total number of instances for time reporting: 282
Total number of "6 trees" instances: 209
Number of pure binary instances for time reporting: 52
Number of pure binary instances in all-6 set: 35


#### DEBUG (check which instances were selected but do not have all six runs)

In [28]:
## DEBUG (check which instances were selected but do not have all six runs)
not_all_6 = [key for key in selected_time_instances_dict.keys() if key not in all6_instances_dict.keys()]
not_all_6

['10teams_presolved',
 '30n20b8_presolved',
 'a1c1s1_presolved',
 'app3_presolved',
 'bnatt400_presolved',
 'bppc8-02_presolved',
 'danoint_presolved',
 'eilB101_presolved',
 'eild76_presolved',
 'f2gap801600_presolved',
 'graphdraw-domain_presolved',
 'ic97_potential_presolved',
 'ic97_tension_presolved',
 'icir97_tension_presolved',
 'lrn_presolved',
 'milo-v12-6-r2-40-1_presolved',
 'mkc1_presolved',
 'n2seq36f_presolved',
 'n4-3_presolved',
 'neos-1058477_presolved',
 'neos-1215259_presolved',
 'neos-1330346_presolved',
 'neos-1396125_presolved',
 'neos-1413153_presolved',
 'neos-1415183_presolved',
 'neos-1480121_presolved',
 'neos-1601936_presolved',
 'neos-1605061_presolved',
 'neos-1605075_presolved',
 'neos-1620807_presolved',
 'neos-3118745-obra_presolved',
 'neos-3216931-puriri_presolved',
 'neos-3381206-awhea_presolved',
 'neos-3665875-lesum_presolved',
 'neos-504815_presolved',
 'neos-5075914-elvire_presolved',
 'neos-5078479-escaut_presolved',
 'neos-5182409-nasivi_presol

In [29]:
import csv

# Define the file paths
gap_file_path = 'selected_gap_instances.csv'
time_file_path = 'selected_time_instances.csv'
rejected_file_path = 'rejected_instances.csv'

# Get the keys from selected_gap_instances_dict and selected_time_instances_dict
selected_gap_keys = list(selected_gap_instances_dict.keys())
selected_time_keys = list(selected_time_instances_dict.keys())

# Write the keys to the CSV files
with open(gap_file_path, 'w', newline='') as gap_file:
  gap_writer = csv.writer(gap_file)
  gap_writer.writerow(['Selected Gap Instances'])
  gap_writer.writerows([[key] for key in selected_gap_keys])

with open(time_file_path, 'w', newline='') as time_file:
  time_writer = csv.writer(time_file)
  time_writer.writerow(['Selected Time Instances'])
  time_writer.writerows([[key] for key in selected_time_keys])

df_rejection_reason.to_csv(rejected_file_path)

print('Selected instances have been written to the CSV files.')


Selected instances have been written to the CSV files.


# Section 2: Gap closed tables

### `calc_gap_closed` function

In [30]:
## Calculate gap closed for GMICs, Gurobi, and VPCs
def calc_gap_closed(gap_df, col):
    return np.where(
        ((gap_df[col] > EPS) & (gap_df[col] < INFINITY)) | ((gap_df[col] < -EPS) & (gap_df[col] > -INFINITY)), # condition
        100. * (gap_df[col] - gap_df[col_lp_obj]) / (gap_df[col_ip_obj] - gap_df[col_lp_obj]), # if condition is true
        0.0 # if condition is false
    )

# def calc_gap_closed2(gap_df, col):
#     conditions = gap_df[col] > EPS & np.isfinite(gap_df[col])
#     choices = 100. * (gap_df[col] - gap_df["LP OBJ"]) / (gap_df["IP OBJ"] - gap_df["LP OBJ"])
#     return np.select(conditions, choices, default=0.0)

### `gap_df`: Calculate gap closed for GMICs, Gurobi, and VPCs

In [31]:
# Create subset of dataframe relevant to gap closed
gap_df = df.loc[:, 
                [
                    col_num_disj_terms,
                    col_num_rows,
                    col_num_cols
                ]
                +
                obj_val_col_list
                +
                [
                    col_num_gmic,
                    col_num_vpc,
                    col_num_obj,
                    col_exit_reason
                ]
               ]

# Calculate some missing % gap closed columns
# gap closed = 100 * (post_cut_opt_val - lp_opt_val) / (ip_opt_val - lp_opt_val)
for cut_type in reg_cut_type_long_list:
    col = cut_type + ' ' + obj_stub
    if col not in df.columns:
        if cut_type == 'MAX(GMIC,VPC)':
            # Add max(G,V) column
            gap_df[cut_type + ' ' + pct_gap_closed_stub] = \
                np.maximum(
                    gap_df['GMIC' + ' ' + pct_gap_closed_stub],
                    gap_df[ 'VPC' + ' ' + pct_gap_closed_stub]
                )
        continue
    gap_df[cut_type + ' ' + pct_gap_closed_stub] = calc_gap_closed(gap_df, col)

# Compare against reference solver
for stat_type in solver_stat_list[:3]:
    for solver_type in solver_stubs:
        for cut_type in solver_cut_type_stubs:
            col = stat_type + ' ' + solver_type + ' ' + cut_type
            gap_df[col + ' ' + pct_gap_closed_stub] = calc_gap_closed(gap_df, col)

display(gap_df.loc[['bm23_presolved','maxgasflow_presolved']][gap_cols])
display(gap_df.loc[("bm23_presolved",2)])

GMIC % GAP CLOSED  ROOT % GAP CLOSED  \
INSTANCE             disj_terms                                         
bm23_presolved       -1                  16.809643       0.000000e+00   
                      0                   0.000000       0.000000e+00   
                      2                  16.809643       0.000000e+00   
                      4                  16.809643       0.000000e+00   
                      8                  16.809643       0.000000e+00   
                      16                 16.809643       0.000000e+00   
                      32                 16.809643       0.000000e+00   
                      64                 16.809643       0.000000e+00   
maxgasflow_presolved -1                  -0.004342       6.226580e-15   
                      0                  22.478700       0.000000e+00   
                      2                  22.478700      -3.223163e-11   
                      4                  22.478700      -3.223163e-11   
                      8                  22.478700      -3.223163e-11   
                      16                 22.478700      -3.223163e-11   
                      32                 22.478700      -3.223163e-11   
                      64                 22.478700      -3.223163e-11   

                                 BEST DISJ % GAP CLOSED  VPC % GAP CLOSED  \
INSTANCE             disj_terms                                             
bm23_presolved       -1                       71.466577         69.698381   
                      0                        0.000000          0.000000   
                      2                        6.797215          6.797215   
                      4                       14.647006         14.647006   
                      8                       17.613150         17.613150   
                      16                      39.543044         38.939251   
                      32                      56.526208         56.526208   
                      64                      71.466577         69.642109   
maxgasflow_presolved -1                       -0.000003         -0.000001   
                      0                        0.000000          0.000000   
                      2                        0.000301          0.000301   
                      4                        0.000408          0.000408   
                      8                        0.001225          0.000909   
                      16                       0.006743          0.001540   
                      32                       0.009301          0.005425   
                      64                       0.015531          0.003080   

                                 MAX(GMIC,VPC) % GAP CLOSED  \
INSTANCE             disj_terms                               
bm23_presolved       -1                           69.698381   
                      0                            0.000000   
                      2                           16.809643   
                      4                           16.809643   
                      8                           17.613150   
                      16                          38.939251   
                      32                          56.526208   
                      64                          69.642109   
maxgasflow_presolved -1                           -0.000001   
                      0                           22.478700   
                      2                           22.478700   
                      4                           22.478700   
                      8                           22.478700   
                      16                          22.478700   
                      32                          22.478700   
                      64                          22.478700   

                                 VPC+GMIC % GAP CLOSED  \
INSTANCE             disj_terms                          
bm23_presolved       -1                      69.698381   
                      

NUM DISJ TERMS                                     2
ROWS                                              20
COLS                                              27
LP OBJ                                     20.570922
IP OBJ                                          34.0
WORST DISJ OBJ                             27.267238
GMIC OBJ                                   22.828302
ROOT OBJ                                   20.570922
BEST DISJ OBJ                              21.483725
VPC OBJ                                    21.483725
VPC+GMIC OBJ                               23.184302
FIRST REF FIRST_CUT_PASS                   23.373228
FIRST REF+V FIRST_CUT_PASS                 23.392329
AVG REF FIRST_CUT_PASS                     23.412523
AVG REF+V FIRST_CUT_PASS                   23.478734
BEST REF FIRST_CUT_PASS                    23.545818
BEST REF+V FIRST_CUT_PASS                  23.583407
FIRST REF LAST_CUT_PASS                     25.60614
FIRST REF+V LAST_CUT_PASS                  25.

### `selected_gap_df`: Gap closed for selected instances, adding 0-row that has best for `V+` cols

In [32]:
## `selected_gap_df`: Gap closed for selected instances, adding 0-row that has best for `V+` cols
## Show the instances that have been selected (and their original index)
## and then set the selected_gap_df as the selected instances from gap_df
## We also set the '0' row to contain the best result for each method
## (including the option of not using VPCs at all)
## and we replace any runs with no VPCs with the values obtained without them
selected_gap_df = gap_df.loc[selected_gap_instances]

# From https://pandas.pydata.org/docs/user_guide/advanced.html#defined-levels
# "The MultiIndex keeps all the defined levels of an index, even if they are not actually used.
# When slicing an index, you may notice this."
# Even without using remove_unused_levels, index was correct with selected_gap_df.index.get_level_values(0).unique()
selected_gap_df.index = selected_gap_df.index.remove_unused_levels()

#display(selected_gap_df.index.difference(gap_df.index))
#selected_gap_df.drop(['22433_presolved'])

# # Check what the selected_gap_df contains for bm23
# inst = "bm23_presolved"
# display(selected_gap_df.loc[inst])

#inst = "10teams_presolved"
# inst = '22433_presolved'
# curr_df = selected_gap_df.loc[inst]
# display(curr_df)
# # for i in curr_df.index:
# #     display(curr_df.loc[i])

#display(selected_gap_df.index.get_level_values(0).unique())

# Do we update the value of the "best" in each column when no VPCs are generated for a run and we use the "no-VPCs" data?
# This may cause the stats in the "best" row to improve
# For example, we replace V+GurF with GurF when no VPCs are generated, since that is what would occur without VPCs
# But if GurF is better than any V+GurF when VPCs are produced, then the average in the max-row is inflated
SHOULD_UPDATE_MAX_WHEN_NO_VPCS = False

# inst_set = selected_gap_df.index.get_level_values(0).unique()
inst_set = selected_gap_df.index.levels[0]
num_inst = len(inst_set)
for curr_inst_ind, inst in enumerate(inst_set):
    print("{}/{}".format(curr_inst_ind+1,num_inst), end='\r', flush=True)
    curr_df = selected_gap_df.loc[inst].copy() # copy needed to not throw SettingWithCopyWarning
    curr_df = curr_df[curr_df.index.get_level_values(0) >= 0] # to remove disjset = -1 row, for example

    # Set 0-row to have max values across all rows for this instance
    max_vals = curr_df[gap_cols].max()
    selected_gap_df.loc[(inst,0),gap_cols] = max_vals

    for ind in curr_df.index:
        if ind <= 0:
            continue

        # Propogate GurF and GurL down
        # sel_gap = [col_first_ref_first, col_first_ref, col_last_ref_first, col_last_ref]
        sel_gap = ref_solver_gap_cols
        selected_gap_df.loc[(inst,ind),sel_gap] = curr_df.loc[0,sel_gap]

        # If no VPCs produced, the values for V+GurF and V+GurL have not been provided
        # We replace these by GurF and GurL
        # Currently disabled: update max for that column too (if disabled, we instead keep max as the value among those that generated VPCs)
        num_vpc = curr_df.loc[ind,col_num_vpc]
        if num_vpc == 0:
            # print("Zero cuts for inst {} at depth {:d}".format(inst, ind))
            ref_gap = [col_first_cut_pass_gap_ref, col_last_cut_pass_gap_ref] # this is where we pull info from
            refinds = [gap_cols.index(colname) for colname in ref_gap] 
            sel_gap = [col_first_cut_pass_gap_ref_v, col_last_cut_pass_gap_ref_v] # this is where we put the info
            selected_gap_df.loc[(inst,ind),sel_gap] = curr_df.loc[0,ref_gap].to_numpy()

            if SHOULD_UPDATE_MAX_WHEN_NO_VPCS:
                for i in refinds:
                    if curr_df.loc[0,gap_cols[i]] > selected_gap_df.loc[(inst,0),gap_cols[i+1]]:
                        # if curr_df.loc[0,gap_cols[i]] > 0:
                            # print("DEBUG: Updating {} for inst {} from {:f} to {:f}".format(
                            #     gap_cols[i+1], 
                            #     inst, 
                            #     selected_gap_df.loc[(inst,0),gap_cols[i+1]], 
                            #     curr_df.loc[0,gap_cols[i]]))
                        selected_gap_df.loc[(inst,0),gap_cols[i+1]] = curr_df.loc[0,gap_cols[i]]

display(selected_gap_df.head(21).loc[:,[col_num_vpc]+gap_cols])

NUM VPC  GMIC % GAP CLOSED  ROOT % GAP CLOSED  \
INSTANCE          disj_terms                                                  
10teams_presolved -1               80         100.000000                0.0   
                   0                0         100.000000                0.0   
                   2               77         100.000000                0.0   
                   4                1         100.000000                0.0   
                   8                2         100.000000                0.0   
                   16              51         100.000000                0.0   
                   32              16         100.000000                0.0   
                   64               0         100.000000                0.0   
23588_presolved   -1              386           5.772831                0.0   
                   0                0           5.772831                0.0   
                   2               11           5.772831                0.0   
                   4               75           5.772831                0.0   
                   8               75           5.772831                0.0   
                   16              75           5.772831                0.0   
                   32              75           5.772831                0.0   
                   64              75           5.772831                0.0   
30n20b8_presolved -1              381          11.099235                0.0   
                   0                0          11.099235                0.0   
                   2              190          11.099235                0.0   
                   4              190          11.099235                0.0   
                   8                1          11.099235                0.0   

                              BEST DISJ % GAP CLOSED  VPC % GAP CLOSED  \
INSTANCE          disj_terms                                             
10teams_presolved -1                    8.120488e-12      1.586743e-09   
                   0                    8.120488e-12      0.000000e+00   
                   2                   -4.872293e-12     -1.689062e-10   
                   4                    8.120488e-12     -2.744725e-10   
                   8                   -1.624098e-11     -8.315380e-10   
                   16                  -9.744586e-12     -2.760966e-11   
                   32                  -9.744586e-12     -3.248195e-12   
                   64                  -9.744586e-12      0.000000e+00   
23588_presolved   -1                    7.218238e+01      7.161482e+01   
                   0                    7.218238e+01      7.161482e+01   
                   2                    2.188689e+01      1.745981e+01   
                   4                    3.409109e+01      2.749366e+01   
                   8                    4.357635e+01      3.475065e+01   
                   16                   4.952913e+01      4.821609e+01   
                   32                   6.209834e+01      6.059920e+01   
                   64                   7.218238e+01      7.161482e+01   
30n20b8_presolved -1                    1.564622e+00      4.717160e-02   
                   0                    1.564622e+00      3.333430e-02   
                   2                    4.865507e-02      3.333430e-02   
                   4                    8.645315e-02      3.062940e-02   
                   8                    8.707174e-02      1.661296e-02   

                              MAX(GMIC,VPC) % GAP CLOSED  \
INSTANCE          disj_terms                               
10teams_presolved -1                          100.000000   
                   0                          100.000000   
                   2                          100.000000   
                   4                          100.000000   
                   8                          100.000000   
                   16                         100.000000   
                   32        

#### DEBUG: Why REF+V is less than REF

In [33]:
### DEBUG
# Why REF+V < REF

# inst = 'f2gap801600_presolved'
#inst = 'neos-1112787_presolved'
inst = 'neos-1330346_presolved'

tmp_df = gap_df.loc[inst,
                      [col_num_vpc]
                      +
                      [col_first_cut_pass_bound_ref, col_first_cut_pass_bound_ref_v]
                      +
                      gap_cols
                      +
                      [col_lp_obj, col_ip_obj]
                    ]

# # Compare against reference solver
# for stat_type in solver_stat_list:
#     for solver_type in solver_stubs:
#         for cut_type in solver_cut_type_list:
#             col = stat_type + ' ' + solver_type + ' ' + cut_type
#             gap_df[col + ' ' + pct_gap_closed_stub] = calc_gap_closed(gap_df, col)

# display(tmp_df)
# display(gap_df.loc[inst,['NUM VPC']+['BEST REF FIRST_CUT_PASS']+['FIRST REF+V FIRST_CUT_PASS']+gap_cols])

# display(selected_gap_df.loc[inst,['NUM VPC']+['BEST REF FIRST_CUT_PASS']+['FIRST REF+V FIRST_CUT_PASS']+gap_cols])

tmp_df[col_first_cut_pass_gap_ref_v] = calc_gap_closed(tmp_df, col_first_cut_pass_bound_ref_v)
display(tmp_df)

# display(gap_df.loc[inst,['NUM VPC']+['BEST REF FIRST_CUT_PASS']+['FIRST REF+V FIRST_CUT_PASS']+gap_cols])

# display(selected_gap_df.loc[inst,['NUM VPC']+['BEST REF FIRST_CUT_PASS']+['FIRST REF+V FIRST_CUT_PASS']+gap_cols])

,NUM VPC,AVG REF FIRST_CUT_PASS,AVG REF+V FIRST_CUT_PASS,GMIC % GAP CLOSED,ROOT % GAP CLOSED,BEST DISJ % GAP CLOSED,VPC % GAP CLOSED,"MAX(GMIC,VPC) % GAP CLOSED",VPC+GMIC % GAP CLOSED,FIRST REF FIRST_CUT_PASS % GAP CLOSED,...,BEST REF FIRST_CUT_PASS % GAP CLOSED,BEST REF+V FIRST_CUT_PASS % GAP CLOSED,FIRST REF LAST_CUT_PASS % GAP CLOSED,FIRST REF+V LAST_CUT_PASS % GAP CLOSED,AVG REF LAST_CUT_PASS % GAP CLOSED,AVG REF+V LAST_CUT_PASS % GAP CLOSED,BEST REF LAST_CUT_PASS % GAP CLOSED,BEST REF+V LAST_CUT_PASS % GAP CLOSED,LP OBJ,IP OBJ
disj_terms,,,,,,,,,,,,,,,,,,,,,
-1,46,4.571429,4.571429,2.537653e-14,0.0,1.428571e+01,0.000000e+00,2.537653e-14,-2.537653e-14,0.0,...,14.285714,14.285714,0.0,14.285714,8.163265,6.802721,14.285714,14.285714,4.5,8.0
0,0,4.571429,0.000000,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,...,14.285714,0.000000,0.0,0.000000,8.163265,0.000000,14.285714,0.000000,4.5,8.0
2,32,4.571429,4.571429,2.537653e-14,0.0,2.537653e-14,2.537653e-14,2.537653e-14,0.000000e+00,0.0,...,14.285714,14.285714,0.0,0.000000,8.163265,6.122449,14.285714,14.285714,4.5,8.0
4,12,4.571429,4.571429,2.537653e-14,0.0,-2.537653e-14,2.537653e-14,2.537653e-14,2.537653e-14,0.0,...,14.285714,14.285714,0.0,0.000000,8.163265,6.122449,14.285714,14.285714,4.5,8.0
8,5,4.571429,4.571429,2.537653e-14,0.0,2.537653e-14,2.537653e-14,2.537653e-14,-2.537653e-14,0.0,...,14.285714,14.285714,0.0,0.000000,8.163265,4.081633,14.285714,14.285714,4.5,8.0
16,0,0.000000,0.000000,2.537653e-14,0.0,2.857143e+00,0.000000e+00,2.537653e-14,2.537653e-14,0.0,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,4.5,8.0
32,0,0.000000,0.000000,2.537653e-14,0.0,1.428571e+01,0.000000e+00,2.537653e-14,2.537653e-14,0.0,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,4.5,8.0
64,0,0.000000,0.000000,2.537653e-14,0.0,1.428571e+01,0.000000e+00,2.537653e-14,2.537653e-14,0.0,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,4.5,8.0


### `best_gap_df`: For each instance, what the best gap closed is (and how that was obtained)

The columns are defined by:
* `gap_cols_short`
* the best disjunction for 'V', 'V+G', 'V+GurF', 'V+GurL'
* number of VPCs leading to best 'V' result
* number of GMICs for the instance

In [34]:
## Create best df = for each instance, what the best gap closed is (and how that was obtained)

# inst_set = selected_gap_instances_dict.keys()
inst_set = selected_gap_df.index.levels[0]
# inst_set = ['neos22_presolved']
# inst_set = ['usAbbrv-8-25_70_presolved']

best_gap_df = pd.DataFrame(
    columns = gap_cols_short+[
        col_best_disj_vpc,
        col_best_disj_gmic_vpc,
        col_best_disj_refv_first_cut_pass,
        col_best_disj_refv_last_cut_pass,
        col_num_vpc,
        col_num_gmic,
    ],
    index = inst_set,
    dtype = float,
)

best_disj_short_types = [ 'V', 'V+G' ] + refv_cut_type_short_list
ind_of_gap_for_best_disj = [
    gap_cols_short.index(cut_type) for cut_type in best_disj_short_types
]

num_inst = len(inst_set)
for i, inst in enumerate(inst_set):
    print("{}/{}".format(i+1,num_inst), end='\r', flush=True)
    # print("Processing instance {:d} with name {}.".format(i, inst))
    best_gaps = [ -1 for _ in range(len(gap_cols_short)) ]
    best_disj = [ -1 for _ in range(len(best_disj_short_types)) ]
    best_num_vpc = -1
    best_num_gmic = -1
    
    # # best_vpc = -1.
    # best_vpc_disj = -1
    # # best_max_gmic_vpc = -1.
    # # best_vpcgmic = -1.
    # best_vpcgmic_disj = -1
    # # best_VGurF = -1.
    # best_VGurF_disj = -1
    # # best_VGurL = -1.
    # best_VGurL_disj = -1
    
    curr_df = selected_gap_df.loc[inst]
    # curr_df = curr_df[curr_df.index.get_level_values(0) > 0]
    
    # Get info for GurF and GurL from the no-VPC row
    # row = curr_df.loc[0]
    # gmic_gap = float(row[col_gmic])
    # GurF_gap = float(row[col_first_cut_pass_gap_ref])
    # GurL_gap = float(row[col_last_cut_pass_gap_ref])
    # root_gap = float(row[col_root])
    # disj_gap = float(row[col_best_disj])

    for index, row in curr_df.iterrows():
        num_disj_terms = int(row[col_num_disj_terms])
        # num_obj_tried  = float(row[col_num_obj])
        num_vpc        = float(row[col_num_vpc])
        if num_disj_terms <= 0:
            continue
        if num_vpc <= 0:
            # Check if we update 'DB' value
            db_ind = gap_cols_short.index('DB')
            if row[col_best_disj] > best_gaps[db_ind]:
                best_gaps[db_ind] = row[col_best_disj]
            continue
            
        # print("Index {:d}: Processing instance {} with {:d} disj terms.".format(index, inst, num_disj_terms))
        curr_gaps = [ float(row[map_short_to_cols_gap[col]]) for col in gap_cols_short ]
        num_gmic    = float(row[col_num_gmic])
        # vpc_gap     = float(row[col_vpc])
        # vpcgmic_gap = float(row[col_vpc_gmic])
        # VGurF_gap   = float(row[col_first_ref_v])
        # VGurL_gap   = float(row[col_last_ref_v])
        # num_vpc     = float(row['NUM VPC'])

        # Update best gap closed for each method
        for j, gap in enumerate(curr_gaps):
            if gap > best_gaps[j]:
                best_gaps[j] = gap
                if j in ind_of_gap_for_best_disj:
                    best_disj[ind_of_gap_for_best_disj.index(j)] = index
                if gap_cols_short[j] == 'V':
                    best_num_vpc = num_vpc
                    best_num_gmic = num_gmic
        
        # if (best_vpc < vpc_gap): #or (is_val(best_vpc, vpc_gap) and best_num_vpc == 0):
        #     best_vpc = vpc_gap
        #     best_vpc_disj = index
        #     best_num_vpc = num_vpc
        #     best_num_gmic = num_gmic
        # if best_vpcgmic < vpcgmic_gap:
        #     best_vpcgmic = vpcgmic_gap
        #     best_vpcgmic_disj = index
        # if best_max_gmic_vpc < max(vpc_gap, gmic_gap):
        #     best_max_gmic_vpc = max(vpc_gap, gmic_gap)
        # if best_VGurF < VGurF_gap:
        #     best_VGurF = VGurF_gap
        #     best_VGurF_disj = index
        # if best_VGurL < VGurL_gap:
        #     best_VGurL = VGurL_gap
        #     best_VGurL_disj = index


    best_gaps = [ gap if gap >= EPS else 0. for gap in best_gaps ]
    best_gap_df.iloc[i] = best_gaps + best_disj + [best_num_vpc, best_num_gmic]
        # gmic_gap if gmic_gap >= EPS else 0.,
        # root_gap if root_gap >= EPS else 0.,
        # disj_gap if disj_gap >= EPS else 0.,
        # best_vpc if best_vpc >= EPS else 0.,
        # best_max_gmic_vpc if best_max_gmic_vpc >= EPS else 0.,
        # best_vpcgmic if best_vpcgmic >= EPS else 0.,
        # GurF_gap if GurF_gap >= EPS else 0.,
        # best_VGurF if best_VGurF >= EPS else 0.,
        # GurL_gap if GurL_gap >= EPS else 0.,
        # best_VGurL if best_VGurL >= EPS else 0.,
    #     best_vpc_disj,
    #     best_vpcgmic_disj,
    #     best_VGurF_disj,
    #     best_VGurL_disj,
    #     best_num_vpc,
    #     best_num_gmic,
    # ]

int_col_list = [ col_best_disj_vpc, col_best_disj_gmic_vpc, col_best_disj_refv_first_cut_pass, col_best_disj_refv_last_cut_pass, col_num_vpc, col_num_gmic ]
#['BEST VPC DISJ', 'BEST GMIC+VPC DISJ', 'BEST V+GurF DISJ', 'BEST V+GurL DISJ', 'NUM VPC', 'NUM GMIC']
for col in int_col_list:
    best_gap_df[col] = best_gap_df[col].astype(np.int64)

display(best_gap_df)

,G,R,DB,V,"max(G,V)",V+G,GurF,V+GurF,GurL,V+GurL,BEST VPC DISJ,BEST GMIC+VPC DISJ,Best V+GurF DISJ,Best V+GurL DISJ,NUM VPC,NUM GMIC
INSTANCE,,,,,,,,,,,,,,,,
10teams_presolved,100.000000,0.0,0.000000,0.000000,100.000000,100.000000,93.877551,93.877551,100.000000,100.000000,32,32,16,2,16,153
23588_presolved,5.772831,0.0,72.182376,71.614817,71.614817,71.624582,15.591428,68.936554,24.381330,71.775864,64,64,64,64,75,75
30n20b8_presolved,11.099235,0.0,1.564622,0.033334,11.099235,11.099235,0.926468,1.331188,16.910036,16.300723,2,2,2,4,190,184
50v-10_presolved,45.753596,0.0,18.008191,11.184101,45.753596,45.823184,49.858925,50.152252,73.169318,74.444520,64,16,8,64,29,29
a1c1s1_presolved,25.105518,0.0,4.895611,1.094331,25.105518,25.382401,45.584474,46.254097,88.806373,88.730339,8,2,4,4,3,154
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
umts_presolved,0.973181,0.0,0.209040,0.108872,0.973181,0.973181,1.412379,1.463163,4.793132,5.023309,64,64,2,8,276,275
usAbbrv-8-25_70_presolved,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8,8,8,8,4,722
vpm1_presolved,16.930266,0.0,7.788162,4.672897,16.930266,16.930266,45.861148,56.942590,68.157543,72.129506,32,2,2,64,10,16


#### DEBUG: Look at `best_gap_df` entries

In [35]:
# best_gap_df.to_csv('best_gap.csv')
tmp_inst_set = ['cap6000_presolved','neos-1330346_presolved']
best_gap_df.loc[tmp_inst_set]

,G,R,DB,V,"max(G,V)",V+G,GurF,V+GurF,GurL,V+GurL,BEST VPC DISJ,BEST GMIC+VPC DISJ,Best V+GurF DISJ,Best V+GurL DISJ,NUM VPC,NUM GMIC
INSTANCE,,,,,,,,,,,,,,,,
cap6000_presolved,41.647393,0.0,67.980383,63.480611,63.480611,63.480611,35.697489,35.697766,36.023078,38.816797,64,64,2,64,2,2
neos-1330346_presolved,0.000000,0.0,14.285714,0.000000,0.000000,0.000000,2.040816,2.040816,8.163265,6.122449,2,4,2,2,32,565


#### DEBUG: In `best_gap_df`, can get V > V+G due to numerical issues

In [36]:
## DEBUG: You can get V > V+G due to numerical issues

col1 = best_gap_df['V']
col2 = best_gap_df['V+G']

display(best_gap_df[(col1 > col2 + EPS) == True])

#df.loc['neos-1058477_presolved'] #.to_csv("neos-1058477_presolved_data.csv")
df.loc['seymour-disj-10_presolved']

,G,R,DB,V,"max(G,V)",V+G,GurF,V+GurF,GurL,V+GurL,BEST VPC DISJ,BEST GMIC+VPC DISJ,Best V+GurF DISJ,Best V+GurL DISJ,NUM VPC,NUM GMIC
INSTANCE,,,,,,,,,,,,,,,,


,cutlimit,gomory,mode,partial_bb_strategy,partial_bb_keep_pruned_nodes,partial_bb_num_strong,preprocess,prlp_flip_beta,rounds,bb_mode,...,REF+V TIME (6),REF+V TIME (7),REF+V NODES (1),REF+V NODES (2),REF+V NODES (3),REF+V NODES (4),REF+V NODES (5),REF+V NODES (6),REF+V NODES (7),AVG REF+V W/CUTGEN TIME
disj_terms,,,,,,,,,,,,,,,,,,,,,
-1,-1,-1,4,4,0,5,0,0,1,11,...,7200.0,7200.0,176702.0,185192.0,177870.0,189538.0,185020.0,175522.0,180558.0,8375.32
0,-1,0,0,4,0,5,0,0,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-1,-1,0,4,0,5,0,0,1,11,...,7200.0,7200.0,232818.0,227640.0,222888.0,237708.0,200720.0,259782.0,251712.0,7206.01
4,-1,-1,0,4,0,5,0,0,1,11,...,7200.0,7200.0,185632.0,221466.0,164602.0,208406.0,212170.0,164904.0,158692.0,7300.39
8,-1,-1,0,4,0,5,0,0,1,11,...,7200.0,7200.0,188514.0,178386.0,155034.0,183128.0,192790.0,165164.0,166266.0,7987.51
16,-1,-1,0,4,0,5,0,0,1,11,...,7200.0,7200.0,198230.0,223000.0,182524.0,204464.0,194386.0,194290.0,235886.0,7255.55
32,-1,-1,0,4,0,5,0,0,1,11,...,7200.0,7200.0,203536.0,226190.0,187740.0,210366.0,198854.0,198460.0,243374.0,7289.23
64,-1,-1,0,4,0,5,0,0,1,11,...,7200.0,7200.0,204014.0,229366.0,189172.0,211730.0,199732.0,198358.0,241568.0,7356.70


#### DEBUG: Find instances in which V+GurF max does not match up

In [37]:
## DEBUG: Find instances in which V+GurF max does not match up
# This causes the value in Table 2 'Best' row to not match Table 1 'All'

# For instance f2gap801600_presolved, the gap closed at the end of the root node is 0% whenever VPCs are used,
# but without VPCs, the gap closed is 50%
# In `best_gap_df`, for an instance in which no VPCs were generated,
# we use the value of GurF/GurL for V+GurF/V+GurL
# In `selected_gap_df`, the "zero" row contains

num_inst = len(best_gap_df.index)
col = 'V+GurF'
origcol = map_short_to_cols_gap[col]
num_errors = 0
avg1 = 0
avg2 = 0
for inst in best_gap_df.index:
    val1 = best_gap_df.loc[inst,col]
    val2 = selected_gap_df.loc[(inst,0),origcol]
    if abs(val1-val2) > EPS:
        print("{} has best_gap_df = {:f} and selected_gap_df = {:f} for col {} (diff = {:e})".format(inst,val1,val2,col,abs(val1-val2)))
        num_errors += 1
    avg1 += val1 / num_inst
    avg2 += val2 / num_inst

print("Average from best_gap_df = {}".format(avg1))
print("Average from selected_gap_df = {}".format(avg2))
print("Total # of errors =", num_errors, flush=True)

Average from best_gap_df = 34.8014543982641
Average from selected_gap_df = 34.80145439826473
Total # of errors = 0


#### DEBUG: Print relevant info from `selected_gap_df` and `best_gap_df` to further debug

In [38]:
## DEBUG
# inst = 'f2gap801600_presolved'
# inst = 'neos22_presolved'
# inst = 'neos-1112787_presolved'
# display(best_gap_df.loc[inst])
# display(selected_gap_df.loc[inst,[col_num_vpcs]+gap_cols])

In [39]:
# ## DEBUG
# gap_cols = [
#     'GMIC % GAP CLOSED',
#     'BEST DISJ % GAP CLOSED',
#     'VPC % GAP CLOSED',
#     'VPC+GMIC % GAP CLOSED',
#     'REF FIRST_CUT_PASS % GAP CLOSED',
#     'REF+V FIRST_CUT_PASS % GAP CLOSED',
#     'REF LAST_CUT_PASS % GAP CLOSED',
#     'REF+V LAST_CUT_PASS % GAP CLOSED',
# ]
# col_num_vpcs = 'NUM VPC'

inst = 'f2gap801600_presolved'
tmp_selected_gap_df = gap_df.loc[selected_gap_instances_dict.keys()]
# Check if inst is in tmp_selected_gap_df
if inst not in tmp_selected_gap_df.index.get_level_values(0).unique():
    print(ValueError("Instance {} is not in tmp_selected_gap_df".format(inst)))
else:
    curr_df = tmp_selected_gap_df.loc[inst].copy() # copy needed to not throw SettingWithCopyWarning

    # Set 0-row to have max values across all rows for this instance
    max_vals = curr_df[gap_cols].max()
    # selected_gap_df.loc[(inst,0),gap_cols] = max_vals

    display(tmp_selected_gap_df.loc[inst])
    display(max_vals)

,NUM DISJ TERMS,ROWS,COLS,LP OBJ,IP OBJ,WORST DISJ OBJ,GMIC OBJ,ROOT OBJ,BEST DISJ OBJ,VPC OBJ,...,FIRST REF+V FIRST_CUT_PASS % GAP CLOSED,FIRST REF+V LAST_CUT_PASS % GAP CLOSED,AVG REF FIRST_CUT_PASS % GAP CLOSED,AVG REF LAST_CUT_PASS % GAP CLOSED,AVG REF+V FIRST_CUT_PASS % GAP CLOSED,AVG REF+V LAST_CUT_PASS % GAP CLOSED,BEST REF FIRST_CUT_PASS % GAP CLOSED,BEST REF LAST_CUT_PASS % GAP CLOSED,BEST REF+V FIRST_CUT_PASS % GAP CLOSED,BEST REF+V LAST_CUT_PASS % GAP CLOSED
disj_terms,,,,,,,,,,,,,,,,,,,,,
-1,-1,80,1600,86570.120867,86678.99998,8.662223e+04,86655.691018,86570.120867,8.657675e+04,86571.335554,...,95.407769,95.407769,49.794643,49.794643,95.407769,95.407769,49.794643,49.794643,95.407769,95.407769
0,0,80,1600,86570.120870,86678.99998,-inf,NaN,NaN,-inf,NaN,...,0.000000,0.000000,49.794639,49.794639,0.000000,0.000000,49.794639,49.794639,0.000000,0.000000
2,2,80,1600,86570.120867,86678.99998,8.657845e+04,86655.691018,86570.120867,8.657127e+04,86571.026383,...,95.407769,95.407769,49.794643,49.794643,95.407769,95.407769,49.794643,49.794643,95.407769,95.407769
4,4,80,1600,86570.120867,86678.99998,8.658275e+04,86655.691018,86570.120867,8.657302e+04,86571.188056,...,95.407769,95.407769,49.794643,49.794643,95.407769,95.407769,49.794643,49.794643,95.407769,95.407769
8,8,80,1600,86570.120867,86678.99998,8.661955e+04,86655.691018,86570.120867,8.657410e+04,86571.326821,...,95.407769,95.407769,49.794643,49.794643,95.407769,95.407769,49.794643,49.794643,95.407769,95.407769
16,16,80,1600,86570.120867,86678.99998,8.661955e+04,86655.691018,86570.120867,8.657505e+04,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
32,32,80,1600,86570.120867,86678.99998,8.661955e+04,86655.691018,86570.120867,8.657588e+04,86571.271636,...,95.407769,95.407769,49.794643,49.794643,95.407769,95.407769,49.794643,49.794643,95.407769,95.407769
64,64,80,1600,86570.120867,86678.99998,8.662223e+04,86655.691018,86570.120867,8.657675e+04,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


GMIC % GAP CLOSED                          78.591888
ROOT % GAP CLOSED                           0.000000
BEST DISJ % GAP CLOSED                      6.086315
VPC % GAP CLOSED                            1.115629
MAX(GMIC,VPC) % GAP CLOSED                 78.591888
VPC+GMIC % GAP CLOSED                      78.607302
FIRST REF FIRST_CUT_PASS % GAP CLOSED      49.794643
FIRST REF+V FIRST_CUT_PASS % GAP CLOSED    95.407769
AVG REF FIRST_CUT_PASS % GAP CLOSED        49.794643
AVG REF+V FIRST_CUT_PASS % GAP CLOSED      95.407769
BEST REF FIRST_CUT_PASS % GAP CLOSED       49.794643
BEST REF+V FIRST_CUT_PASS % GAP CLOSED     95.407769
FIRST REF LAST_CUT_PASS % GAP CLOSED       49.794643
FIRST REF+V LAST_CUT_PASS % GAP CLOSED     95.407769
AVG REF LAST_CUT_PASS % GAP CLOSED         49.794643
AVG REF+V LAST_CUT_PASS % GAP CLOSED       95.407769
BEST REF LAST_CUT_PASS % GAP CLOSED        49.794643
BEST REF+V LAST_CUT_PASS % GAP CLOSED      95.407769
dtype: float64

### Table 1: `avg_gap_df`: average percent gap closed across different combinations of cuts

In [40]:
## TABLE 1: average percent gap closed across different combinations of cuts
## Create avg_gap_df = average gap closed across instances
all_set_name = 'All'
good_vpc_set_name = tex_escape('≥10%')
binary_set_name = 'Binary'
avg_row_name = tex_escape('Avg (%)')
wins_row_name = 'Wins'

idx = pd.MultiIndex.from_product(
    [ [all_set_name, good_vpc_set_name, binary_set_name], [avg_row_name, wins_row_name] ],
    names = ['Set', '']
)
    
ncols = len(best_gap_df.columns)
nrows = len(idx)

col = best_gap_df['V'].astype(float)
good_vpc_df = best_gap_df[col >= 10.]

binary_instances_df = best_gap_df.loc[binary_x_gap_instances]

data = np.zeros((nrows, ncols), dtype=float)
data[0,:] = [best_gap_df[col].mean() for col in best_gap_df.columns]
data[2,:] = [good_vpc_df[col].mean() for col in best_gap_df.columns]
data[4,:] = [binary_instances_df[col].mean() for col in best_gap_df.columns]

# display(best_gap_df.head())
avg_gap_df = pd.DataFrame(
    data,
    columns = best_gap_df.columns,
    index = idx,
    dtype = object
)

inst_col_name = '# inst'
avg_gap_df[inst_col_name] = [len(best_gap_df), 0, len(good_vpc_df), 0, len(binary_instances_df), 0]

avg_gap_df.iloc[1] = ["" for i in range(ncols+1)]
avg_gap_df.iloc[3] = ["" for i in range(ncols+1)]
avg_gap_df.iloc[5] = ["" for i in range(ncols+1)]

display(avg_gap_df)

/var/folders/_8/bgx4yr0n6f1dm56pnvsqqb1s91dzml/T/ipykernel_59358/2782119350.py:38: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  avg_gap_df.iloc[1] = ["" for i in range(ncols+1)]


G         R         DB          V   max(G,V)  \
Set                                                                        
All       Avg (\%)  16.508148  0.256744  18.369423  12.008265  23.007369   
          Wins                                                             
$\ge$10\% Avg (\%)  16.804984  0.692761  38.100165  29.795586  34.219557   
          Wins                                                             
Binary    Avg (\%)  13.641808       0.0  21.798838  17.306146  25.477278   
          Wins                                                             

                          V+G       GurF     V+GurF       GurL     V+GurL  \
Set                                                                         
All       Avg (\%)  23.916435  28.929744  34.801454  48.319147  52.806027   
          Wins                                                              
$\ge$10\% Avg (\%)  35.904073  26.410744  38.588081  45.268302  55.396735   
          Wins                                                              
Binary    Avg (\%)    26.3934  19.229515  31.255048  32.935913  42.795453   
          Wins                                                              

                   BEST VPC DISJ BEST GMIC+VPC DISJ Best V+GurF DISJ  \
Set                                                                    
All       Avg (\%)     39.957831          35.753012        22.987952   
          Wins                                                         
$\ge$10\% Avg (\%)          54.0          51.396552        35.551724   
          Wins                                                         
Binary    Avg (\%)     40.984615          40.492308        26.676923   
          Wins                                                         

                   Best V+GurL DISJ     NUM VPC    NUM GMIC # inst  
Set                                                                 
All       Avg (\%)         24.96988   73.216867  175.879518    332  
          Wins                                                      
$\ge$10\% Avg (\%)        34.344828   64.663793   92.043103    116  
          Wins                                                      
Binary    Avg (\%)        29.661538  163.307692  380.323077     65  
          Wins

### `wins_df`: num wins

In [41]:
## Create num wins df
# x wins over y for an instance if x > y + EPS
#shortcols = avg_gap_df.columns[0:-1]
wins_df = pd.DataFrame(
    np.zeros((len(gap_cols_short), len(gap_cols_short)), dtype=int),
    columns = gap_cols_short,
    index = gap_cols_short,
    dtype = int,
)

WINS_EPS = GAP_DIFF_EPS

from itertools import permutations
for (ind1, ind2) in permutations(range(len(gap_cols_short)), 2):
    wins_df.at[gap_cols_short[ind1],gap_cols_short[ind2]] =\
        int(sum(best_gap_df[gap_cols_short[ind1]] > best_gap_df[gap_cols_short[ind2]] + WINS_EPS))
    wins_df.at[gap_cols_short[ind2],gap_cols_short[ind1]] =\
        int(sum(best_gap_df[gap_cols_short[ind2]] > best_gap_df[gap_cols_short[ind1]] + WINS_EPS))

# Sets we are considering
# all_set = 'Wins (All)'
# good_vpc_set = 'Wins (V ≥ 10%)'
all_set = (all_set_name,wins_row_name)
good_vpc_set = (good_vpc_set_name,wins_row_name)
binary_set = (binary_set_name,wins_row_name)

# "G" are wins relative to "V"
shortrefcol = 'V'
#refcol = 'VPC % GAP CLOSED'
#refcol = map_short_to_cols[shortrefcol]
refcol = shortrefcol
shortdestcol = 'G'
#col = 'GMIC % GAP CLOSED'
#col = map_short_to_cols[shortcol]
destcol = shortdestcol
avg_gap_df.at[all_set,shortdestcol] = wins_df.at[shortdestcol,shortrefcol]
avg_gap_df.at[good_vpc_set,shortdestcol] = sum(good_vpc_df[destcol] > good_vpc_df[refcol] + WINS_EPS)
avg_gap_df.at[binary_set,shortdestcol] = sum(binary_instances_df[destcol] > binary_instances_df[refcol] + WINS_EPS)

# "DB", "V", "V+G": wins are relative to "G"
shortrefcol = 'G'
#refcol = 'GMIC % GAP CLOSED'
#refcol = map_short_to_cols[shortrefcol]
refcol = shortrefcol
shortdestcol = 'DB'
#col = 'BEST DISJ % GAP CLOSED'
#col = map_short_to_cols[shortcol]
destcol = shortdestcol
avg_gap_df.at[all_set,shortdestcol] = wins_df.at[shortdestcol,shortrefcol]
avg_gap_df.at[good_vpc_set,shortdestcol] = sum(good_vpc_df[destcol] > good_vpc_df[refcol] + WINS_EPS)
avg_gap_df.at[binary_set,shortdestcol] = sum(binary_instances_df[destcol] > binary_instances_df[refcol] + WINS_EPS)

shortdestcol = 'V'
#col = 'VPC % GAP CLOSED'
#col = map_short_to_cols[shortcol]
destcol = shortdestcol
avg_gap_df.at[all_set,shortdestcol] = wins_df.at[shortdestcol,shortrefcol]
avg_gap_df.at[good_vpc_set,shortdestcol] = sum(good_vpc_df[destcol] > good_vpc_df[refcol] + WINS_EPS)
avg_gap_df.at[binary_set,shortdestcol] = sum(binary_instances_df[destcol] > binary_instances_df[refcol] + WINS_EPS)

shortdestcol = 'V+G'
#col = 'VPC+GMIC % GAP CLOSED'
#col = map_short_to_cols[shortcol]
destcol = shortdestcol
avg_gap_df.at[all_set,shortdestcol] = wins_df.at[shortdestcol,shortrefcol]
avg_gap_df.at[good_vpc_set,shortdestcol] = sum(good_vpc_df[destcol] > good_vpc_df[refcol] + WINS_EPS)
avg_gap_df.at[binary_set,shortdestcol] = sum(binary_instances_df[destcol] > binary_instances_df[refcol] + WINS_EPS)

# "V+GurF" are wins relative to "GurF"
shortrefcol = 'GurF'
refcol = shortrefcol
shortdestcol = 'V+GurF'
destcol = shortdestcol
#col = map_short_to_cols[shortcol]
avg_gap_df.at[all_set,shortdestcol] = wins_df.at[shortdestcol,shortrefcol]
avg_gap_df.at[good_vpc_set,shortdestcol] = sum(good_vpc_df[destcol] > good_vpc_df[refcol] + WINS_EPS)
avg_gap_df.at[binary_set,shortdestcol] = sum(binary_instances_df[destcol] > binary_instances_df[refcol] + WINS_EPS)

# "V+GurL" are wins relative to "GurL"
shortrefcol = 'GurL'
refcol = shortrefcol
shortdestcol = 'V+GurL'
destcol = shortdestcol
wins_df.at[shortdestcol,shortrefcol] = int(sum(best_gap_df[destcol] > best_gap_df[refcol] + WINS_EPS))
wins_df.at[shortrefcol,shortdestcol] = int(sum(best_gap_df[refcol] > best_gap_df[destcol] + WINS_EPS))
avg_gap_df.at[all_set,shortdestcol] = wins_df.at[shortdestcol,shortrefcol]
avg_gap_df.at[good_vpc_set,shortdestcol] = sum(good_vpc_df[destcol] > good_vpc_df[refcol] + WINS_EPS)
avg_gap_df.at[binary_set,shortdestcol] = sum(binary_instances_df[destcol] > binary_instances_df[refcol] + WINS_EPS)

# Count number of instances that have V+G > 0
shortdestcol = inst_col_name
#col = 'V+GurL'
destcol = 'V+G'
avg_gap_df.at[all_set,shortdestcol] = sum(best_gap_df[destcol] > WINS_EPS)
avg_gap_df.at[good_vpc_set,shortdestcol] = sum(good_vpc_df[destcol] > WINS_EPS)
avg_gap_df.at[binary_set,shortdestcol] = sum(binary_instances_df[destcol] > WINS_EPS)

display(avg_gap_df)
display(wins_df)

G         R         DB          V   max(G,V)  \
Set                                                                        
All       Avg (\%)  16.508148  0.256744  18.369423  12.008265  23.007369   
          Wins            165                  178        132              
$\ge$10\% Avg (\%)  16.804984  0.692761  38.100165  29.795586  34.219557   
          Wins             24                  100         92              
Binary    Avg (\%)  13.641808       0.0  21.798838  17.306146  25.477278   
          Wins             22                   50         37              

                          V+G       GurF     V+GurF       GurL     V+GurL  \
Set                                                                         
All       Avg (\%)  23.916435  28.929744  34.801454  48.319147  52.806027   
          Wins            226                   250                   234   
$\ge$10\% Avg (\%)  35.904073  26.410744  38.588081  45.268302  55.396735   
          Wins            112                   101                   107   
Binary    Avg (\%)    26.3934  19.229515  31.255048  32.935913  42.795453   
          Wins             53                    48                    48   

                   BEST VPC DISJ BEST GMIC+VPC DISJ Best V+GurF DISJ  \
Set                                                                    
All       Avg (\%)     39.957831          35.753012        22.987952   
          Wins                                                         
$\ge$10\% Avg (\%)          54.0          51.396552        35.551724   
          Wins                                                         
Binary    Avg (\%)     40.984615          40.492308        26.676923   
          Wins                                                         

                   Best V+GurL DISJ     NUM VPC    NUM GMIC # inst  
Set                                                                 
All       Avg (\%)         24.96988   73.216867  175.879518    332  
          Wins                                                 299  
$\ge$10\% Avg (\%)        34.344828   64.663793   92.043103    116  
          Wins                                                 116  
Binary    Avg (\%)        29.661538  163.307692  380.323077     65  
          Wins                                                  59

,G,R,DB,V,"max(G,V)",V+G,GurF,V+GurF,GurL,V+GurL
G,0,288,121,165,0,0,68,33,16,9
R,1,0,0,0,0,0,1,0,1,0
DB,178,283,0,264,164,150,121,108,80,62
V,132,264,0,0,0,0,90,57,58,14
"max(G,V)",132,298,121,165,0,0,124,82,67,20
V+G,226,299,138,249,177,0,136,97,69,25
GurF,222,301,185,216,172,165,0,0,0,0
V+GurF,267,306,192,242,210,197,250,0,52,0
GurL,288,308,233,255,240,238,294,249,0,50
V+GurL,298,312,245,293,281,276,302,292,234,0


In [42]:
binary_instances_df.to_csv('binary_instances.csv')

### Analyze instances in which DB > G but V <= G

In [43]:
col1 = 'DB'
col2 = 'G'
tmp_df = best_gap_df.loc[best_gap_df[col1] > best_gap_df[col2] + GAP_DIFF_EPS]

col1 = 'V'
tmp_df = tmp_df[tmp_df[col1] <= tmp_df[col2] + GAP_DIFF_EPS]
display(tmp_df.head())

# inst_set = tmp_df.index
inst_depth_set = [(inst,tmp_df.at[inst,col_best_disj_vpc]) for inst in tmp_df.index]

print("Total num inst with DB > G >= V is {:d}".format(len(tmp_df)))
print("Num times hit cut limit = {:d}".format(sum(df.loc[inst_depth_set,col_exit_reason] == 'CUT_LIMIT')))

# display(df.loc[inst_depth_set])


,G,R,DB,V,"max(G,V)",V+G,GurF,V+GurF,GurL,V+GurL,BEST VPC DISJ,BEST GMIC+VPC DISJ,Best V+GurF DISJ,Best V+GurL DISJ,NUM VPC,NUM GMIC
INSTANCE,,,,,,,,,,,,,,,,
app3_presolved,20.407763,0.0,55.666020,15.523885,20.407763,26.668549,28.427030,34.823870,83.904311,84.426687,32,32,32,16,15,18
beasleyC3_presolved,1.296774,0.0,2.810011,1.236898,1.296774,1.796111,57.739584,63.229491,97.718222,97.873228,16,32,2,32,124,124
bg512142_presolved,0.808097,0.0,2.001413,0.808720,0.808720,1.386087,3.505379,3.812320,10.772976,10.985625,4,4,4,16,63,204
bppc8-09_presolved,3.078078,0.0,3.301803,0.593229,3.078078,3.078078,1.187913,1.588956,3.611834,3.167295,4,32,4,4,30,30
control30-3-2-3_presolved,0.000631,0.0,0.004913,0.000961,0.000961,0.001357,0.015750,0.016448,0.016493,0.016988,16,16,16,16,24,10


Total num inst with DB > G >= V is 46
Num times hit cut limit = 24


### Analyze instances in which V+G <= G

In [44]:
col1 = 'V+G'
col2 = 'G'
tmp_df = best_gap_df.loc[best_gap_df[col1] <= best_gap_df[col2] + GAP_DIFF_EPS]

display(tmp_df)

inst_depth_set = [(inst,tmp_df.at[inst,col_best_disj_vpc]) for inst in tmp_df.index]

print("Total num inst with V+G <= G is {:d}".format(len(tmp_df)))
refval = 100. - GAP_DIFF_EPS
print("Num times with G > {}% gap closed = {:d}".format(refval, sum(tmp_df['G'] > refval)))
print("Num times with V+G = 0% gap closed = {:d}".format(sum(tmp_df['V+G'] == 0.)))
print("Num times hit cut limit = {:d}".format(sum(df.loc[inst_depth_set,col_exit_reason] == 'CUT_LIMIT')))

# display(df.loc[inst_depth_set])

,G,R,DB,V,"max(G,V)",V+G,GurF,V+GurF,GurL,V+GurL,BEST VPC DISJ,BEST GMIC+VPC DISJ,Best V+GurF DISJ,Best V+GurL DISJ,NUM VPC,NUM GMIC
INSTANCE,,,,,,,,,,,,,,,,
10teams_presolved,100.000000,0.0,0.000000,0.000000,100.000000,100.000000,93.877551,93.877551,100.000000,100.000000,32,32,16,2,16,153
30n20b8_presolved,11.099235,0.0,1.564622,0.033334,11.099235,11.099235,0.926468,1.331188,16.910036,16.300723,2,2,2,4,190,184
a2c1s1_presolved,24.372660,0.0,3.337449,0.461882,24.372660,24.372660,42.797493,44.248500,91.529658,92.019454,2,4,16,32,18,157
b1c1s1_presolved,23.005942,0.0,3.738399,0.049882,23.005942,23.006597,22.481832,27.757283,76.316351,76.687606,16,16,64,16,9,240
b2c1s1_presolved,19.812503,0.0,2.007490,0.085124,19.812503,19.812503,23.068346,23.399644,72.082156,72.319120,64,32,16,4,4,238
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
tr12-30_presolved,58.360211,0.0,1.743446,0.477979,58.360211,58.360211,60.236075,60.438883,98.547975,98.689839,64,2,2,4,2,321
traininstance6_presolved,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2,2,2,2,2,21
umts_presolved,0.973181,0.0,0.209040,0.108872,0.973181,0.973181,1.412379,1.463163,4.793132,5.023309,64,64,2,8,276,275


Total num inst with V+G <= G is 106
Num times with G > 99.999% gap closed = 6
Num times with V+G = 0% gap closed = 33
Num times hit cut limit = 24


### Analyze when G > V

In [45]:
col1 = 'G'
col2 = 'V'
tmp_df = best_gap_df.loc[best_gap_df[col1] > best_gap_df[col2] + GAP_DIFF_EPS]

display(tmp_df)

inst_depth_set = [(inst,tmp_df.at[inst,col_best_disj_vpc]) for inst in tmp_df.index]

print("Total num inst with G > V is {:d}".format(len(tmp_df)))
print("Num times with #V < 10 is {:d}".format(len(tmp_df[(tmp_df[col_num_vpc] < 10)])))
print("Num times with #V < 10 while #G > 10 is {:d}".format(len(tmp_df[(tmp_df[col_num_vpc] < 10) & (tmp_df[col_num_gmic] > 10)])))
# print("Num times with #V < 10 is {:d}".format(sum(tmp_df['NUM VPC'] < 10)))

# print("Num times with V+G = 0% gap closed = {:d}".format(sum(tmp_df['V+G'] == 0.)))
print("Num times hit cut limit = {:d}".format(sum(df.loc[inst_depth_set,col_exit_reason] == 'CUT_LIMIT')))

tmp_inst_set = tmp_df[(tmp_df[col_num_vpc] < 10) & (tmp_df[col_num_gmic] > 10)].index
tmp_inst_depth_set = [(inst,tmp_df.at[inst,col_best_disj_vpc]) for inst in tmp_inst_set]
print("Num times hit cut limit when #G > #V = {:d} (should be 0)".format(sum(df.loc[tmp_inst_depth_set,col_exit_reason] == 'CUT_LIMIT')))

# display(df.loc[inst_depth_set])

,G,R,DB,V,"max(G,V)",V+G,GurF,V+GurF,GurL,V+GurL,BEST VPC DISJ,BEST GMIC+VPC DISJ,Best V+GurF DISJ,Best V+GurL DISJ,NUM VPC,NUM GMIC
INSTANCE,,,,,,,,,,,,,,,,
10teams_presolved,100.000000,0.0,0.000000,0.000000,100.000000,100.000000,93.877551,93.877551,100.000000,100.000000,32,32,16,2,16,153
30n20b8_presolved,11.099235,0.0,1.564622,0.033334,11.099235,11.099235,0.926468,1.331188,16.910036,16.300723,2,2,2,4,190,184
50v-10_presolved,45.753596,0.0,18.008191,11.184101,45.753596,45.823184,49.858925,50.152252,73.169318,74.444520,64,16,8,64,29,29
a1c1s1_presolved,25.105518,0.0,4.895611,1.094331,25.105518,25.382401,45.584474,46.254097,88.806373,88.730339,8,2,4,4,3,154
a2c1s1_presolved,24.372660,0.0,3.337449,0.461882,24.372660,24.372660,42.797493,44.248500,91.529658,92.019454,2,4,16,32,18,157
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
tr12-30_presolved,58.360211,0.0,1.743446,0.477979,58.360211,58.360211,60.236075,60.438883,98.547975,98.689839,64,2,2,4,2,321
umts_presolved,0.973181,0.0,0.209040,0.108872,0.973181,0.973181,1.412379,1.463163,4.793132,5.023309,64,64,2,8,276,275
vpm1_presolved,16.930266,0.0,7.788162,4.672897,16.930266,16.930266,45.861148,56.942590,68.157543,72.129506,32,2,2,64,10,16


Total num inst with G > V is 165
Num times with #V < 10 is 64
Num times with #V < 10 while #G > 10 is 60
Num times hit cut limit = 61
Num times hit cut limit when #G > #V = 0 (should be 0)


### Analyze when DB % gap closed nontrivial

In [46]:
# Select instances in selected_gap_df for which value in col_best_disj is at least MIN_DISJ_GAP
MIN_DISJ_GAP = 0.
tmp_df = selected_gap_df.loc[selected_gap_df[col_best_disj_gap] >= MIN_DISJ_GAP]

# Collect instance names with best disj gap >= MIN_DISJ_GAP
tmp_df.index = tmp_df.index.remove_unused_levels()
inst_set_db = tmp_df.index.get_level_values(0).unique()
num_inst_db = len(inst_set_db)
inst_set_orig = selected_gap_df.index.levels[0]
num_inst_orig = len(inst_set)
print("Total num inst with best disj gap >= {:f} is {:d} (out of {:d} total instances).".format(MIN_DISJ_GAP, num_inst_db, num_inst_orig))

# Report average in each column broken down by depth
tmp_df_grouped = tmp_df.groupby(level='disj_terms').mean(numeric_only=True)
display(tmp_df_grouped[gap_cols])

# Repeat with MIN_DISJ_GAP = 1.0
MIN_DISJ_GAP = 1.
tmp_df = selected_gap_df.loc[selected_gap_df[col_best_disj_gap] >= MIN_DISJ_GAP]

# Collect instance names with best disj gap >= MIN_DISJ_GAP
tmp_df.index = tmp_df.index.remove_unused_levels()
inst_set_db = tmp_df.index.get_level_values(0).unique()
num_inst_db = len(inst_set_db)
inst_set_orig = selected_gap_df.index.levels[0]
num_inst_orig = len(inst_set)
print("Total num inst with best disj gap >= {:f} is {:d} (out of {:d} total instances).".format(MIN_DISJ_GAP, num_inst_db, num_inst_orig))

# Report average in each column broken down by depth
tmp_df_grouped = tmp_df.groupby(level='disj_terms').mean(numeric_only=True)
display(tmp_df_grouped[gap_cols])

# Repeat with MIN_DISJ_GAP = 10.
MIN_DISJ_GAP = 10.
tmp_df = selected_gap_df.loc[selected_gap_df[col_best_disj_gap] >= MIN_DISJ_GAP]

# Collect instance names with best disj gap >= MIN_DISJ_GAP
tmp_df.index = tmp_df.index.remove_unused_levels()
inst_set_db = tmp_df.index.get_level_values(0).unique()
num_inst_db = len(inst_set_db)
inst_set_orig = selected_gap_df.index.levels[0]
num_inst_orig = len(inst_set)
print("Total num inst with best disj gap >= {:f} is {:d} (out of {:d} total instances).".format(MIN_DISJ_GAP, num_inst_db, num_inst_orig))

# Report average in each column broken down by depth
tmp_df_grouped = tmp_df.groupby(level='disj_terms').mean(numeric_only=True)
display(tmp_df_grouped[gap_cols])


Total num inst with best disj gap >= 0.000000 is 332 (out of 332 total instances).


,GMIC % GAP CLOSED,ROOT % GAP CLOSED,BEST DISJ % GAP CLOSED,VPC % GAP CLOSED,"MAX(GMIC,VPC) % GAP CLOSED",VPC+GMIC % GAP CLOSED,FIRST REF FIRST_CUT_PASS % GAP CLOSED,FIRST REF+V FIRST_CUT_PASS % GAP CLOSED,AVG REF FIRST_CUT_PASS % GAP CLOSED,AVG REF+V FIRST_CUT_PASS % GAP CLOSED,BEST REF FIRST_CUT_PASS % GAP CLOSED,BEST REF+V FIRST_CUT_PASS % GAP CLOSED,FIRST REF LAST_CUT_PASS % GAP CLOSED,FIRST REF+V LAST_CUT_PASS % GAP CLOSED,AVG REF LAST_CUT_PASS % GAP CLOSED,AVG REF+V LAST_CUT_PASS % GAP CLOSED,BEST REF LAST_CUT_PASS % GAP CLOSED,BEST REF+V LAST_CUT_PASS % GAP CLOSED
disj_terms,,,,,,,,,,,,,,,,,,
-1,16.518152,0.259875,18.012541,12.639294,23.299707,24.226787,28.970755,34.532474,29.055084,34.652712,30.767344,36.181843,48.693809,52.796652,48.650307,52.458325,51.407810,55.338254
0,16.508148,0.256744,18.369423,12.008265,23.007369,23.916435,28.844963,34.681055,28.929746,34.801454,30.625793,36.297842,48.355547,53.590603,48.319149,52.806027,51.094295,55.807675
2,16.834565,0.270600,3.177195,2.332737,17.131359,17.568248,29.493663,30.731209,29.606992,31.716813,31.317964,32.228174,49.539220,48.675216,49.709897,50.652438,52.348941,51.257251
4,16.901518,0.267207,5.485016,3.738900,17.650218,18.230355,29.719965,31.474220,29.800877,32.124022,31.484860,32.917285,49.913805,49.495554,49.827462,50.808698,52.384068,51.960680
8,16.582317,0.270600,8.555967,5.109086,17.973525,18.716862,29.541799,30.454226,29.619399,32.355609,31.250807,31.783415,49.844016,48.298080,49.666977,50.905485,52.137770,50.250215
16,16.652707,0.270600,11.776455,6.804641,19.672333,20.363842,29.319416,30.671470,29.401439,33.132555,31.006155,31.923759,49.565862,48.053495,49.380255,51.423028,51.837141,50.347900
32,16.883934,0.274965,15.712750,9.361390,21.649018,22.522697,29.528632,31.273717,29.537281,34.333400,31.026175,32.440526,49.340920,47.778319,49.183702,52.219020,51.327104,49.720418
64,16.759712,0.273202,19.546950,10.889200,22.751517,23.547545,29.384713,31.151625,29.377316,34.526443,30.820825,32.170993,49.268749,46.390159,48.884498,52.389039,51.231034,48.320855


Total num inst with best disj gap >= 1.000000 is 260 (out of 332 total instances).


,GMIC % GAP CLOSED,ROOT % GAP CLOSED,BEST DISJ % GAP CLOSED,VPC % GAP CLOSED,"MAX(GMIC,VPC) % GAP CLOSED",VPC+GMIC % GAP CLOSED,FIRST REF FIRST_CUT_PASS % GAP CLOSED,FIRST REF+V FIRST_CUT_PASS % GAP CLOSED,AVG REF FIRST_CUT_PASS % GAP CLOSED,AVG REF+V FIRST_CUT_PASS % GAP CLOSED,BEST REF FIRST_CUT_PASS % GAP CLOSED,BEST REF+V FIRST_CUT_PASS % GAP CLOSED,FIRST REF LAST_CUT_PASS % GAP CLOSED,FIRST REF+V LAST_CUT_PASS % GAP CLOSED,AVG REF LAST_CUT_PASS % GAP CLOSED,AVG REF+V LAST_CUT_PASS % GAP CLOSED,BEST REF LAST_CUT_PASS % GAP CLOSED,BEST REF+V LAST_CUT_PASS % GAP CLOSED
disj_terms,,,,,,,,,,,,,,,,,,
-1,18.523127,0.336913,23.319236,16.119603,27.131416,28.318739,30.835459,37.688179,30.796680,37.634954,32.248942,38.903524,51.215661,56.109200,50.948660,55.764170,53.497798,58.217351
0,18.227744,0.327843,23.424177,15.314762,26.523508,27.682019,30.447021,37.652923,30.443140,37.636671,31.916892,38.904128,50.476574,56.676800,50.216553,55.896379,52.708423,58.461357
2,19.703660,0.575591,6.569134,4.811157,20.334941,21.164724,31.905675,34.331944,31.748457,34.822708,32.709692,34.986450,50.825196,51.382231,50.683743,52.235854,52.769820,53.199417
4,19.380455,0.439116,8.922722,6.090190,20.608501,21.550579,32.167897,35.222750,32.143300,35.230919,33.563055,36.498031,52.306664,53.657675,52.195125,53.799831,54.373567,56.026714
8,19.838527,0.396461,12.472576,7.453452,21.873857,22.953615,32.415086,34.450008,32.394052,35.874307,33.819563,35.634712,52.844444,52.942954,52.731758,54.382737,54.875113,54.634336
16,19.289088,0.372223,16.143840,9.334114,23.439234,24.385498,31.711213,34.081754,31.734874,36.387775,33.179033,35.291150,51.971992,51.584491,51.780103,54.566046,53.983955,53.508804
32,19.041903,0.350778,20.005784,11.927472,25.119201,26.229956,31.438155,34.405426,31.519869,37.283500,33.005109,35.578514,51.554021,51.316393,51.454551,55.357594,53.702045,53.092933
64,18.227744,0.327843,23.424177,13.049467,25.414669,26.367784,30.447020,33.203564,30.443138,36.424025,31.916891,34.183920,50.476573,49.001348,50.216551,54.378382,52.708422,50.818714


Total num inst with best disj gap >= 10.000000 is 173 (out of 332 total instances).


,GMIC % GAP CLOSED,ROOT % GAP CLOSED,BEST DISJ % GAP CLOSED,VPC % GAP CLOSED,"MAX(GMIC,VPC) % GAP CLOSED",VPC+GMIC % GAP CLOSED,FIRST REF FIRST_CUT_PASS % GAP CLOSED,FIRST REF+V FIRST_CUT_PASS % GAP CLOSED,AVG REF FIRST_CUT_PASS % GAP CLOSED,AVG REF+V FIRST_CUT_PASS % GAP CLOSED,BEST REF FIRST_CUT_PASS % GAP CLOSED,BEST REF+V FIRST_CUT_PASS % GAP CLOSED,FIRST REF LAST_CUT_PASS % GAP CLOSED,FIRST REF+V LAST_CUT_PASS % GAP CLOSED,AVG REF LAST_CUT_PASS % GAP CLOSED,AVG REF+V LAST_CUT_PASS % GAP CLOSED,BEST REF LAST_CUT_PASS % GAP CLOSED,BEST REF+V LAST_CUT_PASS % GAP CLOSED
disj_terms,,,,,,,,,,,,,,,,,,
-1,18.665451,0.510412,32.629501,22.938458,31.243789,32.774878,29.894644,38.281634,29.882114,38.249750,31.506738,39.624786,51.001849,57.110199,50.928007,56.946163,53.450511,59.081171
0,18.320578,0.492710,32.579625,21.700938,30.357060,31.857749,29.440205,38.306523,29.485523,38.340578,31.137317,39.709601,50.074118,57.783494,50.001361,57.126120,52.446508,59.410083
2,30.605842,4.229435,25.987925,18.312230,34.303646,36.275594,43.921000,48.365510,42.456595,46.945514,44.141244,48.549317,64.871016,67.766646,64.401752,66.459897,67.029127,69.478798
4,22.382298,1.575672,21.675840,15.408665,25.947100,27.997440,34.670844,37.852513,34.136849,37.355811,35.504001,38.581257,56.065488,57.444726,56.207535,58.143703,58.824662,61.420417
8,18.918868,0.923670,24.013431,14.541965,23.223695,25.197161,31.117013,33.625527,30.883206,35.202570,32.332135,34.413032,52.902361,53.032911,52.883085,54.586810,55.744710,54.971022
16,18.816440,0.692580,25.831705,15.321460,26.035793,27.487848,30.230216,34.957507,30.187664,36.513231,31.996109,36.437450,51.666546,52.910699,51.613842,55.280668,54.543775,55.255603
32,18.171154,0.575597,29.438205,18.028349,27.762957,29.343880,28.938839,34.802949,28.934187,36.356955,30.629555,36.095938,50.065182,52.659740,49.996042,55.118266,52.682023,54.523125
64,18.320578,0.492710,32.579625,18.487752,28.730157,29.969568,29.440203,34.036878,29.485520,36.864043,31.137316,35.071426,50.074117,50.422763,50.001358,55.336555,52.446508,51.749328


### Table 2: `gap_by_size_df`: gap closed by num leaves

In [47]:
## TABLE 2: gap closed by num leaves
## Note that ``best'' can be worse than for a single row
## because when no VPCs are generated, we assume the "no VPCs" results hold for Gurobi,
## but we do not count that for the ``best'' calculation,
## since otherwise there is potential bias, as sometimes Gurobi does better without VPCs
shortcols = [
        'DB',
        'V',
        'max(G,V)',
        'V+G'
      ] + solver_cut_type_short_list[1]

gap_by_size_df = pd.DataFrame(
    columns = shortcols,
    index = [0] + sizes + ['Best'],
    # index = [str(size) + " leaves" for size in sizes]+['Best'],
    dtype = float,
)
zero_row_name = 0

# `grouped_df` will collect gap closed across instances, grouped by num terms
grouped_df = selected_gap_df.groupby(level='disj_terms').mean(numeric_only=True)
ungrouped_df = best_gap_df.mean(numeric_only=True)

# For each of the columns (in shortcols),
# save the average value for each size
# (this will put in the right place as the index is based on sizes for both)
for col in shortcols:
    orig_col = map_short_to_cols_gap[col]
    #gap_by_size_df.loc[2]['DB'] = best_gap_df[orig_col].mean()
    gap_by_size_df[col] = grouped_df[orig_col]

# Fill in the 'Best' row, since that is currently stored in `gap_by_size_df` in the "0" row
gap_by_size_df.loc['Best'] = gap_by_size_df.loc[zero_row_name]

# Now update the zero row with correct values
col = 'DB'
gap_by_size_df[col][zero_row_name] = 0.

col = 'V'
gap_by_size_df[col][zero_row_name] = 0.

stubs = ['G', 'GurF', 'GurL']
for stub in stubs:
    col = 'V+'+stub
    # orig_col = map_short_to_cols[stub]
    gap_by_size_df[col][0] = ungrouped_df[stub]

# Also replace the 0-row of the "max(G,V)" column with the value of G, since that corresponds to no VPCs
gap_by_size_df['max(G,V)'][0] = gap_by_size_df['V+G'][0]

# Reindex to add "leaves" to index
idx = ['VPCs disabled']+[str(size) + " leaves" for size in sizes]+['Best']
reidx = {old_id : new_id for old_id, new_id in zip(gap_by_size_df.index,idx)}
gap_by_size_df.rename(reidx, inplace=True)

# display(grouped_df[gap_cols])
display(ungrouped_df)
display(gap_by_size_df)

# Create new df with additional columns:
# (1) the ratio 'V'/'DB'
# (2) the ratio 'max(G,V)'/'V+G'
gap_by_size_df_new = gap_by_size_df.copy()
gap_by_size_df_new['V/DB'] = gap_by_size_df_new['V'] / gap_by_size_df_new['DB']
gap_by_size_df_new['max(G,V)/V+G'] = gap_by_size_df_new['max(G,V)'] / gap_by_size_df_new['V+G']
display(gap_by_size_df_new)

G                      16.508148
R                       0.256744
DB                     18.369423
V                      12.008265
max(G,V)               23.007369
V+G                    23.916435
GurF                   28.929744
V+GurF                 34.801454
GurL                   48.319147
V+GurL                 52.806027
BEST VPC DISJ          39.957831
BEST GMIC+VPC DISJ     35.753012
Best V+GurF DISJ       22.987952
Best V+GurL DISJ       24.969880
NUM VPC                73.216867
NUM GMIC              175.879518
dtype: float64

,DB,V,"max(G,V)",V+G,V+GurF,V+GurL
VPCs disabled,0.000000,0.000000,16.508148,16.508148,28.929744,48.319147
2 leaves,3.014507,2.213289,16.789746,17.206579,30.961171,49.260530
4 leaves,5.270242,3.592498,17.227532,17.785031,31.164402,49.252142
8 leaves,8.117860,4.847477,17.828119,18.533394,31.526034,49.484617
16 leaves,11.173444,6.456210,19.373155,20.029255,32.470786,50.236188
32 leaves,14.671544,8.741057,20.957474,21.773258,33.413550,51.140841
64 leaves,18.369423,10.233224,22.139001,22.887076,33.825053,51.638833
Best,18.369423,12.008265,23.007369,23.916435,34.801454,52.806027


,DB,V,"max(G,V)",V+G,V+GurF,V+GurL,V/DB,"max(G,V)/V+G"
VPCs disabled,0.000000,0.000000,16.508148,16.508148,28.929744,48.319147,NaN,1.000000
2 leaves,3.014507,2.213289,16.789746,17.206579,30.961171,49.260530,0.734213,0.975775
4 leaves,5.270242,3.592498,17.227532,17.785031,31.164402,49.252142,0.681657,0.968654
8 leaves,8.117860,4.847477,17.828119,18.533394,31.526034,49.484617,0.597137,0.961946
16 leaves,11.173444,6.456210,19.373155,20.029255,32.470786,50.236188,0.577817,0.967243
32 leaves,14.671544,8.741057,20.957474,21.773258,33.413550,51.140841,0.595783,0.962533
64 leaves,18.369423,10.233224,22.139001,22.887076,33.825053,51.638833,0.557079,0.967315
Best,18.369423,12.008265,23.007369,23.916435,34.801454,52.806027,0.653709,0.961990


### Table 5: `all_gap_results_df`: complete gap closed results

In [48]:
inst_set = selected_gap_df.index.levels[0]
inst_set.set_names("Instance",inplace=True)

col_idx = pd.MultiIndex.from_arrays(
    [
        ['', '', '# cuts', '# cuts'] + ['% gap closed']*len(gap_cols_short),
        ['Rows', 'Cols', 'G', 'V'] + gap_cols_short
    ],
)

all_gap_results_df = pd.DataFrame(
    columns = col_idx,
    index = inst_set,
    dtype = object,
)

# Enter number of rows and cols
tmp_df = df.xs(0, level='disj_terms').loc[inst_set,[col_num_rows, col_num_cols]]
tmp_df.columns = pd.MultiIndex.from_product([[''],['Rows','Cols']])
all_gap_results_df.loc[:,tmp_df.columns] = tmp_df

# Enter number of cuts
# tmp_df = best_gap_df.xs(0, level='disj_terms').loc[inst_set,['NUM GMIC', 'NUM VPC']]
tmp_df = best_gap_df.loc[inst_set, [col_num_gmic, col_num_vpc]]
tmp_df.columns = pd.MultiIndex.from_product([['# cuts'],['G','V']])
all_gap_results_df.loc[:,tmp_df.columns] = tmp_df

# Enter gap closed
tmp_df = best_gap_df.loc[inst_set, gap_cols_short]
tmp_df.columns = pd.MultiIndex.from_product([['% gap closed'],gap_cols_short])
all_gap_results_df.loc[:,tmp_df.columns] = tmp_df

# Add average row
all_gap_results_df.loc["Average"] = all_gap_results_df.loc[:,('% gap closed',gap_cols_short)].mean()

# Now convert the % gap closed columns to objects so we can add an int row
all_gap_results_df.loc[:,('% gap closed',gap_cols_short)] = all_gap_results_df.loc[:,('% gap closed',gap_cols_short)].astype(object)

# Add wins row
win_gap_cols_short = ['DB', 'V', 'V+G'] + solver_cut_type_short_list[1]
all_gap_results_df.loc['Wins',('% gap closed',win_gap_cols_short)] = avg_gap_df.loc[all_set,win_gap_cols_short].values.tolist()
# all_gap_results_df.loc['Wins',('% gap closed',win_gap_cols_short)] = avg_gap_df.loc[all_set,gap_cols_short].astype(np.int64).values.tolist()
# all_gap_results_df.loc["Wins"] = avg_gap_df.loc[all_set,gap_cols_short]
# wins_df.at[cols[ind1],cols[ind2]] = int(sum(best_gap_df[cols[ind1]] > best_gap_df[cols[ind2]] + EPS))

# Replace missing entries with empty string
all_gap_results_df = all_gap_results_df.fillna('',downcast=False)

# Convert rows, cols, # cuts to int values
tmp_cols = pd.MultiIndex.from_product([[''],['Rows','Cols']])
all_gap_results_df.loc[inst_set,tmp_cols] = all_gap_results_df.loc[inst_set,tmp_cols].astype(np.int64)
tmp_cols = pd.MultiIndex.from_product([['# cuts'],['G','V']])
all_gap_results_df.loc[inst_set,tmp_cols] = all_gap_results_df.loc[inst_set,tmp_cols].astype(np.int64)

display(all_gap_results_df.tail())

print("Num instances =",len(all_gap_results_df)-2)

/var/folders/_8/bgx4yr0n6f1dm56pnvsqqb1s91dzml/T/ipykernel_59358/2559796401.py:47: FutureWarning: The 'downcast' keyword in fillna is deprecated and will be removed in a future version. Use res.infer_objects(copy=False) to infer non-object dtype, or pd.to_numeric with the 'downcast' keyword to downcast numeric results.
  all_gap_results_df = all_gap_results_df.fillna('',downcast=False)


# cuts     % gap closed                       \
                     Rows  Cols      G   V            G         R         DB   
Instance                                                                       
vpm1_presolved        128   188     16  10    16.930266       0.0   7.788162   
vpm2_presolved        127   187     25  25    17.849671       0.0  14.293216   
zib54-UUE_presolved  1114  3726     56  56    10.696523       0.0  17.642953   
Average                                       16.508148  0.256744  18.369423   
Wins                                                                     178   

                                                                            \
                             V   max(G,V)        V+G       GurF     V+GurF   
Instance                                                                     
vpm1_presolved        4.672897  16.930266  16.930266  45.861148   56.94259   
vpm2_presolved        7.834301  17.849671  19.708285   43.30183  50.632749   
zib54-UUE_presolved    7.59568  10.696523  15.599614  43.113981   54.09513   
Average              12.008265  23.007369  23.916435  28.929744  34.801454   
Wins                       132                   226                   250   

                                           
                          GurL     V+GurL  
Instance                                   
vpm1_presolved       68.157543  72.129506  
vpm2_presolved        73.26105   73.52794  
zib54-UUE_presolved  68.331458  68.225269  
Average              48.319147  52.806027  
Wins                                  234

Num instances = 332


# Section 3: Time tables

In [49]:
from statistics import geometric_mean
SHIFT_TIME  = 1
SHIFT_GEN_TIME = 1
SHIFT_NODES = 1000
WIN_BY_TIME_FACTOR = 1.1
WIN_BY_NODES_FACTOR = 1.0

## Prepare variables for row/col names

In [50]:
## Prepare variables for row/col names

# Row names
bb_classes = ['All', '6 trees', 'Binary']
num_bb_classes = len(bb_classes)

bucket_min = [0, 10, 100, 1000]
bucket_max = [3600, 3600, 3600, 3600]
num_buckets = len(bucket_min)
assert(len(bucket_max) == num_buckets)
bb_buckets = ['[' + str(bucket_min[j]) + ',' + str(bucket_max[j]) + ')' for j in range(num_buckets)]
# bucket_names = [classes[i] + ' [' + str(bucket_min[j]) + ',' + str(bucket_max[j]) + ')' for i in range(num_classes) for j in range(num_buckets)]
# display(bucket_names)

bb_metrics = ['Gmean', 'Wins']

# Column names
time_col_header = 'Time (s)'
node_col_header = 'Nodes (#)'


## Function `create_avg_bb_by_depth_df` that can be reused

In [121]:
import warnings
warnings.filterwarnings('ignore')

def create_avg_bb_by_depth_df(inst_set, DEBUG = False):
  bb_classes_by_depth = [str(t) + ' leaves' for t in sizes]
  num_bb_classes_by_depth = len(bb_classes_by_depth)

  bb_buckets_by_depth = bb_buckets
  bb_metrics_by_depth = bb_metrics

  cols_time_by_depth       = time_cols_long #[ref_time_col, refv_time_col, refv_w_cut_time_col]
  shortcols_time_by_depth  = [map_cols_to_short_time[col] for col in cols_time_by_depth]
  cols_nodes_by_depth      = [ref_nodes_col, refv_nodes_col]
  shortcols_nodes_by_depth = [map_cols_to_short_nodes[col] for col in cols_nodes_by_depth]

  avg_bb_cols_by_depth = pd.MultiIndex.from_arrays(
      [
        [time_col_header]*len(shortcols_time_by_depth) + 
        [node_col_header]*len(shortcols_nodes_by_depth), 
        shortcols_time_by_depth + shortcols_nodes_by_depth
      ],
      names = ['criterion', 'type'])

  bb_row_names_by_depth = pd.MultiIndex.from_product(
      [bb_classes_by_depth, bb_buckets_by_depth, bb_metrics_by_depth],
      names=['class', 'bucket', 'metric'])

  tmp_avg_bb_by_depth_df = pd.DataFrame(
      columns = avg_bb_cols_by_depth,
      index = bb_row_names_by_depth,
      dtype = float
  )

  # Make all columns "object" type to allow for integer values
  tmp_avg_bb_by_depth_df.loc[:,(time_col_header,shortcols_time_by_depth)] = tmp_avg_bb_by_depth_df.loc[:,(time_col_header,shortcols_time_by_depth)].astype(object)
  tmp_avg_bb_by_depth_df.loc[:,(node_col_header,shortcols_nodes_by_depth)] = tmp_avg_bb_by_depth_df.loc[:,(node_col_header,shortcols_nodes_by_depth)].astype(object)

  num_inst_by_depth = np.zeros(len(tmp_avg_bb_by_depth_df),dtype = np.int64)
  row_ind = 0

  # Calculate stats for instances by depth
  cols = cols_time_by_depth + cols_nodes_by_depth + [col_num_vpc, ref_timeout_col, refv_timeout_col]
  cols = list(set(cols)) # remove duplicates
  curr_df = df.loc[inst_set,cols]
  curr_df = curr_df[curr_df.index.get_level_values(1) > 0]

  for curr_size_ind in range(0,len(bb_classes_by_depth)):
      # print("{}".format(bb_classes_by_depth[curr_size_ind]))
      curr_by_depth_df = curr_df[curr_df.index.get_level_values(1) == sizes[curr_size_ind]] # take only chosen depth
      
      # Take only instances in which num vpcs > 0
      curr_by_depth_df = curr_by_depth_df[curr_by_depth_df[col_num_vpc] > 0]

      if DEBUG:
          curr_by_depth_timeouts_df = curr_by_depth_df[
              (curr_by_depth_df[ref_timeout_col] >= MAX_TIME)
              & (curr_by_depth_df[refv_timeout_col] >= MAX_TIME)
          ]
          if (len(curr_by_depth_timeouts_df) > 0):
            display(curr_by_depth_timeouts_df)

      # Remove instances in which both Gurobi and VPCs timed out
      # (If inst_set = selected_time_instances, then this is not necessary, as we already filtered out instances that timed out for both solvers)
      curr_by_depth_df = curr_by_depth_df[
          (curr_by_depth_df[ref_timeout_col] < MAX_TIME) 
          | (curr_by_depth_df[refv_timeout_col] < MAX_TIME)
      ]

      if DEBUG:
          print("Num instances selected for depth {:d} = {:d}".format(sizes[curr_size_ind],len(curr_by_depth_df)))
          print("Num instances solved w/i timelimit for Gur: {:d}".format(len(curr_by_depth_df[(curr_by_depth_df[ref_timeout_col] < MAX_TIME)])))
          print("Num instances solved w/i timelimit for Gur+V: {:d}".format(len(curr_by_depth_df[(curr_by_depth_df[refv_timeout_col] < MAX_TIME)])))

      for bucket_ind in range(num_buckets):
          # Decide which instances to include in this bucket
          # This will be the instances for which the average solver time is at least the bucket minimum, for all solvers
          # TODO: is it fair to use "&" or "|" here?
          curr_by_depth_df = curr_by_depth_df[ 
              (curr_by_depth_df[ref_time_col] >= bucket_min[bucket_ind])
              & (curr_by_depth_df[refv_time_col] >= bucket_min[bucket_ind])
            ]
          
          bb_metric_ind = 0
          # For each column, compute the geometric mean of the values in the column
          # Use SHIFT_TIME for the first two time columns, and maybe use a different shift for generation time
          for col_ind in range(len(cols_time_by_depth)):
              col = cols_time_by_depth[col_ind]
              SHIFT = SHIFT_TIME if col_ind < 2 else SHIFT_GEN_TIME
              tmp_avg_bb_by_depth_df.loc[
                  (bb_classes_by_depth[curr_size_ind], bb_buckets_by_depth[bucket_ind], bb_metrics_by_depth[bb_metric_ind]),
                  (time_col_header,shortcols_time_by_depth[col_ind])] = \
              geometric_mean(curr_by_depth_df[col] + SHIFT) - SHIFT

          # display(avg_bb_by_depth_df.loc[
          #         (bb_classes_by_depth[curr_size_ind], bb_buckets_by_depth[bucket_ind], bb_metrics_by_depth[0]),
          #         (time_col_header,shortcols_time_by_depth)].head())
          tmp_avg_bb_by_depth_df.loc[
                  (bb_classes_by_depth[curr_size_ind], bb_buckets_by_depth[bucket_ind], bb_metrics_by_depth[bb_metric_ind]),
                  (node_col_header,shortcols_nodes_by_depth)] = \
              [geometric_mean(curr_by_depth_df[col] + SHIFT_NODES) - SHIFT_NODES for col in cols_nodes_by_depth]
          
          # print("row {:d}: {:d}".format(row_ind,len(curr_by_depth_df)))

          num_inst_by_depth[row_ind:row_ind+len(bb_metrics_by_depth)] = len(bb_metrics_by_depth)*[len(curr_by_depth_df)]
          row_ind += len(bb_metrics_by_depth)

          ## Update wins rows
          # A win in terms of time is counted when the ``Gur'' baseline seconds taken 
          # is at least 10\% slower, to account for some variability in runtimes.
          bb_metric_ind = 1
          refcol = ref_time_col
          tmp_avg_bb_by_depth_df.loc[
                  (bb_classes_by_depth[curr_size_ind], bb_buckets_by_depth[bucket_ind], bb_metrics_by_depth[bb_metric_ind]),
                  (time_col_header,shortcols_time_by_depth)] = \
              [ int(sum( curr_by_depth_df[refv_time_col] > WIN_BY_TIME_FACTOR * curr_by_depth_df[ref_time_col] )),
                int(sum( curr_by_depth_df[ref_time_col] > WIN_BY_TIME_FACTOR * curr_by_depth_df[refv_time_col] )),
                int(sum( curr_by_depth_df[ref_time_col] > WIN_BY_TIME_FACTOR * (curr_by_depth_df[refv_time_col] + curr_by_depth_df[col_vpc_gen_time]) )),
              ]

          # A win in terms of nodes is when the ``Gur'' baseline number of nodes is higher.
          refcol = ref_nodes_col
          tmp_avg_bb_by_depth_df.loc[
                  (bb_classes_by_depth[curr_size_ind], bb_buckets_by_depth[bucket_ind], bb_metrics_by_depth[bb_metric_ind]),
                  (node_col_header,shortcols_nodes_by_depth)] = \
              [ int(sum(curr_by_depth_df[refv_nodes_col] > curr_by_depth_df[ref_nodes_col])) ] + \
              [ int(sum(curr_by_depth_df[refcol] > curr_by_depth_df[col])) for col in cols_nodes_by_depth[1:] ]

  tmp_avg_bb_by_depth_df[inst_col_name] = num_inst_by_depth

  # for i in range(num_buckets):
  #     curr_df = curr_df[curr_df[gur1time_col] > bucket_min[i]]
      
  display(tmp_avg_bb_by_depth_df.loc[(bb_classes_by_depth, bb_buckets_by_depth, bb_metrics_by_depth),:])

  return tmp_avg_bb_by_depth_df


## Table 3: "all": `avg_bb_df_by_depth`

In [127]:
inst_set = selected_time_instances # will be filtered down

avg_bb_by_depth_df = create_avg_bb_by_depth_df(inst_set)

criterion                        Time (s)                          \
type                                  Gur            V        Gen   
class     bucket      metric                                        
2 leaves  [0,3600)    Gmean      9.264507     9.505446   0.561007   
                      Wins             64           52         28   
          [10,3600)   Gmean    129.679451    139.34905    1.18823   
                      Wins             33           22         16   
          [100,3600)  Gmean    503.496056   534.573401   1.003378   
                      Wins             13           10         10   
          [1000,3600) Gmean   1974.557425  1894.638111   0.296151   
                      Wins              4            4          4   
4 leaves  [0,3600)    Gmean      8.813739     8.802836   1.188708   
                      Wins             70           55         26   
          [10,3600)   Gmean     117.38084   115.831482   2.375498   
                      Wins             26           29         21   
          [100,3600)  Gmean    466.634515   458.663953   2.557509   
                      Wins             12           12         10   
          [1000,3600) Gmean   1802.023961  1875.014178   0.962807   
                      Wins              4            4          4   
8 leaves  [0,3600)    Gmean      9.186959     9.681584   2.211241   
                      Wins             89           46         20   
          [10,3600)   Gmean    139.393297   153.050294   4.716767   
                      Wins             42           24         16   
          [100,3600)  Gmean    510.319081    557.74757   5.163097   
                      Wins             24           13         11   
          [1000,3600) Gmean    2046.17191  1965.601753   2.644587   
                      Wins              7            6          6   
16 leaves [0,3600)    Gmean      9.216628     9.462807   3.909299   
                      Wins             78           63         23   
          [10,3600)   Gmean     139.44426   147.970481   6.658429   
                      Wins             30           28         16   
          [100,3600)  Gmean    535.232775   562.072782   7.832284   
                      Wins             15           14         11   
          [1000,3600) Gmean   1891.260914  1824.106626   4.820112   
                      Wins              5            7          7   
32 leaves [0,3600)    Gmean      8.354017     8.660765   7.333595   
                      Wins             85           53         16   
          [10,3600)   Gmean    128.969218   136.264026  13.085646   
                      Wins             30           26         16   
          [100,3600)  Gmean    519.646221   567.703026   12.13202   
                      Wins             14           10          8   
          [1000,3600) Gmean    1876.71803    1940.7952  10.713437   
                      Wins              6            5          5   
64 leaves [0,3600)    Gmean      7.878246     8.349617  10.393096   
                      Wins             90           47         10   
          [10,3600)   Gmean    135.620735   150.597977  18.134402   
                      Wins             35           18         10   
          [100,3600)  Gmean    483.705222   558.892444  22.095454   
                      Wins             14           10          7   
          [1000,3600) Gmean   1804.236845  2254.478262  20.507652   
                      Wins              6            2          2   

criterion                         Nodes (#)                # inst  
type                                    Gur              V         
class     bucket      metric                                       
2 leaves  [0,3600)    Gmean     6309.540478    6452.580675    261  
                      Wins              102            135    261  
          [10,3600)   Gmean    59630.814167   64032.410898    102  
                      Wins               44             58    102  
    

## Table 3 "all 6": `all6_avg_bb_by_depth` for `all6_instances`

In [128]:
inst_set = all6_instances

all6_avg_bb_by_depth_df = create_avg_bb_by_depth_df(inst_set, True)

Num instances selected for depth 2 = 209
Num instances solved w/i timelimit for Gur: 209
Num instances solved w/i timelimit for Gur+V: 209
Num instances selected for depth 4 = 209
Num instances solved w/i timelimit for Gur: 209
Num instances solved w/i timelimit for Gur+V: 209
Num instances selected for depth 8 = 209
Num instances solved w/i timelimit for Gur: 209
Num instances solved w/i timelimit for Gur+V: 209
Num instances selected for depth 16 = 209
Num instances solved w/i timelimit for Gur: 209
Num instances solved w/i timelimit for Gur+V: 209
Num instances selected for depth 32 = 209
Num instances solved w/i timelimit for Gur: 209
Num instances solved w/i timelimit for Gur+V: 209
Num instances selected for depth 64 = 209
Num instances solved w/i timelimit for Gur: 209
Num instances solved w/i timelimit for Gur+V: 209


criterion                        Time (s)                          \
type                                  Gur            V        Gen   
class     bucket      metric                                        
2 leaves  [0,3600)    Gmean      6.297773     6.469383   0.219252   
                      Wins             46           37         21   
          [10,3600)   Gmean    108.560043    119.27533   0.338146   
                      Wins             21           12         11   
          [100,3600)  Gmean    407.050282   445.654589   0.412194   
                      Wins              8            5          5   
          [1000,3600) Gmean   1590.121224  1601.868666    0.42099   
                      Wins              1            1          1   
4 leaves  [0,3600)    Gmean      6.316865     6.333297   0.473271   
                      Wins             49           40         19   
          [10,3600)   Gmean    102.484323   101.939364    0.74797   
                      Wins             15           19         15   
          [100,3600)  Gmean    417.397497   419.124404   0.921479   
                      Wins              7            6          5   
          [1000,3600) Gmean   1522.793553  1536.088601    1.23028   
                      Wins              1            2          2   
8 leaves  [0,3600)    Gmean      6.320597     6.656518   1.043478   
                      Wins             67           36         17   
          [10,3600)   Gmean    109.558889    123.29942   1.809167   
                      Wins             29           16         13   
          [100,3600)  Gmean    380.823491   429.471294    1.98445   
                      Wins             15            8          8   
          [1000,3600) Gmean    1646.14111  1579.110779   3.693587   
                      Wins              2            3          3   
16 leaves [0,3600)    Gmean      6.278304     6.491745   2.214202   
                      Wins             61           52         19   
          [10,3600)   Gmean    104.285737    113.49211   3.889016   
                      Wins             22           21         12   
          [100,3600)  Gmean    433.019671   473.996929   5.520514   
                      Wins              9            9          7   
          [1000,3600) Gmean   1521.685912  1555.355801   7.400474   
                      Wins              2            3          3   
32 leaves [0,3600)    Gmean      6.301682     6.643639   4.760972   
                      Wins             71           41         12   
          [10,3600)   Gmean    108.268834   118.875318   8.130522   
                      Wins             25           18         12   
          [100,3600)  Gmean    384.339077   434.342546  10.147717   
                      Wins             11            6          5   
          [1000,3600) Gmean   1648.477334  1685.631706  11.590214   
                      Wins              3            3          3   
64 leaves [0,3600)    Gmean      6.269067     6.532078  10.182545   
                      Wins             84           41          7   
          [10,3600)   Gmean    107.530414   114.463538  17.987039   
                      Wins             30           14          7   
          [100,3600)  Gmean    403.590283    423.71576  23.633619   
                      Wins             10            7          5   
          [1000,3600) Gmean   1619.416872  1801.370596  32.046766   
                      Wins              3            1          1   

criterion                         Nodes (#)                # inst  
type                                    Gur              V         
class     bucket      metric                                       
2 leaves  [0,3600)    Gmean     5344.393682    5453.409548    209  
                      Wins               79            107    209  
          [10,3600)   Gmean    72659.192537   79023.498582     69  
                      Wins               31             38     69  
    

## Table 3: "binary": `binary_avg_bb_df_by_depth`

In [117]:
inst_set = binary_x_time_instances # will be filtered down

binary_avg_bb_by_depth_df = create_avg_bb_by_depth_df(inst_set)

# bb_classes_by_depth = [str(t) + ' leaves' for t in sizes]
# num_bb_classes_by_depth = len(bb_classes_by_depth)

# bb_buckets_by_depth = bb_buckets
# bb_metrics_by_depth = bb_metrics

# cols_time_by_depth       = time_cols_long #[ref_time_col, refv_time_col, refv_w_cut_time_col]
# shortcols_time_by_depth  = [map_cols_to_short_time[col] for col in cols_time_by_depth]
# cols_nodes_by_depth      = [ref_nodes_col, refv_nodes_col]
# shortcols_nodes_by_depth = [map_cols_to_short_nodes[col] for col in cols_nodes_by_depth]

# avg_bb_cols_by_depth = pd.MultiIndex.from_arrays(
#     [
#       [time_col_header]*len(shortcols_time_by_depth) + 
#       [node_col_header]*len(shortcols_nodes_by_depth), 
#       shortcols_time_by_depth + shortcols_nodes_by_depth
#     ],
#     names = ['criterion', 'type'])

# bb_row_names_by_depth = pd.MultiIndex.from_product(
#     [bb_classes_by_depth, bb_buckets_by_depth, bb_metrics_by_depth],
#     names=['class', 'bucket', 'metric'])

# binary_avg_bb_by_depth_df = pd.DataFrame(
#     columns = avg_bb_cols_by_depth,
#     index = bb_row_names_by_depth,
#     dtype = float
# )

# # Make all columns "object" type to allow for integer values
# binary_avg_bb_by_depth_df.loc[:,(time_col_header,shortcols_time_by_depth)] = avg_bb_by_depth_df.loc[:,(time_col_header,shortcols_time_by_depth)].astype(object)
# binary_avg_bb_by_depth_df.loc[:,(node_col_header,shortcols_nodes_by_depth)] = avg_bb_by_depth_df.loc[:,(node_col_header,shortcols_nodes_by_depth)].astype(object)

# num_inst_by_depth = np.zeros(len(avg_bb_by_depth_df),dtype = np.int64)
# row_ind = 0

# # Calculate stats for instances by depth
# cols = cols_time_by_depth + cols_nodes_by_depth + [col_num_vpc, ref_timeout_col, refv_timeout_col]
# cols = list(set(cols)) # remove duplicates
# curr_df = df.loc[inst_set,cols]
# curr_df = curr_df[curr_df.index.get_level_values(1) > 0]

# for curr_size_ind in range(0,len(bb_classes_by_depth)):
#     # print("{}".format(bb_classes_by_depth[curr_size_ind]))
#     curr_by_depth_df = curr_df[curr_df.index.get_level_values(1) == sizes[curr_size_ind]] # take only chosen depth
    
#     # Take only instances in which num vpcs > 0
#     curr_by_depth_df = curr_by_depth_df[curr_by_depth_df[col_num_vpc] > 0]

#     # Remove instances in which both Gurobi and VPCs timed out
#     curr_by_depth_df = curr_by_depth_df[
#         (curr_by_depth_df[ref_timeout_col] < MAX_TIME)
#         | (curr_by_depth_df[refv_timeout_col] < MAX_TIME)
#     ]

#     for bucket_ind in range(num_buckets):
#         # Decide which instances to include in this bucket
#         # This will be the instances for which the average solver time is at least the bucket minimum, for all solvers
#         # TODO: is it fair to use "&" or "|" here?
#         curr_by_depth_df = curr_by_depth_df[ 
#             (curr_by_depth_df[ref_time_col] >= bucket_min[bucket_ind])
#             & (curr_by_depth_df[refv_time_col] >= bucket_min[bucket_ind])
#           ]
        
#         bb_metric_ind = 0
#         # For each column, compute the geometric mean of the values in the column
#         # Use SHIFT_TIME for the first two time columns, and maybe use a different shift for generation time
#         for col_ind in range(len(cols_time_by_depth)):
#             col = cols_time_by_depth[col_ind]
#             SHIFT = SHIFT_TIME if col_ind < 2 else SHIFT_GEN_TIME
#             binary_avg_bb_by_depth_df.loc[
#                 (bb_classes_by_depth[curr_size_ind], bb_buckets_by_depth[bucket_ind], bb_metrics_by_depth[bb_metric_ind]),
#                 (time_col_header,shortcols_time_by_depth[col_ind])] = \
#             geometric_mean(curr_by_depth_df[col] + SHIFT) - SHIFT

#         # display(avg_bb_by_depth_df.loc[
#         #         (bb_classes_by_depth[curr_size_ind], bb_buckets_by_depth[bucket_ind], bb_metrics_by_depth[0]),
#         #         (time_col_header,shortcols_time_by_depth)].head())
#         binary_avg_bb_by_depth_df.loc[
#                 (bb_classes_by_depth[curr_size_ind], bb_buckets_by_depth[bucket_ind], bb_metrics_by_depth[bb_metric_ind]),
#                 (node_col_header,shortcols_nodes_by_depth)] = \
#             [geometric_mean(curr_by_depth_df[col] + SHIFT_NODES) - SHIFT_NODES for col in cols_nodes_by_depth]
        
#         # print("row {:d}: {:d}".format(row_ind,len(curr_by_depth_df)))

#         num_inst_by_depth[row_ind:row_ind+len(bb_metrics_by_depth)] = len(bb_metrics_by_depth)*[len(curr_by_depth_df)]
#         row_ind += len(bb_metrics_by_depth)

#         ## Update wins rows
#         # A win in terms of time is counted when the ``Gur'' baseline seconds taken 
#         # is at least 10\% slower, to account for some variability in runtimes.
#         bb_metric_ind = 1
#         refcol = ref_time_col
#         binary_avg_bb_by_depth_df.loc[
#                 (bb_classes_by_depth[curr_size_ind], bb_buckets_by_depth[bucket_ind], bb_metrics_by_depth[bb_metric_ind]),
#                 (time_col_header,shortcols_time_by_depth)] = \
#             [ int(sum( curr_by_depth_df[refv_time_col] > WIN_BY_TIME_FACTOR * curr_by_depth_df[ref_time_col] )),
#               int(sum( curr_by_depth_df[ref_time_col] > WIN_BY_TIME_FACTOR * curr_by_depth_df[refv_time_col] )),
#               int(sum( curr_by_depth_df[ref_time_col] > WIN_BY_TIME_FACTOR * (curr_by_depth_df[refv_time_col] + curr_by_depth_df[col_vpc_gen_time]) )),
#             ]

#         # A win in terms of nodes is when the ``Gur'' baseline number of nodes is higher.
#         refcol = ref_nodes_col
#         binary_avg_bb_by_depth_df.loc[
#                 (bb_classes_by_depth[curr_size_ind], bb_buckets_by_depth[bucket_ind], bb_metrics_by_depth[bb_metric_ind]),
#                 (node_col_header,shortcols_nodes_by_depth)] = \
#             [ int(sum(curr_by_depth_df[refv_nodes_col] > curr_by_depth_df[ref_nodes_col])) ] + \
#             [ int(sum(curr_by_depth_df[refcol] > curr_by_depth_df[col])) for col in cols_nodes_by_depth[1:] ]

# binary_avg_bb_by_depth_df[inst_col_name] = num_inst_by_depth

# # for i in range(num_buckets):
# #     curr_df = curr_df[curr_df[gur1time_col] > bucket_min[i]]
    
# display(binary_avg_bb_by_depth_df.loc[(bb_classes_by_depth, bb_buckets_by_depth, bb_metrics_by_depth),:])

criterion                        Time (s)                          \
type                                  Gur            V        Gen   
class     bucket      metric                                        
2 leaves  [0,3600)    Gmean     11.218866    11.528171   0.643067   
                      Wins             13           11          5   
          [10,3600)   Gmean    234.051387   249.900766   1.041755   
                      Wins              6            4          3   
          [100,3600)  Gmean    603.132438   666.291218   1.162814   
                      Wins              4            2          2   
          [1000,3600) Gmean   1312.527411  1402.206184   0.517974   
                      Wins              2            0          0   
4 leaves  [0,3600)    Gmean     10.226922     10.19613   2.052842   
                      Wins             13           11          6   
          [10,3600)   Gmean    182.808155   180.602501   4.467153   
                      Wins              4            5          4   
          [100,3600)  Gmean    514.868546    487.50221   6.692844   
                      Wins              3            3          3   
          [1000,3600) Gmean   1265.346935  1297.537018   1.869564   
                      Wins              0            0          0   
8 leaves  [0,3600)    Gmean     10.641315    11.444254   6.472492   
                      Wins             19            9          1   
          [10,3600)   Gmean     201.39343   233.219973  21.681304   
                      Wins             10            3          1   
          [100,3600)  Gmean    531.096053   608.111793  32.487448   
                      Wins              8            2          1   
          [1000,3600) Gmean   1290.075683   1465.27392   6.483211   
                      Wins              3            0          0   
16 leaves [0,3600)    Gmean       9.72424    10.100529  11.825484   
                      Wins             17           10          1   
          [10,3600)   Gmean    167.332762   187.834268    26.7607   
                      Wins              8            5          1   
          [100,3600)  Gmean    494.466187   541.312715  35.302477   
                      Wins              6            2          0   
          [1000,3600) Gmean   1302.872995  1488.684953  12.415964   
                      Wins              3            0          0   
32 leaves [0,3600)    Gmean      8.223344     8.397861  20.541768   
                      Wins             16           13          2   
          [10,3600)   Gmean    166.915782   181.228995  51.714362   
                      Wins              7            5          2   
          [100,3600)  Gmean    747.735576   775.148569  36.307427   
                      Wins              4            2          1   
          [1000,3600) Gmean   1304.384288  1557.631562  17.507406   
                      Wins              3            0          0   
64 leaves [0,3600)    Gmean      9.519942    10.877538  21.629528   
                      Wins             12           12          1   
          [10,3600)   Gmean    238.180556     321.1035  52.988064   
                      Wins              6            4          1   
          [100,3600)  Gmean    752.918306  1136.039732  91.395504   
                      Wins              4            1          0   
          [1000,3600) Gmean   1353.786355  1602.181393  54.512677   
                      Wins              2            0          0   

criterion                         Nodes (#)                # inst  
type                                    Gur              V         
class     bucket      metric                                       
2 leaves  [0,3600)    Gmean     6716.696632    6928.057352     50  
                      Wins               22             20     50  
          [10,3600)   Gmean    75274.305875   82353.955463     19  
                      Wins                9             10     19  
    

# Section 4: "Combined" results

## Select "Combined" instances

In [54]:
# Create df_nodes dataframe, in which each row is an instance, and each column is a solver (with or w/o cuts, and with a fixed random seed)
# These will be pulled from 'ALL REF TIME' and 'ALL REF+V TIME' columns
inst_set = list(instances)
# inst_set = ['lotsize_presolved']
PRINT_SKIP_REASON = False

disjset_num_gap_errors = 0

selected_disjset_gap_instances_dict = {} # dictionary of (original index, instance)
selected_disjset_time_instances_dict = {} # dictionary of (original index, instance)
for i, inst in enumerate(inst_set):
  print("{}/{}".format(i+1,len(inst_set)), end='\r', flush=True)
  skip_instance = False

  # # Check that -1 depth exists
  # if not (-1 in df.loc[inst].index):
  #   # if PRINT_SKIP_REASON:
  #   print("Skipping instance {:d} -- {}: no -1 depth.".format(
  #       i, inst
  #   ))
  #   skip_instance = True
  #   continue
  if not inst in df_disjset.index:
    if PRINT_SKIP_REASON:
      print("Skipping instance {:d} -- {}: no disjset results.".format(
          i, inst
      ))
    skip_instance = True
    continue
  
  curr_df = df_disjset.loc[(inst,-1)]

  # Ensure nrows and ncols is not too many
  nrows = curr_df[col_num_rows]
  ncols = curr_df[col_num_cols]
  if (nrows > MAX_ROWS) or (ncols > MAX_COLS):
    if PRINT_SKIP_REASON:
        print("Skipping instance {:d} -- {}: nrows = {:d} > {:d} or ncols = {:d} > {:d}.".format(
                i, inst, nrows, ncols, MAX_ROWS, MAX_COLS))
    skip_instance = True

  # Ensure IP objective value is known
  ip_obj = np.float64(df_ipopt.loc[inst,col_ip_obj])
  if not isinstance(ip_obj,float):
    if PRINT_SKIP_REASON:
        print(
            "Skipping instance {:d} -- {}: IP objective value ({}) is not detected to be a float value.".format(
            i, inst, ip_obj))
    skip_instance = True

  check_ip_obj = curr_df[col_ip_obj]
  if not is_val(ip_obj,check_ip_obj):
    print("*** ERROR: Instance {:d} -- {}: IP objective value ({}) does not match value in dataframe ({}). REPLACING WITH KNOWN VALUE!".format(
        i, inst, ip_obj, check_ip_obj))
    df.loc[(inst,-1),col_ip_obj] = ip_obj
    df_disjset.loc[(inst,-1),col_ip_obj] = ip_obj
    # skip_instance = True

  # Check that LP opt < IP opt
  lp_obj = np.float64(df_preprocess.loc[remove_presolved_from_name(inst),col_cleaned_lp_obj])
  YES_GAP = (ip_obj - lp_obj) >= EPS
  if not YES_GAP:
    print("*** ERROR: Instance {:d} -- {}: not YES GAP (lp = {:.10f}; ip = {:.10f}, diff = {:.2f})".format(i, inst, lp_obj, ip_obj, ip_obj-lp_obj))
    skip_instance = True
    disjset_num_gap_errors += 1

  # Check that ExitReason != OPTIMAL_SOLUTION_FOUND
  exitreason = curr_df['ExitReason']
  if exitreason == 'OPTIMAL_SOLUTION_FOUND':
    if PRINT_SKIP_REASON:
      print("Skipping instance {:d} -- {}: optimal IP solution found.".format(
          i, inst
      ))
    skip_instance = True

  # Check that VPCs were generated
  num_vpc = curr_df[col_num_vpc]
  if num_vpc == 0:
    if PRINT_SKIP_REASON:
      print("Skipping instance {:d} -- {}: no VPCs generated.".format(
          i, inst
      ))
    skip_instance = True

  if not skip_instance:
    selected_disjset_gap_instances_dict[inst] = i

    # If either ref or refv times are < MAX_TIME - EPS, include in nodes experiments
    ref_time = curr_df[ref_time_col]
    refv_time = curr_df[refv_time_col]
    if (ref_time < MAX_TIME - EPS) or (refv_time < MAX_TIME - EPS):
      selected_disjset_time_instances_dict[inst] = i

selected_disjset_gap_instances = list(selected_disjset_gap_instances_dict.keys())
print("Total number of errors: {}".format(disjset_num_gap_errors))
print("Num instances selected for disjset gap closed results = {:d}".format(len(selected_disjset_gap_instances)))

selected_disjset_time_instances = list(selected_disjset_time_instances_dict.keys())
print("Num instances selected for disjset time results = {:d}".format(len(selected_disjset_time_instances)))

*** ERROR: Instance 129 -- maxgasflow_presolved: IP objective value (-44565819.158) does not match value in dataframe (-44565819158.0). REPLACING WITH KNOWN VALUE!
*** ERROR: Instance 209 -- neos-3048764-nadi_presolved: IP objective value (-3883988306.366686) does not match value in dataframe (-3883988306.0). REPLACING WITH KNOWN VALUE!
*** ERROR: Instance 228 -- neos-3661949-lesse_presolved: not YES GAP (lp = 689000000.0000000000; ip = 688995225.0000000000, diff = -4775.00)
Total number of errors: 1
Num instances selected for disjset gap closed results = 333
Num instances selected for disjset time results = 288


### List instances that are in exactly one of the two sets `selected_gap_instances` and `selected_disjset_gap_instances`

In [55]:
# Symmetric difference between two sets selected_gap_instances and selected_disjset_gap_instances
print("Instances in selected_disjset_gap_instances that are not in selected_gap_instances:")
newly_selected_instances = [inst for inst in selected_disjset_gap_instances if inst not in selected_gap_instances]
print(newly_selected_instances)

# Remove newly selected instances
for inst in newly_selected_instances:
  selected_disjset_gap_instances.remove(inst)

print("Instances in selected_gap_instances that are not in selected_disjset_gap_instances:")
print([inst for inst in selected_gap_instances if inst not in selected_disjset_gap_instances])

Instances in selected_disjset_gap_instances that are not in selected_gap_instances:
['neos-3214367-sovi_presolved']
Instances in selected_gap_instances that are not in selected_disjset_gap_instances:
[]


## "Combined" gap summary

### `disjset_gap_df`: Analyze disjset gap closed

In [56]:
# Create subset of dataframe relevant to gap closed
disjset_gap_df = df.loc[selected_disjset_gap_instances, 
                [
                    col_num_disj_terms,
                    col_num_rows,
                    col_num_cols
                ]
                +
                obj_val_col_list
                +
                [
                    col_num_gmic,
                    col_num_vpc,
                    col_num_obj,
                    col_exit_reason
                ]
               ]

# Take only -1 depth
disjset_gap_df = disjset_gap_df.xs(-1, level='disj_terms')

# Calculate some missing % gap closed columns
# gap closed = 100 * (post_cut_opt_val - lp_opt_val) / (ip_opt_val - lp_opt_val)
for cut_type in reg_cut_type_long_list:
    col = cut_type + ' ' + obj_stub
    if col not in df.columns:
        if cut_type == 'MAX(GMIC,VPC)':
            # Add max(G,V) column
            disjset_gap_df[cut_type + ' ' + pct_gap_closed_stub] = \
                np.maximum(
                    disjset_gap_df['GMIC' + ' ' + pct_gap_closed_stub],
                    disjset_gap_df[ 'VPC' + ' ' + pct_gap_closed_stub]
                )
        continue
    disjset_gap_df[cut_type + ' ' + pct_gap_closed_stub] = calc_gap_closed(disjset_gap_df, col)

# Compare against reference solver
for stat_type in solver_stat_list[:3]:
    for solver_type in solver_stubs:
        for cut_type in solver_cut_type_stubs:
            col = stat_type + ' ' + solver_type + ' ' + cut_type
            disjset_gap_df[col + ' ' + pct_gap_closed_stub] = calc_gap_closed(disjset_gap_df, col)

# Rename each column with its shortname
disjset_gap_df.rename(columns=map_cols_to_short_gap,inplace=True)

display(disjset_gap_df.loc[['bm23_presolved','maxgasflow_presolved']][gap_cols_short])
display(disjset_gap_df.loc[("bm23_presolved")])

,G,R,DB,V,"max(G,V)",V+G,GurF,V+GurF,GurL,V+GurL
INSTANCE,,,,,,,,,,
bm23_presolved,16.809643,0.000000e+00,71.466577,69.698381,69.698381,69.698381,21.160060,66.200094,39.057502,71.291404
maxgasflow_presolved,22.478700,-3.223163e-11,0.015531,0.006570,22.478700,22.478957,22.500189,22.522284,28.774516,28.025236


NUM DISJ TERMS                                    -1
ROWS                                              20
COLS                                              27
LP OBJ                                     20.570922
IP OBJ                                          34.0
WORST DISJ OBJ                             56.839578
GMIC OBJ                                   22.828302
ROOT OBJ                                   20.570922
BEST DISJ OBJ                              30.168224
VPC OBJ                                    29.930772
VPC+GMIC OBJ                               29.930772
FIRST REF FIRST_CUT_PASS                   23.373228
FIRST REF+V FIRST_CUT_PASS                 29.460984
AVG REF FIRST_CUT_PASS                     23.412523
AVG REF+V FIRST_CUT_PASS                   29.460984
BEST REF FIRST_CUT_PASS                    23.545818
BEST REF+V FIRST_CUT_PASS                  29.460984
FIRST REF LAST_CUT_PASS                     25.60614
FIRST REF+V LAST_CUT_PASS                  30.

In [57]:
disjset_gap_df.to_csv('disjset_gap.csv')

### `disjset_avg_gap_df`: summary of results with disjset instances

In [58]:
## TABLE 1.2: average percent gap closed across different combinations of cuts
## Create disjset_avg_gap_df = average gap closed across instances
idx = pd.MultiIndex.from_product(
    [ [all_set_name, good_vpc_set_name, binary_set_name], [avg_row_name, wins_row_name] ],
    names = ['Set', '']
)
    
ncols = len(gap_cols_short)
nrows = len(idx)

col = disjset_gap_df['V'].astype(float)
disjset_good_vpc_df = disjset_gap_df[col >= 10.]
disjset_binary_instances_df = disjset_gap_df.loc[ [inst for inst in pure_binary_instances if inst in selected_disjset_gap_instances ] ]

data = np.zeros((nrows, ncols), dtype=float)
data[0,:] = [disjset_gap_df[col].mean() for col in gap_cols_short]
data[2,:] = [disjset_good_vpc_df[col].mean() for col in gap_cols_short]
data[4,:] = [disjset_binary_instances_df[col].mean() for col in gap_cols_short]

# display(best_gap_df.head())
disjset_avg_gap_df = pd.DataFrame(
    data,
    columns = gap_cols_short,
    index = idx,
    dtype = object
)

inst_col_name = '# inst'
disjset_avg_gap_df[inst_col_name] = [len(disjset_gap_df), 0, len(disjset_good_vpc_df), 0, len(disjset_binary_instances_df), 0]

disjset_avg_gap_df.iloc[1] = ["" for i in range(ncols+1)]
disjset_avg_gap_df.iloc[3] = ["" for i in range(ncols+1)]
disjset_avg_gap_df.iloc[5] = ["" for i in range(ncols+1)]

display(disjset_avg_gap_df)

/var/folders/_8/bgx4yr0n6f1dm56pnvsqqb1s91dzml/T/ipykernel_59358/2641934701.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  disjset_avg_gap_df.iloc[1] = ["" for i in range(ncols+1)]


G         R         DB          V   max(G,V)  \
Set                                                                        
All       Avg (\%)  16.508148  0.256744  17.795569  12.487033  23.207997   
          Wins                                                             
$\ge$10\% Avg (\%)  17.153438  0.658691   36.83311  29.684385  34.227199   
          Wins                                                             
Binary    Avg (\%)  13.641808       0.0  20.812414  17.641804   25.76521   
          Wins                                                             

                          V+G       GurF     V+GurF       GurL     V+GurL  \
Set                                                                         
All       Avg (\%)  24.123909  28.929745  34.461474  48.319147  52.074567   
          Wins                                                              
$\ge$10\% Avg (\%)  35.911157  27.582378  38.414037  47.277516   56.33864   
          Wins                                                              
Binary    Avg (\%)  26.612777  19.229524  30.563731  32.935918   42.59244   
          Wins                                                              

                   # inst  
Set                        
All       Avg (\%)    332  
          Wins             
$\ge$10\% Avg (\%)    122  
          Wins             
Binary    Avg (\%)     65  
          Wins

### `disjset_wins_df`: number of wins across methods

In [59]:
## Create num wins df
# x wins over y for an instance if x > y + EPS
#shortcols = avg_gap_df.columns[0:-1]
disjset_wins_df = pd.DataFrame(
    np.zeros((len(gap_cols_short), len(gap_cols_short)), dtype=int),
    columns = gap_cols_short,
    index = gap_cols_short,
    dtype = int,
)

WINS_EPS = GAP_DIFF_EPS

from itertools import permutations
for (ind1, ind2) in permutations(range(len(gap_cols_short)), 2):
    disjset_wins_df.at[gap_cols_short[ind1],gap_cols_short[ind2]] =\
        int(sum(disjset_gap_df[gap_cols_short[ind1]] > disjset_gap_df[gap_cols_short[ind2]] + WINS_EPS))
    disjset_wins_df.at[gap_cols_short[ind2],gap_cols_short[ind1]] =\
        int(sum(disjset_gap_df[gap_cols_short[ind2]] > disjset_gap_df[gap_cols_short[ind1]] + WINS_EPS))

# Sets we are considering
# all_set = 'Wins (All)'
# good_vpc_set = 'Wins (V ≥ 10%)'
disjset_all_set = (all_set_name,wins_row_name)
disjset_good_vpc_set = (good_vpc_set_name,wins_row_name)
disjset_binary_set = (binary_set_name,wins_row_name)

# "G" are wins relative to "V"
shortrefcol = 'V'
#refcol = 'VPC % GAP CLOSED'
#refcol = map_short_to_cols[shortrefcol]
refcol = shortrefcol
shortdestcol = 'G'
#col = 'GMIC % GAP CLOSED'
#col = map_short_to_cols[shortcol]
destcol = shortdestcol
disjset_avg_gap_df.at[disjset_all_set,shortdestcol] = disjset_wins_df.at[shortdestcol,shortrefcol]
disjset_avg_gap_df.at[disjset_good_vpc_set,shortdestcol] = sum(disjset_good_vpc_df[destcol] > disjset_good_vpc_df[refcol] + WINS_EPS)
disjset_avg_gap_df.at[disjset_binary_set,shortdestcol] = sum(disjset_binary_instances_df[destcol] > disjset_binary_instances_df[refcol] + WINS_EPS)

# "DB", "V", "V+G": wins are relative to "G"
shortrefcol = 'G'
#refcol = 'GMIC % GAP CLOSED'
#refcol = map_short_to_cols[shortrefcol]
refcol = shortrefcol
shortdestcol = 'DB'
#col = 'BEST DISJ % GAP CLOSED'
#col = map_short_to_cols[shortcol]
destcol = shortdestcol
disjset_avg_gap_df.at[disjset_all_set,shortdestcol] = disjset_wins_df.at[shortdestcol,shortrefcol]
disjset_avg_gap_df.at[disjset_good_vpc_set,shortdestcol] = sum(disjset_good_vpc_df[destcol] > disjset_good_vpc_df[refcol] + WINS_EPS)
disjset_avg_gap_df.at[disjset_binary_set,shortdestcol] = sum(disjset_binary_instances_df[destcol] > disjset_binary_instances_df[refcol] + WINS_EPS)

shortdestcol = 'V'
#col = 'VPC % GAP CLOSED'
#col = map_short_to_cols[shortcol]
destcol = shortdestcol
disjset_avg_gap_df.at[disjset_all_set,shortdestcol] = disjset_wins_df.at[shortdestcol,shortrefcol]
disjset_avg_gap_df.at[disjset_good_vpc_set,shortdestcol] = sum(disjset_good_vpc_df[destcol] > disjset_good_vpc_df[refcol] + WINS_EPS)
disjset_avg_gap_df.at[disjset_binary_set,shortdestcol] = sum(disjset_binary_instances_df[destcol] > disjset_binary_instances_df[refcol] + WINS_EPS)

shortdestcol = 'V+G'
#col = 'VPC+GMIC % GAP CLOSED'
#col = map_short_to_cols[shortcol]
destcol = shortdestcol
disjset_avg_gap_df.at[disjset_all_set,shortdestcol] = disjset_wins_df.at[shortdestcol,shortrefcol]
disjset_avg_gap_df.at[disjset_good_vpc_set,shortdestcol] = sum(disjset_good_vpc_df[destcol] > disjset_good_vpc_df[refcol] + WINS_EPS)
disjset_avg_gap_df.at[disjset_binary_set,shortdestcol] = sum(disjset_binary_instances_df[destcol] > disjset_binary_instances_df[refcol] + WINS_EPS)

# "V+GurF" are wins relative to "GurF"
shortrefcol = 'GurF'
refcol = shortrefcol
shortdestcol = 'V+GurF'
destcol = shortdestcol
#col = map_short_to_cols[shortcol]
disjset_avg_gap_df.at[disjset_all_set,shortdestcol] = disjset_wins_df.at[shortdestcol,shortrefcol]
disjset_avg_gap_df.at[disjset_good_vpc_set,shortdestcol] = sum(disjset_good_vpc_df[destcol] > disjset_good_vpc_df[refcol] + WINS_EPS)
disjset_avg_gap_df.at[disjset_binary_set,shortdestcol] = sum(disjset_binary_instances_df[destcol] > disjset_binary_instances_df[refcol] + WINS_EPS)

# "V+GurL" are wins relative to "GurL"
shortrefcol = 'GurL'
refcol = shortrefcol
shortdestcol = 'V+GurL'
destcol = shortdestcol
disjset_wins_df.at[shortdestcol,shortrefcol] = int(sum(disjset_gap_df[destcol] > disjset_gap_df[refcol] + WINS_EPS))
disjset_wins_df.at[shortrefcol,shortdestcol] = int(sum(disjset_gap_df[refcol] > disjset_gap_df[destcol] + WINS_EPS))
disjset_avg_gap_df.at[disjset_all_set,shortdestcol] = disjset_wins_df.at[shortdestcol,shortrefcol]
disjset_avg_gap_df.at[disjset_good_vpc_set,shortdestcol] = sum(disjset_good_vpc_df[destcol] > disjset_good_vpc_df[refcol] + WINS_EPS)
disjset_avg_gap_df.at[disjset_binary_set,shortdestcol] = sum(disjset_binary_instances_df[destcol] > disjset_binary_instances_df[refcol] + WINS_EPS)

# Count number of instances that have V+G > 0
shortdestcol = inst_col_name
#col = 'V+GurL'
destcol = 'V+G'
disjset_avg_gap_df.at[disjset_all_set,shortdestcol] = sum(disjset_gap_df[destcol] > WINS_EPS)
disjset_avg_gap_df.at[disjset_good_vpc_set,shortdestcol] = sum(disjset_good_vpc_df[destcol] > WINS_EPS)
disjset_avg_gap_df.at[disjset_binary_set,shortdestcol] = sum(disjset_binary_instances_df[destcol] > WINS_EPS)

display(disjset_avg_gap_df)
display(disjset_wins_df)

G         R         DB          V   max(G,V)  \
Set                                                                        
All       Avg (\%)  16.508148  0.256744  17.795569  12.487033  23.207997   
          Wins            163                  172        136              
$\ge$10\% Avg (\%)  17.153438  0.658691   36.83311  29.684385  34.227199   
          Wins             28                  100         94              
Binary    Avg (\%)  13.641808       0.0  20.812414  17.641804   25.76521   
          Wins             24                   46         35              

                          V+G       GurF     V+GurF       GurL     V+GurL  \
Set                                                                         
All       Avg (\%)  24.123909  28.929745  34.461474  48.319147  52.074567   
          Wins            228                   250                   168   
$\ge$10\% Avg (\%)  35.911157  27.582378  38.414037  47.277516   56.33864   
          Wins            118                   106                    84   
Binary    Avg (\%)  26.612777  19.229524  30.563731  32.935918   42.59244   
          Wins             53                    48                    40   

                   # inst  
Set                        
All       Avg (\%)    332  
          Wins        299  
$\ge$10\% Avg (\%)    122  
          Wins        122  
Binary    Avg (\%)     65  
          Wins         59

,G,R,DB,V,"max(G,V)",V+G,GurF,V+GurF,GurL,V+GurL
G,0,288,126,163,0,0,68,34,16,9
R,1,0,0,0,0,0,1,0,1,0
DB,172,276,0,257,159,144,116,105,75,61
V,136,265,5,0,0,0,93,66,59,14
"max(G,V)",136,299,126,163,0,0,127,90,68,20
V+G,228,299,144,247,176,0,140,105,69,25
GurF,222,301,189,213,170,161,0,0,0,0
V+GurF,265,306,196,234,203,190,250,0,51,0
GurL,288,308,237,254,239,238,294,250,0,117
V+GurL,298,312,247,293,281,276,302,292,168,0


### Update Table 2 `gap_by_size_df` with "Combined" results

In [60]:
# Create a new row with label "Combined"
all_row = pd.DataFrame(columns=gap_by_size_df.columns, index=["Combined"])

# Calculate the average values for each column from the `disjset` dataframe
all_row = disjset_avg_gap_df.loc['All',gap_by_size_df.columns][0:1].to_numpy()

# Append the new row to the `gap_by_size` dataframe
gap_by_size_df.loc['Combined'] = all_row[0]

# Display the updated `gap_by_size` dataframe
display(gap_by_size_df)


,DB,V,"max(G,V)",V+G,V+GurF,V+GurL
VPCs disabled,0.000000,0.000000,16.508148,16.508148,28.929744,48.319147
2 leaves,3.014507,2.213289,16.789746,17.206579,30.961171,49.260530
4 leaves,5.270242,3.592498,17.227532,17.785031,31.164402,49.252142
8 leaves,8.117860,4.847477,17.828119,18.533394,31.526034,49.484617
16 leaves,11.173444,6.456210,19.373155,20.029255,32.470786,50.236188
32 leaves,14.671544,8.741057,20.957474,21.773258,33.413550,51.140841
64 leaves,18.369423,10.233224,22.139001,22.887076,33.825053,51.638833
Best,18.369423,12.008265,23.007369,23.916435,34.801454,52.806027
Combined,17.795569,12.487033,23.207997,24.123909,34.461474,52.074567


## "Combined" timing summary

### `disjset_timing`

In [61]:
inst_set = selected_disjset_time_instances
NUM_SEEDS = 7
disjset_timing_cols = \
    ['ALL' + ' ' + solver_type + ' ' + time_stub for solver_type in solver_stubs] \
        + \
        time_cols_long \
        + \
        [col_num_vpc]
new_disjset_timing_cols = [ 
        [
            solver_type + ' ' + '(%d)' % seed + ' ' + time_stub
            for seed in range(1,NUM_SEEDS+1)
        ]
        for solver_type in solver_stubs
    ]
new_disjset_wins_cols = [
    "Wins ({} over {})".format(solver_stubs[0], solver_stubs[1]),
    "Wins ({} over {})".format(solver_stubs[1], solver_stubs[0])
]
disjset_timing_df = df_disjset.loc[inst_set, disjset_timing_cols]
# display(disjet_timing_df.head())

# Add columns solver_type (%d) + ' ' + time_stub for the 7 random seeds
# These will be parsed from the ALL REF TIME and ALL REF+V TIME columns,
# which contain semicolon-separated values
for seed_ind in range(NUM_SEEDS):
    for solver_ind in range(len(solver_stubs)):
        solver_type = solver_stubs[solver_ind]
        orig_col = disjset_timing_cols[solver_ind]
        new_col = new_disjset_timing_cols[solver_ind][seed_ind]
        disjset_timing_df[new_col] = disjset_timing_df[orig_col].str.split(';').str[seed_ind]
        disjset_timing_df[new_col] = disjset_timing_df[new_col].astype(float)

# Add wins columns
for solver_ind in range(len(solver_stubs)):
    other_solver_ind = 1 - solver_ind

    disjset_timing_df[new_disjset_wins_cols[solver_ind]] = sum(
        disjset_timing_df[new_disjset_timing_cols[other_solver_ind][seed_ind]]
        > 
        WIN_BY_TIME_FACTOR * disjset_timing_df[new_disjset_timing_cols[solver_ind][seed_ind]]
        for seed_ind in range(NUM_SEEDS)
    )

display(disjset_timing_df.head())

,,ALL REF TIME,ALL REF+V TIME,AVG REF TIME,AVG REF+V TIME,VPC_GEN_TIME,NUM VPC,REF (1) TIME,REF+V (1) TIME,REF (2) TIME,REF+V (2) TIME,...,REF (4) TIME,REF+V (4) TIME,REF (5) TIME,REF+V (5) TIME,REF (6) TIME,REF+V (6) TIME,REF (7) TIME,REF+V (7) TIME,Wins (REF over REF+V),Wins (REF+V over REF)
INSTANCE,disj_terms,,,,,,,,,,,,,,,,,,,,,
10teams_presolved,-1,0.303684;2.535196;0.989145;0.593072;4.379637;0...,0.626467;0.459655;3.815446;2.182773;4.006704;1...,1.829,2.004,466.44,80,0.303684,0.626467,2.535196,0.459655,...,0.593072,2.182773,4.379637,4.006704,0.914021,1.230319,3.090965,1.703434,4,2
23588_presolved,-1,0.484330;0.425422;0.400010;0.514678;0.474368;0...,0.571225;0.747228;0.528576;0.644340;0.552846;0...,0.519,0.666,40.93,386,0.484330,0.571225,0.425422,0.747228,...,0.514678,0.644340,0.474368,0.552846,0.890932,0.981250,0.445473,0.634306,7,0
30n20b8_presolved,-1,10.868823;20.671000;16.525700;15.207604;15.926...,8.117330;11.577305;11.595140;7.415187;8.538931...,15.586,10.489,779.69,381,10.868823,8.117330,20.671000,11.577305,...,15.207604,7.415187,15.926806,8.538931,19.154248,12.912346,10.747110,13.270065,1,6
50v-10_presolved,-1,3600.000062;1573.493537;3600.000113;2817.90685...,2510.967950;1966.919215;3038.140497;2182.05285...,2893.849,2333.387,8.54,72,3600.000062,2510.967950,1573.493537,1966.919215,...,2817.906855,2182.052850,1939.052187,2201.952075,3126.488219,2450.375285,3600.000120,1983.302788,2,5
a1c1s1_presolved,-1,807.966638;431.850770;453.278678;639.477005;53...,655.902565;527.671430;520.796736;729.955668;65...,531.318,605.311,11.06,22,807.966638,655.902565,431.850770,527.671430,...,639.477005,729.955668,536.304876,650.134424,416.303954,560.778815,434.046231,591.937187,6,1


In [62]:
disjset_timing_df.to_csv("disjet_timing.csv")

### `disjset_time_geomean_df`

In [63]:
# Row names
disjset_bb_classes = ['Combined', 'Combined-Binary']
disjset_bb_classes_lists = [ selected_disjset_time_instances, [inst for inst in pure_binary_instances if inst in selected_disjset_time_instances] ]
disjset_num_bb_classes = len(disjset_bb_classes)

# Rows are geomean and wins for each bucket
disjset_bb_row_names = pd.MultiIndex.from_product(
    [disjset_bb_classes, bb_buckets, bb_metrics],
    names=['class', 'bucket', 'metric'])
disjset_time_geomean_cols = [
    [
        new_disjset_timing_cols[solver_ind][seed_ind] 
        for seed_ind in range(NUM_SEEDS) 
        for solver_ind in range(len(solver_stubs))
    ]
    + time_cols_long
    + new_disjset_wins_cols
]
disjset_time_geomean_df = pd.DataFrame(
    columns = disjset_time_geomean_cols,
    index = disjset_bb_row_names,
    dtype = float
)

# Make all columns "object" type to allow for integer values
disjset_time_geomean_df.loc[:,(disjset_time_geomean_cols[0])] = disjset_time_geomean_df.loc[:,(disjset_time_geomean_cols[0])].astype(object)

# Prepare num_inst columns
disjset_num_inst_by_seed_bucket_class = [
    np.zeros(len(disjset_time_geomean_df),dtype = np.int64)
    for _ in range(NUM_SEEDS) 
    # for _ in range(num_buckets) 
    # for _ in range(disjset_num_bb_classes)
]
disjset_num_inst_avg = np.zeros(len(disjset_time_geomean_df),dtype = np.int64)
disjset_num_ref_seed_wins = np.zeros(len(disjset_time_geomean_df),dtype = np.int64)
disjset_num_refv_seed_wins = np.zeros(len(disjset_time_geomean_df),dtype = np.int64)

# For each class, calculate geomean
for seed_ind in range(NUM_SEEDS):
    row_ind = 0
    for class_ind in range(disjset_num_bb_classes):
        print("\nClass: {}".format(disjset_bb_classes[class_ind]))
        curr_df = disjset_timing_df.loc[disjset_bb_classes_lists[class_ind]]
        seed_cols = [ new_disjset_timing_cols[solver_ind][seed_ind] for solver_ind in range(len(solver_stubs)) ]
        seed_curr_df = curr_df[seed_cols]

        # For every instance in which curr_by_depth_df[col] >= MAX_TIME - EPS, multiply by TIMEOUT_TIME_FACTOR
        for col_ind in range(len(seed_cols)):
            time_col = seed_cols[col_ind]
            seed_curr_df.loc[seed_curr_df[time_col] >= MAX_TIME - EPS, time_col] = TIMEOUT_TIME_FACTOR * MAX_TIME

        for bucket_ind in range(num_buckets):
            print("Bucket: [{:d},{:d})".format(bucket_min[bucket_ind],bucket_max[bucket_ind]))

            # Take subset of instances for which both solvers (with this seed) solve the instance in the time frame for the bucket
            seed_curr_df = seed_curr_df[
                (seed_curr_df[seed_cols].min(axis=1) >= bucket_min[bucket_ind])
                & (seed_curr_df[seed_cols].min(axis=1) < bucket_max[bucket_ind] - EPS)
            ]

            # Calculate geomean
            metric_ind = 0
            disjset_time_geomean_df.loc[
                (disjset_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
                seed_cols
            ] = [
                    geometric_mean(seed_curr_df[col] + SHIFT_TIME) - SHIFT_TIME
                    for col in seed_cols
                ]
            
            # Calculate wins
            metric_ind = 1
            for solver_ind in range(len(solver_stubs)):
                other_solver_ind = 1 - solver_ind
                disjset_time_geomean_df.loc[
                    (disjset_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
                    seed_cols[solver_ind]
                ] = sum(
                    seed_curr_df[seed_cols[other_solver_ind]]
                    > 
                    WIN_BY_TIME_FACTOR * seed_curr_df[seed_cols[solver_ind]]
                )

            print("row {:d}: {:d}".format(row_ind,len(seed_curr_df)))
            disjset_num_inst_by_seed_bucket_class[seed_ind][row_ind:row_ind+len(bb_metrics)] = [len(seed_curr_df)] * len(bb_metrics)

            row_ind += len(bb_metrics)

            # for solver_ind in range(len(solver_stubs)):
            #     other_solver_ind = 1 - solver_ind
            #     disjset_time_geomean_df.loc[
            #         (disjset_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
            #         new_disjset_wins_cols[solver_ind]
            #     ] = sum(
            #         seed_curr_df[new_disjset_timing_cols[other_solver_ind][seed_ind]]
            #         > 
            #         WIN_BY_TIME_FACTOR * seed_curr_df[new_disjset_timing_cols[solver_ind][seed_ind]]
            #     )


row_ind = 0
only_time_cols = [ ref_time_col, refv_time_col ]
cols = time_cols_long
wins_cols = new_disjset_wins_cols
for class_ind in range(disjset_num_bb_classes):
    print("\nClass: {}".format(disjset_bb_classes[class_ind]))
    curr_df = disjset_timing_df.loc[disjset_bb_classes_lists[class_ind]]

    # For every instance in which curr_by_depth_df[col] >= MAX_TIME - EPS, multiply by TIMEOUT_TIME_FACTOR
    for col_ind in range(len(only_time_cols)):
        time_col = only_time_cols[col_ind]
        curr_df.loc[curr_df[time_col] >= MAX_TIME - EPS, time_col] = TIMEOUT_TIME_FACTOR * MAX_TIME

    for bucket_ind in range(num_buckets):
        print("Bucket: [{:d},{:d})".format(bucket_min[bucket_ind],bucket_max[bucket_ind]))

        # Take subset of instances for which both solvers (with this seed) solve the instance in the time frame for the bucket
        curr_df = curr_df[
            (curr_df[only_time_cols].min(axis=1) >= bucket_min[bucket_ind])
            & (curr_df[only_time_cols].min(axis=1) < bucket_max[bucket_ind] - EPS)
        ]

        # Calculate geomean
        metric_ind = 0
        disjset_time_geomean_df.loc[
            (disjset_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
            cols
        ] = [
                geometric_mean(curr_df[col] + SHIFT_TIME) - SHIFT_TIME
                for col in cols
            ]
        
        # Calculate wins
        metric_ind = 1
        for solver_ind in range(len(solver_stubs)):
            other_solver_ind = 1 - solver_ind
            disjset_time_geomean_df.loc[
                (disjset_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
                only_time_cols[solver_ind]
            ] = sum(
                curr_df[only_time_cols[other_solver_ind]]
                > 
                WIN_BY_TIME_FACTOR * curr_df[only_time_cols[solver_ind]]
            )

            # Also report average number of seeds for which REF and REF+V win
            disjset_time_geomean_df.loc[
                (disjset_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[0:len(bb_metrics)]),
                wins_cols[solver_ind]
            ] = sum(curr_df[wins_cols[solver_ind]]) / len(curr_df)

        print("row {:d}: {:d}".format(row_ind,len(curr_df)))
        disjset_num_inst_avg[row_ind:row_ind+len(bb_metrics)] = [len(curr_df)] * len(bb_metrics)

        row_ind += len(bb_metrics)

# Insert columns with # inst (seed)
for seed_ind in range(NUM_SEEDS):
    disjset_time_geomean_df.insert(
        2 + seed_ind * (len(solver_stubs) + 1),
        "# inst ({:d})".format(seed_ind+1),
        disjset_num_inst_by_seed_bucket_class[seed_ind]
    )

# Insert columns with # inst (avg)
disjset_time_geomean_df.insert(
    2 + NUM_SEEDS * (len(solver_stubs) + 1),
    "# inst (avg)",
    disjset_num_inst_avg
)

disjset_time_geomean_df
    


Class: Combined
Bucket: [0,3600)
row 0: 285
Bucket: [10,3600)
row 2: 111
Bucket: [100,3600)
row 4: 57
Bucket: [1000,3600)
row 6: 12

Class: Combined-Binary
Bucket: [0,3600)
row 8: 52
Bucket: [10,3600)
row 10: 19
Bucket: [100,3600)
row 12: 13
Bucket: [1000,3600)
row 14: 2

Class: Combined
Bucket: [0,3600)
row 0: 284
Bucket: [10,3600)
row 2: 107
Bucket: [100,3600)
row 4: 48
Bucket: [1000,3600)
row 6: 15

Class: Combined-Binary
Bucket: [0,3600)
row 8: 52
Bucket: [10,3600)
row 10: 21
Bucket: [100,3600)
row 12: 10
Bucket: [1000,3600)
row 14: 4

Class: Combined
Bucket: [0,3600)
row 0: 284
Bucket: [10,3600)
row 2: 109
Bucket: [100,3600)
row 4: 52
Bucket: [1000,3600)
row 6: 13

Class: Combined-Binary
Bucket: [0,3600)
row 8: 52
Bucket: [10,3600)
row 10: 20
Bucket: [100,3600)
row 12: 13
Bucket: [1000,3600)
row 14: 3

Class: Combined
Bucket: [0,3600)
row 0: 284
Bucket: [10,3600)
row 2: 106
Bucket: [100,3600)
row 4: 55
Bucket: [1000,3600)
row 6: 13

Class: Combined-Binary
Bucket: [0,3600)
row 8: 

REF (1) TIME REF+V (1) TIME # inst (1)  \
class           bucket      metric                                          
Combined        [0,3600)    Gmean     10.016998      10.186793        285   
                            Wins            140             79        285   
                [10,3600)   Gmean     139.07065     146.223591        111   
                            Wins             55             35        111   
                [100,3600)  Gmean    517.758125     575.186954         57   
                            Wins             34             14         57   
                [1000,3600) Gmean   2599.137139    2160.009045         12   
                            Wins              6              4         12   
Combined-Binary [0,3600)    Gmean     10.859753      11.213015         52   
                            Wins             25             17         52   
                [10,3600)   Gmean    233.143262     298.284986         19   
                            Wins             12              6         19   
                [100,3600)  Gmean    526.536003     834.423695         13   
                            Wins             10              2         13   
                [1000,3600) Gmean   1442.016712    1854.384563          2   
                            Wins              2              0          2   

                                   REF (2) TIME REF+V (2) TIME # inst (2)  \
class           bucket      metric                                          
Combined        [0,3600)    Gmean      9.499052       9.965717        284   
                            Wins            130             88        284   
                [10,3600)   Gmean    131.349465     148.113342        107   
                            Wins             43             41        107   
                [100,3600)  Gmean    514.579899     617.982046         48   
                            Wins             22             16         48   
                [1000,3600) Gmean   1774.128963    1792.262362         15   
                            Wins              6              6         15   
Combined-Binary [0,3600)    Gmean     10.397683      12.178336         52   
                            Wins             20             19         52   
                [10,3600)   Gmean    172.365455     261.723238         21   
                            Wins              9             10         21   
                [100,3600)  Gmean    764.742026    1104.103659         10   
                            Wins              6              4         10   
                [1000,3600) Gmean   1474.350786    1856.132668          4   
                            Wins              3              1          4   

                                   REF (3) TIME REF+V (3) TIME # inst (3)  \
class           bucket      metric                                          
Combined        [0,3600)    Gmean      9.568452       9.796906        284   
                            Wins            128             90        284   
                [10,3600)   Gmean    131.149726     136.943696        109   
                            Wins             43             44        109   
                [100,3600)  Gmean     545.87874     566.983642         52   
                            Wins             23             19         52   
                [1000,3600) Gmean   2171.273801     2265.36422         13   
                            Wins              7              4         13   
Combined-Binary [0,3600)    Gmean     10.862729      11.759193         52   
                            Wins             27             15         52   
                [10,3600)   Gmean    200.125232     247.077893         20   
                            Wins             12              5         20   
                [100,3600)  Gmean    540.342737     638.998761         13   
                            Wins              9              3         13   
                [1000,3600) G

## "Combined" nodes summary

### disjset_nodes

In [64]:
inst_set = selected_disjset_time_instances

disjset_nodes_cols = \
    ['ALL' + ' ' + solver_type + ' ' + nodes_stub for solver_type in solver_stubs] \
    + \
    [ref_nodes_col, refv_nodes_col] \
    + \
    [col_num_vpc]
new_disjset_nodes_cols = [ 
        [
            solver_type + ' ' + '(%d)' % seed + ' ' + nodes_stub
            for seed in range(1,NUM_SEEDS+1)
        ]
        for solver_type in solver_stubs
    ]
new_disjset_wins_cols = [
    "Wins ({} over {})".format(solver_stubs[0], solver_stubs[1]),
    "Wins ({} over {})".format(solver_stubs[1], solver_stubs[0])
]
disjset_nodes_df = df_disjset.loc[inst_set, disjset_nodes_cols]
# display(disjet_timing_df.head())

# Add columns solver_type (%d) + ' ' + time_stub for the 7 random seeds
# These will be parsed from the ALL REF TIME and ALL REF+V TIME columns,
# which contain semicolon-separated values
for seed_ind in range(NUM_SEEDS):
    for solver_ind in range(len(solver_stubs)):
        solver_type = solver_stubs[solver_ind]
        orig_col = disjset_nodes_cols[solver_ind]
        new_col = new_disjset_nodes_cols[solver_ind][seed_ind]
        disjset_nodes_df[new_col] = disjset_nodes_df[orig_col].str.split(';').str[seed_ind]
        disjset_nodes_df[new_col] = disjset_nodes_df[new_col].astype(float)

# Add wins columns
for solver_ind in range(len(solver_stubs)):
    other_solver_ind = 1 - solver_ind

    disjset_nodes_df[new_disjset_wins_cols[solver_ind]] = sum(
        disjset_nodes_df[new_disjset_nodes_cols[other_solver_ind][seed_ind]]
        > 
        WIN_BY_TIME_FACTOR * disjset_nodes_df[new_disjset_nodes_cols[solver_ind][seed_ind]]
        for seed_ind in range(NUM_SEEDS)
    )

display(disjset_nodes_df.head())

,,ALL REF NODES,ALL REF+V NODES,AVG REF NODES,AVG REF+V NODES,NUM VPC,REF (1) NODES,REF+V (1) NODES,REF (2) NODES,REF+V (2) NODES,REF (3) NODES,...,REF (4) NODES,REF+V (4) NODES,REF (5) NODES,REF+V (5) NODES,REF (6) NODES,REF+V (6) NODES,REF (7) NODES,REF+V (7) NODES,Wins (REF over REF+V),Wins (REF+V over REF)
INSTANCE,disj_terms,,,,,,,,,,,,,,,,,,,,,
10teams_presolved,-1,1;515;226;1;577;123;517,29;1;384;399;1039;132;299,280,326,80,1.0,29.0,515.0,1.0,226.0,...,1.0,399.0,577.0,1039.0,123.0,132.0,517.0,299.0,4,2
23588_presolved,-1,654;906;668;997;813;1293;844,1209;1898;1309;1670;1121;1959;1493,882,1522,386,654.0,1209.0,906.0,1898.0,668.0,...,997.0,1670.0,813.0,1121.0,1293.0,1959.0,844.0,1493.0,7,0
30n20b8_presolved,-1,536;812;550;541;598;2072;557,446;635;540;760;746;744;964,809,690,381,536.0,446.0,812.0,635.0,550.0,...,541.0,760.0,598.0,746.0,2072.0,744.0,557.0,964.0,3,3
50v-10_presolved,-1,1698842;860977;1674112;1347590;1040288;1860918...,1408816;1194175;1530595;1236973;1187890;136470...,1495008,1294072,72,1698842.0,1408816.0,860977.0,1194175.0,1674112.0,...,1347590.0,1236973.0,1040288.0,1187890.0,1860918.0,1364702.0,1982335.0,1135356.0,2,3
a1c1s1_presolved,-1,63571;41748;46488;54947;40220;35137;40351,62100;51654;49067;50284;49242;44500;53462,46066,51472,22,63571.0,62100.0,41748.0,51654.0,46488.0,...,54947.0,50284.0,40220.0,49242.0,35137.0,44500.0,40351.0,53462.0,4,0


In [65]:
disjset_nodes_df.to_csv("disjset_nodes.csv")

### disjset_nodes_geomean_df

In [66]:
# Row names
disjset_nodes_geomean_cols = [
    [
        new_disjset_nodes_cols[solver_ind][seed_ind] 
        for seed_ind in range(NUM_SEEDS) 
        for solver_ind in range(len(solver_stubs))
    ]
    + [ref_nodes_col, refv_nodes_col]
    + new_disjset_wins_cols
]
disjset_nodes_geomean_df = pd.DataFrame(
    columns = disjset_nodes_geomean_cols,
    index = disjset_bb_row_names,
    dtype = float
)

# Make all columns "object" type to allow for integer values
disjset_nodes_geomean_df.loc[:,(disjset_nodes_geomean_cols[0])] = disjset_nodes_geomean_df.loc[:,(disjset_nodes_geomean_cols[0])].astype(object)

# Prepare num_inst columns
disjset_num_inst_by_seed_bucket_class = [
    np.zeros(len(disjset_nodes_geomean_df),dtype = np.int64)
    for _ in range(NUM_SEEDS) 
]
disjset_num_inst_avg = np.zeros(len(disjset_nodes_geomean_df),dtype = np.int64)
disjset_num_ref_seed_wins = np.zeros(len(disjset_nodes_geomean_df),dtype = np.int64)
disjset_num_refv_seed_wins = np.zeros

# For each class, calculate geomean
for seed_ind in range(NUM_SEEDS):
    row_ind = 0
    for class_ind in range(disjset_num_bb_classes):
        print("\nClass: {}".format(disjset_bb_classes[class_ind]))
        curr_df = disjset_nodes_df.loc[disjset_bb_classes_lists[class_ind]]
        seed_cols = [ new_disjset_nodes_cols[solver_ind][seed_ind] for solver_ind in range(len(solver_stubs)) ]
        seed_curr_df = curr_df[seed_cols]
        
        time_curr_df = disjset_timing_df.loc[disjset_bb_classes_lists[class_ind]]
        time_seed_cols = [ new_disjset_timing_cols[solver_ind][seed_ind] for solver_ind in range(len(solver_stubs)) ]
        seed_time_curr_df = time_curr_df[time_seed_cols]

        # if (len(seed_curr_df) != len(seed_time_curr_df)):
        #     display(len(seed_curr_df))
        #     display(len(seed_time_curr_df))
        # break

        # For every instance in which curr_by_depth_df[col] >= MAX_TIME - EPS, multiply nodes processed by TIMEOUT_NODE_FACTOR
        for col_ind in range(len(time_seed_cols)):
            time_col = time_seed_cols[col_ind]
            node_col = seed_cols[col_ind]
            curr_selected_indices = seed_time_curr_df[time_col] >= MAX_TIME - EPS
            seed_curr_df.loc[curr_selected_indices, node_col] = TIMEOUT_NODE_FACTOR * seed_curr_df[node_col]

        for bucket_ind in range(num_buckets):
            print("Bucket: [{:d},{:d})".format(bucket_min[bucket_ind],bucket_max[bucket_ind]))

            # Take subset of instances for which both solvers (with this seed) solve the instance in the time frame for the bucket
            curr_selected_indices = \
                (seed_time_curr_df[time_seed_cols].min(axis=1) >= bucket_min[bucket_ind]) \
                & (seed_time_curr_df[time_seed_cols].min(axis=1) < bucket_max[bucket_ind] - EPS)
            seed_curr_df = seed_curr_df[curr_selected_indices]
            seed_time_curr_df = seed_time_curr_df[curr_selected_indices]

            # Calculate geomean
            metric_ind = 0
            disjset_nodes_geomean_df.loc[
                (disjset_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
                seed_cols
            ] = [
                    geometric_mean(seed_curr_df[col] + SHIFT_NODES) - SHIFT_NODES
                    for col in seed_cols
                ]
            
            # Calculate wins
            metric_ind = 1
            for solver_ind in range(len(solver_stubs)):
                other_solver_ind = 1 - solver_ind
                disjset_nodes_geomean_df.loc[
                    (disjset_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
                    seed_cols[solver_ind]
                ] = sum(
                    seed_curr_df[seed_cols[other_solver_ind]]
                    > 
                    WIN_BY_NODES_FACTOR * seed_curr_df[seed_cols[solver_ind]]
                )

            print("row {:d}: {:d}".format(row_ind,len(seed_curr_df)))
            disjset_num_inst_by_seed_bucket_class[seed_ind][row_ind:row_ind+len(bb_metrics)] = [len(seed_curr_df)] * len(bb_metrics)

            row_ind += len(bb_metrics)

row_ind = 0
time_cols = [ ref_time_col, refv_time_col ]
nodes_cols = [ ref_nodes_col, refv_nodes_col ]
wins_cols = new_disjset_wins_cols
for class_ind in range(disjset_num_bb_classes):
    print("\nClass: {}".format(disjset_bb_classes[class_ind]))
    curr_df = disjset_nodes_df.loc[disjset_bb_classes_lists[class_ind]]
    time_curr_df = disjset_timing_df.loc[disjset_bb_classes_lists[class_ind]]

    # For every instance in which curr_by_depth_df[col] >= MAX_TIME - EPS, multiply nodes processed by TIMEOUT_NODE_FACTOR
    for col_ind in range(len(time_cols)):
        time_col = time_cols[col_ind]
        node_col = nodes_cols[col_ind]
        curr_selected_indices = time_curr_df[time_col] >= MAX_TIME - EPS
        curr_df.loc[curr_selected_indices, node_col] = TIMEOUT_NODE_FACTOR * curr_df[node_col]

    for bucket_ind in range(num_buckets):
        print("Bucket: [{:d},{:d})".format(bucket_min[bucket_ind],bucket_max[bucket_ind]))

        # Take subset of instances for which both solvers (with this seed) solve the instance in the time frame for the bucket
        curr_selected_indices = \
            (time_curr_df[time_cols].min(axis=1) >= bucket_min[bucket_ind]) \
            & (time_curr_df[time_cols].min(axis=1) < bucket_max[bucket_ind] - EPS)
        curr_df = curr_df[curr_selected_indices]
        time_curr_df = time_curr_df[curr_selected_indices]

        # Calculate geomean
        metric_ind = 0
        disjset_nodes_geomean_df.loc[
            (disjset_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
            nodes_cols
        ] = [
                geometric_mean(curr_df[col] + SHIFT_NODES) - SHIFT_NODES
                for col in nodes_cols
            ]
        
        # Calculate wins
        metric_ind = 1
        for solver_ind in range(len(solver_stubs)):
            other_solver_ind = 1 - solver_ind
            disjset_nodes_geomean_df.loc[
                (disjset_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
                nodes_cols[solver_ind]
            ] = sum(
                curr_df[nodes_cols[other_solver_ind]]
                > 
                WIN_BY_NODES_FACTOR * curr_df[nodes_cols[solver_ind]]
            )

            # Also report average number of seeds for which REF and REF+V win
            disjset_nodes_geomean_df.loc[
                (disjset_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[0:len(bb_metrics)]),
                wins_cols[solver_ind]
            ] = sum(curr_df[wins_cols[solver_ind]]) / len(curr_df)

        print("row {:d}: {:d}".format(row_ind,len(curr_df)))
        disjset_num_inst_avg[row_ind:row_ind+len(bb_metrics)] = [len(curr_df)] * len(bb_metrics)

        row_ind += len(bb_metrics)

# Insert columns with # inst (seed)
for seed_ind in range(NUM_SEEDS):
    disjset_nodes_geomean_df.insert(
        2 + seed_ind * (len(solver_stubs) + 1),
        "# inst ({:d})".format(seed_ind+1),
        disjset_num_inst_by_seed_bucket_class[seed_ind]
    )

# Insert columns with # inst (avg)
disjset_nodes_geomean_df.insert(
    2 + NUM_SEEDS * (len(solver_stubs) + 1),
    "# inst (avg)",
    disjset_num_inst_avg
)

disjset_nodes_geomean_df



Class: Combined
Bucket: [0,3600)
row 0: 285
Bucket: [10,3600)
row 2: 111
Bucket: [100,3600)
row 4: 57
Bucket: [1000,3600)
row 6: 12

Class: Combined-Binary
Bucket: [0,3600)
row 8: 52
Bucket: [10,3600)
row 10: 19
Bucket: [100,3600)
row 12: 13
Bucket: [1000,3600)
row 14: 2

Class: Combined
Bucket: [0,3600)
row 0: 284
Bucket: [10,3600)
row 2: 107
Bucket: [100,3600)
row 4: 48
Bucket: [1000,3600)
row 6: 15

Class: Combined-Binary
Bucket: [0,3600)
row 8: 52
Bucket: [10,3600)
row 10: 21
Bucket: [100,3600)
row 12: 10
Bucket: [1000,3600)
row 14: 4

Class: Combined
Bucket: [0,3600)
row 0: 284
Bucket: [10,3600)
row 2: 109
Bucket: [100,3600)
row 4: 52
Bucket: [1000,3600)
row 6: 13

Class: Combined-Binary
Bucket: [0,3600)
row 8: 52
Bucket: [10,3600)
row 10: 20
Bucket: [100,3600)
row 12: 13
Bucket: [1000,3600)
row 14: 3

Class: Combined
Bucket: [0,3600)
row 0: 284
Bucket: [10,3600)
row 2: 106
Bucket: [100,3600)
row 4: 55
Bucket: [1000,3600)
row 6: 13

Class: Combined-Binary
Bucket: [0,3600)
row 8: 

REF (1) NODES REF+V (1) NODES # inst (1)  \
class           bucket      metric                                              
Combined        [0,3600)    Gmean      6900.447494     6603.141935        285   
                            Wins               127             125        285   
                [10,3600)   Gmean     69439.439933    65748.606336        111   
                            Wins                52              58        111   
                [100,3600)  Gmean    243545.260111   253238.204694         57   
                            Wins                29              27         57   
                [1000,3600) Gmean   1115003.092376   874798.812311         12   
                            Wins                 5               6         12   
Combined-Binary [0,3600)    Gmean      6179.694596     6178.401313         52   
                            Wins                23              20         52   
                [10,3600)   Gmean     68388.675373    80026.738743         19   
                            Wins                10               9         19   
                [100,3600)  Gmean    140369.856146    204967.53804         13   
                            Wins                 8               5         13   
                [1000,3600) Gmean    713016.304567   570367.453384          2   
                            Wins                 1               1          2   

                                    REF (2) NODES REF+V (2) NODES # inst (2)  \
class           bucket      metric                                             
Combined        [0,3600)    Gmean     6431.491608     6539.856096        284   
                            Wins              121             126        284   
                [10,3600)   Gmean     64655.93218    70191.238838        107   
                            Wins               50              56        107   
                [100,3600)  Gmean   230256.482657    269483.69226         48   
                            Wins               24              23         48   
                [1000,3600) Gmean   594049.630887   585567.279211         15   
                            Wins                7               7         15   
Combined-Binary [0,3600)    Gmean     5986.124724     6579.884713         52   
                            Wins               18              23         52   
                [10,3600)   Gmean    54984.927011    70561.114326         21   
                            Wins                9              12         21   
                [100,3600)  Gmean   188894.811975   228175.385844         10   
                            Wins                5               5         10   
                [1000,3600) Gmean   283290.408192   277062.590852          4   
                            Wins                2               2          4   

                                    REF (3) NODES REF+V (3) NODES # inst (3)  \
class           bucket      metric                                             
Combined        [0,3600)    Gmean     6434.639223     6193.289188        284   
                            Wins              119             128        284   
                [10,3600)   Gmean     58348.42236    56662.282582        109   
                            Wins               49              59        109   
                [100,3600)  Gmean   251979.752762   239808.467586         52   
                            Wins               21              30         52   
                [1000,3600) Gmean   551339.431759   521257.504944         13   
                            Wins                8               5         13   
Combined-Binary [0,3600)    Gmean     5926.008288     6168.625582         52   
                            Wins               21              20         52   
                [10,3600)   Gmean    56529.119917    64216.873582         20   
                            Wins               10              10         20   
                [

## Table 3: `disjset_avg_bb_df` for "Combined" instances

In [67]:
## Set up empty `disjset_avg_bb_df`

disjset_time_cols_short = time_cols_short
disjset_node_cols_short = node_cols_short

disjset_avg_bb_cols = pd.MultiIndex.from_arrays(
    [
      [time_col_header]*len(disjset_time_cols_short) + [node_col_header]*len(disjset_node_cols_short),
      disjset_time_cols_short + disjset_node_cols_short
    ],
    names = ['criterion', 'type'])

#bb_row_names = pd.MultiIndex.from_product([bb_buckets, bb_row_names], names=['bucket', 'metric'])
disjset_bb_row_names = pd.MultiIndex.from_product(
    [disjset_bb_classes, bb_buckets, bb_metrics],
    names=['class', 'bucket', 'metric'])

disjset_avg_bb_df = pd.DataFrame(
    columns = disjset_avg_bb_cols,
    index = disjset_bb_row_names,
    dtype = float
)

# Make all columns "object" type to allow for integer values
disjset_avg_bb_df.loc[:,(time_col_header,disjset_time_cols_short)] = disjset_avg_bb_df.loc[:,(time_col_header,disjset_time_cols_short)].astype(object)
disjset_avg_bb_df.loc[:,(node_col_header,disjset_node_cols_short)] = disjset_avg_bb_df.loc[:,(node_col_header,disjset_node_cols_short)].astype(object)

num_inst = np.zeros(len(disjset_avg_bb_df),dtype = np.int64)
# cols = disjset_time_cols_short + disjset_node_cols_short

## Fill in values from disjset_time_geomean_df and disjset_nodes_geomean_df (from the ref and refv cols)
# Fill in values from disjset_time_geomean_df
for index, row in disjset_time_geomean_df.iterrows():
    class_val, bucket_val, metric_val = index
    for col in disjset_time_cols_short:
        disjset_avg_bb_df.loc[(class_val, bucket_val, metric_val), (time_col_header, col)] = \
            row[map_short_to_cols_time[col]]

# Fill in values from disjset_nodes_geomean_df
for index, row in disjset_nodes_geomean_df.iterrows():
    class_val, bucket_val, metric_val = index
    for col in disjset_node_cols_short:
        disjset_avg_bb_df.loc[(class_val, bucket_val, metric_val), (node_col_header, col)] = \
            row[map_short_to_cols_nodes[col]]
        
# Add in number of instances in each bucket as '# inst' col
disjset_avg_bb_df[inst_col_name] = disjset_num_inst_avg

display(disjset_avg_bb_df.loc[:,disjset_avg_bb_df.columns.get_level_values(0)==node_col_header].head(8))
#display(avg_bb_df.loc[(bb_classes[0], bb_buckets[1], bb_metrics[0]),:])
display(disjset_avg_bb_df.loc[(disjset_bb_classes, bb_buckets, bb_metrics[0]),:])
display(disjset_avg_bb_df)


criterion                         Nodes (#)               
type                                    Gur              V
class    bucket      metric                               
Combined [0,3600)    Gmean      7681.913224    7793.801266
                     Wins               128            131
         [10,3600)   Gmean     75193.842935   76955.769766
                     Wins                54             65
         [100,3600)  Gmean    244802.368364  255028.026777
                     Wins                38             31
         [1000,3600) Gmean   1008339.705759  996884.899183
                     Wins                11             13

criterion                              Time (s)                           \
type                                        Gur            V         Gen   
class           bucket      metric                                         
Combined        [0,3600)    Gmean     11.256821    11.839758   27.617049   
                [10,3600)   Gmean    157.263482    168.45359   55.301806   
                [100,3600)  Gmean    505.119151   550.557326   43.695307   
                [1000,3600) Gmean   1972.953117  2073.896266   35.858057   
Combined-Binary [0,3600)    Gmean     11.404711    12.320784   72.859344   
                [10,3600)   Gmean    206.231182   252.802845  200.191878   
                [100,3600)  Gmean    505.793757   655.610711  239.674786   
                [1000,3600) Gmean   1249.808935  1493.336448  184.759389   

criterion                                Nodes (#)                # inst  
type                                           Gur              V         
class           bucket      metric                                        
Combined        [0,3600)    Gmean      7681.913224    7793.801266    288  
                [10,3600)   Gmean     75193.842935   76955.769766    119  
                [100,3600)  Gmean    244802.368364  255028.026777     69  
                [1000,3600) Gmean   1008339.705759  996884.899183     24  
Combined-Binary [0,3600)    Gmean      6409.804727    6606.771462     52  
                [10,3600)   Gmean     63749.847854   70951.409932     21  
                [100,3600)  Gmean     124041.27542  148827.302683     15  
                [1000,3600) Gmean    208512.296594  198921.973161      5

criterion                              Time (s)                           \
type                                        Gur            V         Gen   
class           bucket      metric                                         
Combined        [0,3600)    Gmean     11.256821    11.839758   27.617049   
                            Wins            117           62         NaN   
                [10,3600)   Gmean    157.263482    168.45359   55.301806   
                            Wins             44           32         NaN   
                [100,3600)  Gmean    505.119151   550.557326   43.695307   
                            Wins             32           15         NaN   
                [1000,3600) Gmean   1972.953117  2073.896266   35.858057   
                            Wins             10            5         NaN   
Combined-Binary [0,3600)    Gmean     11.404711    12.320784   72.859344   
                            Wins             22           13         NaN   
                [10,3600)   Gmean    206.231182   252.802845  200.191878   
                            Wins             10            6         NaN   
                [100,3600)  Gmean    505.793757   655.610711  239.674786   
                            Wins              9            3         NaN   
                [1000,3600) Gmean   1249.808935  1493.336448  184.759389   
                            Wins              3            0         NaN   

criterion                                Nodes (#)                # inst  
type                                           Gur              V         
class           bucket      metric                                        
Combined        [0,3600)    Gmean      7681.913224    7793.801266    288  
                            Wins               128            131    288  
                [10,3600)   Gmean     75193.842935   76955.769766    119  
                            Wins                54             65    119  
                [100,3600)  Gmean    244802.368364  255028.026777     69  
                            Wins                38             31     69  
                [1000,3600) Gmean   1008339.705759  996884.899183     24  
                            Wins                11             13     24  
Combined-Binary [0,3600)    Gmean      6409.804727    6606.771462     52  
                            Wins                23             21     52  
                [10,3600)   Gmean     63749.847854   70951.409932     21  
                            Wins                10             11     21  
                [100,3600)  Gmean     124041.27542  148827.302683     15  
                            Wins                 9              6     15  
                [1000,3600) Gmean    208512.296594  198921.973161      5  
                            Wins                 2              3      5

# Section 5: rounds results

## Select rounds instances

In [68]:
## Read in rounds data
#df_rounds = pd.read_csv(RESULTS_DIR + '/' + 'vpc-rounds.csv', index_col=0)
df_rounds = initialize_df(RESULTS_DIR + '/' + 'vpc-rounds.csv')

## Change disj_terms (second level of index) to -1
df_rounds.index = df_rounds.index.set_levels([-1], level=1)

## Update column types
col_list = [col_best_disj_obj, col_worst_disj_obj]
for col in col_list:
    df_rounds[col] = pd.to_numeric(df_rounds[col])

## Create new column for number of disjunctive terms since original one is now index
df_rounds[col_num_disj_terms] = df_rounds.index.get_level_values(1)

## Identify pure binary instances, which are those where 'CLEANED BINARY' column equals 'CLEANED COLS'
df_rounds[col_pure_binary] = (df_rounds[col_binary] == df_rounds[col_num_cols])

## Identify mixed binary instances, which are those where 'CLEANED GEN INT' column = 0
df_rounds[col_mixed_binary] = (df_rounds[col_gen_int] == 0)

## Add col for avg time with cuts
df_rounds[refv_w_cut_time_col] = df_rounds[refv_time_col] + df_rounds[col_vpc_gen_time]

display(df_rounds.head(2))
display(df_rounds.loc['mas284_presolved'][obj_val_col_list])

,,cutlimit,gomory,mode,partial_bb_strategy,partial_bb_keep_pruned_nodes,partial_bb_num_strong,preprocess,prlp_flip_beta,rounds,bb_mode,...,cpu_model,cpu_id,ExitReason,end_time_string,time elapsed,instname,end,IS PURE BINARY,IS MIXED BINARY,AVG REF+V W/CUTGEN TIME
INSTANCE,disj_terms,,,,,,,,,,,,,,,,,,,,,
10teams_presolved,-1,-2,-1,4,4,0,5,0,0,2,10,...,13th Gen Intel(R) Core(TM) i9-13900K,10,TIME_LIMIT,Mon Dec 4 21:51:15 2023,4130,10teams_presolved,DONE,True,True,4115.985
22433_presolved,-1,-2,-1,4,4,0,5,0,0,2,10,...,13th Gen Intel(R) Core(TM) i9-13900K,6,OPTIMAL_SOLUTION_FOUND,Mon Dec 4 11:16:14 2023,1,22433_presolved,DONE,False,False,0.298


,LP OBJ,IP OBJ,WORST DISJ OBJ,GMIC OBJ,ROOT OBJ,BEST DISJ OBJ,VPC OBJ,VPC+GMIC OBJ,FIRST REF FIRST_CUT_PASS,FIRST REF+V FIRST_CUT_PASS,AVG REF FIRST_CUT_PASS,AVG REF+V FIRST_CUT_PASS,BEST REF FIRST_CUT_PASS,BEST REF+V FIRST_CUT_PASS,FIRST REF LAST_CUT_PASS,FIRST REF+V LAST_CUT_PASS,AVG REF LAST_CUT_PASS,AVG REF+V LAST_CUT_PASS,BEST REF LAST_CUT_PASS,BEST REF+V LAST_CUT_PASS
disj_terms,,,,,,,,,,,,,,,,,,,,
-1,86195.863028,91405.72368,91724.670004,87973.466711,87972.851173,88080.047754,88059.374848,88059.374848,0.0,87911.453992,0.0,87911.453992,0.0,87911.453992,0.0,88040.684627,0.0,88041.087001,0.0,88042.659519


In [69]:
# Create df_nodes dataframe, in which each row is an instance, and each column is a solver (with or w/o cuts, and with a fixed random seed)
# These will be pulled from 'ALL REF TIME' and 'ALL REF+V TIME' columns
inst_set = list(instances)
# inst_set = ['bm23_presolved', 'cvs16r106-72_presolved', 'cvs16r128-89_presolved', 'mine-90-10_presolved', 'misc03_presolved', 'neos-1058477_presolved', 'neos-3083819-nubu_presolved', 'neos-593853_presolved', 'neos-880324_presolved', 'ns1688347_presolved']
PRINT_SKIP_REASON = False

rounds_num_gap_errors = 0

selected_rounds_gap_instances_dict = {} # dictionary of (original index, instance)
selected_rounds_time_instances_dict = {} # dictionary of (original index, instance)
for i, inst in enumerate(inst_set):
  print("{}/{}".format(i+1,len(inst_set)), end='\r', flush=True)
  skip_instance = False

  # # Check that -1 depth exists
  # if not (-1 in df.loc[inst].index):
  #   # if PRINT_SKIP_REASON:
  #   print("Skipping instance {:d} -- {}: no -1 depth.".format(
  #       i, inst
  #   ))
  #   skip_instance = True
  #   continue
  if not inst in df_rounds.index:
    if PRINT_SKIP_REASON:
      print("Skipping instance {:d} -- {}: no rounds results.".format(
          i, inst
      ))
    skip_instance = True
    continue
  
  curr_df = df_rounds.loc[(inst,-1)]

  # Ensure nrows and ncols is not too many
  nrows = curr_df[col_num_rows]
  ncols = curr_df[col_num_cols]
  if (nrows > MAX_ROWS) or (ncols > MAX_COLS):
    if PRINT_SKIP_REASON:
        print("Skipping instance {:d} -- {}: nrows = {:d} > {:d} or ncols = {:d} > {:d}.".format(
                i, inst, nrows, ncols, MAX_ROWS, MAX_COLS))
    skip_instance = True

  # Ensure IP objective value is known
  ip_obj = np.float64(df_ipopt.loc[inst,col_ip_obj])
  if not isinstance(ip_obj,float):
    if PRINT_SKIP_REASON:
        print(
            "Skipping instance {:d} -- {}: IP objective value ({}) is not detected to be a float value.".format(
            i, inst, ip_obj))
    skip_instance = True

  check_ip_obj = curr_df[col_ip_obj]
  if not is_val(ip_obj,check_ip_obj):
    print("*** ERROR: Instance {:d} -- {}: IP objective value ({}) does not match value in dataframe ({}). REPLACING WITH KNOWN VALUE!".format(
        i, inst, ip_obj, check_ip_obj))
    df_rounds.loc[(inst,-1),col_ip_obj] = ip_obj
    # skip_instance = True

  # Check that LP opt < IP opt
  lp_obj = np.float64(df_preprocess.loc[remove_presolved_from_name(inst),col_cleaned_lp_obj])
  YES_GAP = (ip_obj - lp_obj) >= EPS
  if not YES_GAP:
    print("*** ERROR: Instance {:d} -- {}: not YES GAP (lp = {:.10f}; ip = {:.10f}, diff = {:.2f})".format(i, inst, lp_obj, ip_obj, ip_obj-lp_obj))
    skip_instance = True
    rounds_num_gap_errors += 1

  # Check that ExitReason != OPTIMAL_SOLUTION_FOUND
  exitreason = curr_df['ExitReason']
  if exitreason == 'OPTIMAL_SOLUTION_FOUND':
    if PRINT_SKIP_REASON:
      print("Skipping instance {:d} -- {}: optimal IP solution found.".format(
          i, inst
      ))
    skip_instance = True

  # Check that VPCs were generated
  num_vpc = curr_df[col_num_vpc]
  if num_vpc == 0:
    if PRINT_SKIP_REASON:
      print("Skipping instance {:d} -- {}: no VPCs generated.".format(
          i, inst
      ))
    skip_instance = True

  if not skip_instance:
    selected_rounds_gap_instances_dict[inst] = i

    # If both ref and refv times are < MAX_TIME - EPS, include in nodes experiments
    ref_time = curr_df[ref_time_col]
    refv_time = curr_df[refv_time_col]
    if (ref_time < MAX_TIME - EPS) and (refv_time < MAX_TIME - EPS):
      selected_rounds_time_instances_dict[inst] = i

selected_rounds_gap_instances = list(selected_rounds_gap_instances_dict.keys())
print("Total number of errors: {}".format(rounds_num_gap_errors))
print("Num instances selected for rounds gap closed results = {:d}".format(len(selected_rounds_gap_instances)))

selected_rounds_time_instances = list(selected_rounds_time_instances_dict.keys())
print("Num instances selected for rounds nodes results = {:d}".format(len(selected_rounds_time_instances)))

*** ERROR: Instance 129 -- maxgasflow_presolved: IP objective value (-44565819.158) does not match value in dataframe (-44565819158.0). REPLACING WITH KNOWN VALUE!
*** ERROR: Instance 209 -- neos-3048764-nadi_presolved: IP objective value (-3883988306.366686) does not match value in dataframe (-3883988306.0). REPLACING WITH KNOWN VALUE!
*** ERROR: Instance 228 -- neos-3661949-lesse_presolved: not YES GAP (lp = 689000000.0000000000; ip = 688995225.0000000000, diff = -4775.00)
Total number of errors: 1
Num instances selected for rounds gap closed results = 323
Num instances selected for rounds nodes results = 279


### List instances that are in exactly one of the two sets `selected_gap_instances` and `selected_rounds_gap_instances`

In [70]:
# Symmetric difference between two sets selected_gap_instances and selected_rounds_gap_instances
print("Instances in selected_rounds_gap_instances that are not in selected_gap_instances:")
newly_selected_instances = [inst for inst in selected_rounds_gap_instances if inst not in selected_gap_instances]
print(newly_selected_instances)

# Remove newly selected instances
for inst in newly_selected_instances:
  selected_rounds_gap_instances.remove(inst)

print("Instances in selected_gap_instances that are not in selected_rounds_gap_instances:")
print([inst for inst in selected_gap_instances if inst not in selected_rounds_gap_instances])

print("Instances in selected_rounds_time_instances that are not in selected_disjset_time_instances:")
print([inst for inst in selected_rounds_time_instances if inst not in selected_disjset_time_instances])

Instances in selected_rounds_gap_instances that are not in selected_gap_instances:
['neos-3214367-sovi_presolved']
Instances in selected_gap_instances that are not in selected_rounds_gap_instances:
['bm23_presolved', 'cvs16r106-72_presolved', 'cvs16r128-89_presolved', 'mine-90-10_presolved', 'misc03_presolved', 'neos-1058477_presolved', 'neos-3083819-nubu_presolved', 'neos-593853_presolved', 'neos-880324_presolved', 'ns1688347_presolved']
Instances in selected_rounds_time_instances that are not in selected_disjset_time_instances:
[]


## rounds gap summary

### `rounds_gap_df`: Analyze rounds gap closed

In [71]:
# Create subset of dataframe relevant to gap closed
rounds_gap_df = df_rounds.loc[selected_rounds_gap_instances, 
                [
                    col_num_disj_terms,
                    col_num_rows,
                    col_num_cols
                ]
                +
                obj_val_col_list
                +
                [
                    col_num_gmic,
                    col_num_vpc,
                    col_num_obj,
                    col_exit_reason
                ]
               ]

# Take only -1 depth
rounds_gap_df = rounds_gap_df.xs(-1, level='disj_terms')

# Calculate some missing % gap closed columns
# gap closed = 100 * (post_cut_opt_val - lp_opt_val) / (ip_opt_val - lp_opt_val)
for cut_type in reg_cut_type_long_list:
    col = cut_type + ' ' + obj_stub
    if col not in df.columns:
        if cut_type == 'MAX(GMIC,VPC)':
            # Add max(G,V) column
            rounds_gap_df[cut_type + ' ' + pct_gap_closed_stub] = \
                np.maximum(
                    rounds_gap_df['GMIC' + ' ' + pct_gap_closed_stub],
                    rounds_gap_df[ 'VPC' + ' ' + pct_gap_closed_stub]
                )
        continue
    rounds_gap_df[cut_type + ' ' + pct_gap_closed_stub] = calc_gap_closed(rounds_gap_df, col)

# Compare against reference solver
for stat_type in solver_stat_list[:3]:
    for solver_type in solver_stubs:
        for cut_type in solver_cut_type_stubs:
            col = stat_type + ' ' + solver_type + ' ' + cut_type
            rounds_gap_df[col + ' ' + pct_gap_closed_stub] = calc_gap_closed(rounds_gap_df, col)

# Rename each column with its shortname
rounds_gap_df.rename(columns=map_cols_to_short_gap,inplace=True)

# Fill in values of col_first_cut_pass_gap_ref and col_last_cut_pass_gap_ref from disjset_gap_df
for inst in selected_rounds_gap_instances:
    for col in [col_first_cut_pass_gap_ref, col_last_cut_pass_gap_ref]:
        col = map_cols_to_short_gap[col]
        rounds_gap_df.loc[inst,col] = disjset_gap_df.loc[inst,col]

tmp_inst_set = ['mas284_presolved','maxgasflow_presolved']
display(disjset_gap_df.loc[tmp_inst_set][gap_cols_short])
display(rounds_gap_df.loc[tmp_inst_set][gap_cols_short])
display(rounds_gap_df.loc[("mas284_presolved")])

,G,R,DB,V,"max(G,V)",V+G,GurF,V+GurF,GurL,V+GurL
INSTANCE,,,,,,,,,,
mas284_presolved,0.866419,0.000000e+00,34.867672,34.097198,34.097198,34.097198,0.508365,33.801872,10.066505,33.923311
maxgasflow_presolved,22.478700,-3.223163e-11,0.015531,0.006570,22.478700,22.478957,22.500189,22.522284,28.774516,28.025236


,G,R,DB,V,"max(G,V)",V+G,GurF,V+GurF,GurL,V+GurL
INSTANCE,,,,,,,,,,
mas284_presolved,34.119985,34.10817,36.165741,35.768938,35.768938,35.768938,0.508365,32.929690,10.066505,35.417914
maxgasflow_presolved,22.526628,0.00657,0.015531,0.006578,22.526628,22.526912,22.500189,22.522284,28.774516,28.547865


NUM DISJ TERMS                                       -1
ROWS                                                 68
COLS                                                148
LP OBJ                                     86195.863028
IP OBJ                                      91405.72368
WORST DISJ OBJ                             91724.670004
GMIC OBJ                                   87973.466711
ROOT OBJ                                   87972.851173
BEST DISJ OBJ                              88080.047754
VPC OBJ                                    88059.374848
VPC+GMIC OBJ                               88059.374848
FIRST REF FIRST_CUT_PASS                            0.0
FIRST REF+V FIRST_CUT_PASS                 87911.453992
AVG REF FIRST_CUT_PASS                              0.0
AVG REF+V FIRST_CUT_PASS                   87911.453992
BEST REF FIRST_CUT_PASS                             0.0
BEST REF+V FIRST_CUT_PASS                  87911.453992
FIRST REF LAST_CUT_PASS                         

### `rounds_avg_gap_df`: summary of results with rounds instances

In [72]:
## TABLE 1.2: average percent gap closed across different combinations of cuts
## Create rounds_avg_gap_df = average gap closed across instances
idx = pd.MultiIndex.from_product(
    [ [all_set_name, good_vpc_set_name, binary_set_name], [avg_row_name, wins_row_name] ],
    names = ['Set', '']
)
    
ncols = len(gap_cols_short)
nrows = len(idx)

col = rounds_gap_df['V'].astype(float)
rounds_good_vpc_df = rounds_gap_df[col >= 10.]
rounds_binary_instances_df = rounds_gap_df.loc[ [inst for inst in pure_binary_instances if inst in selected_rounds_gap_instances ] ]

data = np.zeros((nrows, ncols), dtype=float)
data[0,:] = [rounds_gap_df[col].mean() for col in gap_cols_short]
data[2,:] = [rounds_good_vpc_df[col].mean() for col in gap_cols_short]
data[4,:] = [rounds_binary_instances_df[col].mean() for col in gap_cols_short]

# display(best_gap_df.head())
rounds_avg_gap_df = pd.DataFrame(
    data,
    columns = gap_cols_short,
    index = idx,
    dtype = object
)

inst_col_name = '# inst'
rounds_avg_gap_df[inst_col_name] = [len(rounds_gap_df), 0, len(rounds_good_vpc_df), 0, len(rounds_binary_instances_df), 0]

rounds_avg_gap_df.iloc[1] = ["" for i in range(ncols+1)]
rounds_avg_gap_df.iloc[3] = ["" for i in range(ncols+1)]
rounds_avg_gap_df.iloc[5] = ["" for i in range(ncols+1)]

display(rounds_avg_gap_df)

/var/folders/_8/bgx4yr0n6f1dm56pnvsqqb1s91dzml/T/ipykernel_59358/4122335831.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  rounds_avg_gap_df.iloc[1] = ["" for i in range(ncols+1)]


G          R         DB          V   max(G,V)  \
Set                                                                         
All       Avg (\%)  22.272732  11.674826  18.425291  15.282947   25.55777   
          Wins                                                              
$\ge$10\% Avg (\%)  28.879002  23.758833   34.77734  30.547997  35.796323   
          Wins                                                              
Binary    Avg (\%)  22.620802  15.191587  19.508952  18.162172  27.456728   
          Wins                                                              

                          V+G       GurF     V+GurF       GurL     V+GurL  \
Set                                                                         
All       Avg (\%)  26.306358  29.157278  35.211859  48.484295  52.675635   
          Wins                                                              
$\ge$10\% Avg (\%)  37.090814  29.940066  40.209613  50.333952  58.692071   
          Wins                                                              
Binary    Avg (\%)  28.339375  19.578793  31.504306  33.608415  43.877678   
          Wins                                                              

                   # inst  
Set                        
All       Avg (\%)    322  
          Wins             
$\ge$10\% Avg (\%)    146  
          Wins             
Binary    Avg (\%)     60  
          Wins

### `rounds_wins_df`: number of wins across methods

In [73]:
## Create num wins df
# x wins over y for an instance if x > y + EPS
#shortcols = avg_gap_df.columns[0:-1]
rounds_wins_df = pd.DataFrame(
    np.zeros((len(gap_cols_short), len(gap_cols_short)), dtype=int),
    columns = gap_cols_short,
    index = gap_cols_short,
    dtype = int,
)

WINS_EPS = GAP_DIFF_EPS

from itertools import permutations
for (ind1, ind2) in permutations(range(len(gap_cols_short)), 2):
    rounds_wins_df.at[gap_cols_short[ind1],gap_cols_short[ind2]] =\
        int(sum(rounds_gap_df[gap_cols_short[ind1]] > rounds_gap_df[gap_cols_short[ind2]] + WINS_EPS))
    rounds_wins_df.at[gap_cols_short[ind2],gap_cols_short[ind1]] =\
        int(sum(rounds_gap_df[gap_cols_short[ind2]] > rounds_gap_df[gap_cols_short[ind1]] + WINS_EPS))

# Sets we are considering
# all_set = 'Wins (All)'
# good_vpc_set = 'Wins (V ≥ 10%)'
rounds_all_set = (all_set_name,wins_row_name)
rounds_good_vpc_set = (good_vpc_set_name,wins_row_name)
rounds_binary_set = (binary_set_name,wins_row_name)

# "G" are wins relative to "V"
shortrefcol = 'V'
#refcol = 'VPC % GAP CLOSED'
#refcol = map_short_to_cols[shortrefcol]
refcol = shortrefcol
shortdestcol = 'G'
#col = 'GMIC % GAP CLOSED'
#col = map_short_to_cols[shortcol]
destcol = shortdestcol
rounds_avg_gap_df.at[rounds_all_set,shortdestcol] = rounds_wins_df.at[shortdestcol,shortrefcol]
rounds_avg_gap_df.at[rounds_good_vpc_set,shortdestcol] = sum(rounds_good_vpc_df[destcol] > rounds_good_vpc_df[refcol] + WINS_EPS)
rounds_avg_gap_df.at[rounds_binary_set,shortdestcol] = sum(rounds_binary_instances_df[destcol] > rounds_binary_instances_df[refcol] + WINS_EPS)

# "DB", "V", "V+G": wins are relative to "G"
shortrefcol = 'G'
#refcol = 'GMIC % GAP CLOSED'
#refcol = map_short_to_cols[shortrefcol]
refcol = shortrefcol
shortdestcol = 'DB'
#col = 'BEST DISJ % GAP CLOSED'
#col = map_short_to_cols[shortcol]
destcol = shortdestcol
rounds_avg_gap_df.at[rounds_all_set,shortdestcol] = rounds_wins_df.at[shortdestcol,shortrefcol]
rounds_avg_gap_df.at[rounds_good_vpc_set,shortdestcol] = sum(rounds_good_vpc_df[destcol] > rounds_good_vpc_df[refcol] + WINS_EPS)
rounds_avg_gap_df.at[rounds_binary_set,shortdestcol] = sum(rounds_binary_instances_df[destcol] > rounds_binary_instances_df[refcol] + WINS_EPS)

shortdestcol = 'V'
#col = 'VPC % GAP CLOSED'
#col = map_short_to_cols[shortcol]
destcol = shortdestcol
rounds_avg_gap_df.at[rounds_all_set,shortdestcol] = rounds_wins_df.at[shortdestcol,shortrefcol]
rounds_avg_gap_df.at[rounds_good_vpc_set,shortdestcol] = sum(rounds_good_vpc_df[destcol] > rounds_good_vpc_df[refcol] + WINS_EPS)
rounds_avg_gap_df.at[rounds_binary_set,shortdestcol] = sum(rounds_binary_instances_df[destcol] > rounds_binary_instances_df[refcol] + WINS_EPS)

shortdestcol = 'V+G'
#col = 'VPC+GMIC % GAP CLOSED'
#col = map_short_to_cols[shortcol]
destcol = shortdestcol
rounds_avg_gap_df.at[rounds_all_set,shortdestcol] = rounds_wins_df.at[shortdestcol,shortrefcol]
rounds_avg_gap_df.at[rounds_good_vpc_set,shortdestcol] = sum(rounds_good_vpc_df[destcol] > rounds_good_vpc_df[refcol] + WINS_EPS)
rounds_avg_gap_df.at[rounds_binary_set,shortdestcol] = sum(rounds_binary_instances_df[destcol] > rounds_binary_instances_df[refcol] + WINS_EPS)

# "V+GurF" are wins relative to "GurF"
shortrefcol = 'GurF'
refcol = shortrefcol
shortdestcol = 'V+GurF'
destcol = shortdestcol
#col = map_short_to_cols[shortcol]
rounds_avg_gap_df.at[rounds_all_set,shortdestcol] = rounds_wins_df.at[shortdestcol,shortrefcol]
rounds_avg_gap_df.at[rounds_good_vpc_set,shortdestcol] = sum(rounds_good_vpc_df[destcol] > rounds_good_vpc_df[refcol] + WINS_EPS)
rounds_avg_gap_df.at[rounds_binary_set,shortdestcol] = sum(rounds_binary_instances_df[destcol] > rounds_binary_instances_df[refcol] + WINS_EPS)

# "V+GurL" are wins relative to "GurL"
shortrefcol = 'GurL'
refcol = shortrefcol
shortdestcol = 'V+GurL'
destcol = shortdestcol
rounds_wins_df.at[shortdestcol,shortrefcol] = int(sum(rounds_gap_df[destcol] > rounds_gap_df[refcol] + WINS_EPS))
rounds_wins_df.at[shortrefcol,shortdestcol] = int(sum(rounds_gap_df[refcol] > rounds_gap_df[destcol] + WINS_EPS))
rounds_avg_gap_df.at[rounds_all_set,shortdestcol] = rounds_wins_df.at[shortdestcol,shortrefcol]
rounds_avg_gap_df.at[rounds_good_vpc_set,shortdestcol] = sum(rounds_good_vpc_df[destcol] > rounds_good_vpc_df[refcol] + WINS_EPS)
rounds_avg_gap_df.at[rounds_binary_set,shortdestcol] = sum(rounds_binary_instances_df[destcol] > rounds_binary_instances_df[refcol] + WINS_EPS)

# Count number of instances that have V+G > 0
shortdestcol = inst_col_name
#col = 'V+GurL'
destcol = 'V+G'
rounds_avg_gap_df.at[rounds_all_set,shortdestcol] = sum(rounds_gap_df[destcol] > WINS_EPS)
rounds_avg_gap_df.at[rounds_good_vpc_set,shortdestcol] = sum(rounds_good_vpc_df[destcol] > WINS_EPS)
rounds_avg_gap_df.at[rounds_binary_set,shortdestcol] = sum(rounds_binary_instances_df[destcol] > WINS_EPS)

display(rounds_avg_gap_df)
display(rounds_wins_df)

G          R         DB          V   max(G,V)  \
Set                                                                         
All       Avg (\%)  22.272732  11.674826  18.425291  15.282947   25.55777   
          Wins            156                   153        134              
$\ge$10\% Avg (\%)  28.879002  23.758833   34.77734  30.547997  35.796323   
          Wins             41                   111        105              
Binary    Avg (\%)  22.620802  15.191587  19.508952  18.162172  27.456728   
          Wins             22                    37         32              

                          V+G       GurF     V+GurF       GurL     V+GurL  \
Set                                                                         
All       Avg (\%)  26.306358  29.157278  35.211859  48.484295  52.675635   
          Wins            228                   246                   166   
$\ge$10\% Avg (\%)  37.090814  29.940066  40.209613  50.333952  58.692071   
          Wins            138                   130                    99   
Binary    Avg (\%)  28.339375  19.578793  31.504306  33.608415  43.877678   
          Wins             47                    46                    38   

                   # inst  
Set                        
All       Avg (\%)    322  
          Wins        290  
$\ge$10\% Avg (\%)    146  
          Wins        146  
Binary    Avg (\%)     60  
          Wins         54

,G,R,DB,V,"max(G,V)",V+G,GurF,V+GurF,GurL,V+GurL
G,0,201,137,156,0,0,121,71,54,13
R,86,0,9,0,0,0,83,48,55,6
DB,153,248,0,242,145,128,111,98,71,58
V,134,238,19,0,0,0,103,76,65,21
"max(G,V)",134,280,137,156,0,0,144,103,74,28
V+G,228,283,154,248,186,0,149,115,75,33
GurF,169,213,184,193,146,141,0,0,0,0
V+GurF,220,245,195,216,183,171,246,0,54,0
GurL,241,248,231,238,222,221,285,238,0,113
V+GurL,284,296,242,280,266,261,293,282,166,0


### Update Table 2 `gap_by_size_df` with rounds results

In [74]:
# Create a new row with label "Rounds"
rounds_row = pd.DataFrame(columns=gap_by_size_df.columns, index=["Rounds"])

# Calculate the average values for each column from the `rounds` dataframe
rounds_row = rounds_avg_gap_df.loc['All',gap_by_size_df.columns][0:1].to_numpy()

# Append the new row to the `gap_by_size` dataframe
gap_by_size_df.loc['Rounds'] = rounds_row[0]

# Display the updated `gap_by_size` dataframe
display(gap_by_size_df)


,DB,V,"max(G,V)",V+G,V+GurF,V+GurL
VPCs disabled,0.000000,0.000000,16.508148,16.508148,28.929744,48.319147
2 leaves,3.014507,2.213289,16.789746,17.206579,30.961171,49.260530
4 leaves,5.270242,3.592498,17.227532,17.785031,31.164402,49.252142
8 leaves,8.117860,4.847477,17.828119,18.533394,31.526034,49.484617
16 leaves,11.173444,6.456210,19.373155,20.029255,32.470786,50.236188
32 leaves,14.671544,8.741057,20.957474,21.773258,33.413550,51.140841
64 leaves,18.369423,10.233224,22.139001,22.887076,33.825053,51.638833
Best,18.369423,12.008265,23.007369,23.916435,34.801454,52.806027
Combined,17.795569,12.487033,23.207997,24.123909,34.461474,52.074567
Rounds,18.425291,15.282947,25.557770,26.306358,35.211859,52.675635


### How long did it take to generate these cuts, on average, compared to just one round with half the cut limit?

In [75]:
intersection_disjset_rounds_gap_instances = list(set(selected_disjset_gap_instances) & set(selected_rounds_gap_instances))
disjset_avg_gen_time = df_disjset.loc[intersection_disjset_rounds_gap_instances,[col_vpc_gen_time]].mean().iloc[0]
rounds_avg_gen_time = df_rounds.loc[intersection_disjset_rounds_gap_instances,[col_vpc_gen_time]].mean().iloc[0]
print("Average VPC generation time for disjset instances: {:.2f}".format(disjset_avg_gen_time))
print("Average VPC generation time for rounds instances: {:.2f}".format(rounds_avg_gen_time))

# Repeat for instances by disj size
curr_df = df.loc[intersection_disjset_rounds_gap_instances,[col_vpc_gen_time]]
for size in sizes:
    # report average time to generate vpcs from df for this size
    curr_size_avg_gen_time = curr_df.xs(size, level='disj_terms')[col_vpc_gen_time].mean()
    print("Average VPC generation time for instances of size {:d}: {:.2f}".format(size,curr_size_avg_gen_time))

Average VPC generation time for disjset instances: 400.99
Average VPC generation time for rounds instances: 1219.63
Average VPC generation time for instances of size 2: 18.14
Average VPC generation time for instances of size 4: 53.56
Average VPC generation time for instances of size 8: 83.89
Average VPC generation time for instances of size 16: 90.70
Average VPC generation time for instances of size 32: 146.76
Average VPC generation time for instances of size 64: 201.36


## rounds timing summary

### `rounds_timing`

In [76]:
inst_set = selected_rounds_time_instances
NUM_SEEDS = 7
rounds_timing_cols = ['ALL' + ' ' + solver_type + ' ' + time_stub for solver_type in solver_stubs] + [ref_time_col, refv_time_col] + [col_vpc_gen_time] + [col_num_vpc]
new_rounds_timing_cols = [ 
        [
            solver_type + ' ' + '(%d)' % seed + ' ' + time_stub
            for seed in range(1,NUM_SEEDS+1)
        ]
        for solver_type in solver_stubs
    ]
new_rounds_wins_cols = [
    "Wins ({} over {})".format(solver_stubs[0], solver_stubs[1]),
    "Wins ({} over {})".format(solver_stubs[1], solver_stubs[0])
]
rounds_timing_df = df_rounds.loc[inst_set, rounds_timing_cols]

# Replace values of rounds_timing_df associated to ref with the values from disjset_timing_df for the same columns
cols = ['ALL' + ' ' + solver_stubs[0] + ' ' + time_stub, ref_time_col]
rounds_timing_df.loc[:,cols] = disjset_timing_df.loc[:,cols]

# display(disjet_timing_df.head())

# Add columns solver_type (%d) + ' ' + time_stub for the 7 random seeds
# These will be parsed from the ALL REF TIME and ALL REF+V TIME columns,
# which contain semicolon-separated values
for seed_ind in range(NUM_SEEDS):
    for solver_ind in range(len(solver_stubs)):
        solver_type = solver_stubs[solver_ind]
        orig_col = rounds_timing_cols[solver_ind]
        new_col = new_rounds_timing_cols[solver_ind][seed_ind]
        rounds_timing_df[new_col] = rounds_timing_df[orig_col].str.split(';').str[seed_ind]
        rounds_timing_df[new_col] = rounds_timing_df[new_col].astype(float)

# Add wins columns
for solver_ind in range(len(solver_stubs)):
    other_solver_ind = 1 - solver_ind

    rounds_timing_df[new_rounds_wins_cols[solver_ind]] = sum(
        rounds_timing_df[new_rounds_timing_cols[other_solver_ind][seed_ind]]
        > 
        WIN_BY_TIME_FACTOR * rounds_timing_df[new_rounds_timing_cols[solver_ind][seed_ind]]
        for seed_ind in range(NUM_SEEDS)
    )

display(rounds_timing_df.head())

,,ALL REF TIME,ALL REF+V TIME,AVG REF TIME,AVG REF+V TIME,VPC_GEN_TIME,NUM VPC,REF (1) TIME,REF+V (1) TIME,REF (2) TIME,REF+V (2) TIME,...,REF (4) TIME,REF+V (4) TIME,REF (5) TIME,REF+V (5) TIME,REF (6) TIME,REF+V (6) TIME,REF (7) TIME,REF+V (7) TIME,Wins (REF over REF+V),Wins (REF+V over REF)
INSTANCE,disj_terms,,,,,,,,,,,,,,,,,,,,,
10teams_presolved,-1,0.303684;2.535196;0.989145;0.593072;4.379637;0...,3.114999;1.029250;0.540013;1.451543;5.481126;0...,1.829,2.035,4113.95,614,0.303684,3.114999,2.535196,1.029250,...,0.593072,1.451543,4.379637,5.481126,0.914021,0.819071,3.090965,1.811770,3,4
23588_presolved,-1,0.484330;0.425422;0.400010;0.514678;0.474368;0...,1.453255;1.912989;1.212225;2.229727;0.927447;0...,0.519,1.310,252.76,1616,0.484330,1.453255,0.425422,1.912989,...,0.514678,2.229727,0.474368,0.927447,0.890932,0.737682,0.445473,0.697561,6,1
30n20b8_presolved,-1,10.868823;20.671000;16.525700;15.207604;15.926...,22.476277;19.315350;20.683366;20.214293;23.203...,15.586,19.917,6200.41,1569,10.868823,22.476277,20.671000,19.315350,...,15.207604,20.214293,15.926806,23.203611,19.154248,21.492474,10.747110,12.032215,6,0
50v-10_presolved,-1,3600.000062;1573.493537;3600.000113;2817.90685...,2876.623524;3600.000106;1096.497703;3289.76020...,2893.849,2462.218,409.76,539,3600.000062,2876.623524,1573.493537,3600.000106,...,2817.906855,3289.760203,1939.052187,1821.783865,3126.488219,2588.944642,3600.000120,1961.912704,2,4
a1c1s1_presolved,-1,807.966638;431.850770;453.278678;639.477005;53...,585.815723;577.521918;638.560913;464.078481;41...,531.318,529.772,18.25,45,807.966638,585.815723,431.850770,577.521918,...,639.477005,464.078481,536.304876,418.991635,416.303954,428.449108,434.046231,594.985613,3,3


In [77]:
rounds_timing_df.to_csv("disjet_timing.csv")

### `rounds_time_geomean_df`

In [78]:
# Row names
rounds_bb_classes = ['Rounds','RoundsBinary']
rounds_bb_classes_lists = [ selected_rounds_time_instances, [inst for inst in pure_binary_instances if inst in selected_rounds_time_instances] ]
rounds_num_bb_classes = len(rounds_bb_classes)

# Rows are geomean and wins for each bucket
rounds_bb_row_names = pd.MultiIndex.from_product(
    [rounds_bb_classes, bb_buckets, bb_metrics],
    names=['class', 'bucket', 'metric'])
rounds_time_geomean_cols = [
    [
        new_rounds_timing_cols[solver_ind][seed_ind] 
        for seed_ind in range(NUM_SEEDS) 
        for solver_ind in range(len(solver_stubs))
    ]
    + [ref_time_col, refv_time_col]
    + [col_vpc_gen_time]
    + new_rounds_wins_cols
]
rounds_time_geomean_df = pd.DataFrame(
    columns = rounds_time_geomean_cols,
    index = rounds_bb_row_names,
    dtype = float
)

# Make all columns "object" type to allow for integer values
rounds_time_geomean_df.loc[:,(rounds_time_geomean_cols[0])] = rounds_time_geomean_df.loc[:,(rounds_time_geomean_cols[0])].astype(object)

# Prepare num_inst columns
rounds_num_inst_by_seed_bucket_class = [
    np.zeros(len(rounds_time_geomean_df),dtype = np.int64)
    for _ in range(NUM_SEEDS) 
    # for _ in range(num_buckets) 
    # for _ in range(rounds_num_bb_classes)
]
rounds_num_inst_avg = np.zeros(len(rounds_time_geomean_df),dtype = np.int64)
rounds_num_ref_seed_wins = np.zeros(len(rounds_time_geomean_df),dtype = np.int64)
rounds_num_refv_seed_wins = np.zeros(len(rounds_time_geomean_df),dtype = np.int64)

# For each class, calculate geomean
for seed_ind in range(NUM_SEEDS):
    row_ind = 0
    for class_ind in range(rounds_num_bb_classes):
        print("\nClass: {}".format(rounds_bb_classes[class_ind]))
        curr_df = rounds_timing_df.loc[rounds_bb_classes_lists[class_ind]]
        seed_cols = [ new_rounds_timing_cols[solver_ind][seed_ind] for solver_ind in range(len(solver_stubs)) ]
        seed_curr_df = curr_df[seed_cols]

        # For every instance in which curr_by_depth_df[col] >= MAX_TIME - EPS, multiply by TIMEOUT_TIME_FACTOR
        for col_ind in range(len(seed_cols)):
            time_col = seed_cols[col_ind]
            seed_curr_df.loc[seed_curr_df[time_col] >= MAX_TIME - EPS, time_col] = TIMEOUT_TIME_FACTOR * MAX_TIME

        for bucket_ind in range(num_buckets):
            print("Bucket: [{:d},{:d})".format(bucket_min[bucket_ind],bucket_max[bucket_ind]))

            # Take subset of instances for which both solvers (with this seed) solve the instance in the time frame for the bucket
            seed_curr_df = seed_curr_df[
                (seed_curr_df[seed_cols].min(axis=1) >= bucket_min[bucket_ind])
                & (seed_curr_df[seed_cols].min(axis=1) < bucket_max[bucket_ind] - EPS)
            ]

            # Calculate geomean
            metric_ind = 0
            rounds_time_geomean_df.loc[
                (rounds_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
                seed_cols
            ] = [
                    geometric_mean(seed_curr_df[col] + SHIFT_TIME) - SHIFT_TIME
                    for col in seed_cols
                ]
            
            # Calculate wins
            metric_ind = 1
            for solver_ind in range(len(solver_stubs)):
                other_solver_ind = 1 - solver_ind
                rounds_time_geomean_df.loc[
                    (rounds_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
                    seed_cols[solver_ind]
                ] = sum(
                    seed_curr_df[seed_cols[other_solver_ind]]
                    > 
                    WIN_BY_TIME_FACTOR * seed_curr_df[seed_cols[solver_ind]]
                )

            print("row {:d}: {:d}".format(row_ind,len(seed_curr_df)))
            rounds_num_inst_by_seed_bucket_class[seed_ind][row_ind:row_ind+len(bb_metrics)] = [len(seed_curr_df)] * len(bb_metrics)

            row_ind += len(bb_metrics)

            # for solver_ind in range(len(solver_stubs)):
            #     other_solver_ind = 1 - solver_ind
            #     rounds_time_geomean_df.loc[
            #         (rounds_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
            #         new_rounds_wins_cols[solver_ind]
            #     ] = sum(
            #         seed_curr_df[new_rounds_timing_cols[other_solver_ind][seed_ind]]
            #         > 
            #         WIN_BY_TIME_FACTOR * seed_curr_df[new_rounds_timing_cols[solver_ind][seed_ind]]
            #     )


row_ind = 0
only_time_cols = [ ref_time_col, refv_time_col ]
cols = time_cols_long
wins_cols = new_rounds_wins_cols
for class_ind in range(rounds_num_bb_classes):
    print("\nClass: {}".format(rounds_bb_classes[class_ind]))
    curr_df = rounds_timing_df.loc[rounds_bb_classes_lists[class_ind]]

    # For every instance in which curr_by_depth_df[col] >= MAX_TIME - EPS, multiply by TIMEOUT_TIME_FACTOR
    for col_ind in range(len(only_time_cols)):
        time_col = only_time_cols[col_ind]
        curr_df.loc[curr_df[time_col] >= MAX_TIME - EPS, time_col] = TIMEOUT_TIME_FACTOR * MAX_TIME
        
    for bucket_ind in range(num_buckets):
        print("Bucket: [{:d},{:d})".format(bucket_min[bucket_ind],bucket_max[bucket_ind]))

        # Take subset of instances for which both solvers (with this seed) solve the instance in the time frame for the bucket
        curr_df = curr_df[
            (curr_df[only_time_cols].min(axis=1) >= bucket_min[bucket_ind])
            & (curr_df[only_time_cols].min(axis=1) < bucket_max[bucket_ind] - EPS)
        ]

        # Calculate geomean
        metric_ind = 0
        rounds_time_geomean_df.loc[
            (rounds_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
            cols
        ] = [
                geometric_mean(curr_df[col] + SHIFT_TIME) - SHIFT_TIME
                for col in cols
            ]
        
        # Calculate wins
        metric_ind = 1
        for solver_ind in range(len(solver_stubs)):
            other_solver_ind = 1 - solver_ind
            rounds_time_geomean_df.loc[
                (rounds_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
                only_time_cols[solver_ind]
            ] = sum(
                curr_df[only_time_cols[other_solver_ind]]
                > 
                WIN_BY_TIME_FACTOR * curr_df[only_time_cols[solver_ind]]
            )

            # Also report average number of seeds for which REF and REF+V win
            rounds_time_geomean_df.loc[
                (rounds_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[0:len(bb_metrics)]),
                wins_cols[solver_ind]
            ] = sum(curr_df[wins_cols[solver_ind]]) / len(curr_df)

        print("row {:d}: {:d}".format(row_ind,len(curr_df)))
        rounds_num_inst_avg[row_ind:row_ind+len(bb_metrics)] = [len(curr_df)] * len(bb_metrics)

        row_ind += len(bb_metrics)

# Insert columns with # inst (seed)
for seed_ind in range(NUM_SEEDS):
    rounds_time_geomean_df.insert(
        2 + seed_ind * (len(solver_stubs) + 1),
        "# inst ({:d})".format(seed_ind+1),
        rounds_num_inst_by_seed_bucket_class[seed_ind]
    )

# Insert columns with # inst (avg)
rounds_time_geomean_df.insert(
    2 + NUM_SEEDS * (len(solver_stubs) + 1),
    "# inst (avg)",
    rounds_num_inst_avg
)

rounds_time_geomean_df
    


Class: Rounds
Bucket: [0,3600)
row 0: 277
Bucket: [10,3600)
row 2: 108
Bucket: [100,3600)
row 4: 51
Bucket: [1000,3600)
row 6: 13

Class: RoundsBinary
Bucket: [0,3600)
row 8: 49
Bucket: [10,3600)
row 10: 20
Bucket: [100,3600)
row 12: 11
Bucket: [1000,3600)
row 14: 3

Class: Rounds
Bucket: [0,3600)
row 0: 276
Bucket: [10,3600)
row 2: 111
Bucket: [100,3600)
row 4: 48
Bucket: [1000,3600)
row 6: 14

Class: RoundsBinary
Bucket: [0,3600)
row 8: 49
Bucket: [10,3600)
row 10: 20
Bucket: [100,3600)
row 12: 11
Bucket: [1000,3600)
row 14: 4

Class: Rounds
Bucket: [0,3600)
row 0: 276
Bucket: [10,3600)
row 2: 110
Bucket: [100,3600)
row 4: 50
Bucket: [1000,3600)
row 6: 15

Class: RoundsBinary
Bucket: [0,3600)
row 8: 49
Bucket: [10,3600)
row 10: 22
Bucket: [100,3600)
row 12: 12
Bucket: [1000,3600)
row 14: 4

Class: Rounds
Bucket: [0,3600)
row 0: 276
Bucket: [10,3600)
row 2: 108
Bucket: [100,3600)
row 4: 51
Bucket: [1000,3600)
row 6: 13

Class: RoundsBinary
Bucket: [0,3600)
row 8: 49
Bucket: [10,3600)

REF (1) TIME REF+V (1) TIME # inst (1)  \
class        bucket      metric                                          
Rounds       [0,3600)    Gmean      10.57909      10.541608        277   
                         Wins            138             82        277   
             [10,3600)   Gmean    135.266348     142.889406        108   
                         Wins             46             41        108   
             [100,3600)  Gmean     554.72825      640.79419         51   
                         Wins             22             16         51   
             [1000,3600) Gmean   2159.708614    2039.134341         13   
                         Wins              6              4         13   
RoundsBinary [0,3600)    Gmean     12.238309      12.717076         49   
                         Wins             28             14         49   
             [10,3600)   Gmean    207.340604     219.885377         20   
                         Wins             10              8         20   
             [100,3600)  Gmean     643.46797      972.31086         11   
                         Wins              7              3         11   
             [1000,3600) Gmean   1283.104123    1527.204131          3   
                         Wins              3              0          3   

                                REF (2) TIME REF+V (2) TIME # inst (2)  \
class        bucket      metric                                          
Rounds       [0,3600)    Gmean     10.034644      10.813111        276   
                         Wins            142             85        276   
             [10,3600)   Gmean     130.38982     141.450562        111   
                         Wins             45             45        111   
             [100,3600)  Gmean    594.330099      662.68182         48   
                         Wins             21             18         48   
             [1000,3600) Gmean   2078.098668    2113.973276         14   
                         Wins              5              8         14   
RoundsBinary [0,3600)    Gmean     11.662561      11.426557         49   
                         Wins             28             15         49   
             [10,3600)   Gmean    191.408508      194.77399         20   
                         Wins             10              8         20   
             [100,3600)  Gmean    639.057795      693.32154         11   
                         Wins              6              4         11   
             [1000,3600) Gmean   1513.797962    1419.694694          4   
                         Wins              2              2          4   

                                REF (3) TIME REF+V (3) TIME # inst (3)  \
class        bucket      metric                                          
Rounds       [0,3600)    Gmean      10.07846      10.856004        276   
                         Wins            136             79        276   
             [10,3600)   Gmean    126.432743     137.386336        110   
                         Wins             50             39        110   
             [100,3600)  Gmean    562.071208     626.025244         50   
                         Wins             25             13         50   
             [1000,3600) Gmean   2005.732393    1853.688202         15   
                         Wins              7              6         15   
RoundsBinary [0,3600)    Gmean     12.192341      15.219341         49   
                         Wins             33             11         49   
             [10,3600)   Gmean    160.017607     235.206558         22   
                         Wins             16              4         22   
             [100,3600)  Gmean    617.502814     1002.13326         12   
                         Wins             10              2         12   
             [1000,3600) Gmean   1355.736872     1508.86059          4   
                         Wins              3              1          4   

                                R

## rounds nodes summary

### rounds_nodes

In [79]:
inst_set = selected_rounds_time_instances

rounds_nodes_cols = \
    ['ALL' + ' ' + solver_type + ' ' + nodes_stub for solver_type in solver_stubs] \
    + \
    [ref_nodes_col, refv_nodes_col] \
    + \
    [col_num_vpc]
new_rounds_nodes_cols = [ 
        [
            solver_type + ' ' + '(%d)' % seed + ' ' + nodes_stub
            for seed in range(1,NUM_SEEDS+1)
        ]
        for solver_type in solver_stubs
    ]
new_rounds_wins_cols = [
    "Wins ({} over {})".format(solver_stubs[0], solver_stubs[1]),
    "Wins ({} over {})".format(solver_stubs[1], solver_stubs[0])
]
rounds_nodes_df = df_rounds.loc[inst_set, rounds_nodes_cols]
# display(disjet_timing_df.head())

# Replace values of rounds_timing_df associated to ref with the values from disjset_timing_df for the same columns
cols = ['ALL' + ' ' + solver_stubs[0] + ' ' + nodes_stub, ref_nodes_col]
rounds_nodes_df.loc[:,cols] = disjset_nodes_df.loc[:,cols]

# Add columns solver_type (%d) + ' ' + time_stub for the 7 random seeds
# These will be parsed from the ALL REF TIME and ALL REF+V TIME columns,
# which contain semicolon-separated values
for seed_ind in range(NUM_SEEDS):
    for solver_ind in range(len(solver_stubs)):
        solver_type = solver_stubs[solver_ind]
        orig_col = rounds_nodes_cols[solver_ind]
        new_col = new_rounds_nodes_cols[solver_ind][seed_ind]
        rounds_nodes_df[new_col] = rounds_nodes_df[orig_col].str.split(';').str[seed_ind]
        rounds_nodes_df[new_col] = rounds_nodes_df[new_col].astype(float)

# Add wins columns
for solver_ind in range(len(solver_stubs)):
    other_solver_ind = 1 - solver_ind

    rounds_nodes_df[new_rounds_wins_cols[solver_ind]] = sum(
        rounds_nodes_df[new_rounds_nodes_cols[other_solver_ind][seed_ind]]
        > 
        WIN_BY_TIME_FACTOR * rounds_nodes_df[new_rounds_nodes_cols[solver_ind][seed_ind]]
        for seed_ind in range(NUM_SEEDS)
    )

display(rounds_nodes_df.head())

,,ALL REF NODES,ALL REF+V NODES,AVG REF NODES,AVG REF+V NODES,NUM VPC,REF (1) NODES,REF+V (1) NODES,REF (2) NODES,REF+V (2) NODES,REF (3) NODES,...,REF (4) NODES,REF+V (4) NODES,REF (5) NODES,REF+V (5) NODES,REF (6) NODES,REF+V (6) NODES,REF (7) NODES,REF+V (7) NODES,Wins (REF over REF+V),Wins (REF+V over REF)
INSTANCE,disj_terms,,,,,,,,,,,,,,,,,,,,,
10teams_presolved,-1,1;515;226;1;577;123;517,528;186;1;515;580;77;299,280,312,614,1.0,528.0,515.0,186.0,226.0,...,1.0,515.0,577.0,580.0,123.0,77.0,517.0,299.0,2,4
23588_presolved,-1,654;906;668;997;813;1293;844,2431;2346;2153;1721;1648;1365;1347,882,1858,1616,654.0,2431.0,906.0,2346.0,668.0,...,997.0,1721.0,813.0,1648.0,1293.0,1365.0,844.0,1347.0,6,0
30n20b8_presolved,-1,536;812;550;541;598;2072;557,1986;1924;1763;1789;2108;1771;539,809,1697,1569,536.0,1986.0,812.0,1924.0,550.0,...,541.0,1789.0,598.0,2108.0,2072.0,1771.0,557.0,539.0,5,1
50v-10_presolved,-1,1698842;860977;1674112;1347590;1040288;1860918...,1319752;2132440;507235;1784481;1098538;1446033...,1495008,1321042,539,1698842.0,1319752.0,860977.0,2132440.0,1674112.0,...,1347590.0,1784481.0,1040288.0,1098538.0,1860918.0,1446033.0,1982335.0,958819.0,2,4
a1c1s1_presolved,-1,63571;41748;46488;54947;40220;35137;40351,50236;52005;65006;40834;38099;42305;50076,46066,48365,45,63571.0,50236.0,41748.0,52005.0,46488.0,...,54947.0,40834.0,40220.0,38099.0,35137.0,42305.0,40351.0,50076.0,4,2


In [80]:
rounds_nodes_df.to_csv("rounds_nodes.csv")

### rounds_nodes_geomean_df

In [81]:
# Row names
rounds_nodes_geomean_cols = [
    [
        new_rounds_nodes_cols[solver_ind][seed_ind] 
        for seed_ind in range(NUM_SEEDS) 
        for solver_ind in range(len(solver_stubs))
    ]
    + [ref_nodes_col, refv_nodes_col]
    + new_rounds_wins_cols
]
rounds_nodes_geomean_df = pd.DataFrame(
    columns = rounds_nodes_geomean_cols,
    index = rounds_bb_row_names,
    dtype = float
)

# Make all columns "object" type to allow for integer values
rounds_nodes_geomean_df.loc[:,(rounds_nodes_geomean_cols[0])] = rounds_nodes_geomean_df.loc[:,(rounds_nodes_geomean_cols[0])].astype(object)

# Prepare num_inst columns
rounds_num_inst_by_seed_bucket_class = [
    np.zeros(len(rounds_nodes_geomean_df),dtype = np.int64)
    for _ in range(NUM_SEEDS) 
]
rounds_num_inst_avg = np.zeros(len(rounds_nodes_geomean_df),dtype = np.int64)
rounds_num_ref_seed_wins = np.zeros(len(rounds_nodes_geomean_df),dtype = np.int64)
rounds_num_refv_seed_wins = np.zeros(len(rounds_nodes_geomean_df),dtype = np.int64)

# For each class, calculate geomean
for seed_ind in range(NUM_SEEDS):
    row_ind = 0
    for class_ind in range(rounds_num_bb_classes):
        print("\nClass: {}".format(rounds_bb_classes[class_ind]))
        curr_df = rounds_nodes_df.loc[rounds_bb_classes_lists[class_ind]]
        seed_cols = [ new_rounds_nodes_cols[solver_ind][seed_ind] for solver_ind in range(len(solver_stubs)) ]
        seed_curr_df = curr_df[seed_cols]
        
        time_curr_df = rounds_timing_df.loc[rounds_bb_classes_lists[class_ind]]
        time_seed_cols = [ new_rounds_timing_cols[solver_ind][seed_ind] for solver_ind in range(len(solver_stubs)) ]
        seed_time_curr_df = time_curr_df[time_seed_cols]

        # For every instance in which curr_by_depth_df[col] >= MAX_TIME - EPS, multiply nodes processed by TIMEOUT_NODE_FACTOR
        for col_ind in range(len(time_seed_cols)):
            time_col = time_seed_cols[col_ind]
            node_col = seed_cols[col_ind]
            curr_selected_indices = seed_time_curr_df[time_col] >= MAX_TIME - EPS
            seed_curr_df.loc[curr_selected_indices, node_col] = TIMEOUT_NODE_FACTOR * seed_curr_df[node_col]

        for bucket_ind in range(num_buckets):
            print("Bucket: [{:d},{:d})".format(bucket_min[bucket_ind],bucket_max[bucket_ind]))

            # Take subset of instances for which both solvers (with this seed) solve the instance in the time frame for the bucket
            curr_selected_indices = \
                (seed_time_curr_df[time_seed_cols].min(axis=1) >= bucket_min[bucket_ind]) \
                & (seed_time_curr_df[time_seed_cols].min(axis=1) < bucket_max[bucket_ind] - EPS)
            seed_curr_df = seed_curr_df[curr_selected_indices]
            seed_time_curr_df = seed_time_curr_df[curr_selected_indices]

            # Calculate geomean
            metric_ind = 0
            rounds_nodes_geomean_df.loc[
                (rounds_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
                seed_cols
            ] = [
                    geometric_mean(seed_curr_df[col] + SHIFT_NODES) - SHIFT_NODES
                    for col in seed_cols
                ]
            
            # Calculate wins
            metric_ind = 1
            for solver_ind in range(len(solver_stubs)):
                other_solver_ind = 1 - solver_ind
                rounds_nodes_geomean_df.loc[
                    (rounds_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
                    seed_cols[solver_ind]
                ] = sum(
                    seed_curr_df[seed_cols[other_solver_ind]]
                    > 
                    WIN_BY_NODES_FACTOR * seed_curr_df[seed_cols[solver_ind]]
                )

            print("row {:d}: {:d}".format(row_ind,len(seed_curr_df)))
            rounds_num_inst_by_seed_bucket_class[seed_ind][row_ind:row_ind+len(bb_metrics)] = [len(seed_curr_df)] * len(bb_metrics)

            row_ind += len(bb_metrics)

row_ind = 0
nodes_cols = [ ref_nodes_col, refv_nodes_col ]
time_cols = [ ref_time_col, refv_time_col ]
wins_cols = new_rounds_wins_cols
for class_ind in range(rounds_num_bb_classes):
    print("\nClass: {}".format(rounds_bb_classes[class_ind]))
    curr_df = rounds_nodes_df.loc[rounds_bb_classes_lists[class_ind]]
    time_curr_df = rounds_timing_df.loc[rounds_bb_classes_lists[class_ind]]

    # For every instance in which curr_by_depth_df[col] >= MAX_TIME - EPS, multiply nodes processed by TIMEOUT_NODE_FACTOR
    for col_ind in range(len(time_cols)):
        time_col = time_cols[col_ind]
        node_col = nodes_cols[col_ind]
        curr_selected_indices = time_curr_df[time_col] >= MAX_TIME - EPS
        curr_df.loc[curr_selected_indices, node_col] = TIMEOUT_NODE_FACTOR * curr_df[node_col]
        
    for bucket_ind in range(num_buckets):
        print("Bucket: [{:d},{:d})".format(bucket_min[bucket_ind],bucket_max[bucket_ind]))

        # Take subset of instances for which both solvers (with this seed) solve the instance in the time frame for the bucket
        curr_selected_indices = \
            (time_curr_df[time_cols].min(axis=1) >= bucket_min[bucket_ind]) \
            & (time_curr_df[time_cols].min(axis=1) < bucket_max[bucket_ind] - EPS)
        curr_df = curr_df[curr_selected_indices]
        time_curr_df = time_curr_df[curr_selected_indices]

        # Calculate geomean
        metric_ind = 0
        rounds_nodes_geomean_df.loc[
            (rounds_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
            nodes_cols
        ] = [
                geometric_mean(curr_df[col] + SHIFT_NODES) - SHIFT_NODES
                for col in nodes_cols
            ]
        
        # Calculate wins
        metric_ind = 1
        for solver_ind in range(len(solver_stubs)):
            other_solver_ind = 1 - solver_ind
            rounds_nodes_geomean_df.loc[
                (rounds_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[metric_ind]),
                nodes_cols[solver_ind]
            ] = sum(
                curr_df[nodes_cols[other_solver_ind]]
                > 
                WIN_BY_NODES_FACTOR * curr_df[nodes_cols[solver_ind]]
            )

            # Also report average number of seeds for which REF and REF+V win
            rounds_nodes_geomean_df.loc[
                (rounds_bb_classes[class_ind], bb_buckets[bucket_ind], bb_metrics[0:len(bb_metrics)]),
                wins_cols[solver_ind]
            ] = sum(curr_df[wins_cols[solver_ind]]) / len(curr_df)

        print("row {:d}: {:d}".format(row_ind,len(curr_df)))
        rounds_num_inst_avg[row_ind:row_ind+len(bb_metrics)] = [len(curr_df)] * len(bb_metrics)

        row_ind += len(bb_metrics)

# Insert columns with # inst (seed)
for seed_ind in range(NUM_SEEDS):
    rounds_nodes_geomean_df.insert(
        2 + seed_ind * (len(solver_stubs) + 1),
        "# inst ({:d})".format(seed_ind+1),
        rounds_num_inst_by_seed_bucket_class[seed_ind]
    )

# Insert columns with # inst (avg)
rounds_nodes_geomean_df.insert(
    2 + NUM_SEEDS * (len(solver_stubs) + 1),
    "# inst (avg)",
    rounds_num_inst_avg
)

rounds_nodes_geomean_df



Class: Rounds
Bucket: [0,3600)
row 0: 277
Bucket: [10,3600)
row 2: 108
Bucket: [100,3600)
row 4: 51
Bucket: [1000,3600)
row 6: 13

Class: RoundsBinary
Bucket: [0,3600)
row 8: 49
Bucket: [10,3600)
row 10: 20
Bucket: [100,3600)
row 12: 11
Bucket: [1000,3600)
row 14: 3

Class: Rounds
Bucket: [0,3600)
row 0: 276
Bucket: [10,3600)
row 2: 111
Bucket: [100,3600)
row 4: 48
Bucket: [1000,3600)
row 6: 14

Class: RoundsBinary
Bucket: [0,3600)
row 8: 49
Bucket: [10,3600)
row 10: 20
Bucket: [100,3600)
row 12: 11
Bucket: [1000,3600)
row 14: 4

Class: Rounds
Bucket: [0,3600)
row 0: 276
Bucket: [10,3600)
row 2: 110
Bucket: [100,3600)
row 4: 50
Bucket: [1000,3600)
row 6: 15

Class: RoundsBinary
Bucket: [0,3600)
row 8: 49
Bucket: [10,3600)
row 10: 22
Bucket: [100,3600)
row 12: 12
Bucket: [1000,3600)
row 14: 4

Class: Rounds
Bucket: [0,3600)
row 0: 276
Bucket: [10,3600)
row 2: 108
Bucket: [100,3600)
row 4: 51
Bucket: [1000,3600)
row 6: 13

Class: RoundsBinary
Bucket: [0,3600)
row 8: 49
Bucket: [10,3600)

REF (1) NODES REF+V (1) NODES # inst (1)  \
class        bucket      metric                                             
Rounds       [0,3600)    Gmean     7267.907101     6679.166985        277   
                         Wins              114             129        277   
             [10,3600)   Gmean    69063.703612    66665.656369        108   
                         Wins               51              57        108   
             [100,3600)  Gmean   258949.132708   268707.763338         51   
                         Wins               28              23         51   
             [1000,3600) Gmean   817619.005523   704762.716812         13   
                         Wins                7               6         13   
RoundsBinary [0,3600)    Gmean     6884.547029     6400.506685         49   
                         Wins               19              21         49   
             [10,3600)   Gmean    65533.057559     61151.36461         20   
                         Wins               10              10         20   
             [100,3600)  Gmean   194093.113341   252643.812257         11   
                         Wins                6               5         11   
             [1000,3600) Gmean   850044.108629   635960.336403          3   
                         Wins                0               3          3   

                                 REF (2) NODES REF+V (2) NODES # inst (2)  \
class        bucket      metric                                             
Rounds       [0,3600)    Gmean     6667.542371     6850.530049        276   
                         Wins              124             117        276   
             [10,3600)   Gmean    62615.270907     63941.53735        111   
                         Wins               46              64        111   
             [100,3600)  Gmean   227011.197531   248435.245292         48   
                         Wins               21              26         48   
             [1000,3600) Gmean    831013.90649    830311.37477         14   
                         Wins                4               9         14   
RoundsBinary [0,3600)    Gmean     6579.959033     5791.415818         49   
                         Wins               17              21         49   
             [10,3600)   Gmean    62852.383634    54178.613798         20   
                         Wins                8              12         20   
             [100,3600)  Gmean   142841.051549   121453.868345         11   
                         Wins                4               7         11   
             [1000,3600) Gmean   616557.578682   413440.054629          4   
                         Wins                0               4          4   

                                 REF (3) NODES REF+V (3) NODES # inst (3)  \
class        bucket      metric                                             
Rounds       [0,3600)    Gmean      6728.18476     6644.382405        276   
                         Wins              124             118        276   
             [10,3600)   Gmean    56939.785568    57224.432199        110   
                         Wins               55              54        110   
             [100,3600)  Gmean   210251.607773     218046.7561         50   
                         Wins               26              23         50   
             [1000,3600) Gmean    560536.82068   497089.415336         15   
                         Wins                5              10         15   
RoundsBinary [0,3600)    Gmean     6545.862513     7552.403172         49   
                         Wins               25              14         49   
             [10,3600)   Gmean    49791.416371    67410.497945         22   
                         Wins               14               8         22   
             [100,3600)  Gmean    149681.07802   223726.320335         12   
                         Wins                8               4         12   
             [1000,3600) Gm

## Table 3: `rounds_avg_bb_df` for rounds instances

In [82]:
## Set up empty `rounds_avg_bb_df`

rounds_time_cols_short = time_cols_short
rounds_node_cols_short = node_cols_short

rounds_avg_bb_cols = pd.MultiIndex.from_arrays(
    [
      [time_col_header]*len(rounds_time_cols_short) + [node_col_header]*len(rounds_node_cols_short),
      rounds_time_cols_short + rounds_node_cols_short
    ],
    names = ['criterion', 'type'])

#bb_row_names = pd.MultiIndex.from_product([bb_buckets, bb_row_names], names=['bucket', 'metric'])
rounds_bb_row_names = pd.MultiIndex.from_product(
    [rounds_bb_classes, bb_buckets, bb_metrics],
    names=['class', 'bucket', 'metric'])

rounds_avg_bb_df = pd.DataFrame(
    columns = rounds_avg_bb_cols,
    index = rounds_bb_row_names,
    dtype = float
)

# Make all columns "object" type to allow for integer values
rounds_avg_bb_df.loc[:,(time_col_header,rounds_time_cols_short)] = rounds_avg_bb_df.loc[:,(time_col_header,rounds_time_cols_short)].astype(object)
rounds_avg_bb_df.loc[:,(node_col_header,rounds_node_cols_short)] = rounds_avg_bb_df.loc[:,(node_col_header,rounds_node_cols_short)].astype(object)

num_inst = np.zeros(len(rounds_avg_bb_df),dtype = np.int64)
row_ind = 0
cols = rounds_time_cols_short + rounds_node_cols_short

# Fill in values from rounds_time_geomean_df and rounds_nodes_geomean_df (from the ref and refv cols)
# Fill in values from rounds_time_geomean_df
for index, row in rounds_time_geomean_df.iterrows():
    class_val, bucket_val, metric_val = index
    for col in rounds_time_cols_short:
        rounds_avg_bb_df.loc[(class_val, bucket_val, metric_val), (time_col_header, col)] = \
            row[map_short_to_cols_time[col]]

# Fill in values from rounds_nodes_geomean_df
for index, row in rounds_nodes_geomean_df.iterrows():
    class_val, bucket_val, metric_val = index
    for col in rounds_node_cols_short:
        rounds_avg_bb_df.loc[(class_val, bucket_val, metric_val), (node_col_header, col)] = \
            row[map_short_to_cols_nodes[col]]
        
# Fill in num inst
rounds_avg_bb_df[inst_col_name] = rounds_num_inst_avg

display(rounds_avg_bb_df.loc[:,rounds_avg_bb_df.columns.get_level_values(0)==node_col_header].head(6))
#display(avg_bb_df.loc[(bb_classes[0], bb_buckets[1], bb_metrics[0]),:])
display(rounds_avg_bb_df.loc[(rounds_bb_classes, bb_buckets, bb_metrics[0]),:])
display(rounds_avg_bb_df)


criterion                     Nodes (#)               
type                                Gur              V
class  bucket     metric                              
Rounds [0,3600)   Gmean     7845.546959    7919.763758
                  Wins              129            123
       [10,3600)  Gmean    74002.424463    74706.00832
                  Wins               58             59
       [100,3600) Gmean   235024.497883  241114.762824
                  Wins               36             28

criterion                           Time (s)                            \
type                                     Gur            V          Gen   
class        bucket      metric                                          
Rounds       [0,3600)    Gmean     11.621243    12.470732   128.876724   
             [10,3600)   Gmean    155.225984   165.437991   216.395625   
             [100,3600)  Gmean    526.969209   558.173541   193.578409   
             [1000,3600) Gmean   1868.614466  1982.474085   166.964579   
RoundsBinary [0,3600)    Gmean     12.856481    14.338427   323.799927   
             [10,3600)   Gmean    206.231182   243.259074   748.739262   
             [100,3600)  Gmean    560.020607   691.487405   1185.58029   
             [1000,3600) Gmean   1261.153599  1420.617794  1090.439245   

criterion                             Nodes (#)                 # inst  
type                                        Gur               V         
class        bucket      metric                                         
Rounds       [0,3600)    Gmean      7845.546959     7919.763758    279  
             [10,3600)   Gmean     74002.424463     74706.00832    117  
             [100,3600)  Gmean    235024.497883   241114.762824     64  
             [1000,3600) Gmean   1041127.748276  1032613.929929     23  
RoundsBinary [0,3600)    Gmean      7107.026915     7327.794716     49  
             [10,3600)   Gmean     63749.847854    69978.656564     21  
             [100,3600)  Gmean    123394.190094   146607.698896     14  
             [1000,3600) Gmean    373596.969176    334943.38527      5

criterion                           Time (s)                            \
type                                     Gur            V          Gen   
class        bucket      metric                                          
Rounds       [0,3600)    Gmean     11.621243    12.470732   128.876724   
                         Wins            144           61          NaN   
             [10,3600)   Gmean    155.225984   165.437991   216.395625   
                         Wins             45           34          NaN   
             [100,3600)  Gmean    526.969209   558.173541   193.578409   
                         Wins             22           16          NaN   
             [1000,3600) Gmean   1868.614466  1982.474085   166.964579   
                         Wins              8            5          NaN   
RoundsBinary [0,3600)    Gmean     12.856481    14.338427   323.799927   
                         Wins             29            9          NaN   
             [10,3600)   Gmean    206.231182   243.259074   748.739262   
                         Wins              9            5          NaN   
             [100,3600)  Gmean    560.020607   691.487405   1185.58029   
                         Wins              6            3          NaN   
             [1000,3600) Gmean   1261.153599  1420.617794  1090.439245   
                         Wins              2            0          NaN   

criterion                             Nodes (#)                 # inst  
type                                        Gur               V         
class        bucket      metric                                         
Rounds       [0,3600)    Gmean      7845.546959     7919.763758    279  
                         Wins               129             123    279  
             [10,3600)   Gmean     74002.424463     74706.00832    117  
                         Wins                58              59    117  
             [100,3600)  Gmean    235024.497883   241114.762824     64  
                         Wins                36              28     64  
             [1000,3600) Gmean   1041127.748276  1032613.929929     23  
                         Wins                13              10     23  
RoundsBinary [0,3600)    Gmean      7107.026915     7327.794716     49  
                         Wins                24              17     49  
             [10,3600)   Gmean     63749.847854    69978.656564     21  
                         Wins                13               8     21  
             [100,3600)  Gmean    123394.190094   146607.698896     14  
                         Wins                 9               5     14  
             [1000,3600) Gmean    373596.969176    334943.38527      5  
                         Wins                 2               3      5

## Create `avg_bb_df` which will have values from disjset, rounds, and avg_bb_df_by_depth

In [83]:
# Merge disjset, rounds, and avg_bb_df_by_depth for 'All' class
# avg_bb_df = disjset_avg_bb_df.loc[(disjset_bb_classes[0], bb_buckets, bb_metrics),:]
avg_bb_df = pd.concat([
        disjset_avg_bb_df.loc[(disjset_bb_classes[0], bb_buckets, bb_metrics),:],
        rounds_avg_bb_df.loc[(rounds_bb_classes[0], bb_buckets, bb_metrics),:],
        avg_bb_by_depth_df.loc[:,avg_bb_by_depth_df.columns != ('Time (s)', 'Total')]
    ], 
    axis=0)

# Fill missing values with 0
avg_bb_df = avg_bb_df.fillna(0)

# Display the resulting dataframe
avg_bb_df.head(20)


criterion                       Time (s)                           \
type                                 Gur            V         Gen   
class    bucket      metric                                         
Combined [0,3600)    Gmean     11.256821    11.839758   27.617049   
                     Wins     117.000000    62.000000    0.000000   
         [10,3600)   Gmean    157.263482   168.453590   55.301806   
                     Wins      44.000000    32.000000    0.000000   
         [100,3600)  Gmean    505.119151   550.557326   43.695307   
                     Wins      32.000000    15.000000    0.000000   
         [1000,3600) Gmean   1972.953117  2073.896266   35.858057   
                     Wins      10.000000     5.000000    0.000000   
Rounds   [0,3600)    Gmean     11.621243    12.470732  128.876724   
                     Wins     144.000000    61.000000    0.000000   
         [10,3600)   Gmean    155.225984   165.437991  216.395625   
                     Wins      45.000000    34.000000    0.000000   
         [100,3600)  Gmean    526.969209   558.173541  193.578409   
                     Wins      22.000000    16.000000    0.000000   
         [1000,3600) Gmean   1868.614466  1982.474085  166.964579   
                     Wins       8.000000     5.000000    0.000000   
2 leaves [0,3600)    Gmean      9.264507     9.505446    0.561007   
                     Wins      64.000000    52.000000   28.000000   
         [10,3600)   Gmean    129.679451   139.349050    1.188230   
                     Wins      33.000000    22.000000   16.000000   

criterion                       Nodes (#)               # inst  
type                                  Gur             V         
class    bucket      metric                                     
Combined [0,3600)    Gmean   7.681913e+03  7.793801e+03    288  
                     Wins    1.280000e+02  1.310000e+02    288  
         [10,3600)   Gmean   7.519384e+04  7.695577e+04    119  
                     Wins    5.400000e+01  6.500000e+01    119  
         [100,3600)  Gmean   2.448024e+05  2.550280e+05     69  
                     Wins    3.800000e+01  3.100000e+01     69  
         [1000,3600) Gmean   1.008340e+06  9.968849e+05     24  
                     Wins    1.100000e+01  1.300000e+01     24  
Rounds   [0,3600)    Gmean   7.845547e+03  7.919764e+03    279  
                     Wins    1.290000e+02  1.230000e+02    279  
         [10,3600)   Gmean   7.400242e+04  7.470601e+04    117  
                     Wins    5.800000e+01  5.900000e+01    117  
         [100,3600)  Gmean   2.350245e+05  2.411148e+05     64  
                     Wins    3.600000e+01  2.800000e+01     64  
         [1000,3600) Gmean   1.041128e+06  1.032614e+06     23  
                     Wins    1.300000e+01  1.000000e+01     23  
2 leaves [0,3600)    Gmean   6.309540e+03  6.452581e+03    261  
                     Wins    1.020000e+02  1.350000e+02    261  
         [10,3600)   Gmean   5.963081e+04  6.403241e+04    102  
                     Wins    4.400000e+01  5.800000e+01    102

# Section 6: Objective and time analysis

## `obj_and_time_df`: objectives, successes, fails, and time per obj or cut

In [84]:
inst_set = best_gap_df.index
# inst_set = ['10teams_presolved', 'bm23_presolved', 'vpm1_presolved']

# Define rows to add
inst_depth_set = [(inst, best_gap_df.loc[inst, 'BEST VPC DISJ']) for inst in inst_set]

# Define columns to add
fail_rate_col_name = 'Fail rate (%)'
time_col_name = 'Time (s)'
sec_per_obj_col_name = '(s) / obj'
sec_per_cut_col_name = '(s) / cut'
obj_and_time_new_cols = [
    fail_rate_col_name,
    time_col_name,
    sec_per_obj_col_name,
    sec_per_cut_col_name,
]

obj_and_time_df = df.loc[inst_depth_set,[col_num_obj, col_num_vpc, col_num_fails]].copy(deep=True)
obj_and_time_df[fail_rate_col_name] = 100. * obj_and_time_df[col_num_fails] / obj_and_time_df[col_num_obj]
obj_and_time_df[time_col_name] = df[col_vpc_gen_time]
obj_and_time_df[sec_per_obj_col_name] = obj_and_time_df[time_col_name] / obj_and_time_df[col_num_obj]
obj_and_time_df[sec_per_cut_col_name] = obj_and_time_df[time_col_name] / obj_and_time_df[col_num_vpc]

# Replace Fail rate = NaN when all cuts are one-sided cuts
SKIP_CHAR = '-'
obj_and_time_df.fillna(SKIP_CHAR, inplace = True)
obj_and_time_df.replace(np.inf, SKIP_CHAR, inplace = True)

# Add average row
# obj_and_time_df.loc['Average'] = 0
obj_and_time_df.loc['Average', obj_and_time_new_cols] =\
    [obj_and_time_df[obj_and_time_df[col] != SKIP_CHAR][col].mean() for col in obj_and_time_new_cols]
# for col in obj_and_time_new_cols:
#     obj_and_time_df.at[('Average',0),col] =\
#         obj_and_time_df[obj_and_time_df[col] != SKIP_CHAR][col].mean()

obj_and_time_df.loc['Average',[col_num_obj, col_num_vpc, col_num_fails]] = ""

display(obj_and_time_df)
# obj_and_time_df[obj_and_time_df['NUM CUTS'] == 0]
# obj_and_time_df[obj_and_time_df['(s) / obj'] > 100000]
# obj_and_time_df.loc['neos18_presolved']
# obj_and_time_df[obj_and_time_df['(s) / obj'] != SKIP_CHAR]['(s) / obj'].max()

/var/folders/_8/bgx4yr0n6f1dm56pnvsqqb1s91dzml/T/ipykernel_59358/3615557839.py:27: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  obj_and_time_df.fillna(SKIP_CHAR, inplace = True)
/var/folders/_8/bgx4yr0n6f1dm56pnvsqqb1s91dzml/T/ipykernel_59358/3615557839.py:38: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  obj_and_time_df.loc['Average',[col_num_obj, col_num_vpc, col_num_fails]] = ""


,,NUM OBJ,NUM VPC,NUM FAILS,Fail rate (%),Time (s),(s) / obj,(s) / cut
INSTANCE,disj_terms,,,,,,,
10teams_presolved,32,214.0,16.0,198.0,92.523364,3604.560000,16.843738,225.285000
23588_presolved,64,76.0,75.0,1.0,1.315789,28.600000,0.376316,0.381333
30n20b8_presolved,2,193.0,190.0,3.0,1.554404,39.990000,0.207202,0.210474
50v-10_presolved,64,30.0,29.0,1.0,3.333333,4.450000,0.148333,0.153448
a1c1s1_presolved,8,3.0,3.0,0.0,0.0,0.540000,0.18,0.180000
...,...,...,...,...,...,...,...,...
usAbbrv-8-25_70_presolved,8,4.0,4.0,0.0,0.0,0.550000,0.1375,0.137500
vpm1_presolved,32,34.0,10.0,24.0,70.588235,0.070000,0.002059,0.007000
vpm2_presolved,64,36.0,25.0,11.0,30.555556,0.240000,0.006667,0.009600


In [85]:
### DEBUG finding max difference in time between TOTAL_TIME and sum of individual times
# cuts_cols = [col for col in df.columns if col.startswith('NUM CUTS')]
# time_cols = [
#     'INIT_SOLVE_TIME',
#     'VPC_GEN_TIME',
#     'VPC_APPLY_TIME',
#     'BB_TIME',
#     'TOTAL_TIME'
# ]
# display(df.loc['bell3b_presolved',['NUM OBJ', 'NUM FAILS'] + cuts_cols])

# obj_and_time_df = df.loc[inst_depth_set].copy(deep = True)['NUM OBJ', 'NUM CUTS', 'NUM FAILS', 'VPC_GEN_TIME']
# display(obj_and_time_df)

# max_diff_time = 0.
# max_diff_inst = ''
# for inst in best_gap_df.index:
#     depth = best_gap_df.loc[inst, 'BEST VPC DISJ']
#     curr_num_obj   = df.loc[(inst,depth)]['NUM OBJ']
#     curr_num_vpc   = df.loc[(inst,depth)]['NUM VPC']
#     curr_num_1side = df.loc[(inst,depth)]['NUM CUTS ONE_SIDED']
#     curr_num_fails = df.loc[(inst,depth)]['NUM FAILS']
#     if curr_num_vpc + curr_num_fails != curr_num_obj + curr_num_1side:
#         raise ValueError("{}: curr_num_vpc ({:d}) + curr_num_fails ({:d}) != curr_num_obj ({:d}) + curr_num_1side ({:d})".format(inst, curr_num_vpc, curr_num_fails, curr_num_obj, curr_num_1side))
    
#     curr_fail_pct = 100. * curr_num_fails / curr_num_obj
#     curr_init_solve = df.loc[(inst,depth)]['INIT_SOLVE_TIME']
#     curr_vpc_gen = df.loc[(inst,depth)]['VPC_GEN_TIME']
#     curr_vpc_apply = df.loc[(inst,depth)]['VPC_APPLY_TIME']
#     curr_bb_time = df.loc[(inst,depth)]['BB_TIME']
#     curr_total_time = df.loc[(inst,depth)]['TOTAL_TIME']

#     curr_diff_time = curr_total_time - (curr_init_solve + curr_vpc_gen + curr_vpc_apply + curr_bb_time)
#     if curr_diff_time < -EPS:
#         display(df.loc[inst,time_cols])
#         raise ValueError("{} (depth {:d}): curr_diff_time {} < 0.".format(inst,depth,curr_diff_time))
    
#     if max_diff_time < curr_diff_time:
#         max_diff_inst = inst
#         max_diff_time = curr_diff_time

# print("Max diff time = {} for inst {}".format(max_diff_time,max_diff_inst))
# display(df.loc[(max_diff_inst,best_gap_df.loc[max_diff_inst, 'BEST VPC DISJ']),time_cols])


## `best_disj_gap_df`: Number of times a particular depth achieves best result and beats baseline by at least EPS

In [86]:
long_cols_to_compare = {
    col_vpc:                      col_gmic,
    col_vpc_gmic:                 col_gmic,
    col_first_cut_pass_gap_ref_v: col_first_cut_pass_gap_ref,
    col_last_cut_pass_gap_ref_v:  col_last_cut_pass_gap_ref,
}
short_cols_to_compare = [map_cols_to_short_gap[col] for col in long_cols_to_compare.keys()]
row_no_improvement = 'No improvement'

best_disj_gap_df = pd.DataFrame(
    columns = short_cols_to_compare,
    index = [row_no_improvement] + [0] + sizes,
    dtype = int,
)

for curr_depth in [0] + sizes:
    curr_depth_df = selected_gap_df.xs(curr_depth,level='disj_terms')
    for col in long_cols_to_compare.keys():
        shortcol = map_cols_to_short_gap[col]
        refcol = long_cols_to_compare[col]
        
        # Calculate num times this depth yielded the best result
        curr_num_wins = sum(
            (curr_depth_df[col] == best_gap_df[shortcol]) & 
            (best_gap_df[shortcol] > best_gap_df[map_cols_to_short_gap[refcol]] + GAP_DIFF_EPS)
        )
        best_disj_gap_df.at[curr_depth,shortcol] = curr_num_wins
        
# Add no improvement row
curr_depth = row_no_improvement
for col in long_cols_to_compare.keys():
    shortcol = map_cols_to_short_gap[col]
    refcol = long_cols_to_compare[col]

    # Calculate num times no improvement over the baseline
    curr_num_wins = sum(
            (best_gap_df[shortcol] <= best_gap_df[map_cols_to_short_gap[refcol]] + GAP_DIFF_EPS)
        )
    best_disj_gap_df.at[curr_depth,shortcol] = curr_num_wins


# Reindex to add "leaves" to index
idx = [row_no_improvement] + ['Best'] + [str(size) + " leaves" for size in sizes]
reidx = {old_id : new_id for old_id, new_id in zip(best_disj_gap_df.index,idx)}
best_disj_gap_df.rename(reidx, inplace=True)

# Remove best row (it is good to verify this is the same as the relevant entries in win_df or Table 1)
best_disj_gap_df.drop('Best', axis=0, inplace=True)

# Make sure all cols are int
best_disj_gap_df = best_disj_gap_df.astype(int)

best_disj_gap_df

,V,V+G,V+GurF,V+GurL
No improvement,200,106,82,98
2 leaves,0,10,53,39
4 leaves,2,8,65,42
8 leaves,5,20,60,37
16 leaves,4,18,79,44
32 leaves,16,37,72,44
64 leaves,105,143,122,96


## `best_disj_time_df`: Number of times depth is best and improvement is at least 10%

In [87]:
row_no_improvement = 'No improvement'

# sizes_to_check = [0] + sizes
sizes_to_check = sizes

best_disj_time_df = pd.DataFrame(
    columns = bb_classes,
    index = [row_no_improvement] + sizes_to_check,
    dtype = int,
)

cols_time_by_depth = time_cols_long[:2] #[ref_time_col, refv_time_col, refv_w_cut_time_col]
curr_df = df.loc[selected_time_instances, cols_time_by_depth + [col_num_vpc]]
curr_df = curr_df[curr_df.index.get_level_values(1) > 0]

# Calculate minimum value for each column, per instance (index level 0)
best_time_df = curr_df.groupby(level=0).min()
display(best_time_df)

# Report shifted geometric mean of both columns
for col in cols_time_by_depth:
    print("{}: {}".format(col, geometric_mean(best_time_df[col] + SHIFT_TIME) - SHIFT_TIME))

inst_sets = [selected_time_instances, all6_instances, binary_x_time_instances]

# display(curr_df.xs(2,level='disj_terms'))
# display(curr_df[curr_df.index.get_level_values(1) == 2])

for curr_size in sizes_to_check:
    curr_by_depth_df = curr_df.xs(curr_size,level='disj_terms')

    # Check if the value for this depth is roughly the best value overall (in best_time_df)
    bb_class_ind = 0
    for inst_set in inst_sets:
        curr_class_df = curr_by_depth_df.loc[inst_set]
        curr_best_df = best_time_df.loc[inst_set]

        # Give wiggle room of ~5% for the best time
        curr_num_wins = sum(
            (curr_class_df[refv_time_col] < 1.05 * curr_best_df[refv_time_col]) & 
            (curr_best_df[ref_time_col] > WIN_BY_TIME_FACTOR * curr_best_df[refv_time_col])
        )
        best_disj_time_df.at[curr_size,bb_classes[bb_class_ind]] = curr_num_wins
        
        # Increment bb_class_ind for next set
        bb_class_ind += 1

# Add no improvement row
# Calculate num times no improvement over the baseline
curr_size = row_no_improvement

bb_class_ind = 0
for inst_set in inst_sets:
    curr_best_df = best_time_df.loc[inst_set]
    curr_num_wins = sum(
            (curr_best_df[ref_time_col] <= WIN_BY_TIME_FACTOR * curr_best_df[refv_time_col])
        )
    best_disj_time_df.at[curr_size,bb_classes[bb_class_ind]] = curr_num_wins
    bb_class_ind += 1

# Convert to int
best_disj_time_df = best_disj_time_df.astype(int)

# Reindex to add "leaves" to index
idx = [row_no_improvement] + [str(size) + " leaves" for size in sizes]
reidx = {old_id : new_id for old_id, new_id in zip(best_disj_time_df.index,idx)}
best_disj_time_df.rename(reidx, inplace=True)

# Remove best row (it is good to verify this is the same as the relevant entries in win_df or Table 1)
# best_disj_time_df.drop('Best', axis=0, inplace=True)

best_disj_time_df

,AVG REF TIME,AVG REF+V TIME,NUM VPC
INSTANCE,,,
10teams_presolved,1.821695,1.749950,0
23588_presolved,0.538193,0.474737,11
30n20b8_presolved,15.494462,9.722237,0
50v-10_presolved,4422.104332,2236.296629,2
a1c1s1_presolved,519.457900,499.183995,0
...,...,...,...
uct-subprob_presolved,1176.324622,1145.540321,0
umts_presolved,68.983241,71.111423,276
vpm1_presolved,0.004912,0.004515,3


AVG REF TIME: 9.699458422114915
AVG REF+V TIME: 8.813540522567733


,All,6 trees,Binary
No improvement,178,134,30
2 leaves,27,22,5
4 leaves,37,26,10
8 leaves,23,20,5
16 leaves,25,20,5
32 leaves,32,25,9
64 leaves,27,25,10


## `best_disj_nodes_df`: Number of times depth is best for nodes

In [88]:
# cols_nodes_by_depth = nodes_cols[:2] #[ref_time_col, refv_time_col, refv_w_cut_time_col]
# curr_df = df.loc[selected_time_instances, cols_time_by_depth + cols_nodes_by_depth + [col_num_vpc]]
# curr_df = curr_df[curr_df.index.get_level_values(1) > 0]

# # Calculate minimum value for each column, per instance (index level 0)
# curr_argmin_df = curr_df.groupby(level=0).idxmin()
# best_df = curr_df.loc[curr_argmin_df[refv_time_col]]
#cols_nodes_by_depth

['AVG REF NODES', 'AVG REF+V NODES']

In [98]:
row_no_improvement = 'No improvement'

# sizes_to_check = [0] + sizes
sizes_to_check = sizes

best_disj_nodes_df = pd.DataFrame(
    columns = bb_classes,
    index = [row_no_improvement] + sizes_to_check,
    dtype = int,
)

cols_nodes_by_depth = nodes_cols[:2] #[ref_time_col, refv_time_col, refv_w_cut_time_col]
curr_df = df.loc[selected_time_instances, cols_nodes_by_depth + [col_num_vpc]]
curr_df = curr_df[curr_df.index.get_level_values(1) > 0]

# Calculate minimum value for each column, per instance (index level 0)
best_nodes_df = curr_df.groupby(level=0).min()
display(best_nodes_df)

# Report shifted geometric mean of both columns
for col in cols_nodes_by_depth:
    print("{}: {}".format(col, geometric_mean(best_nodes_df[col] + SHIFT_NODES) - SHIFT_NODES))

inst_sets = [selected_time_instances, all6_instances, binary_x_time_instances]

# display(curr_df.xs(2,level='disj_terms'))
# display(curr_df[curr_df.index.get_level_values(1) == 2])

for curr_size in sizes_to_check:
    curr_by_depth_df = curr_df.xs(curr_size,level='disj_terms')

    # Check if the value for this depth is roughly the best value overall (in best_time_df)
    bb_class_ind = 0
    for inst_set in inst_sets:
        curr_class_df = curr_by_depth_df.loc[inst_set]
        curr_best_df = best_nodes_df.loc[inst_set]

        # Give wiggle room of ~5% for the best time
        curr_num_wins = sum(
            (curr_class_df[refv_nodes_col] <= curr_best_df[refv_nodes_col]) & 
            (curr_best_df[ref_nodes_col] > WIN_BY_NODES_FACTOR * curr_best_df[refv_nodes_col])
        )
        best_disj_nodes_df.at[curr_size,bb_classes[bb_class_ind]] = curr_num_wins
        
        # Increment bb_class_ind for next set
        bb_class_ind += 1

# Add no improvement row
# Calculate num times no improvement over the baseline
curr_size = row_no_improvement

bb_class_ind = 0
for inst_set in inst_sets:
    curr_best_df = best_nodes_df.loc[inst_set]
    curr_num_wins = sum(
            (curr_best_df[ref_nodes_col] <= WIN_BY_NODES_FACTOR * curr_best_df[refv_nodes_col])
        )
    best_disj_nodes_df.at[curr_size,bb_classes[bb_class_ind]] = curr_num_wins
    bb_class_ind += 1

# Convert to int
best_disj_nodes_df = best_disj_nodes_df.astype(int)

# Reindex to add "leaves" to index
idx = [row_no_improvement] + [str(size) + " leaves" for size in sizes]
reidx = {old_id : new_id for old_id, new_id in zip(best_disj_nodes_df.index,idx)}
best_disj_nodes_df.rename(reidx, inplace=True)

# Remove best row (it is good to verify this is the same as the relevant entries in win_df or Table 1)
# best_disj_time_df.drop('Best', axis=0, inplace=True)

best_disj_nodes_df

,AVG REF NODES,AVG REF+V NODES,NUM VPC
INSTANCE,,,
10teams_presolved,2.800000e+02,2.187143e+02,0
23588_presolved,8.821429e+02,8.935714e+02,11
30n20b8_presolved,8.094286e+02,5.662857e+02,0
50v-10_presolved,2.284218e+06,1.103440e+06,2
a1c1s1_presolved,4.606600e+04,4.427271e+04,0
...,...,...,...
uct-subprob_presolved,1.198120e+05,1.080696e+05,0
umts_presolved,1.073990e+05,9.810671e+04,276
vpm1_presolved,1.000000e+00,1.000000e+00,3


AVG REF NODES: 6760.17242563644
AVG REF+V NODES: 5946.715318986759


,All,6 trees,Binary
No improvement,79,56,18
2 leaves,38,31,6
4 leaves,37,26,7
8 leaves,35,27,3
16 leaves,41,33,4
32 leaves,40,32,6
64 leaves,38,34,9


## `density_df`

In [87]:
rows = [
    '\# inst w/VPCs and time < 3600s',
    '\# wins by time',
    'Avg min cut density',
    'Avg max cut density',
    'Avg avg cut density',
    'Avg avg cut density (win by time)',
    'Avg avg cut density (lose by time)',
]

columns = ['V ({:d})'.format(size) for size in sizes]

density_df = pd.DataFrame(
    columns = columns,
    index = rows,
    dtype = float
)

# Calculate stats for 6 trees instances by depth
# inst_set = all6_instances_dict.keys()
inst_set = selected_time_instances
support_cols = [col for col in df.columns if "SUPPORT VPC" in col]
selected_cols = [ref_time_col, refv_time_col]+support_cols+[col_num_cols,col_num_vpc,ref_timeout_col,refv_timeout_col]
selected_cols = list(set(selected_cols)) # remove duplicates
curr_df = df.loc[inst_set,selected_cols]

# Take only indices that are > 0
curr_df = curr_df[curr_df.index.get_level_values(1) > 0]

# curr_df0 = curr_df.xs(0,level='disj_terms')

for curr_size_ind in range(0,len(sizes)):
    # Select only this depth
    # curr_by_depth_df = curr_df[curr_df.index.get_level_values(1) == sizes[curr_size_ind]]
    curr_by_depth_df = curr_df.xs(sizes[curr_size_ind], level='disj_terms')
    
    # Remove instances that take more than an hour
    INSTANCES_TO_KEEP = curr_by_depth_df[[ref_timeout_col, refv_timeout_col]].min(axis=1) < MAX_TIME
    curr_by_depth_df = curr_by_depth_df[INSTANCES_TO_KEEP]

    # Count number of instances having cuts
    curr_row_ind = 0
    density_df.iloc[curr_row_ind,curr_size_ind] = sum(curr_by_depth_df[col_num_vpc] > 0)
    
    # Mean of min, max, avg density
    curr_row_ind = 2
    for col_ind in range(len(support_cols)):
        curr_series = curr_by_depth_df[support_cols[col_ind]] / curr_by_depth_df[col_num_cols]
        density_df.iloc[curr_row_ind,curr_size_ind] = curr_series.mean()
        curr_row_ind += 1

    ## Count wins1 (should be same as in avg_bb_by_depth_df)
    # A win in terms of time is counted when the ``Gur1'' baseline seconds taken 
    # is at least 10\% slower, to account for some variability in runtimes.
    # A win in terms of nodes is when the ``Gur1'' baseline number of nodes is higher.
    curr_wins_df = curr_by_depth_df[curr_by_depth_df[ref_time_col] > WIN_BY_TIME_FACTOR * curr_by_depth_df[refv_time_col]]
    curr_row_ind = 1
    density_df.iloc[curr_row_ind,curr_size_ind] = len(curr_wins_df)
    curr_row_ind = len(rows)-2
    density_df.iloc[curr_row_ind,curr_size_ind] = (curr_wins_df[support_cols[2]] / curr_wins_df[col_num_cols]).mean()

    # curr_lose_df = curr_by_depth_df[1.1*curr_df0[gur1time_col] < curr_by_depth_df[gur1vtime_col]]
    curr_lose_df = curr_by_depth_df[WIN_BY_TIME_FACTOR * curr_by_depth_df[ref_time_col] < curr_by_depth_df[refv_time_col]]
    #curr_lose_df = curr_by_depth_df[curr_by_depth_df[ref_time_col] <= WIN_BY_TIME_FACTOR * curr_by_depth_df[refv_time_col]]
    curr_row_ind = len(rows)-1
    density_df.iloc[curr_row_ind,curr_size_ind] = (curr_lose_df[support_cols[2]] / curr_lose_df[col_num_cols]).mean()

density_df

,V (2),V (4),V (8),V (16),V (32),V (64)
\# inst w/VPCs and time < 3600s,261.000000,266.000000,259.000000,261.000000,246.000000,229.000000
\# wins by time,52.000000,55.000000,46.000000,63.000000,53.000000,47.000000
Avg min cut density,0.216980,0.236145,0.274479,0.301382,0.349120,0.391576
Avg max cut density,0.343876,0.382824,0.404883,0.426546,0.484096,0.487735
Avg avg cut density,0.279550,0.310963,0.343381,0.373185,0.426838,0.453378
Avg avg cut density (win by time),0.289946,0.298028,0.212830,0.276679,0.360823,0.398727
Avg avg cut density (lose by time),0.278624,0.311245,0.346565,0.398178,0.486850,0.501303


## `obj_fails_df`

In [88]:
inst_set = best_gap_df.index
# inst_set = ['10teams_presolved', 'bm23_presolved', 'vpm1_presolved']

# Define rows to add
inst_depth_set = [(inst, best_gap_df.loc[inst, 'BEST VPC DISJ']) for inst in inst_set]

rows = [
    '\# inst w/obj',
    '\# inst w/succ obj',
    '\# inst no obj',
    '\# inst all obj fail',
    '\# inst all obj succ',
    '\% obj fails',
    '\% fails dup',
    '\% fails unbdd',
    '\% fails tilim',
    '\% fails dyn',
    '\% fails all ones',
    '\% fails post-GMIC obj',
    '\% fails DB',
    '\# obj / cut',
    '(s) / obj',
    '(s) / cut',
]

columns = ['V ({:d})'.format(size) for size in sizes]

obj_fails_df = pd.DataFrame(
    columns = columns,
    index = rows,
    dtype = float
)

# Choose columns to pull
selected_cols = [
    col_num_obj,
    col_num_vpc,
    'NUM FAILS DUMMY_OBJ',
    'NUM FAILS ALL_ONES',
    'NUM FAILS CUT_VERTICES',
    'NUM FAILS ITER_BILINEAR',
    'NUM FAILS UNIT_VECTORS',
    'NUM FAILS DISJ_LB',
    'NUM FAILS TIGHT_POINTS',
    'NUM FAILS TIGHT_RAYS',
    'NUM FAILS TIGHT_POINTS2',
    'NUM FAILS TIGHT_RAYS2',
    'NUM FAILS USER',
    'NUM FAILS OBJ_CUT',
    'NUM FAILS ONE_SIDED',
    'NUM FAILS',
    'ABANDONED',
    'BAD_DYNAMISM',
    'BAD_SUPPORT',
    'BAD_VIOLATION',
    'CUT_LIMIT',
    'DUAL_INFEASIBLE',
    'DUPLICATE_SIC',
    'DUPLICATE_VPC',
    'ITERATION_LIMIT',
    'ORTHOGONALITY_SIC',
    'ORTHOGONALITY_VPC',
    'PRIMAL_INFEASIBLE',
    'TIME_LIMIT',
    'NUMERICAL_ISSUES_WARNING',
    'DLB_EQUALS_DUB_NO_OBJ',
    'DLB_EQUALS_LPOPT_NO_OBJ',
    'PRIMAL_INFEASIBLE_NO_OBJ',
    'NUMERICAL_ISSUES_NO_OBJ',
    'UNKNOWN',
    col_vpc_gen_time,
]

curr_df = df.loc[inst_set,selected_cols]
curr_df0 = curr_df.xs(0,level='disj_terms')

for curr_size_ind in range(0,len(sizes)):
    # Select only this depth
    # curr_by_depth_df = curr_df[curr_df.index.get_level_values(1) == sizes[curr_size_ind]]
    curr_by_depth_df = curr_df.xs(sizes[curr_size_ind], level='disj_terms')

    # Num inst with objectives tried
    obj_fails_df.iloc[0,curr_size_ind] =\
        sum(curr_by_depth_df[col_num_obj] > 0)

    # Num inst with successful objectives
    obj_fails_df.iloc[1,curr_size_ind] =\
        sum((curr_by_depth_df[col_num_obj] > 0) & (curr_by_depth_df[col_num_vpc] > 0))

    # Num inst with no objectives tried
    obj_fails_df.iloc[2,curr_size_ind] =\
        sum(curr_by_depth_df[col_num_obj] == 0)

    # Num inst with objectives tried but all failed
    obj_fails_df.iloc[3,curr_size_ind] =\
        sum((curr_by_depth_df[col_num_obj] > 0) & (curr_by_depth_df[col_num_vpc] == 0))

    # Num inst with objectives tried and all succeeded
    obj_fails_df.iloc[4,curr_size_ind] =\
        sum((curr_by_depth_df[col_num_obj] > 0) & (curr_by_depth_df[col_num_obj] == curr_by_depth_df[col_num_vpc]))

    # Percent objective failures
    inst_w_obj_df = curr_by_depth_df[curr_by_depth_df[col_num_obj] > 0]
    obj_fails_df.iloc[5,curr_size_ind] =\
        (100. * inst_w_obj_df[col_num_fails] / inst_w_obj_df[col_num_obj]).mean()

    ## Percent of failures caused by:
    inst_w_fails_df = curr_by_depth_df[curr_by_depth_df[col_num_fails] > 0]

    # duplicates
    obj_fails_df.iloc[6,curr_size_ind] =\
        (100. * (inst_w_fails_df['DUPLICATE_VPC']+inst_w_fails_df['DUPLICATE_SIC']) / inst_w_fails_df[col_num_fails]).mean()

    # unbdd
    obj_fails_df.iloc[7,curr_size_ind] =\
        (100. * (inst_w_fails_df['DUAL_INFEASIBLE']) / inst_w_fails_df[col_num_fails]).mean()

    # tilim
    obj_fails_df.iloc[8,curr_size_ind] =\
        (100. * (inst_w_fails_df['TIME_LIMIT']) / inst_w_fails_df[col_num_fails]).mean()
    
    # dynamism
    obj_fails_df.iloc[9,curr_size_ind] =\
        (100. * (inst_w_fails_df['BAD_DYNAMISM']) / inst_w_fails_df[col_num_fails]).mean()

    # all-ones
    obj_fails_df.iloc[10,curr_size_ind] =\
        (100. * (inst_w_fails_df['NUM FAILS ALL_ONES']) / inst_w_fails_df[col_num_fails]).mean()

    # post-GMIC
    obj_fails_df.iloc[11,curr_size_ind] =\
        (100. * (inst_w_fails_df['NUM FAILS ITER_BILINEAR']) / inst_w_fails_df[col_num_fails]).mean()

    # disj_lb
    obj_fails_df.iloc[12,curr_size_ind] =\
        (100. * (inst_w_fails_df['NUM FAILS DISJ_LB']) / inst_w_fails_df[col_num_fails]).mean()
    
    # num obj / cut
    inst_w_cuts_df = curr_by_depth_df[curr_by_depth_df[col_num_vpc] > 0]
    obj_fails_df.iloc[13,curr_size_ind] = (inst_w_cuts_df[col_num_obj] / inst_w_cuts_df[col_num_vpc]).mean()

    # (s) / obj
    obj_fails_df.iloc[14,curr_size_ind] = (inst_w_obj_df[col_vpc_gen_time] / inst_w_obj_df[col_num_obj]).mean()

    # (s) / cut
    obj_fails_df.iloc[15,curr_size_ind] = (inst_w_cuts_df[col_vpc_gen_time] / inst_w_cuts_df[col_num_vpc]).mean()

obj_fails_df

,V (2),V (4),V (8),V (16),V (32),V (64)
\# inst w/obj,311.000000,318.000000,304.000000,306.000000,289.000000,271.000000
\# inst w/succ obj,308.000000,315.000000,300.000000,302.000000,285.000000,263.000000
\# inst no obj,21.000000,14.000000,28.000000,26.000000,43.000000,61.000000
\# inst all obj fail,3.000000,3.000000,4.000000,4.000000,4.000000,8.000000
\# inst all obj succ,30.000000,41.000000,31.000000,34.000000,27.000000,24.000000
\% obj fails,28.572839,25.279546,26.495366,30.774286,31.833289,34.084307
\% fails dup,50.672298,39.337152,41.105995,48.784992,51.021596,56.847203
\% fails unbdd,38.737356,47.589754,45.409934,35.735784,32.210717,27.495963
\% fails tilim,1.379326,3.673926,3.607452,5.130045,6.267595,5.910538
\% fails dyn,9.075878,8.736189,9.609179,9.418319,9.312240,9.377769


## `active_cuts_df`: when generated cuts are active, by objective

In [89]:
inst_set = best_gap_df.index
# inst_set = ['10teams_presolved', 'bm23_presolved', 'vpm1_presolved']

# Define rows to add
inst_depth_set = [(inst, best_gap_df.loc[inst, 'BEST VPC DISJ']) for inst in inst_set]

rows = [
    '\% active GMIC',
    '\% active VPC',
    '\% cuts one-sided',
    '\% active one-sided',
    '\% cuts all ones',
    '\% active all ones',
    '\% cuts post-GMIC opt',
    '\% active post-GMIC opt',
    '\% cuts DB',
    '\% active DB',
]

columns = ['V+G ({:d})'.format(size) for size in sizes]

active_cuts_df = pd.DataFrame(
    columns = columns,
    index = rows,
    dtype = float
)

# Choose columns to pull
active_gmic_col = 'ACTIVE GMIC (all cuts)'
active_vpc_col = 'ACTIVE VPC (all cuts)'
selected_cols = [
    col_num_gmic,
    col_num_vpc,
    active_gmic_col,
    active_vpc_col,
] + [col for col in df.columns if "NUM CUTS " in col] + [col for col in df.columns if "NUM ACTIVE" in col]

curr_df = df.loc[inst_set,selected_cols]

num_inst_with_one_sided_cuts = [0 for size in sizes]
for curr_size_ind in range(0,len(sizes)):
    # Select only this depth
    # curr_by_depth_df = curr_df[curr_df.index.get_level_values(1) == sizes[curr_size_ind]]
    curr_by_depth_df = curr_df.xs(sizes[curr_size_ind], level='disj_terms')

    # active gmic
    active_cuts_df.iloc[0,curr_size_ind] =\
        (100. * curr_by_depth_df[active_gmic_col] / curr_by_depth_df[col_num_gmic]).mean()

    # active vpc
    active_cuts_df.iloc[1,curr_size_ind] =\
        (100. * curr_by_depth_df[active_vpc_col] / curr_by_depth_df[col_num_vpc]).mean()

    # percent of active cuts among those generated by a specific objective type
    obj_types = ['ONE_SIDED', 'ALL_ONES', 'ITER_BILINEAR', 'DISJ_LB']
    curr_row_index = 2
    inst_w_vpc = curr_by_depth_df[curr_by_depth_df['NUM VPC'] > 0]
    for obj in obj_types:
        active_cuts_df.iloc[curr_row_index,curr_size_ind] =\
            (100. * inst_w_vpc['NUM CUTS '+obj] / inst_w_vpc[col_num_vpc]).mean()
        curr_row_index += 1

        inst_w_cuts = inst_w_vpc[inst_w_vpc['NUM CUTS '+obj] > 0]
        
        active_cuts_df.iloc[curr_row_index,curr_size_ind] =\
            (100. * inst_w_cuts['NUM ACTIVE '+obj] / inst_w_cuts['NUM CUTS '+obj]).mean()
        curr_row_index += 1
    
    # num one-sided cuts
    num_inst_with_one_sided_cuts[curr_size_ind] = sum(curr_by_depth_df['NUM CUTS ONE_SIDED'] > 0)

display(active_cuts_df)

print("Num inst with one-sided cuts (should be same across partial trees) =",num_inst_with_one_sided_cuts)
print("Total num one-sided cuts =", sum(curr_by_depth_df['NUM CUTS ONE_SIDED']))

,V+G (2),V+G (4),V+G (8),V+G (16),V+G (32),V+G (64)
\% active GMIC,44.088577,43.515863,42.170715,41.598561,40.879039,40.675613
\% active VPC,30.196454,30.591121,31.897898,35.499314,34.478032,33.401584
\% cuts one-sided,0.767677,0.722595,0.741979,1.107456,0.815851,0.792411
\% active one-sided,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
\% cuts all ones,11.822462,6.723898,7.806594,9.095018,8.404258,8.995381
\% active all ones,91.447368,84.782609,81.818182,77.018634,79.629630,79.268293
\% cuts post-GMIC opt,2.615979,3.397466,2.420045,2.186830,1.703320,2.739358
\% active post-GMIC opt,85.897436,71.428571,67.924528,62.745098,63.829787,61.904762
\% cuts DB,84.793883,89.156040,89.031382,87.610696,89.076571,87.472850
\% active DB,63.745545,58.323106,56.359239,51.850133,49.306592,43.000601


Num inst with one-sided cuts (should be same across partial trees) = [5, 5, 5, 5, 5, 5]
Total num one-sided cuts = 6


# Section 7: Export tables to LaTeX

## Format Table 1: gap closed and num wins

In [90]:
# Format Table 1: gap closed and num wins

# Create copy of table then remove values we do not want (wins for 'G)
# TABLE1 = avg_df.copy(deep=True)[[inst_col_name, 'G', 'DB', 'V', 'V+G', 'GurF', 'V+GurF', 'GurL', 'V+GurL']]
TABLE1 = avg_gap_df.copy(deep=True)[[inst_col_name]+gap_cols_short]

TABLE1['G'].loc[:,wins_row_name] = ""

# Process the column with # inst to only report number of instances for each set
TABLE1[inst_col_name].loc[:,wins_row_name] = ""
val = TABLE1[inst_col_name].loc[all_set_name,avg_row_name]
TABLE1[inst_col_name].loc[all_set_name,avg_row_name] = \
    create_multirow_string(str(val), extra_format=r"\tablenum[table-format=3]")
val = TABLE1[inst_col_name].loc[good_vpc_set_name,avg_row_name]
TABLE1[inst_col_name].loc[good_vpc_set_name,avg_row_name] = \
    create_multirow_string(str(val), extra_format=r"\tablenum[table-format=3]")
val = TABLE1[inst_col_name].loc[binary_set_name,avg_row_name]
TABLE1[inst_col_name].loc[binary_set_name,avg_row_name] = \
    create_multirow_string(str(val), extra_format=r"\tablenum[table-format=3]")

# Reset index to appear as cols
TABLE1.reset_index(inplace=True)

# Place column with # inst as second column
inst_col = TABLE1[inst_col_name]
TABLE1.drop(columns=[inst_col_name], inplace=True)
TABLE1.insert(loc=1, column=inst_col_name, value=inst_col)

# Set column should have multirow
setseries = TABLE1['Set']
format_col_as_multirow(setseries)

# for i in TABLE1.index:
#     curr_name = tex_escape(str(i))
#     print("Changing {} to {}".format(i, curr_name))
#     TABLE1.rename({i: curr_name}, inplace=True)
# print("")

# If we are not using the automatic tex-escaper, we need to do it ourselves
for col in TABLE1.columns:
    # curr_col = '{' + tex_escape(col) + '}'
    curr_col = tex_escape(str(col))
    TABLE1.rename({col: curr_col}, inplace=True, axis=1)

# Finally, apply the desired style
# styler.format({
#     ("Numeric", "Integers"): '\${}',
#     ("Numeric", "Floats"): '{:.3f}',
#     ("Non-Numeric", "Strings"): str.upper
# })
# styler.format_index(escape="latex", axis=0).format_index(escape="latex", axis=1)
# styler.hide(level=0,axis=0)
table1_str = TABLE1.style.\
    hide(axis=0).\
    format(formatter = int_format).\
    to_latex(
        #@{}l@{\hskip 5pt}
        column_format="""
        @{}l@{}
        S[table-format=2.0,table-auto-round,table-number-alignment=center]
        l
        *{1}{S[table-auto-round]}
        H
        *{8}{S[table-auto-round]}
        @{}""",
        hrules = True,
        #clines = "skip-last;data",
        sparse_index = True,
        multirow_align = "c",
        # float_format="%.2f", 
        # escape=False, 
        siunitx=True,
        # index_names=False,
        #columns=['\# inst', 'G', 'R', 'DB', 'V', 'max(G,V)', 'V+G', 'GurF', 'V+GurF', 'GurL', 'V+GurL']
        convert_css = True,
        environment = "table",
        position_float = "centering",
        label = "tab:gap-closed-summary",
        caption = """
            Summary statistics for percent gap closed by VPCs.
            The wins row reports how many instances close at least $\epsilon$ more gap when comparing DB, V, V+G to G on its own, V+GurF to GurF, and V+GurL to GurL.
        """,
        )

# Add a midrule between the two sets; the "3" is hand-coded but can be automated
table1_str = add_midrule(table1_str, -3)
table1_str = add_midrule(table1_str, -6)

# Adjustbox environment sets width to pagewidth
table1_str = add_adjustbox_environment(table1_str)

# Set default siunitx options for this table
table1_str = add_sisetup(table1_str)

print(table1_str)


{
\sisetup{
    table-alignment-mode = format,
    table-number-alignment = center,
    table-format = 2.2,
}
\begin{table}
\centering
\caption{
            Summary statistics for percent gap closed by VPCs.
            The wins row reports how many instances close at least $\epsilon$ more gap when comparing DB, V, V+G to G on its own, V+GurF to GurF, and V+GurL to GurL.
        }
\label{tab:gap-closed-summary}
\begin{adjustbox}{width=1\textwidth}
\begin{tabular}{@{}l@{}
        S[table-format=2.0,table-auto-round,table-number-alignment=center]
        l
        *{1}{S[table-auto-round]}
        H
        *{8}{S[table-auto-round]}
        @{}}
\toprule
{Set} & {\# inst} & {} & {G} & {R} & {DB} & {V} & {max(G,V)} & {V+G} & {GurF} & {V+GurF} & {GurL} & {V+GurL} \\
\midrule
{\multirow[c]{2}{*}{All}} & {\multirow[c]{2}{*}{\tablenum[table-format=3]{332}}} & Avg (\%) & 16.508148432312172 & 0.25674423491955295 & 18.369422595516696 & 12.00826462388178 & 23.00736907616349 & 23.91643511610843 &

## Format Table 2: depth x gap

In [91]:
# Format Table 2: percent gap closed by depth
TABLE2 = gap_by_size_df.copy(deep=True)

# If we are not using the automatic tex-escaper, we need to do it ourselves
for col in TABLE2.columns:
    # curr_col = '{' + tex_escape(col) + '}'
    curr_col = tex_escape(str(col))
    TABLE2.rename({col: curr_col}, inplace=True, axis=1)

# Finally, apply the desired style
table2_str = TABLE2.style.\
    format(formatter = int_format).\
    to_latex(
        column_format="""
        @{}l
        *{6}{S[table-auto-round]}
        @{}""",
        hrules = True,
        sparse_index = True,
        multirow_align = "c",
        siunitx=True,
        convert_css = True,
        environment = "table",
        position_float = "centering",
        label = "tab:depth",
        caption = """
            Average percent gap closed broken down by the number of leaf nodes used to construct the partial branch-and-bound tree,
            for VPCs with and without GMICs, as well as at the root by \Gurobi{} after the first and last round of cuts. 
            ``Best'' refers to the maximum gap closed across all partial tree sizes.
        """,
        )

# Adjustbox environment sets width to pagewidth
# table2_str = add_adjustbox_environment(table2_str)

# Set default siunitx options for this table
table2_str = add_sisetup(table2_str)

# Add midrule after 2nd row and before last two rows
toprule_row_ind = table2_str.count("\n", 0, table2_str.index("\\toprule")) + 1
bottomrule_row_ind = table2_str.count("\n", 0, table2_str.index("\\bottomrule")) + 1
table2_str = add_midrule(table2_str, toprule_row_ind + 3)
table2_str = add_midrule(table2_str, bottomrule_row_ind + (-2))

print(table2_str)


{
\sisetup{
    table-alignment-mode = format,
    table-number-alignment = center,
    table-format = 2.2,
}
\begin{table}
\centering
\caption{
            Average percent gap closed broken down by the number of leaf nodes used to construct the partial branch-and-bound tree,
            for VPCs with and without GMICs, as well as at the root by \Gurobi{} after the first and last round of cuts. 
            ``Best'' refers to the maximum gap closed across all partial tree sizes.
        }
\label{tab:depth}
\begin{tabular}{@{}l
        *{6}{S[table-auto-round]}
        @{}}
\toprule
{} & {DB} & {V} & {max(G,V)} & {V+G} & {V+GurF} & {V+GurL} \\
\midrule
VPCs disabled & 0.0 & 0.0 & 16.508148432312172 & 16.508148432312172 & 28.92974415029431 & 48.319147189366504 \\
\midrule
2 leaves & 3.0145069679615464 & 2.213289365939913 & 16.78974579780376 & 17.206578963213257 & 30.96117050218173 & 49.260529886136254 \\
4 leaves & 5.270241662205749 & 3.5924977627287134 & 17.227532496610067 & 17.7850308

## Format Table 3: summary of b&b results

In [92]:
# Display full avg_bb_df for reference

# Set the maximum number of rows to be displayed
pd.set_option('display.max_rows', None)

# Display only up to two significant digits
pd.options.display.float_format = '{:.2f}'.format

# Display the DataFrame
display(avg_bb_df)

# Reset the maximum number of rows to be displayed
pd.reset_option('display.max_rows')


criterion                    Time (s)                 Nodes (#)             \
type                              Gur       V    Gen        Gur          V   
class     bucket      metric                                                 
Combined  [0,3600)    Gmean     11.26   11.84  27.62    7681.91    7793.80   
                      Wins     117.00   62.00   0.00     128.00     131.00   
          [10,3600)   Gmean    157.26  168.45  55.30   75193.84   76955.77   
                      Wins      44.00   32.00   0.00      54.00      65.00   
          [100,3600)  Gmean    505.12  550.56  43.70  244802.37  255028.03   
                      Wins      32.00   15.00   0.00      38.00      31.00   
          [1000,3600) Gmean   1972.95 2073.90  35.86 1008339.71  996884.90   
                      Wins      10.00    5.00   0.00      11.00      13.00   
Rounds    [0,3600)    Gmean     11.62   12.47 128.88    7845.55    7919.76   
                      Wins     144.00   61.00   0.00     129.00     123.00   
          [10,3600)   Gmean    155.23  165.44 216.40   74002.42   74706.01   
                      Wins      45.00   34.00   0.00      58.00      59.00   
          [100,3600)  Gmean    526.97  558.17 193.58  235024.50  241114.76   
                      Wins      22.00   16.00   0.00      36.00      28.00   
          [1000,3600) Gmean   1868.61 1982.47 166.96 1041127.75 1032613.93   
                      Wins       8.00    5.00   0.00      13.00      10.00   
2 leaves  [0,3600)    Gmean      9.26    9.51   0.56    6309.54    6452.58   
                      Wins      64.00   52.00  28.00     102.00     135.00   
          [10,3600)   Gmean    129.68  139.35   1.19   59630.81   64032.41   
                      Wins      33.00   22.00  16.00      44.00      58.00   
          [100,3600)  Gmean    503.50  534.57   1.00  186171.78  199675.89   
                      Wins      13.00   10.00  10.00      18.00      32.00   
          [1000,3600) Gmean   1974.56 1894.64   0.30  799031.96  729535.07   
                      Wins       4.00    4.00   4.00       4.00      11.00   
4 leaves  [0,3600)    Gmean      8.81    8.80   1.19    6017.40    5963.03   
                      Wins      70.00   55.00  26.00     112.00     127.00   
          [10,3600)   Gmean    117.38  115.83   2.38   55206.09   53955.81   
                      Wins      26.00   29.00  21.00      49.00      55.00   
          [100,3600)  Gmean    466.63  458.66   2.56  166328.21  162475.08   
                      Wins      12.00   12.00  10.00      19.00      29.00   
          [1000,3600) Gmean   1802.02 1875.01   0.96  790760.04  792726.31   
                      Wins       4.00    4.00   4.00       7.00       6.00   
8 leaves  [0,3600)    Gmean      9.19    9.68   2.21    6717.70    7034.71   
                      Wins      89.00   46.00  20.00     127.00     107.00   
          [10,3600)   Gmean    139.39  153.05   4.72   74523.48   82415.76   
                      Wins      42.00   24.00  16.00      58.00      41.00   
          [100,3600)  Gmean    510.32  557.75   5.16  213765.20  241449.82   
                      Wins      24.00   13.00  11.00      29.00      23.00   
          [1000,3600) Gmean   2046.17 1965.60   2.64  750097.42  709701.10   
                      Wins       7.00    6.00   6.00       7.00       9.00   
16 leaves [0,3600)    Gmean      9.22    9.46   3.91    6964.50    7019.52   
                      Wins      78.00   63.00  23.00     104.00     133.00   
          [10,3600)   Gmean    139.44  147.97   6.66   80057.85   83603.49   
                      Wins      30.00   28.00  16.00      45.00      55.00   
          [100,3600)  Gmean    535.23  562.07   7.83  226992.48  236017.94   
                      Wins      15.00   14.00  11.00      21.00      29.00   
          [1000,3600) Gmean   1891.26 1824.11   4.82  995349.75  889380.21   
                      Wins       5.00    7.00   7.00       4.00      14.00   
32 leaves [0,3600)    

In [93]:
# Format Table 3: summary of b&b results
TABLE3 = avg_bb_df.copy(deep=True)

# Retrieve classes, buckets, and metrics
final_bb_classes = avg_bb_df.index.get_level_values(0).unique()
final_bb_buckets = avg_bb_df.index.get_level_values(1).unique()
final_bb_metrics = avg_bb_df.index.get_level_values(2).unique()

# display(final_bb_classes)
# display(final_bb_buckets)
# display(final_bb_metrics)

# Indices of wins metrics
wins_metric_ind = [ ind for ind, metric in enumerate(final_bb_metrics) if metric.startswith('Wins') ]

# Change dtype of inst_col_name to string
TABLE3.loc[:, inst_col_name] = TABLE3.loc[:, inst_col_name].astype(str)

# Process the column with # inst to only report number of instances for each set
TABLE3.loc[(slice(None), slice(None), final_bb_metrics[wins_metric_ind]), inst_col_name] = ""

for curr_class in final_bb_classes:
    for curr_bucket in final_bb_buckets:
        curr_name = (curr_class, curr_bucket, final_bb_metrics[0])
        val = TABLE3.loc[curr_name, inst_col_name]
        TABLE3.loc[curr_name, inst_col_name] = \
            create_multirow_string(str(val), num_rows = len(final_bb_metrics), extra_format=r"\tablenum[table-format=3]")

# Set num wins in int format or enclose in braces (center)
# tmp_df = TABLE3.loc[(slice(None), slice(None), bb_metrics[1:3]),time_col_header].applymap(int_format, num_digits=6)
tmp_df = TABLE3.loc[(slice(None), slice(None), final_bb_metrics[wins_metric_ind]),time_col_header].map(int_format, num_digits=4, add_phantom=True)
# tmp_df = TABLE3.loc[(slice(None), slice(None), bb_metrics[1:3]),time_col_header].applymap(enclose_in_braces)
tmp_df.columns = pd.MultiIndex.from_product([[time_col_header],tmp_df.columns])
TABLE3.loc[(slice(None), slice(None), final_bb_metrics[wins_metric_ind]),time_col_header] = TABLE3.loc[(slice(None), slice(None), final_bb_metrics[wins_metric_ind]),time_col_header].astype(str)
TABLE3.loc[(slice(None), slice(None), final_bb_metrics[wins_metric_ind]),time_col_header] = tmp_df

# tmp_df = TABLE3.loc[(slice(None), slice(None), bb_metrics[1:3]),node_col_header].applymap(int_format, num_digits=6)
tmp_df = TABLE3.loc[(slice(None), slice(None), bb_metrics[1:3]),node_col_header].map(int_format, num_digits=6, add_phantom=False)
# tmp_df = TABLE3.loc[(slice(None), slice(None), bb_metrics[1:3]),node_col_header].applymap(enclose_in_braces)
tmp_df.columns = pd.MultiIndex.from_product([[node_col_header],tmp_df.columns])
TABLE3.loc[(slice(None), slice(None), final_bb_metrics[wins_metric_ind]),node_col_header] = tmp_df

# Remove unnecessary entries, which are the wins entries for the last time col
TABLE3.loc[(slice(None), slice(None), final_bb_metrics[wins_metric_ind]),([time_col_header,node_col_header],map_cols_to_short_time[col_vpc_gen_time])] = ""

# Reset index to appear as cols
TABLE3.reset_index(inplace=True)

# Add new col combining class and bucket in one
class_bucket_col = "\multirow{" + str(len(final_bb_metrics)) + "}{*}{\shortstack[l]{" + TABLE3['class'] + "\\\\\\relax " + TABLE3['bucket'] + "}}"
for i in range(len(class_bucket_col)):
    if i%len(final_bb_metrics)!=0:
        class_bucket_col[i] = ""
TABLE3.drop(columns = ['class', 'bucket'], inplace = True, level = 0)
TABLE3.insert(loc=0, column="Set", value=class_bucket_col)

# Place column with # inst as second column
inst_col = TABLE3[inst_col_name]
TABLE3.drop(columns=[inst_col_name], inplace=True, level=0)
TABLE3.insert(loc=1, column=inst_col_name, value=inst_col)

# If we are not using the automatic tex-escaper, we need to do it ourselves
for col in TABLE3.columns:
    if isinstance(col, tuple):
        for lvl_ind, lvl_col in enumerate(col):
            curr_col = tex_escape(str(lvl_col))
            TABLE3.rename({lvl_col: curr_col}, inplace=True, axis=1, level=lvl_ind)
    else:
        # curr_col = '{' + tex_escape(col) + '}'
        curr_col = tex_escape(str(col))
        TABLE3.rename({col: curr_col}, inplace=True, axis=1)

# Finally, apply the desired style
    # format(formatter = int_format).\
table3_str = TABLE3.style.\
    hide(axis=0).\
    to_latex(
        column_format="""
        @{}l    % set
        c       % inst
        l       % stat
        *{2}{S[table-auto-round,table-format=4.2]}
        *{2}{H}
        *{2}{S[table-auto-round,table-format=4.2]}        
        *{2}{S[table-auto-round,table-format=6.0]}
        *{1}{H}
        *{1}{S[table-auto-round,table-format=6.0]}
        @{}}""",
        hrules = True,
        sparse_index = True,
        multirow_align = "c",
        siunitx=True,
        convert_css = True,
        environment = "table",
        position_float = "centering",
        label = "tab:bb-summary",
        caption = """
            Summary statistics for time to solve instances with branch-and-bound.
        """,
        )

# Add a midrule between the two sets; the "9" is hand-coded but can be automated
for ind in [-57,-49,-41,-33,-25,-17,-9]:
    table3_str = add_midrule(table3_str, ind)

# Adjustbox environment sets width to pagewidth
table3_str = add_adjustbox_environment(table3_str)

# Set default siunitx options for this table
table3_str = add_sisetup(table3_str, table_format="4.2")

print(table3_str)


{
\sisetup{
    table-alignment-mode = format,
    table-number-alignment = center,
    table-format = 4.2,
}
\begin{table}
\centering
\caption{
            Summary statistics for time to solve instances with branch-and-bound.
        }
\label{tab:bb-summary}
\begin{adjustbox}{width=1\textwidth}
\begin{tabular}{@{}l    % set
        c       % inst
        l       % stat
        *{2}{S[table-auto-round,table-format=4.2]}
        *{2}{H}
        *{2}{S[table-auto-round,table-format=4.2]}        
        *{2}{S[table-auto-round,table-format=6.0]}
        *{1}{H}
        *{1}{S[table-auto-round,table-format=6.0]}
        @{}}}
\toprule
{Set} & {\# inst} & {metric} & \multicolumn{3}{r}{Time (s)} & \multicolumn{2}{r}{Nodes (\#)} \\
{} & {} & {} & {Gur} & {V} & {Gen} & {Gur} & {V} \\
\midrule
\multirow{2}{*}{\shortstack[l]{Combined\\\relax [0,3600)}} & {\multirow[c]{2}{*}{\tablenum[table-format=3]{288}}} & Gmean & 11.256821 & 11.839758 & 27.617049 & 7681.913224 & 7793.801266 \\
 &  & Wins & 

/var/folders/_8/bgx4yr0n6f1dm56pnvsqqb1s91dzml/T/ipykernel_54123/3158147712.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  TABLE3.loc[curr_name, inst_col_name] = \
/var/folders/_8/bgx4yr0n6f1dm56pnvsqqb1s91dzml/T/ipykernel_54123/3158147712.py:35: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['{\\tablenum[table-format=4.0]{117.0}\\phantom{.00}}'
 '{\\tablenum[table-format=4.0]{44.0}\\phantom{.00}}'
 '{\\tablenum[table-format=4.0]{32.0}\\phantom{.00}}'
 '{\\tablenum[table-format=4.0]{10.0}\\phantom{.00}}'
 '{\\tablenum[table-format=4.0]{144.0}\\phantom{.00}}'
 '{\\tablenum[table-format=4.0]{45.0}\\phantom{.00}}'
 '{\\tablenum[table-format=4.0]{22.0}\\phantom{.00}}'
 '{\\tablenum[table-format=4.0]{8.0}\\phantom{.00}}'
 '{\\tablenum[table-format=4.0]{64.0}\\phantom{.00}}'
 '{\\tablenum[table-format=4.0]{33.0}\\phantom{.00}}'
 '{\\tablenum[table-format=4.0]{13.0}\\phantom{.00}}'
 '{\\t

## Format Table 4: number of leaf nodes yielding the best result for each experiment per instance

## Format Table 5: density statistics

## Format Table 6: failures

## Format Table 7: active cuts

## Format Table 8: objective + time analysis per instance

In [94]:
# Format Table 8: obj and time analysis
TABLE8 = obj_and_time_df.copy(deep=True)

# Move instance names into a column
TABLE8.reset_index(inplace=True)
TABLE8.drop('disj_terms',axis=1,inplace=True)

# Create new column index
TABLE8.columns = pd.MultiIndex.from_tuples(
    [('','Instance'),
    ('Objectives','Obj'),
    ('Objectives','Succ'),
    ('Objectives','Fails'),
    ('Objectives','\% fails'),
    ('Time (s)','Total'),
    ('Time (s)','(s) / obj'),
    ('Time (s)','(s) / cut')]
)

# Format instance column correctly
TABLE8[('',"Instance")] = TABLE8[('',"Instance")].apply(remove_presolved_from_name)
TABLE8[('',"Instance")] = TABLE8[('',"Instance")].apply(tex_escape)

# Format SKIP_CHAR correctly
for col in TABLE8.columns:
    TABLE8[col] = TABLE8[col].apply(enclose_in_braces, val_to_match=SKIP_CHAR)

# If we are not using the automatic tex-escaper, we need to do it ourselves
for col in TABLE8.columns:
    # curr_col = '{' + tex_escape(col) + '}'
    curr_col = tex_escape(str(col))
    TABLE8.rename({col: curr_col}, inplace=True, axis=1)

# Finally, apply the desired style
table8_str = TABLE8.style.\
    hide(axis=0).\
    to_latex(
        column_format="""
        @{}
        l
        *{3}{S[table-format=3.0,table-auto-round,table-number-alignment=center]}
        *{1}{S[table-format=2.1,table-auto-round,table-number-alignment=center]}
        *{1}{S[table-format=4.1,table-auto-round,table-number-alignment=center]}
        *{2}{S[table-format=4.1,table-auto-round,table-number-alignment=center]}
        @{}""",
        hrules = True,
        sparse_index = True,
        multirow_align = "c",
        siunitx=True,
        convert_css = True,
        environment = "table",
        position_float = "centering",
        label = "app:tab:obj-and-time-best",
        caption = """
            Information about objectives and time to generate cuts corresponding to the results in Table~\ref{app:tab:gap-closed}.
        """,
        )

# Adjustbox environment sets width to pagewidth
# table8_str = add_adjustbox_environment(table8_str)

table8_str = add_midrule(table8_str, -2)

# Set default siunitx options for this table
table8_str = add_sisetup(table8_str)

print(table8_str)


{
\sisetup{
    table-alignment-mode = format,
    table-number-alignment = center,
    table-format = 2.2,
}
\begin{table}
\centering
\caption{
            Information about objectives and time to generate cuts corresponding to the results in Table~
ef{app:tab:gap-closed}.
        }
\label{app:tab:obj-and-time-best}
\begin{tabular}{@{}
        l
        *{3}{S[table-format=3.0,table-auto-round,table-number-alignment=center]}
        *{1}{S[table-format=2.1,table-auto-round,table-number-alignment=center]}
        *{1}{S[table-format=4.1,table-auto-round,table-number-alignment=center]}
        *{2}{S[table-format=4.1,table-auto-round,table-number-alignment=center]}
        @{}}
\toprule
{} & \multicolumn{4}{r}{Objectives} & \multicolumn{3}{r}{Time (s)} \\
{Instance} & {Obj} & {Succ} & {Fails} & {\% fails} & {Total} & {(s) / obj} & {(s) / cut} \\
\midrule
10teams & 214.0 & 16.0 & 198.0 & 92.5233644859813 & 3604.56 & 16.84373831775701 & 225.285 \\
23588 & 76.0 & 75.0 & 1.0 & 1.3157894736

In [95]:
# Format Table 12: active cuts
TABLE12 = active_cuts_df.copy(deep=True)

# Finally, apply the desired style
    # hide(axis=0).\
table12_str = TABLE12.style.\
    to_latex(
        column_format="""@{}l*{6}{S[table-format=3.2,table-auto-round,table-number-alignment=center]}@{}""",
        hrules = True,
        sparse_index = True,
        multirow_align = "c",
        siunitx=True,
        convert_css = True,
        environment = "table",
        position_float = "centering",
        label = "app:tab:activity",
        caption = """
            Statistics about when generated cuts are active, broken down by partial tree size.
        """,
        )

# Adjustbox environment sets width to pagewidth
# table12_str = add_adjustbox_environment(table12_str)

# Set default siunitx options for this table
table12_str = add_sisetup(table12_str, table_format="2.2")

print(table12_str)


{
\sisetup{
    table-alignment-mode = format,
    table-number-alignment = center,
    table-format = 2.2,
}
\begin{table}
\centering
\caption{
            Statistics about when generated cuts are active, broken down by partial tree size.
        }
\label{app:tab:activity}
\begin{tabular}{@{}l*{6}{S[table-format=3.2,table-auto-round,table-number-alignment=center]}@{}}
\toprule
{} & {V+G (2)} & {V+G (4)} & {V+G (8)} & {V+G (16)} & {V+G (32)} & {V+G (64)} \\
\midrule
\% active GMIC & 44.088577 & 43.515863 & 42.170715 & 41.598561 & 40.879039 & 40.675613 \\
\% active VPC & 30.196454 & 30.591121 & 31.897898 & 35.499314 & 34.478032 & 33.401584 \\
\% cuts one-sided & 0.767677 & 0.722595 & 0.741979 & 1.107456 & 0.815851 & 0.792411 \\
\% active one-sided & 100.000000 & 100.000000 & 100.000000 & 100.000000 & 100.000000 & 100.000000 \\
\% cuts all ones & 11.822462 & 6.723898 & 7.806594 & 9.095018 & 8.404258 & 8.995381 \\
\% active all ones & 91.447368 & 84.782609 & 81.818182 & 77.018634 & 79.62

In [96]:
# Format Table 11: failures
TABLE11 = obj_fails_df.copy(deep=True)

# Make num inst rows int
tmp_df = TABLE11.iloc[0:5].map(int_format, num_digits=3, add_phantom=False)
TABLE11.iloc[0:5] = tmp_df

# Finally, apply the desired style
    # hide(axis=0).\
table11_str = TABLE11.style.\
    to_latex(
        column_format="""@{}l*{6}{S[table-format=2.2,table-auto-round,table-number-alignment=center]}@{}""",
        hrules = True,
        sparse_index = True,
        multirow_align = "c",
        siunitx=True,
        convert_css = True,
        environment = "table",
        position_float = "centering",
        label = "app:tab:objectives",
        caption = """
            Statistics about the objectives leading to failures, broken down by partial tree size used for cut generation.
        """,
        )

# Adjustbox environment sets width to pagewidth
# table11_str = add_adjustbox_environment(table11_str)

table11_str = add_midrule(table11_str, -4)
table11_str = add_midrule(table11_str, -8)
table11_str = add_midrule(table11_str, -13)
table11_str = add_midrule(table11_str, -15)

# Set default siunitx options for this table
table11_str = add_sisetup(table11_str, table_format="2.2")

print(table11_str)


{
\sisetup{
    table-alignment-mode = format,
    table-number-alignment = center,
    table-format = 2.2,
}
\begin{table}
\centering
\caption{
            Statistics about the objectives leading to failures, broken down by partial tree size used for cut generation.
        }
\label{app:tab:objectives}
\begin{tabular}{@{}l*{6}{S[table-format=2.2,table-auto-round,table-number-alignment=center]}@{}}
\toprule
{} & {V (2)} & {V (4)} & {V (8)} & {V (16)} & {V (32)} & {V (64)} \\
\midrule
\# inst w/obj & {\tablenum[table-format=3.0]{311.0}} & {\tablenum[table-format=3.0]{318.0}} & {\tablenum[table-format=3.0]{304.0}} & {\tablenum[table-format=3.0]{306.0}} & {\tablenum[table-format=3.0]{289.0}} & {\tablenum[table-format=3.0]{271.0}} \\
\# inst w/succ obj & {\tablenum[table-format=3.0]{308.0}} & {\tablenum[table-format=3.0]{315.0}} & {\tablenum[table-format=3.0]{300.0}} & {\tablenum[table-format=3.0]{302.0}} & {\tablenum[table-format=3.0]{285.0}} & {\tablenum[table-format=3.0]{263.0}} \\
\# 

/var/folders/_8/bgx4yr0n6f1dm56pnvsqqb1s91dzml/T/ipykernel_54123/2693163282.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['{\\tablenum[table-format=3.0]{311.0}}'
 '{\\tablenum[table-format=3.0]{308.0}}'
 '{\\tablenum[table-format=3.0]{21.0}}'
 '{\\tablenum[table-format=3.0]{3.0}}'
 '{\\tablenum[table-format=3.0]{30.0}}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  TABLE11.iloc[0:5] = tmp_df
/var/folders/_8/bgx4yr0n6f1dm56pnvsqqb1s91dzml/T/ipykernel_54123/2693163282.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['{\\tablenum[table-format=3.0]{318.0}}'
 '{\\tablenum[table-format=3.0]{315.0}}'
 '{\\tablenum[table-format=3.0]{14.0}}'
 '{\\tablenum[table-format=3.0]{3.0}}'
 '{\\tablenum[table-format=3.0]{41.0}}']' has dtype incompatible with float64, please explicitly cast to a compatible 

In [97]:
# Format Table 10: density statistics
TABLE10 = density_df.copy(deep=True)

# Make # inst and wins row int
#tmp_df = TABLE10.iloc[0:2].applymap(int_format, num_digits=3, add_phantom=False)
tmp_df = TABLE10.iloc[0:2].map(int_format, num_digits=3, add_phantom=False)

TABLE10.iloc[0:2] = tmp_df
# tmp_df = TABLE10.loc['\# wins by time',:].apply(int_format, num_digits=2, add_phantom=False)
# TABLE10.loc['\# wins by time'] = tmp_df

# Finally, apply the desired style
    # hide(axis=0).\
table10_str = TABLE10.style.\
    to_latex(
        column_format="""@{}l*{6}{S[table-format=0.3,table-auto-round,table-number-alignment=center]}@{}""",
        hrules = True,
        sparse_index = True,
        multirow_align = "c",
        siunitx=True,
        convert_css = True,
        environment = "table",
        position_float = "centering",
        label = "app:tab:density",
        caption = """
            Statistics about the density of generated cuts broken down by partial tree size.
        """,
        )

# Adjustbox environment sets width to pagewidth
# table10_str = add_adjustbox_environment(table10_str)

# table10_str = add_midrule(table10_str, -2)

# Set default siunitx options for this table
table10_str = add_sisetup(table10_str, table_format="0.3")

print(table10_str)


{
\sisetup{
    table-alignment-mode = format,
    table-number-alignment = center,
    table-format = 0.3,
}
\begin{table}
\centering
\caption{
            Statistics about the density of generated cuts broken down by partial tree size.
        }
\label{app:tab:density}
\begin{tabular}{@{}l*{6}{S[table-format=0.3,table-auto-round,table-number-alignment=center]}@{}}
\toprule
{} & {V (2)} & {V (4)} & {V (8)} & {V (16)} & {V (32)} & {V (64)} \\
\midrule
\# inst w/VPCs and time < 3600s & {\tablenum[table-format=3.0]{261.0}} & {\tablenum[table-format=3.0]{266.0}} & {\tablenum[table-format=3.0]{259.0}} & {\tablenum[table-format=3.0]{261.0}} & {\tablenum[table-format=3.0]{246.0}} & {\tablenum[table-format=3.0]{229.0}} \\
\# wins by time & {\tablenum[table-format=3.0]{52.0}} & {\tablenum[table-format=3.0]{55.0}} & {\tablenum[table-format=3.0]{46.0}} & {\tablenum[table-format=3.0]{63.0}} & {\tablenum[table-format=3.0]{53.0}} & {\tablenum[table-format=3.0]{47.0}} \\
Avg min cut density & 0.216

/var/folders/_8/bgx4yr0n6f1dm56pnvsqqb1s91dzml/T/ipykernel_54123/2710154175.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['{\\tablenum[table-format=3.0]{261.0}}'
 '{\\tablenum[table-format=3.0]{52.0}}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  TABLE10.iloc[0:2] = tmp_df
/var/folders/_8/bgx4yr0n6f1dm56pnvsqqb1s91dzml/T/ipykernel_54123/2710154175.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['{\\tablenum[table-format=3.0]{266.0}}'
 '{\\tablenum[table-format=3.0]{55.0}}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  TABLE10.iloc[0:2] = tmp_df
/var/folders/_8/bgx4yr0n6f1dm56pnvsqqb1s91dzml/T/ipykernel_54123/2710154175.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value

In [98]:
best_disj_gap_df

,V,V+G,V+GurF,V+GurL
No improvement,200,106,82,98
2 leaves,0,10,53,39
4 leaves,2,8,65,42
8 leaves,5,20,60,37
16 leaves,4,18,79,44
32 leaves,16,37,72,44
64 leaves,105,143,122,96


In [99]:
best_disj_time_df

,All,6 trees,Binary
No improvement,178,134,30
2 leaves,27,22,5
4 leaves,37,26,10
8 leaves,23,20,5
16 leaves,25,20,5
32 leaves,32,25,9
64 leaves,27,25,10


In [100]:
# Format Table 9: frequency of when each size is best
TABLE9 = pd.concat([best_disj_gap_df,best_disj_time_df],axis=1)
# TABLE9 = best_disj_gap_df.copy(deep=True)
# TABLE9.drop('Best', axis=0, inplace=True)

# Create new column index
TABLE9.columns = pd.MultiIndex.from_tuples(
    [('Gap',col) for col in best_disj_gap_df.columns]
    + [('Time',col) for col in best_disj_time_df.columns]
)

# Drop all but 'Time' 'All' column
TABLE9.drop(columns=[col for col in TABLE9.columns if col[0] == 'Time' and col[1] != 'All'], inplace=True)

# If we are not using the automatic tex-escaper, we need to do it ourselves
for col in TABLE9.columns:
    # curr_col = '{' + tex_escape(col) + '}'
    curr_col = tex_escape(str(col))
    TABLE9.rename({col: curr_col}, inplace=True, axis=1)

# Finally, apply the desired style
    # hide(axis=0).\
table9_str = TABLE9.style.\
    to_latex(
        column_format="""@{}l*4{S}*1{S}@{}""",
        hrules = True,
        sparse_index = True,
        multirow_align = "c",
        siunitx=True,
        convert_css = True,
        environment = "table",
        position_float = "centering",
        label = "app:tab:size",
        caption = """
            Number of leaf nodes yielding the best result for each experiment per instance.
        """,
        )

# Adjustbox environment sets width to pagewidth
# table9_str = add_adjustbox_environment(table9_str)

# table9_str = add_midrule(table9_str, -2)

# Set default siunitx options for this table
table9_str = add_sisetup(table9_str, table_format="3.0")

print(table9_str)


{
\sisetup{
    table-alignment-mode = format,
    table-number-alignment = center,
    table-format = 3.0,
}
\begin{table}
\centering
\caption{
            Number of leaf nodes yielding the best result for each experiment per instance.
        }
\label{app:tab:size}
\begin{tabular}{@{}l*4{S}*1{S}@{}}
\toprule
{} & \multicolumn{4}{r}{Gap} & {Time} \\
{} & {V} & {V+G} & {V+GurF} & {V+GurL} & {All} \\
\midrule
No improvement & 200 & 106 & 82 & 98 & 178 \\
2 leaves & 0 & 10 & 53 & 39 & 27 \\
4 leaves & 2 & 8 & 65 & 42 & 37 \\
8 leaves & 5 & 20 & 60 & 37 & 23 \\
16 leaves & 4 & 18 & 79 & 44 & 25 \\
32 leaves & 16 & 37 & 72 & 44 & 32 \\
64 leaves & 105 & 143 & 122 & 96 & 27 \\
\bottomrule
\end{tabular}
\end{table}

}


## Prepare Table 9: rejected instances

In [101]:
df_rejection_reason.to_csv('rejection_reason.csv', index=True)

#### Verbose version

In [102]:
## *Verbose version*: For each instance that was not selected, print the reason
df_rejection_reason_rejected = df_rejection_reason[df_rejection_reason['SELECTED_GAP'] == False]
rejected_instance_list = df_rejection_reason_rejected.index
rejected_instance_list.name = 'Instance'
cols = ['Set', 'Reason']
df_rejected_instances = pd.DataFrame(columns=cols, index=rejected_instance_list)
df_rejected_instances['Set'] = df_ipopt.loc[rejected_instance_list,'SET']
df_rejected_instances.loc[df_rejection_reason_rejected['OPTIMAL_SOLUTION_FOUND'] > 0, 
                            'Reason'] = "Integer-optimal solution found constructing partial tree"
df_rejected_instances.loc[(df_rejection_reason_rejected['OPTIMAL_SOLUTION_FOUND'] == 0) 
                                & (df_rejection_reason_rejected['LP=DLB=DUB'] == 6), 
                            'Reason'] = "Max leaf value = LP value"
df_rejected_instances.loc[(df_rejection_reason_rejected['OPTIMAL_SOLUTION_FOUND'] == 0) 
                                & (df_rejection_reason_rejected['LP=DLB=DUB'] < 6) 
                                & (df_rejection_reason_rejected['LP=DLB=DUB'] + df_rejection_reason_rejected['PRLP_INFEASIBLE'] == 6), 
                            'Reason'] = "Max leaf value = LP value or PRLP primal infeasible"
df_rejected_instances.loc[(df_rejection_reason_rejected['OPTIMAL_SOLUTION_FOUND'] == 0)
                                & (df_rejection_reason_rejected['LP=DLB=DUB'] < 6) 
                                & (df_rejection_reason_rejected['LP=DLB=DUB'] + df_rejection_reason_rejected['PRLP_INFEASIBLE'] < 6)
                                & (df_rejection_reason_rejected['LP=DLB=DUB'] + df_rejection_reason_rejected['PRLP_INFEASIBLE'] + df_rejection_reason_rejected['PRLP_TIME_LIMIT'] == 6), 
                            'Reason'] = "Max leaf value = LP value or PRLP primal infeasible / hits time limit"
df_rejected_instances.loc[df_rejection_reason_rejected['<7_ATTEMPTS'] > 0, 
                            'Reason'] = "Numerical issues"
display(df_rejected_instances.head())
col_format = """@{}*{2}{l}X@{}"""

tmp_df_remaining_rejected_instances = df_rejection_reason.loc[df_rejected_instances[df_rejected_instances['Reason'].isna()].index]
if len(tmp_df_remaining_rejected_instances) > 0:
    display(tmp_df_remaining_rejected_instances)

,Set,Reason
Instance,,
22433_presolved,miplib2017,Integer-optimal solution found constructing pa...
air01_presolved,miplib2,Integer-optimal solution found constructing pa...
app1-1_presolved,miplib2017,Integer-optimal solution found constructing pa...
b-ball_presolved,miplib2017,Max leaf value = LP value or PRLP primal infea...
bppc4-08_presolved,miplib2017,Max leaf value = LP value or PRLP primal infea...


,SELECTED_GAP,SELECTED_TIME,SELECTED_6TREES,NUM_WITH_OBJS,NUM_WITH_CUTS,IP_OPT_UNKNOWN,TOO_MANY_ROWS_OR_COLS,OPTIMAL_SOLUTION_FOUND,LP_OPT_IS_NOT_CUT,DLB=DUB,LP=DLB=DUB,PRLP_INFEASIBLE,PRLP_TIME_LIMIT,NO_CUTS,NO_GAP,GUR_TIMEOUT,<7_ATTEMPTS
Instance,,,,,,,,,,,,,,,,,
neos-1112782_presolved,False,False,False,6,0,False,False,0,0,0,0,0,0,True,False,0,False
neos-1112787_presolved,False,False,False,6,0,False,False,0,0,0,0,0,0,True,False,0,False


#### Succinct version

In [103]:
## *Succinct version*: For each instance that was not selected, print the reason
df_rejected_instances = df_status_by_depth.loc[df_rejection_reason[df_rejection_reason['SELECTED_GAP'] == False].index]
df_rejected_instances.insert(loc = 0, column = 'Set', value = df_ipopt.loc[rejected_instance_list,'SET'])
col_format="""@{}*{2}{l}*{6}{c}@{}"""

### Print Table 9

In [104]:
# Format Table 4: rejected instances reasons
TABLE4 = df_rejected_instances.copy(deep=True)
TABLE4.reset_index(inplace=True)

TABLE4["Instance"] = TABLE4["Instance"].apply(remove_presolved_from_name)
TABLE4["Instance"] = TABLE4["Instance"].apply(tex_escape)

# If we are not using the automatic tex-escaper, we need to do it ourselves
for col in TABLE4.columns:
    # curr_col = '{' + tex_escape(col) + '}'
    curr_col = tex_escape(str(col))
    TABLE4.rename({col: curr_col}, inplace=True, axis=1)

# Finally, apply the desired style
    # format_index(escape="latex", axis=0).\
table4_str = TABLE4.style.\
    hide(axis=0).\
    to_latex(
        column_format=col_format,
        hrules = True,
        sparse_index = True,
        multirow_align = "c",
        siunitx=False,
        convert_css = True,
        environment = "table",
        position_float = "centering",
        label = "app:tab:discarded-instances",
        caption = """
            Instances that were not considered with reason for being discarded.
        """,
        )
        
print(table4_str)

\begin{table}
\centering
\caption{
            Instances that were not considered with reason for being discarded.
        }
\label{app:tab:discarded-instances}
\begin{tabular}{@{}*{2}{l}*{6}{c}@{}}
\toprule
Instance & Set & 2 & 4 & 8 & 16 & 32 & 64 \\
\midrule
22433 & miplib2017 &  &  & \ref{selection-criterion:partial-tree-does-not-find-opt} & \ref{selection-criterion:partial-tree-does-not-find-opt} & \ref{selection-criterion:partial-tree-does-not-find-opt} & \ref{selection-criterion:partial-tree-does-not-find-opt} \\
air01 & miplib2 & \ref{selection-criterion:partial-tree-does-not-find-opt} & \ref{selection-criterion:partial-tree-does-not-find-opt} & \ref{selection-criterion:partial-tree-does-not-find-opt} & \ref{selection-criterion:partial-tree-does-not-find-opt} & \ref{selection-criterion:partial-tree-does-not-find-opt} & \ref{selection-criterion:partial-tree-does-not-find-opt} \\
app1-1 & miplib2017 &  &  &  &  & \ref{selection-criterion:partial-tree-does-not-find-opt} & \ref{sel

### DEBUG: Test Table 4 code and make sure "set" is properly identified

In [105]:
#### DEBUG
# df_rejection_reason[df_rejection_reason['NUM_WITH_OBJS'] != df_rejection_reason['NUM_WITH_CUTS']]
# df_rejection_reason[(df_rejection_reason['NUM_WITH_CUTS'] > 0) & (df_rejection_reason['DLB=DUB'] > 0) & (df_rejection_reason['OPTIMAL_SOLUTION_FOUND'] == 0)]
# df_rejection_reason[(df_rejection_reason['LP=DLB=DUB'] == 6)]

# inst = 'chromaticindex32-8_presolved'
# # df_rejection_reason.loc[inst]
# tmp = df_bb.loc[(inst,64)]
# tmp[25:50]

# len(df_rejection_reason[df_rejection_reason['SELECTED'] == True])
# inst = 'berlin_5_8_0_presolved'
# gap_df.loc[inst]
#df_rejection_reason.loc['bnatt400_presolved']

In [106]:
##### DEBUG: Verify "Set" col is correct
for inst in rejected_instance_list:
    curr_set = df_ipopt.loc[inst,'SET']
    has_error = False
    if isinstance(curr_set, pd.Series):
        # check that all sets are same, then just take first
        first_set = curr_set[0]
        for tmp_set in curr_set:
            if tmp_set != first_set:
                print("*** ERROR: not all sets are equal ({} != {})".format(first_set, tmp_set))
                has_error = True
                break
        curr_set = first_set
    ref_set = df_rejected_instances.loc[inst, 'Set']
    if ref_set != curr_set:
        print("*** ERROR: for inst {}, df_rej_inst set {} != df_ipopt set {}".format(inst, ref_set, curr_set))
        has_error = True
    
    if has_error:
        break

## Format Table 10: app:tab:gap-closed: full gap closed results

In [107]:
# Take all but column named 'R' of all_gap_results_df
subset_all_gap_results_df = all_gap_results_df.drop(columns=('% gap closed','R'), inplace=False)
subset_all_gap_results_df.head(5)

# cuts      % gap closed                       \
                   Rows  Cols      G    V            G    DB     V max(G,V)   
Instance                                                                      
10teams_presolved   210  1600    153   16       100.00  0.00  0.00   100.00   
23588_presolved     137   237     75   75         5.77 72.18 71.61    71.61   
30n20b8_presolved   387  4191    184  190        11.10  1.56  0.03    11.10   
50v-10_presolved    233  2013     29   29        45.75 18.01 11.18    45.75   
a1c1s1_presolved   1876  2489    154    3        25.11  4.90  1.09    25.11   

                                                     
                     V+G  GurF V+GurF   GurL V+GurL  
Instance                                             
10teams_presolved 100.00 93.88  93.88 100.00 100.00  
23588_presolved    71.62 15.59  68.94  24.38  71.78  
30n20b8_presolved  11.10  0.93   1.33  16.91  16.30  
50v-10_presolved   45.82 49.86  50.15  73.17  74.44  
a1c1s1_presolved   25.38 45.58  46.25  88.81  88.73

In [108]:
# Format Table 5: full gap closed results
TABLE5 = subset_all_gap_results_df.copy(deep=True)

# Set wins row to be integer valued
TABLE5.loc['Wins'] = TABLE5.loc['Wins'].apply(int_format)
# TABLE5.iloc[len(TABLE5)-1] = TABLE5.iloc[len(TABLE5)-1].apply(int_format)

# Move instance names into a column
TABLE5.reset_index(inplace=True, col_level=1)

TABLE5[('',"Instance")] = TABLE5[('',"Instance")].apply(remove_presolved_from_name)
TABLE5[('',"Instance")] = TABLE5[('',"Instance")].apply(tex_escape)

# If we are not using the automatic tex-escaper, we need to do it ourselves
for col in TABLE5.columns:
    # curr_col = '{' + tex_escape(col) + '}'
    curr_col = tex_escape(str(col))
    TABLE5.rename({col: curr_col}, inplace=True, axis=1)

# Finally, apply the desired style
    # format(formatter = int_format).\
table5_str = TABLE5.style.\
    hide(axis=0).\
    to_latex(
        column_format="""
	@{}l*{2}{S[table-format=4.0,table-auto-round,table-number-alignment=center]}
	*{2}{S[table-format=4.0,table-auto-round,table-number-alignment=center]}
	*{8}{S[table-auto-round]}
	@{}
        """,
        hrules = True,
        sparse_index = True,
        multirow_align = "c",
        siunitx=True,
        convert_css = True,
        environment = "table",
        position_float = "centering",
        label = "app:tab:gap-closed",
        caption = """
            Percent gap closed by instance for GMICs (G), VPCs (V), both VPCs and GMICs used together, 
            and the bound implied by the partial branch-and-bound tree with 64 leaf nodes (DB).
            Also shown are the sizes of the instances, the number of cuts added, and the percent gap closed by 
            \Gurobi{} at the root (after one round (GurF) and after the last round (GurL)). 
            Entries in which DB appears to be 0.00 are actually small strictly positive numbers.
        """,
        )

# Adjustbox environment sets width to pagewidth
# table2_str = add_adjustbox_environment(table2_str)

# Set default siunitx options for this table
table5_str = add_sisetup(table5_str)

# Add a midrule between the instances and 3 summary rows; the "5" is hand-coded but can be automated
table5_str = add_midrule(table5_str, -5)

print(table5_str)


{
\sisetup{
    table-alignment-mode = format,
    table-number-alignment = center,
    table-format = 2.2,
}
\begin{table}
\centering
\caption{
            Percent gap closed by instance for GMICs (G), VPCs (V), both VPCs and GMICs used together, 
            and the bound implied by the partial branch-and-bound tree with 64 leaf nodes (DB).
            Also shown are the sizes of the instances, the number of cuts added, and the percent gap closed by 
            \Gurobi{} at the root (after one round (GurF) and after the last round (GurL)). 
            Entries in which DB appears to be 0.00 are actually small strictly positive numbers.
        }
\label{app:tab:gap-closed}
\begin{tabular}{@{}l*{2}{S[table-format=4.0,table-auto-round,table-number-alignment=center]}
	*{2}{S[table-format=4.0,table-auto-round,table-number-alignment=center]}
	*{8}{S[table-auto-round]}
	@{}}
\toprule
\multicolumn{3}{r}{} & \multicolumn{2}{r}{# cuts} & \multicolumn{9}{r}{% gap closed} \\
{Instance} & {Rows

## Format Table 11: "all" time/nodes results

In [ ]:
%%script false --no-raise-error

# Format Table 6: "all" time/nodes results
TABLE6 = all_bb_results_df.copy(deep=True)

# Rename summary rows to reflect the set
rename_metrics_all = {metric : metric + ' (All)' for metric in bb_metrics}
TABLE6.rename(rename_metrics_all, inplace=True)

# Add summary rows from 6 trees set
summary_metrics_6trees = all6_bb_results_df.tail(3).copy(deep=True)
rename_metrics_6trees = {metric : metric + ' (6 trees)' for metric in bb_metrics}
summary_metrics_6trees.rename(rename_metrics_6trees, inplace=True)

TABLE6 = pd.concat([TABLE6, summary_metrics_6trees])

# Drop rows, cols, (time,V7)
TABLE6.drop([('','Rows'),('','Cols'),(node_col_header,map_cols_to_short_time[mintime_col])], axis=1, inplace=True)

# Set wins row to be integer valued
TABLE6.loc['Wins1 (All)'] = TABLE6.loc['Wins1 (All)'].apply(int_format)
TABLE6.loc['Wins7 (All)'] = TABLE6.loc['Wins7 (All)'].apply(int_format)
TABLE6.loc['Wins1 (6 trees)'] = TABLE6.loc['Wins1 (6 trees)'].apply(int_format)
TABLE6.loc['Wins7 (6 trees)'] = TABLE6.loc['Wins7 (6 trees)'].apply(int_format)
# TABLE6.iloc[len(TABLE6)-1] = TABLE6.iloc[len(TABLE6)-1].apply(int_format)

# Move instance names into a column
TABLE6.reset_index(inplace=True, col_level=1)

# Store indices of rows of 6-tree instances
six_trees_instances = list(all6_instances_dict.keys())
# mask = TABLE6[('','Instance')].isin(six_trees_instances)
# six_trees_indices = TABLE6.loc[mask, :].index.tolist()

# Remove presolved from name and escape
TABLE6[('',"Instance")] = TABLE6[('',"Instance")].apply(remove_presolved_from_name)
TABLE6[('',"Instance")] = TABLE6[('',"Instance")].apply(tex_escape)

# If we are not using the automatic tex-escaper, we need to do it ourselves
for col in TABLE6.columns:
    # curr_col = '{' + tex_escape(col) + '}'
    curr_col = tex_escape(str(col))
    TABLE6.rename({col: curr_col}, inplace=True, axis=1)

# Finally, apply the desired style
    # format(formatter = int_format).\
table6_str = TABLE6.style.\
    hide(axis=0).\
    to_latex(
        column_format="""
	@{}l % instance
	*{1}{S[table-format=4.0,table-auto-round,table-number-alignment=center]} % # cuts
	*{2}{S[table-format=4.2,table-auto-round]} % Gur1, Gur7
	*{2}{H} % V, Total
	*{2}{S[table-format=4.2,table-auto-round]} % V7, Total7
	*{3}{S[table-format=8.0,table-auto-round,table-number-alignment=center]} % Nodes
	@{}
        """,
        hrules = True,
        sparse_index = True,
        multirow_align = "c",
        siunitx=True,
        convert_css = True,
        environment = "table",
        position_float = "centering",
        label = "app:tab:bb",
        caption = """
            Time (in seconds) and number nodes taken to solve each instance.
            The table is sorted by column 4 (``V'' under ``Time (s)'').
            ``Gur1'' indicates \Gurobi{} run with one random seed.
            ``Gur7'' indicates the minimum from seven runs of \Gurobi{} with different random seeds.
        """,
        )

# Adjustbox environment sets width to pagewidth
# table6_str = add_adjustbox_environment(table6_str)

# Set default siunitx options for this table
table6_str = add_sisetup(table6_str)

# Add a midrule between the instances and 3 summary rows; the "6" is hand-coded but can be automated
table6_str = add_midrule(table6_str, -6)
table6_str = add_midrule(table6_str, -10)

# Add color to six tree instances
splitlines = table6_str.splitlines()
for i in range(len(splitlines)):
    line = splitlines[i]
    curr_line = line.split('&')
    if len(curr_line) > 0 and curr_line[0].strip()+'_presolved' in six_trees_instances:
        splitlines[i] = '\\rowcolor{lightgray!30} ' + line
table6_str = '\n'.join(splitlines).replace('NaN', '')

print(table6_str)

## XXX Format Table X: "6 trees" time/nodes results

In [ ]:
%%script false --no-raise-error

# Format Table 7: "6 trees" time/nodes results
TABLEX = all6_bb_results_df.copy(deep=True)

# Set wins row to be integer valued
TABLEX.loc['Wins1'] = TABLEX.loc['Wins1'].apply(int_format)
TABLEX.loc['Wins7'] = TABLEX.loc['Wins7'].apply(int_format)
# TABLEX.iloc[len(TABLEX)-1] = TABLEX.iloc[len(TABLEX)-1].apply(int_format)

# Move instance names into a column
TABLEX.reset_index(inplace=True, col_level=1)

TABLEX[('',"Instance")] = TABLEX[('',"Instance")].apply(remove_presolved_from_name)
TABLEX[('',"Instance")] = TABLEX[('',"Instance")].apply(tex_escape)

# If we are not using the automatic tex-escaper, we need to do it ourselves
for col in TABLEX.columns:
    # curr_col = '{' + tex_escape(col) + '}'
    curr_col = tex_escape(str(col))
    TABLEX.rename({col: curr_col}, inplace=True, axis=1)

# Finally, apply the desired style
    # format(formatter = int_format).\
tableX_str = TABLEX.style.\
    hide(axis=0).\
    to_latex(
        column_format="""@{}l*{2}{c}*{2}{c}H*{8}{c}@{}""",
        hrules = True,
        sparse_index = True,
        multirow_align = "c",
        siunitx=True,
        convert_css = True,
        environment = "table",
        position_float = "centering",
        label = "app:tab:bb-7trees",
        caption = """
  Time (in seconds) and number nodes taken to solve each of the instances for which all six branch-and-bound trees successfully yielded VPCs.
  %The columns with V1x are those in which we do not terminate the VPC computation as soon as the time exceeds \Gurobi{}'s time.  
  The table is sorted by column 4 (``V7'' under ``Time (s)'').
  ``Gur1'' indicates Gurobi run with one random seed.
  ``Gur7'' indicates the minimum from seven runs of Gurobi with different random seeds.
        """,
        )

# Adjustbox environment sets width to pagewidth
# tableX_str = add_adjustbox_environment(tableX_str)

# Set default siunitx options for this table
tableX_str = add_sisetup(tableX_str)

# Add a midrule between the instances and 3 summary rows; the "6" is hand-coded but can be automated
tableX_str = add_midrule(tableX_str, -6)

print(tableX_str)

## XXX Format Table X: (now Table 3) b&b summary by depth

In [ ]:
# # Format Table 7: summary of b&b results
# TABLE7 = avg_bb_by_depth_df.copy(deep=True)

# # Remove unnecessary entries
# # TABLE7.loc[(slice(None), slice(None), bb_metrics_by_depth[1:]),([time_col_header,node_col_header],map_cols_to_short_time[gur1time_col])] = ""
# # TABLE7.loc[(slice(None), slice(None), bb_metrics_by_depth[2]),([time_col_header,node_col_header],map_cols_to_short_time[gur7time_col])] = ""

# # Process the column with # inst to only report number of instances for each set
# TABLE7.loc[(slice(None), slice(None), bb_metrics_by_depth[1:]), inst_col_name] = ""

# for curr_class in bb_classes_by_depth:
#     for curr_bucket in bb_buckets_by_depth:
#         curr_name = (curr_class, curr_bucket, bb_metrics_by_depth[0])
#         val = TABLE7.loc[curr_name, inst_col_name]
#         TABLE7.loc[curr_name, inst_col_name] = \
#             create_multirow_string(str(val), num_rows = 2, extra_format=r"\tablenum[table-format=3]")

# # Set num wins in int format or enclose in braces (center)
# # tmp_df = TABLE7.loc[(slice(None), slice(None), bb_metrics_by_depth[1:3]),time_col_header].applymap(int_format, num_digits=6)
# tmp_df = TABLE7.loc[(slice(None), slice(None), bb_metrics_by_depth[1:3]),time_col_header].applymap(int_format, num_digits=4, add_phantom=True)
# # tmp_df = TABLE7.loc[(slice(None), slice(None), bb_metrics_by_depth[1:3]),time_col_header].applymap(enclose_in_braces)
# tmp_df.columns = pd.MultiIndex.from_product([[time_col_header],tmp_df.columns])
# TABLE7.loc[(slice(None), slice(None), bb_metrics_by_depth[1:3]),time_col_header] = tmp_df

# # tmp_df = TABLE7.loc[(slice(None), slice(None), bb_metrics_by_depth[1:3]),node_col_header].applymap(int_format, num_digits=6)
# tmp_df = TABLE7.loc[(slice(None), slice(None), bb_metrics_by_depth[1:3]),node_col_header].applymap(int_format, num_digits=6, add_phantom=False)
# # tmp_df = TABLE7.loc[(slice(None), slice(None), bb_metrics_by_depth[1:3]),node_col_header ].applymap(enclose_in_braces)
# tmp_df.columns = pd.MultiIndex.from_product([[node_col_header],tmp_df.columns])
# TABLE7.loc[(slice(None), slice(None), bb_metrics_by_depth[1:3]),node_col_header] = tmp_df

# # Reset index to appear as cols
# TABLE7.reset_index(inplace=True)

# # Add new col combining class and bucket in one
# class_bucket_col = "\multirow{2}{*}{\shortstack[l]{" + TABLE7['class'] + "\\\\\\relax " + TABLE7['bucket'] + "}}"
# for i in range(len(class_bucket_col)):
#     if i%len(bb_metrics_by_depth)!=0:
#         class_bucket_col[i] = ""
# TABLE7.drop(columns = ['class', 'bucket'], inplace = True, level = 0)
# TABLE7.insert(loc=0, column="Set", value=class_bucket_col)

# # Place column with # inst as second column
# inst_col = TABLE7[inst_col_name]
# TABLE7.drop(columns=[inst_col_name], inplace=True, level=0)
# TABLE7.insert(loc=1, column=inst_col_name, value=inst_col)

# # If we are not using the automatic tex-escaper, we need to do it ourselves
# for col in TABLE7.columns:
#     if isinstance(col, tuple):
#         for lvl_ind, lvl_col in enumerate(col):
#             curr_col = tex_escape(str(lvl_col))
#             TABLE7.rename({lvl_col: curr_col}, inplace=True, axis=1, level=lvl_ind)
#     else:
#         # curr_col = '{' + tex_escape(col) + '}'
#         # curr_col = col
#         curr_col = tex_escape(str(col))
#         TABLE7.rename({col: curr_col}, inplace=True, axis=1)

# # Finally, apply the desired style
#     # format(formatter = int_format).\
# table7_str = TABLE7.style.\
#     hide(axis=0).\
#     to_latex(
#         column_format="""
#         @{}l    % set
#         c       % inst
#         l       % stat
#         *{3}{S[table-auto-round,table-format=4.2]}
#         *{2}{S[table-auto-round,table-format=6.0]}
#         @{}""",
#         hrules = True,
#         sparse_index = True,
#         multirow_align = "c",
#         siunitx=True,
#         convert_css = True,
#         environment = "table",
#         position_float = "centering",
#         label = "tab:bb-summary",
#         caption = """
#             Summary statistics for time to solve instances with branch-and-bound.
#         """,
#         )

# # Add a midrule between the two sets; the "9" is hand-coded but can be automated
# table7_str = add_midrule(table7_str, -41)
# table7_str = add_midrule(table7_str, -33)
# table7_str = add_midrule(table7_str, -25)
# table7_str = add_midrule(table7_str, -17)
# table7_str = add_midrule(table7_str, -9)

# # Adjustbox environment sets width to pagewidth
# table7_str = add_adjustbox_environment(table7_str)

# # Set default siunitx options for this table
# table7_str = add_sisetup(table7_str, table_format="4.2")

# print(table7_str)

## Format Table 13: instances with best bb improvement from VPCs

In [ ]:
%%script false --no-raise-error
# Format Table 13: "best" time/nodes results
TABLE13 = all_bb_results_df.copy(deep=True)

# Drop all rows in which (time_col_header, 'Gur7') is ''
TABLE13 = TABLE13[TABLE13[(time_col_header, 'Gur7')] != '']

# TABLE13[('','V7-Gur7')]
# Change column (time_col_header, V7) to be float valued
#TABLE13[(time_col_header, 'V7')] = TABLE13[(time_col_header, 'V7')].apply(float_format, num_digits=4)
TABLE13 = TABLE13.astype({(time_col_header, 'V7'): float})
TABLE13 = TABLE13.astype({(time_col_header, 'Gur7'): float})

# Add new column for difference between V7 and Gur7
TABLE13[('Time (s)','V7 - Gur7')] = TABLE13[('Time (s)','V7')] - TABLE13[('Time (s)','Gur7')]

# Sort by V7 - Gur7
TABLE13.sort_values(by=[(time_col_header,'V7 - Gur7')], inplace=True)

TABLE13.head(15)

In [ ]:
%%script false --no-raise-error
### DEBUG DEBUG DEBUG
# inst = 'cost266-UUE_presolved'
# hawea instance
inst = 'neos-3592146-hawea_presolved'
if inst in all_bb_results_df.index:
    display(all_bb_results_df.loc[inst])

In [ ]:
%%script false --no-raise-error
# Format Table 13: "best" time/nodes results
TABLE13 = all_bb_results_df.copy(deep=True)

# Drop all rows in which (time_col_header, 'Gur7') is ''
TABLE13 = TABLE13[TABLE13[(time_col_header, 'Gur7')] != '']

# TABLE13[('','V7-Gur7')]
# Change column (time_col_header, V7) to be float valued
#TABLE13[(time_col_header, 'V7')] = TABLE13[(time_col_header, 'V7')].apply(float_format, num_digits=4)
TABLE13 = TABLE13.astype({(time_col_header, 'V7'): float})
TABLE13 = TABLE13.astype({(time_col_header, 'Gur7'): float})

# Add new column for difference between V7 and Gur7
TABLE13[('Time (s)','V7 - Gur7')] = TABLE13[('Time (s)','V7')] - TABLE13[('Time (s)','Gur7')]

# Sort by V7 - Gur7
TABLE13.sort_values(by=[(time_col_header,'V7 - Gur7')], inplace=True)

# Rename summary rows to reflect the set
rename_metrics_all = {metric : metric + ' (All)' for metric in bb_metrics}
TABLE13.rename(rename_metrics_all, inplace=True)

# Add summary rows from 6 trees set
summary_metrics_6trees = all6_bb_results_df.tail(3).copy(deep=True)
rename_metrics_6trees = {metric : metric + ' (6 trees)' for metric in bb_metrics}
summary_metrics_6trees.rename(rename_metrics_6trees, inplace=True)

TABLE13 = pd.concat([TABLE13, summary_metrics_6trees])

# Drop rows, cols, (time,V7)
TABLE13.drop([('','Rows'),('','Cols'),(node_col_header,map_cols_to_short_time[mintime_col])], axis=1, inplace=True)

# Set wins row to be integer valued
TABLE13.loc['Wins1 (All)'] = TABLE13.loc['Wins1 (All)'].apply(int_format)
# TABLE13.loc['Wins7 (All)'] = TABLE13.loc['Wins7 (All)'].apply(int_format)
# TABLE13.loc['Wins1 (6 trees)'] = TABLE13.loc['Wins1 (6 trees)'].apply(int_format)
# TABLE13.loc['Wins7 (6 trees)'] = TABLE13.loc['Wins7 (6 trees)'].apply(int_format)
# TABLE13.iloc[len(TABLE13)-1] = TABLE13.iloc[len(TABLE13)-1].apply(int_format)

# Move instance names into a column
TABLE13.reset_index(inplace=True, col_level=1)

# Store indices of rows of 6-tree instances
six_trees_instances = list(all6_instances_dict.keys())
# mask = TABLE13[('','Instance')].isin(six_trees_instances)
# six_trees_indices = TABLE13.loc[mask, :].index.tolist()

# Remove presolved from name and escape
TABLE13[('',"Instance")] = TABLE13[('',"Instance")].apply(remove_presolved_from_name)
TABLE13[('',"Instance")] = TABLE13[('',"Instance")].apply(tex_escape)

# If we are not using the automatic tex-escaper, we need to do it ourselves
for col in TABLE13.columns:
    # curr_col = '{' + tex_escape(col) + '}'
    curr_col = tex_escape(str(col))
    TABLE13.rename({col: curr_col}, inplace=True, axis=1)

# Finally, apply the desired style
    # format(formatter = int_format).\
table13_str = TABLE13.style.\
    hide(axis=0).\
    to_latex(
        column_format="""
	@{}l % instance
	*{1}{S[table-format=4.0,table-auto-round,table-number-alignment=center]} % # cuts
	*{2}{S[table-format=4.2,table-auto-round]} % Gur1, Gur7
	*{2}{H} % V, Total
	*{2}{S[table-format=4.2,table-auto-round]} % V7, Total7
	*{3}{S[table-format=8.0,table-auto-round,table-number-alignment=center]} % Nodes
	@{}
        """,
        hrules = True,
        sparse_index = True,
        multirow_align = "c",
        siunitx=True,
        convert_css = True,
        environment = "table",
        position_float = "centering",
        label = "app:tab:bb",
        caption = """
            Time (in seconds) and number nodes taken to solve each instance.
            The table is sorted by column 4 (``V'' under ``Time (s)'').
            ``Gur1'' indicates \Gurobi{} run with one random seed.
            ``Gur7'' indicates the minimum from seven runs of \Gurobi{} with different random seeds.
        """,
        )

# Adjustbox environment sets width to pagewidth
# table13_str = add_adjustbox_environment(table13_str)

# Set default siunitx options for this table
table13_str = add_sisetup(table13_str)

# Add a midrule between the instances and 3 summary rows; the "6" is hand-coded but can be automated
table13_str = add_midrule(table13_str, -6)
table13_str = add_midrule(table13_str, -10)

# Add color to six tree instances
splitlines = table13_str.splitlines()
for i in range(len(splitlines)):
    line = splitlines[i]
    curr_line = line.split('&')
    if len(curr_line) > 0 and curr_line[0].strip()+'_presolved' in six_trees_instances:
        splitlines[i] = '\\rowcolor{lightgray!30} ' + line
table13_str = '\n'.join(splitlines).replace('NaN', '')

print(table13_str)

# OLD Section 3: Time tables

## `time_df`: Create subset of dataframe relevant to time

In [ ]:
%%script false --no-raise-error

## Create subset of dataframe relevant to time
time_df = df.loc[:,
                [
                    'NUM DISJ TERMS',
                    'ROWS',
                    'COLS',
                    'LP OBJ',
                    'IP OBJ',
                    'FIRST REF OBJ',
                    'AVG REF OBJ',
                    'BEST REF OBJ',
                    'FIRST REF+V OBJ',
                    'AVG REF+V OBJ',
                    'FIRST REF BOUND',
                    'AVG REF BOUND',
                    'BEST REF BOUND',
                    'FIRST REF+V BOUND',
                    'AVG REF+V BOUND',
                    'FIRST REF ITERS',
                    'AVG REF ITERS',
                    'BEST REF ITERS',
                    'FIRST REF+V ITERS',
                    'AVG REF+V ITERS',
                    'FIRST REF NODES',
                    'AVG REF NODES',
                    'BEST REF NODES',
                    'FIRST REF+V NODES',
                    'AVG REF+V NODES',
                    'FIRST REF TIME',
                    'AVG REF TIME',
                    'BEST REF TIME',
                    'FIRST REF+V TIME',
                    'AVG REF+V TIME',
                    'VPC_GEN_TIME',
                    'NUM GMIC',
                    'NUM VPC',
                    'NUM OBJ',
                    'ALL REF TIME',
                    'ALL REF+V TIME',
                    'ExitReason']
               ]
#display(time_df.loc[("bm23_presolved",2)])

## Prepare short/long column names for time dfs
1. First run of Gurobi without VPCs
2. Best among 7 runs of Gurobi without VPCs
3. First run of Gurobi with VPCs for each disjunction size
4. First run of Gurobi with VPCs for each disjunction size, adding cut generation time
5. Best run across first Gurobi without VPCs and first Gurobi with VPCs (across all terms)

In [ ]:
%%script false --no-raise-error

# Gur1/Gur7 names
gur1_col_stub = 'AVG REF' # Should we change to AVG?
gur7_col_stub = 'BEST REF'
# gur_w_v_col_stub = 'AVG REF'
gur1v_col_stub = gur1_col_stub + '+V'
gur1v_w_cut_col_stub = gur1v_col_stub + ' W/CUTGEN'

# gur1time: first run of Gurobi without VPCs
gur1time_col = gur1_col_stub + ' TIME'
gur1nodes_col = gur1_col_stub + ' NODES'

# gur7time: best among 7 runs of Gurobi without VPCs
gur7time_col = gur7_col_stub + ' TIME'
gur7nodes_col = gur7_col_stub + ' NODES'

# gur1vtime: first run of Gurobi w/VPCs for each disj size
gur1vtime_col = gur1v_col_stub + ' TIME'
gur1vnodes_col = gur1v_col_stub + ' NODES'

# gur1v_w_cut_time: first run of Gurobi w/VPCs for each disj size, counting cut generation time
gur1v_w_cut_time_col = gur1v_w_cut_col_stub + ' TIME'

# Track best disjunction used in 0-row
gurv_disj_col = gur1v_col_stub + ' DISJ'
gurv_w_cut_disj_col = gur1v_w_cut_col_stub + ' DISJ'

# Best Gurobi run across the first without VPCs and first w/VPCs for each disj size
mintime_col       = 'MIN BB TIME'
mintime_w_cut_col = 'MIN BB W/CUTGEN TIME'
mintime_disj_col  = 'MIN BB TIME DISJ'
minnodes_col      = 'MIN BB NODES'

map_cols_to_short_time = {
    gur1time_col         : 'Gur1',
    gur7time_col         : 'Gur7',
    gur1vtime_col        : 'V',
    gur1v_w_cut_time_col : 'Total',
    mintime_col          : 'V7',
    mintime_w_cut_col    : 'Total7',
}

map_cols_to_short_nodes = {
    gur1nodes_col        : 'Gur1',
    gur7nodes_col        : 'Gur7',
    gur1vnodes_col       : 'V',
    minnodes_col         : 'V7',
}

map_short_to_cols_time = {v: k for k, v in map_cols_to_short_time.items()}
map_short_to_cols_nodes = {v: k for k, v in map_cols_to_short_nodes.items()}

time_cols_short = list(map_short_to_cols_time.keys())
node_cols_short = list(map_short_to_cols_nodes.keys())
# display(time_cols, node_cols)

# Select a subset of columns for the "long" list used when updating the 0-row
time_cols_long = [map_short_to_cols_time[col] for col in time_cols_short]
node_cols_long = [map_short_to_cols_nodes[col] for col in node_cols_short]

# # Update list of columns with mintime cols
# newshortcol1 = 'V7'
# newshortcol2 = 'Total7'
# newshortcol3 = 'V7'
# map_cols_to_short_time [mintime_col]       = newshortcol1
# map_cols_to_short_time [mintime_w_cut_col] = newshortcol2
# map_cols_to_short_nodes[minnodes_col]      = newshortcol3

# map_short_to_cols_time [newshortcol1]      = mintime_col
# map_short_to_cols_time [newshortcol2]      = mintime_w_cut_col
# map_short_to_cols_nodes[newshortcol3]      = minnodes_col

# time_cols_short.append(newshortcol1)
# time_cols_short.append(newshortcol2)
# node_cols_short.append(newshortcol3)

## Add total time for running solver + generating cuts

In [ ]:
%%script false --no-raise-error

# Add total time for running solver + generating cuts
time_df[gur1v_w_cut_time_col] = time_df[gur1vtime_col] + time_df[col_vpc_gen_time]

display(time_df.loc['bm23_presolved'])

## `selected_time_df`: Solving and cut-generation time for instances selected for time reporting; 0-row with min values across all rows

In [ ]:
%%script false --no-raise-error

## Solving and cut-generation time for instances selected for time reporting
selected_time_df = time_df.loc[selected_time_instances_dict.keys()]
selected_time_df.index = selected_time_df.index.remove_unused_levels()
selected_time_df[minnodes_col] = 0

## Fill in 0-row with min values across all rows
## Also fill in gur1 values (present only in 0 row currently) for all disj terms
comparison_time_cols = [gur1vtime_col, gur1v_w_cut_time_col]
comparison_node_cols = [gur1vnodes_col]
cols_to_display = [col_num_vpc]+[gur1time_col,gur1vtime_col]+[gur1nodes_col,gur1vnodes_col]+[mintime_col,mintime_w_cut_col,minnodes_col,gurv_disj_col,gurv_w_cut_disj_col,mintime_disj_col]
inst_set = selected_time_df.index.levels[0]
# tmp_inst = '23588_presolved'
# inst_set = ['10teams_presolved',tmp_inst]
for i, inst in enumerate(inst_set):
    print("{}/{}".format(i+1,len(inst_set)), end='\r', flush=True)
    curr_df = selected_time_df.loc[inst].copy() # copy needed to not throw SettingWithCopyWarning
    
    # Select only the rows in which VPCs were generated
    curr_df_with_vpcs = curr_df[curr_df[col_num_vpc] > 0]
    
    # display(inst)
    # display(curr_df_with_vpcs[[col_num_vpcs]+[gur1time_col,gur1vtime_col]])

    # Set 0-row to have min time values across all (non-0-vpc) rows for this instance
    # best_vals = curr_df_with_vpcs[comparison_time_cols].min()
    # selected_time_df.loc[(inst,0),comparison_time_cols] = best_vals
    best_vals_idx = curr_df_with_vpcs[comparison_time_cols].idxmin()
    for curr_col, curr_disj_id in zip(comparison_time_cols, best_vals_idx):
        selected_time_df.at[(inst,0),curr_col] = curr_df_with_vpcs.at[curr_disj_id, curr_col]
    
    # display(best_vals_idx)
    # print("selected_time_df.at[('{}',0),gur1vtime_col] = {}".format(inst,selected_time_df.at[(inst,0),gur1vtime_col]))
    # display(selected_time_df[[col_num_vpcs]+[gur1time_col,gur1vtime_col]].head(14))
    # print("selected_time_df.at[('{}',0),gur1vtime_col] = {}".format(inst,selected_time_df.at[(inst,0),gur1vtime_col]))


    # Also add id of the best disj to the 0-row
    selected_time_df.at[(inst,0),gurv_disj_col]       = int(best_vals_idx.iloc[0])
    selected_time_df.at[(inst,0),gurv_w_cut_disj_col] = int(best_vals_idx.iloc[1])

    # Update 0-row of mintime (V7) entries
    curr_gur1time       = selected_time_df.at[(inst,0),gur1time_col]
    curr_gur1vtime      = selected_time_df.at[(inst,0),gur1vtime_col]
    curr_gur1vcuts_time = selected_time_df.at[(inst,0),gur1v_w_cut_time_col]
    
    curr_vals = [curr_gur1time, curr_gur1vtime]
    min_id = np.argmin(curr_vals)

    # If min_id is 0, then no cuts are used and we report the gur1 time
    # If min_id is 1, then gur1v < gur1 and we can report the number of cuts used
    selected_time_df.at[(inst,0),mintime_col] = curr_vals[min_id]

    # Add num cuts from mintime disj into num vpc col
    best_disj_size = 0 if min_id == 0 else best_vals_idx.iloc[0]
    selected_time_df.at[(inst,0),mintime_disj_col] = best_disj_size
    best_num_cuts = selected_time_df.at[(inst,best_disj_size),col_num_vpc]
    selected_time_df.at[(inst,0),col_num_vpc] = best_num_cuts

    # Update with cuts into Total7 column
    curr_vals = [curr_gur1time, curr_gur1vcuts_time]
    selected_time_df.at[(inst,0),mintime_w_cut_col] = min(curr_vals)

    # Repeat for nodes
    best_vals = curr_df_with_vpcs[comparison_node_cols].min()
    selected_time_df.loc[(inst,0),comparison_node_cols] = best_vals
    # selected_time_df.at[(inst,0),minnodes_col] = int(selected_time_df.loc[(inst,0),[gur1nodes_col,gur1vnodes_col]].min())

    curr_gur1nodes       = selected_time_df.at[(inst,0),gur1nodes_col]
    curr_gur1vnodes      = selected_time_df.at[(inst,0),gur1vnodes_col]
    curr_vals = [curr_gur1nodes, curr_gur1vnodes]
    min_id = np.argmin(curr_vals)
    selected_time_df.at[(inst,0),minnodes_col] = int(curr_vals[min_id])

    # Propogate down 0-row values for gur1 columns
    selected_time_df.loc[inst, gur1time_col] = curr_gur1time
    selected_time_df.loc[inst, gur1nodes_col] = curr_gur1nodes

    #### FOR SOME REASON, THE BELOW ZEROES OUT selected_time_df.loc[[(inst,0)]][gur1vtime_col]
    # display(selected_time_df.loc[(inst,0),[gur1nodes_col,gur1vnodes_col]])

    ## OLD CODE BELOW
    # best_vals_idx = curr_df_with_vpcs[comparison_node_cols].idxmin()
    # for curr_col, curr_disj_id in zip(comparison_node_cols, best_vals_idx):
    #     selected_time_df.at[(inst,0),curr_col] = curr_df_with_vpcs.loc[curr_disj_id, curr_col]

    # # Also add id of the best disj to the 0-row
    # selected_time_df.at[(inst,0),gurv_disj_col + ' (NODES)'] = int(best_vals_idx[0])
    # selected_time_df.at[(inst,0),gurv_w_cut_disj_col+ ' (NODES)'] = int(best_vals_idx[1])

    # for ind in curr_df.index:
    #     if ind == 0:
    #         continue

    #     # Propogate GurF and GurL down
    #     subinds = [4,6]
    #     sel_gap = [gap_cols[i] for i in subinds]
    #     selected_gap_df.loc[(inst,ind),sel_gap] = curr_df.loc[0,sel_gap]

    #     # If no VPCs produced, the values for V+GurF and V+GurL have not been provided
    #     # We replace these by GurF and GurL
    #     # Currently disabled: update max for that column too (if disabled, we instead keep max as the value among those that generated VPCs)
    #     num_vpc = curr_df.loc[ind,col_num_vpcs]
    #     if num_vpc == 0:
    #         # print("Zero cuts for inst {} at depth {:d}".format(inst, ind))
    #         subinds = [5,7]
    #         refinds = [4,6]
    #         sel_gap = [gap_cols[i] for i in subinds]
    #         selected_gap_df.loc[(inst,ind),sel_gap] = curr_df.loc[0,[gap_cols[i] for i in refinds]].to_numpy()

    #         # for i in refinds:
    #         #     if curr_df.loc[0,gap_cols[i]] > selected_gap_df.loc[(inst,0),gap_cols[i+1]]:
    #         #         if curr_df.loc[0,gap_cols[i]] > 0:
    #         #             # print("DEBUG: Updating {} for inst {} from {:f} to {:f}".format(
    #         #             #     gap_cols[i+1], 
    #         #             #     inst, 
    #         #             #     selected_gap_df.loc[(inst,0),gap_cols[i+1]], 
    #         #             #     curr_df.loc[0,gap_cols[i]]))
    #         #         selected_gap_df.loc[(inst,0),gap_cols[i+1]] = curr_df.loc[0,gap_cols[i]]

# Add minimum time when using cuts and when not using cuts
# selected_time_df[mintime_col] = selected_time_df[[gur1time_col, gur1vtime_col]].min(axis=1)
# selected_time_df[mintime_w_cut_col] = selected_time_df[[gur1time_col, gur1v_w_cut_time_col]].min(axis=1)
# selected_time_df[minnodes_col] = selected_time_df[[gur1nodes_col,gur1vnodes_col]].min(axis=1)

display(selected_time_df.head(35).loc[:,[col_num_vpc]+[gur1time_col,gur1vtime_col]+[gur1nodes_col,gur1vnodes_col]+[mintime_col,mintime_w_cut_col,minnodes_col,gurv_disj_col,gurv_w_cut_disj_col,mintime_disj_col]])
# display(selected_time_df.loc['10teams_presolved',[col_num_vpcs]+[gur1time_col,gur1vtime_col]+[gur1nodes_col,gur1vnodes_col]+[mintime_col,mintime_w_cut_col,minnodes_col,gurv_disj_col,gurv_w_cut_disj_col,mintime_disj_col]])
# display(selected_time_df.loc[inst_set,cols_to_display])

In [ ]:
# ### DEBUGGING that first ref+v time gets zeroed out for some reason?
# tmp_df = selected_time_df[[col_num_vpcs]+[gur1time_col,gur1vtime_col]].head(14).copy(deep=True)
# display(tmp_df)

# print(tmp_df.loc[('23588_presolved',0),gur1vtime_col])
# display(tmp_df.loc[[('23588_presolved',0)]][gur1vtime_col])

# tmp_df = selected_time_df
# print(tmp_df.loc[('23588_presolved',0),gur1vtime_col])
# display(tmp_df.loc[[('23588_presolved',0)]][gur1vtime_col])
# display(tmp_df.loc['23588_presolved'])

## Table 3: `avg_bb_df`: average time/nodes taken

### Prepare variables for row/col names

In [ ]:
%%script false --no-raise-error

## Prepare variables for row/col names

bb_classes = ['All', '6 trees', 'Binary']
num_bb_classes = len(bb_classes)

bucket_min = [0, 10, 100, 1000]
bucket_max = [3600, 3600, 3600, 3600]
num_buckets = len(bucket_min)
assert(len(bucket_max) == num_buckets)
bb_buckets = ['[' + str(bucket_min[j]) + ',' + str(bucket_max[j]) + ')' for j in range(num_buckets)]
# bucket_names = [classes[i] + ' [' + str(bucket_min[j]) + ',' + str(bucket_max[j]) + ')' for i in range(num_classes) for j in range(num_buckets)]
# display(bucket_names)

bb_metrics = ['Gmean', 'Wins1', 'Wins7']

time_col_header = 'Time (s)'
node_col_header = 'Nodes (#)'

### Set up empty `avg_bb_df`

In [ ]:
%%script false --no-raise-error

## Prepare avg_bb_df

avg_bb_cols = pd.MultiIndex.from_arrays(
    [[time_col_header]*len(time_cols_short) + [node_col_header]*len(node_cols_short), time_cols_short + node_cols_short],
    names = ['criterion', 'type'])

#bb_row_names = pd.MultiIndex.from_product([bb_buckets, bb_row_names], names=['bucket', 'metric'])
bb_row_names = pd.MultiIndex.from_product(
    [bb_classes, bb_buckets, bb_metrics],
    names=['class', 'bucket', 'metric'])

avg_bb_df = pd.DataFrame(
    columns = avg_bb_cols,
    index = bb_row_names,
    dtype = float
)

display(avg_bb_df.loc[:,avg_bb_cols.get_level_values(0)==node_col_header].head(6))
#display(avg_bb_df.loc[(bb_classes[0], bb_buckets[1], bb_metrics[0]),:])
display(avg_bb_df.loc[(bb_classes[0], bb_buckets, bb_metrics[0]),:])

### `avg_bb_df`: shifted geometric mean of time taken across instances, in various buckets, and geomean of nodes too

In [ ]:
%%script false --no-raise-error

## Create gmean_df
#   = shifted geometric mean of time taken across instances, in various buckets
#     and geomean of nodes too

# Custom functions for prior to python 3.8
# def geo_mean(iterable):
#     a = np.array(iterable)
#     return a.prod()**(1.0/len(a))
# def geo_mean_overflow(iterable):
#     return np.exp(np.log(iterable).mean())
from statistics import geometric_mean
SHIFT_TIME  = 60
SHIFT_NODES = 1000

num_inst = np.zeros(len(avg_bb_df),dtype = np.int64)
row_ind = 0

#avg_bb_df.loc[(bb_classes[0], bb_buckets, bb_metrics[0]),:] = \
shortcols_time = time_cols_short
cols_time = [map_short_to_cols_time[shortcol] for shortcol in shortcols_time]
shortcols_nodes = node_cols_short
cols_nodes = [map_short_to_cols_nodes[shortcol] for shortcol in shortcols_nodes]

cols = cols_time + cols_nodes
shortcols = shortcols_time + shortcols_nodes

# First calculate stats for "all" instances
curr_df = selected_time_df.loc[:,cols_time + cols_nodes]
curr_df = curr_df[curr_df.index.get_level_values(1) == 0] # take only "best" values

for i in range(num_buckets):
    curr_df = curr_df[curr_df[gur1time_col] > bucket_min[i]]
    
    avg_bb_df.loc[(bb_classes[0], bb_buckets[i], bb_metrics[0]),(time_col_header,shortcols_time)] = \
        [geometric_mean(curr_df[col] + SHIFT_TIME) - SHIFT_TIME for col in cols_time]
    avg_bb_df.loc[(bb_classes[0], bb_buckets[i], bb_metrics[0]),(node_col_header,shortcols_nodes)] = \
        [geometric_mean(curr_df[col] + SHIFT_NODES) - SHIFT_NODES for col in cols_nodes]
    
    print("row {:d}: {:d}".format(row_ind,len(curr_df)))
    
    num_inst[row_ind:row_ind+len(bb_metrics)] = len(bb_metrics)*[len(curr_df)]
    row_ind += len(bb_metrics)

# Now calculate stats for "6 trees" instances
curr_df = selected_time_df.loc[all6_instances_dict.keys(),cols_time + cols_nodes]
curr_df = curr_df[curr_df.index.get_level_values(1) == 0] # take only best values

for i in range(num_buckets):
    curr_df = curr_df[curr_df[gur1time_col] > bucket_min[i]]
    
    avg_bb_df.loc[(bb_classes[1], bb_buckets[i], bb_metrics[0]),(time_col_header,shortcols_time)] = \
        [geometric_mean(curr_df[col] + SHIFT_TIME) - SHIFT_TIME for col in cols_time]
    avg_bb_df.loc[(bb_classes[1], bb_buckets[i], bb_metrics[0]),(node_col_header,shortcols_nodes)] = \
        [geometric_mean(curr_df[col] + SHIFT_NODES) - SHIFT_NODES for col in cols_nodes]
    
    print("row {:d}: {:d}".format(row_ind,len(curr_df)))

    num_inst[row_ind:row_ind+len(bb_metrics)] = len(bb_metrics)*[len(curr_df)]
    row_ind += len(bb_metrics)

avg_bb_df[inst_col_name] = num_inst
# avg_bb_df['NUM INST'] = avg_bb_df['NUM INST'].astype(np.int64)

# Repeat for "binary" instances
# identify pure_binary_instances that are in selected_time_instances_dict
binary_x_time_instances = [inst for inst in pure_binary_instances if inst in selected_time_instances_dict.keys()]
curr_df = selected_time_df.loc[binary_x_time_instances,cols_time + cols_nodes]
curr_df = curr_df[curr_df.index.get_level_values(1) == 0] # take only best values

for i in range(num_buckets):
    curr_df = curr_df[curr_df[gur1time_col] > bucket_min[i]]
    
    avg_bb_df.loc[(bb_classes[2], bb_buckets[i], bb_metrics[0]),(time_col_header,shortcols_time)] = \
        [geometric_mean(curr_df[col] + SHIFT_TIME) - SHIFT_TIME for col in cols_time]
    avg_bb_df.loc[(bb_classes[2], bb_buckets[i], bb_metrics[0]),(node_col_header,shortcols_nodes)] = \
        [geometric_mean(curr_df[col] + SHIFT_NODES) - SHIFT_NODES for col in cols_nodes]
    
    print("row {:d}: {:d}".format(row_ind,len(curr_df)))

    num_inst[row_ind:row_ind+len(bb_metrics)] = len(bb_metrics)*[len(curr_df)]
    row_ind += len(bb_metrics)

avg_bb_df[inst_col_name] = num_inst
# avg_bb_df['NUM INST'] = avg_bb_df['NUM INST'].astype(np.int64)

display(avg_bb_df.loc[(bb_classes, bb_buckets, bb_metrics[0]),:])

### Update wins1 rows

In [ ]:
%%script false --no-raise-error

## Update wins1 rows
# A win in terms of time is counted when the ``Gur1'' baseline seconds taken 
# is at least 10\% slower, to account for some variability in runtimes.
# A win in terms of nodes is when the ``Gur1'' baseline number of nodes is higher.

# Make all columns "object" type to allow for integer values
avg_bb_df.loc[:,(time_col_header,shortcols_time)] = avg_bb_df.loc[:,(time_col_header,shortcols_time)].astype(object)
avg_bb_df.loc[:,(node_col_header,shortcols_nodes)] = avg_bb_df.loc[:,(node_col_header,shortcols_nodes)].astype(object)

# First calculate stats for "all" instances
curr_df = selected_time_df.loc[:,cols_time + cols_nodes]
curr_df = curr_df[curr_df.index.get_level_values(1) == 0] # take only best values

for i in range(num_buckets):
    curr_df = curr_df[curr_df[gur1time_col] > bucket_min[i]]
    
    refcol = gur1time_col
    avg_bb_df.loc[(bb_classes[0], bb_buckets[i], bb_metrics[1]),(time_col_header,shortcols_time)] = \
        [ int(sum(curr_df[refcol] > 1.1*curr_df[col])) for col in cols_time ]

    refcol = gur1nodes_col
    avg_bb_df.loc[(bb_classes[0], bb_buckets[i], bb_metrics[1]),(node_col_header,shortcols_nodes)] = \
        [ int(sum(curr_df[refcol] > curr_df[col])) for col in cols_nodes ]

# Now calculate stats for "6 trees" instances
curr_df = selected_time_df.loc[all6_instances_dict.keys(),cols_time + cols_nodes]
curr_df = curr_df[curr_df.index.get_level_values(1) == 0] # take only best values

for i in range(num_buckets):
    curr_df = curr_df[curr_df[gur1time_col] > bucket_min[i]]
    
    refcol = gur1time_col
    avg_bb_df.loc[(bb_classes[1], bb_buckets[i], bb_metrics[1]),(time_col_header,shortcols_time)] = \
        [ int(sum(curr_df[refcol] > 1.1*curr_df[col])) for col in cols_time ]

    refcol = gur1nodes_col
    avg_bb_df.loc[(bb_classes[1], bb_buckets[i], bb_metrics[1]),(node_col_header,shortcols_nodes)] = \
        [ int(sum(curr_df[refcol] > curr_df[col])) for col in cols_nodes ]
    
# Repeat for binary instances
curr_df = selected_time_df.loc[binary_x_time_instances,cols_time + cols_nodes]
curr_df = curr_df[curr_df.index.get_level_values(1) == 0] # take only best values

for i in range(num_buckets):
    curr_df = curr_df[curr_df[gur1time_col] > bucket_min[i]]
    
    refcol = gur1time_col
    avg_bb_df.loc[(bb_classes[2], bb_buckets[i], bb_metrics[1]),(time_col_header,shortcols_time)] = \
        [ int(sum(curr_df[refcol] > 1.1*curr_df[col])) for col in cols_time ]

    refcol = gur1nodes_col
    avg_bb_df.loc[(bb_classes[2], bb_buckets[i], bb_metrics[1]),(node_col_header,shortcols_nodes)] = \
        [ int(sum(curr_df[refcol] > curr_df[col])) for col in cols_nodes ]

### Update wins7 rows

In [ ]:
%%script false --no-raise-error

## Update wins7 rows
# A win in terms of time is counted when the ``Gur1'' baseline seconds taken 
# is at least 10\% slower, to account for some variability in runtimes.
# A win in terms of nodes is when the ``Gur1'' baseline number of nodes is higher.

# First calculate stats for "all" instances
curr_df = selected_time_df.loc[:,cols_time + cols_nodes]
curr_df = curr_df[curr_df.index.get_level_values(1) == 0] # take only best values
for i in range(num_buckets):
    curr_df = curr_df[curr_df[gur1time_col] > bucket_min[i]]
    
    refcol = gur7time_col
    avg_bb_df.loc[(bb_classes[0], bb_buckets[i], bb_metrics[2]),(time_col_header,shortcols_time)] = \
        [ int(sum(curr_df[refcol] > 1.1*curr_df[col])) for col in cols_time ]

    refcol = gur7nodes_col
    avg_bb_df.loc[(bb_classes[0], bb_buckets[i], bb_metrics[2]),(node_col_header,shortcols_nodes)] = \
        [ int(sum(curr_df[refcol] > curr_df[col])) for col in cols_nodes ]

# Now calculate stats for "6 trees" instances
curr_df = selected_time_df.loc[all6_instances_dict.keys(),cols_time + cols_nodes]
curr_df = curr_df[curr_df.index.get_level_values(1) == 0] # take only best values

for i in range(num_buckets):
    curr_df = curr_df[curr_df[gur1time_col] > bucket_min[i]]
    
    refcol = gur7time_col
    avg_bb_df.loc[(bb_classes[1], bb_buckets[i], bb_metrics[2]),(time_col_header,shortcols_time)] = \
        [ int(sum(curr_df[refcol] > 1.1*curr_df[col])) for col in cols_time ]

    refcol = gur7nodes_col
    avg_bb_df.loc[(bb_classes[1], bb_buckets[i], bb_metrics[2]),(node_col_header,shortcols_nodes)] = \
        [ int(sum(curr_df[refcol] > curr_df[col])) for col in cols_nodes ]
    
# Repeat for binary instances
curr_df = selected_time_df.loc[binary_x_time_instances,cols_time + cols_nodes]
curr_df = curr_df[curr_df.index.get_level_values(1) == 0] # take only best values

for i in range(num_buckets):
    curr_df = curr_df[curr_df[gur1time_col] > bucket_min[i]]
    
    refcol = gur7time_col
    avg_bb_df.loc[(bb_classes[2], bb_buckets[i], bb_metrics[2]),(time_col_header,shortcols_time)] = \
        [ int(sum(curr_df[refcol] > 1.1*curr_df[col])) for col in cols_time ]

    refcol = gur7nodes_col
    avg_bb_df.loc[(bb_classes[2], bb_buckets[i], bb_metrics[2]),(node_col_header,shortcols_nodes)] = \
        [ int(sum(curr_df[refcol] > curr_df[col])) for col in cols_nodes ]

In [ ]:
%%script false --no-raise-error

# display(avg_bb_df.loc[:,cols.get_level_values(0)=='Nodes'].head(6))
display(avg_bb_df.loc[(bb_classes[0:2], bb_buckets, bb_metrics[0:3]),:])

## Table 6: `all_bb_results_df`: all time/nodes results

In [ ]:
%%script false --no-raise-error

inst_set = selected_time_df.index.levels[0]
inst_set.set_names("Instance",inplace=True)
numcuts_col_header = '# cuts'

col_idx = pd.MultiIndex.from_arrays(
    [
        ['', '', numcuts_col_header] + [time_col_header]*len(time_cols_short) + [node_col_header]*len(node_cols_short),
        ['Rows', 'Cols', map_cols_to_short_time[gur1vtime_col]] + time_cols_short + node_cols_short
    ],
)

all_bb_results_df = pd.DataFrame(
    columns = col_idx,
    index = inst_set,
    dtype = object,
)

# Enter number of rows and cols
tmp_df = df.xs(0, level='disj_terms').loc[inst_set,['ROWS','COLS']]
tmp_df.columns = pd.MultiIndex.from_product([[''],['Rows','Cols']])
all_bb_results_df.loc[:,tmp_df.columns] = tmp_df

# Enter number of cuts
# tmp_df = selected_time_df.loc[(inst_set,0), ['NUM VPC']]
tmp_df = selected_time_df.xs(0, level='disj_terms')['NUM VPC']
tmp_df.columns = pd.MultiIndex.from_product([[numcuts_col_header],[map_cols_to_short_time[gur1vtime_col]]])
all_bb_results_df.loc[:,tmp_df.columns] = tmp_df

# Enter time
tmp_df = selected_time_df.xs(0, level='disj_terms')[time_cols_long]
tmp_df.columns = pd.MultiIndex.from_product([[time_col_header],time_cols_short])
all_bb_results_df.loc[:,tmp_df.columns] = tmp_df

# Enter nodes
tmp_df = selected_time_df.xs(0, level='disj_terms')[node_cols_long]
tmp_df.columns = pd.MultiIndex.from_product([[node_col_header],node_cols_short])
all_bb_results_df.loc[:,tmp_df.columns] = tmp_df

all_bb_results_df = all_bb_results_df.sort_values(by=[(time_col_header, map_cols_to_short_time[mintime_col])])

# Add average + wins rows
# Replace missing entries with empty string
tmp_df = avg_bb_df.xs((bb_classes[0],bb_buckets[0])).copy(deep=True)
tmp_df.drop(inst_col_name, axis=1, level=0, inplace=True)
all_bb_results_df = pd.concat([all_bb_results_df, tmp_df]).fillna('',downcast=False)

# Remove unnecessary entries
all_bb_results_df.loc['Wins1',[
        (time_col_header,map_cols_to_short_time[gur1time_col]),
        (node_col_header,map_cols_to_short_nodes[gur1nodes_col])
    ]] = ""
# all_bb_results_df.loc['Wins1',([time_col_header,node_col_header],'Gur1')] = ""
all_bb_results_df.loc['Wins7',[
        (time_col_header,map_cols_to_short_time[gur1time_col]),
        (time_col_header,map_cols_to_short_time[gur7time_col]),
        (node_col_header,map_cols_to_short_nodes[gur1nodes_col]),
        (node_col_header,map_cols_to_short_nodes[gur7nodes_col]),
    ]] = ""
# all_bb_results_df.loc['Wins7',([time_col_header,node_col_header],['Gur1','Gur7'])] = ""
# all_bb_results_df = all_bb_results_df.fillna('',downcast=False)

# Convert rows, cols, # cuts to int values
tmp_cols = pd.MultiIndex.from_product([[''],['Rows','Cols']])
all_bb_results_df.loc[inst_set,tmp_cols] = all_bb_results_df.loc[inst_set,tmp_cols].astype(np.int64)
tmp_cols = pd.MultiIndex.from_product([[numcuts_col_header],[map_cols_to_short_time[gur1vtime_col]]])
all_bb_results_df.loc[inst_set,tmp_cols] = all_bb_results_df.loc[inst_set,tmp_cols].astype(np.int64)

# Rename inst col back to inst_row_name
all_bb_results_df.index.set_names("Instance",inplace=True)

display(all_bb_results_df.head(15))
display(all_bb_results_df.tail(10))

## Table 7: ``all6_bb_results_df``: 6-trees time/nodes results

In [ ]:
%%script false --no-raise-error

inst_set = all6_instances_dict.keys()
all6_bb_results_df = all_bb_results_df.loc[inst_set]

all6_bb_results_df = all6_bb_results_df.sort_values(by=[(time_col_header, map_cols_to_short_time[mintime_col])])

# Add average + wins rows
# Replace missing entries with empty string
tmp_df = avg_bb_df.xs((bb_classes[1],bb_buckets[0])).copy(deep=True)
tmp_df.drop(inst_col_name, axis=1, level=0, inplace=True)
all6_bb_results_df = pd.concat([all6_bb_results_df, tmp_df]).fillna('',downcast=False)

# Remove unnecessary entries
all6_bb_results_df.loc['Wins1',[
        (time_col_header,map_cols_to_short_time[gur1time_col]),
        (node_col_header,map_cols_to_short_nodes[gur1nodes_col])
    ]] = ""
all6_bb_results_df.loc['Wins7',[
        (time_col_header,map_cols_to_short_time[gur1time_col]),
        (time_col_header,map_cols_to_short_time[gur7time_col]),
        (node_col_header,map_cols_to_short_nodes[gur1nodes_col]),
        (node_col_header,map_cols_to_short_nodes[gur7nodes_col]),
    ]] = ""

# Convert rows, cols, # cuts to int values
tmp_cols = pd.MultiIndex.from_product([[''],['Rows','Cols']])
all6_bb_results_df.loc[inst_set,tmp_cols] = all6_bb_results_df.loc[inst_set,tmp_cols].astype(np.int64)
tmp_cols = pd.MultiIndex.from_product([[numcuts_col_header],[map_cols_to_short_time[gur1vtime_col]]])
all6_bb_results_df.loc[inst_set,tmp_cols] = all6_bb_results_df.loc[inst_set,tmp_cols].astype(np.int64)

# Rename inst col back to inst_row_name
all6_bb_results_df.index.set_names("Instance",inplace=True)

display(all6_bb_results_df.head(15))
display(all6_bb_results_df.tail(10))

## Table 8: `avg_bb_by_depth_df`: average time/nodes by depth for all-six set

In [ ]:
%%script false --no-raise-error

## Prepare avg_bb_by_depth_df
## Prepare variables for row/col names
inst_set = all6_instances_dict.keys()

bb_classes_by_depth = [str(t) + ' leaves' for t in sizes]
num_bb_classes_by_depth = len(bb_classes_by_depth)

bb_buckets_by_depth = bb_buckets
bb_metrics_by_depth = bb_metrics[0:2]

cols_time_by_depth       = [gur1time_col, gur1vtime_col, gur1v_w_cut_time_col]
shortcols_time_by_depth  = [map_cols_to_short_time[col] for col in cols_time_by_depth]
cols_nodes_by_depth      = [gur1nodes_col, gur1vnodes_col]
shortcols_nodes_by_depth = [map_cols_to_short_nodes[col] for col in cols_nodes_by_depth]

avg_bb_cols_by_depth = pd.MultiIndex.from_arrays(
    [[time_col_header]*len(shortcols_time_by_depth) + 
     [node_col_header]*len(shortcols_nodes_by_depth), 
     shortcols_time_by_depth + shortcols_nodes_by_depth],
    names = ['criterion', 'type'])

# bucket_min = [0, 10, 100, 1000]
# bucket_max = [3600, 3600, 3600, 3600]
# num_buckets = len(bucket_min)
# assert(len(bucket_max) == num_buckets)
# bb_buckets = ['[' + str(bucket_min[j]) + ',' + str(bucket_max[j]) + ')' for j in range(num_buckets)]
# # bucket_names = [classes[i] + ' [' + str(bucket_min[j]) + ',' + str(bucket_max[j]) + ')' for i in range(num_classes) for j in range(num_buckets)]
# # display(bucket_names)

# bb_metrics = ['Gmean', 'Wins1', 'Wins7']

# time_col_header = 'Time (s)'
# node_col_header = 'Nodes (\\#)'

#bb_row_names = pd.MultiIndex.from_product([bb_buckets, bb_row_names], names=['bucket', 'metric'])
bb_row_names_by_depth = pd.MultiIndex.from_product(
    [bb_classes_by_depth, bb_buckets_by_depth, bb_metrics_by_depth],
    names=['class', 'bucket', 'metric'])

avg_bb_by_depth_df = pd.DataFrame(
    columns = avg_bb_cols_by_depth,
    index = bb_row_names_by_depth,
    dtype = float
)

# Fill in values for Gur1 from avg_bb_df
# display(
#     avg_bb_df.loc[
#         (bb_classes[1], bb_buckets, bb_metrics[0:2]),
#         [(time_col_header,map_cols_to_short_time[gur1time_col]),
#         (node_col_header,map_cols_to_short_nodes[gur1nodes_col])]
#     ]
# )

# Make all columns "object" type to allow for integer values
avg_bb_by_depth_df.loc[:,(time_col_header,shortcols_time_by_depth)] = avg_bb_by_depth_df.loc[:,(time_col_header,shortcols_time_by_depth)].astype(object)
avg_bb_by_depth_df.loc[:,(node_col_header,shortcols_nodes_by_depth)] = avg_bb_by_depth_df.loc[:,(node_col_header,shortcols_nodes_by_depth)].astype(object)

## Create gmean_df by depth
#   = shifted geometric mean of time taken across instances, in various buckets
#     and geomean of nodes too

num_inst_by_depth = np.zeros(len(avg_bb_by_depth_df),dtype = np.int64)
row_ind = 0

cols = cols_time_by_depth + cols_nodes_by_depth
shortcols = shortcols_time_by_depth + shortcols_nodes_by_depth

# Calculate stats for 6 trees instances by depth
curr_df = selected_time_df.loc[inst_set,cols]
# curr_df = selected_time_df.loc[all6_binary_x_time_instances,cols]
for curr_size_ind in range(0,len(bb_classes_by_depth)):
    # print("{}".format(bb_classes_by_depth[curr_size_ind]))
    curr_by_depth_df = curr_df[curr_df.index.get_level_values(1) == sizes[curr_size_ind]] # take only best values

    for i in range(num_buckets):
        curr_by_depth_df = curr_by_depth_df[curr_by_depth_df[gur1time_col] > bucket_min[i]]
        avg_bb_by_depth_df.loc[
                (bb_classes_by_depth[curr_size_ind], bb_buckets_by_depth[i], bb_metrics_by_depth[0]),
                (time_col_header,shortcols_time_by_depth)] = \
            [geometric_mean(curr_by_depth_df[col] + SHIFT_TIME) - SHIFT_TIME for col in cols_time_by_depth]

        # display(avg_bb_by_depth_df.loc[
        #         (bb_classes_by_depth[curr_size_ind], bb_buckets_by_depth[i], bb_metrics_by_depth[0]),
        #         (time_col_header,shortcols_time_by_depth)].head())
        avg_bb_by_depth_df.loc[
                (bb_classes_by_depth[curr_size_ind], bb_buckets_by_depth[i], bb_metrics_by_depth[0]),
                (node_col_header,shortcols_nodes_by_depth)] = \
            [geometric_mean(curr_by_depth_df[col] + SHIFT_NODES) - SHIFT_NODES for col in cols_nodes_by_depth]
        
        # print("row {:d}: {:d}".format(row_ind,len(curr_by_depth_df)))

        num_inst_by_depth[row_ind:row_ind+len(bb_metrics_by_depth)] = len(bb_metrics_by_depth)*[len(curr_by_depth_df)]
        row_ind += len(bb_metrics_by_depth)

        ## Update wins1 rows
        # A win in terms of time is counted when the ``Gur1'' baseline seconds taken 
        # is at least 10\% slower, to account for some variability in runtimes.
        # A win in terms of nodes is when the ``Gur1'' baseline number of nodes is higher.
        refcol = gur1time_col
        avg_bb_by_depth_df.loc[
                (bb_classes_by_depth[curr_size_ind], bb_buckets_by_depth[i], bb_metrics_by_depth[1]),
                (time_col_header,shortcols_time_by_depth)] = \
            [ int(sum(curr_by_depth_df[refcol] > 1.1*curr_by_depth_df[col])) for col in cols_time_by_depth ]

        refcol = gur1nodes_col
        avg_bb_by_depth_df.loc[
                (bb_classes_by_depth[curr_size_ind], bb_buckets_by_depth[i], bb_metrics_by_depth[1]),
                (node_col_header,shortcols_nodes_by_depth)] = \
            [ int(sum(curr_by_depth_df[refcol] > curr_by_depth_df[col])) for col in cols_nodes_by_depth ]

avg_bb_by_depth_df[inst_col_name] = num_inst_by_depth

# for i in range(num_buckets):
#     curr_df = curr_df[curr_df[gur1time_col] > bucket_min[i]]
    
display(avg_bb_by_depth_df.loc[(bb_classes_by_depth, bb_buckets_by_depth, bb_metrics_by_depth),:])

## Results for selected instances

In [ ]:
gap_df.loc['bell5_presolved']

In [ ]:
df_preprocess.loc[[inst for inst in df_preprocess.index if 'fast' in inst]]